In [ ]:
# ---- 1. SETUP: environment, imports, budgets, helpers --------------------------------
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import ctypes
import time
import json
import math
import hashlib
import marimo as mo
import numpy as np

try:
    import psutil
except ImportError:
    psutil = None
try:
    import torch
except ImportError:
    torch = None

RAM_CAP_GB = 28.0  # the user's hard limit
RAM_SOFT_GB = 24.0  # what this notebook treats as "full"
VRAM_CAP_GB = 70.0  # allocator budget on the RTX 6000-class card

def rss_gb(include_children=True):
    if psutil is not None:
        p = psutil.Process()
        total = p.memory_info().rss
        if include_children:
            for c in p.children(recursive=True):
                try:
                    total += c.memory_info().rss
                except psutil.Error:
                    pass
        return total / 2**30
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 2**20
    return float("nan")

def ram_guard(extra_gb, what="allocation"):
    used = rss_gb()
    if used + extra_gb > RAM_SOFT_GB:
        raise MemoryError(
            f"{what}: need ~{extra_gb:.1f} GB on top of {used:.1f} GB in use; "
            f"soft limit is {RAM_SOFT_GB:.0f} GB (hard cap {RAM_CAP_GB:.0f} GB)."
        )

def trim_ram():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except OSError:
        pass
    return rss_gb()

def gpu_setup():
    if torch is None or not torch.cuda.is_available():
        return {"device": "cpu", "torch": getattr(torch, "__version__", None)}
    _free, _total = torch.cuda.mem_get_info()
    _total_gb = _total / 2**30
    _cap_gb = min(VRAM_CAP_GB, _total_gb * 0.95)
    torch.cuda.set_per_process_memory_fraction(min(1.0, _cap_gb / _total_gb))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    return {
        "device": "cuda",
        "name": torch.cuda.get_device_name(0),
        "total_gb": round(_total_gb, 1),
        "cap_gb": round(_cap_gb, 1),
        "torch": torch.__version__,
    }

def gpu_free():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

def vram_report(reset_peak=False):
    if torch is None or not torch.cuda.is_available():
        return {}
    out = {
        "allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
        "reserved_gb": round(torch.cuda.memory_reserved() / 2**30, 2),
        "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    }
    if reset_peak:
        torch.cuda.reset_peak_memory_stats()
    return out

def atomic_write(path, writer):
    base, ext = os.path.splitext(path)
    tmp = f"{base}.tmp{ext}"
    writer(tmp)
    os.replace(tmp, path)

# The Absorption Principle
### When can symbolic knowledge help a graph neural network? An information-theoretic answer, tested on real graphs

**Thesis.** Let $\sigma$ be everything a GNN architecture can compute from its input (its receptive field
times its expressive power), and let $S$ be any symbolic or relational channel: observed labels, logic rules,
a compatibility matrix, motif counts, KG facts. The most any neuro-symbolic combination can add, in expected
log-loss over the best neural predictor, is **exactly**
$$\Delta^*(S)=I(Y;S\mid\sigma)=\underbrace{I(Y;S)}_{\text{symbolic value}}-\underbrace{I(S;\sigma)}_{\text{absorbed}}+\underbrace{I(S;\sigma\mid Y)}_{\text{synergy}},$$
and it is **sandwiched** by quantities measurable before any symbolic module is built:
$$\big[I(Y;S)-I(S;\sigma)\big]_+\;\le\;\Delta^*(S)\;\le\;\min\{H(S\mid\sigma),\,H(Y\mid\sigma)\},$$
with gap exactly $H(S\mid Y)$, so the bounds coincide whenever $S$ is determined by $Y$. Every nat the network
learns about $S$ removes up to one nat of symbolic value: a conservation law between the neural and symbolic channels.

**Consequences tested here, all on public data** (11 node-classification graphs, 7 TU graph-classification sets;
the three synthetic graphs are controls for Prop. 1 only):

| level | what $\sigma$ cannot reach | prediction | tests |
|---|---|---|---|
| node | observed labels; features beyond $L$ hops | a compatibility/BP layer is largely absorbed by message passing; what survives is capped by neighbours' residual entropy | H1–H6 |
| node | — | homophily's sign is irrelevant; label informativeness governs | H1, H2b |
| graph | non-1-WL-invariant structure | 1-WL-computable rules are absorbed; on real benchmarks 1-WL already fixes cycle counts, so their gains can only be *statistical* and must shrink with data | H7, H8 |

**Decomposition (conjecture tested by H8b and the low-label node runs).** Observed gain = representational part
$\Delta^*$ (computable in advance; zero for expressible knowledge) + statistical part (inductive bias; vanishes
as training data grows). If this holds, a neuro-symbolic method should be judged by the representational part
alone, and measuring absorption becomes the first step of designing one.

Run order: check the hardware and memory plan, then press **Run experiments** (or run headless with
`RUN_HEAVY=1 python evidence_not_reasoning.py`). Every unit is checkpointed, so a restart resumes.

In [ ]:
# ---- 2. RESOURCE REPORT ----------------------------------------------------------------
HW = gpu_setup()
DEVICE = HW["device"]
mo.md(f"**Hardware:** `{HW}`  \n**RAM in use at start:** {rss_gb():.2f} GB")

In [ ]:
# ---- 3. CONFIG ---------------------------------------------------------------------------
# SMOKE=1 runs the whole pipeline on tiny synthetic graphs + Cora in a few CPU minutes.
SMOKE = os.environ.get("SMOKE", "0") == "1"
REAL = [
    "cora", "citeseer", "pubmed", "photo", "computers", "actor",
    "roman_empire", "amazon_ratings", "minesweeper", "tolokers", "questions",
]
SYN = ["syn_homo", "syn_perm", "syn_rand"]
# Optional overrides, e.g. NSG_DATASETS=cora,actor NSG_SEEDS=0,1 NSG_QUICK=1 for a quick pilot.
_ds_env, _seed_env = os.environ.get("NSG_DATASETS"), os.environ.get("NSG_SEEDS")
QUICK = os.environ.get("NSG_QUICK", "0") == "1"  # smaller models, shorter training
CFG = {
    "datasets": _ds_env.split(",") if _ds_env else ((SYN + ["cora"]) if SMOKE else (SYN + REAL)),
    "label_rates": [0.05, 0.5],  # fraction of nodes whose labels are observed
    "seeds": [int(t) for t in _seed_env.split(",")] if _seed_env else ([0] if SMOKE else [0, 1, 2, 3, 4]),
    "hidden": 64 if SMOKE else (128 if QUICK else 256),
    "dropout": 0.3,
    "lr": 3e-3,
    "wd": 5e-5,
    "epochs": 40 if SMOKE else (200 if QUICK else 500),
    "patience": 20 if SMOKE else (40 if QUICK else 80),
    "models": [("mlp", 2), ("gcn", 2), ("sage", 1), ("sage", 2), ("sage", 3)],
    "bp_iters": 8,
    "bp_damp": 0.5,
    "betas": [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0],
    "em_lambda": 0.5,
    "evidence_rhos": [0.0, 0.25, 0.5, 0.75, 1.0],
    # synthetic contextual SBM (Prop. 1 test)
    "syn_n": 600 if SMOKE else 3000,
    "syn_k": 6,
    "syn_deg": 10,
    "syn_dim": 32,
    "syn_signal": 1.0,
    "syn_a": 0.6,
    "data_dir": "nsgnn_data",
    "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
    "deadline_h": 11.0,
    "code_version": 4,
    # graph level (Part II)
    "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
    "graph_folds": 2 if SMOKE else 10,
    "graph_hidden": 32 if SMOKE else 64,
    "graph_layers": 4,
    "graph_lr": 1e-2,
    "graph_dropout": 0.5,
    "graph_epochs": 30 if SMOKE else 350,
    "graph_patience": 15 if SMOKE else 100,
    "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
    "graph_wl_rounds": 4,
}
# Hash only what changes results, so editing paths or deadline keeps checkpoints valid.
_skip = ("datasets", "seeds", "data_dir", "out_dir", "deadline_h")
_keyed = {k: v for k, v in CFG.items() if k not in _skip and not k.startswith("graph_")}
CFG_HASH = hashlib.sha1(json.dumps(_keyed, sort_keys=True).encode()).hexdigest()[:10]
_gkeyed = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
_gkeyed.update(v=3)
GRAPH_HASH = hashlib.sha1(json.dumps(_gkeyed, sort_keys=True).encode()).hexdigest()[:10]
os.makedirs(CFG["data_dir"], exist_ok=True)
os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
{"SMOKE": SMOKE, "cfg_hash": CFG_HASH, "n_datasets": len(CFG["datasets"])}

In [ ]:
# ---- 4. DATA: raw-file loaders (no torch_geometric) + synthetic contextual SBM --------------
import io
import pickle
import urllib.request
import scipy.sparse as sp

_GH = "https://raw.githubusercontent.com"
_URLS = {
    "planetoid": _GH + "/kimiyoung/planetoid/master/data/ind.{name}.{part}",
    "amazon": _GH + "/shchur/gnn-benchmark/master/data/npz/amazon_electronics_{name}.npz",
    "actor_feat": _GH + "/graphdml-uiuc-jlu/geom-gcn/master/new_data/film/out1_node_feature_label.txt",
    "actor_edge": _GH + "/graphdml-uiuc-jlu/geom-gcn/master/new_data/film/out1_graph_edges.txt",
    "hetero": _GH + "/yandex-research/heterophilous-graphs/main/data/{name}.npz",
}

def fetch(url, path, tries=4):
    if os.path.exists(path):
        return path
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for _t in range(tries):
        try:
            with urllib.request.urlopen(url, timeout=120) as r:
                _blob = r.read()
            with open(path + ".part", "wb") as f:
                f.write(_blob)
            os.replace(path + ".part", path)
            return path
        except Exception as e:  # network hiccup: back off and retry
            _err = e
            import time as _time

            _time.sleep(2 ** (_t + 1))
    raise RuntimeError(f"download failed: {url}: {_err}")

def _row_normalize(x):
    s = x.sum(1, keepdims=True)
    s[s == 0] = 1
    return x / s

def _load_planetoid(name, root):
    parts = {}
    for p in ["x", "tx", "allx", "y", "ty", "ally", "graph", "test.index"]:
        fp = fetch(_URLS["planetoid"].format(name=name, part=p), os.path.join(root, "planetoid", f"ind.{name}.{p}"))
        if p == "test.index":
            parts[p] = np.loadtxt(fp, dtype=np.int64)
        else:
            with open(fp, "rb") as f:
                parts[p] = pickle.load(f, encoding="latin1")
    tx, ty, allx, ally = parts["tx"], parts["ty"], parts["allx"], parts["ally"]
    ti = parts["test.index"]
    st = np.sort(ti)
    if name == "citeseer":  # isolated test nodes are missing from tx/ty
        full = np.arange(st.min(), st.max() + 1)
        tx2 = sp.lil_matrix((len(full), tx.shape[1]))
        tx2[st - st.min(), :] = tx
        ty2 = np.zeros((len(full), ty.shape[1]))
        ty2[st - st.min(), :] = ty
        tx, ty = tx2, ty2
    x = sp.vstack([allx, tx]).tolil()
    x[ti, :] = x[st, :]
    y = np.vstack([ally, ty])
    y[ti, :] = y[st, :]
    y = y.argmax(1)
    g = parts["graph"]
    src = [u for u, vs in g.items() for _v in vs]
    dst = [v for u, vs in g.items() for v in vs]
    x = _row_normalize(np.asarray(x.todense(), dtype=np.float32))
    return x, y, np.array([src, dst])

def _load_amazon(name, root):
    fp = fetch(_URLS["amazon"].format(name=name), os.path.join(root, "amazon", f"{name}.npz"))
    d = np.load(fp, allow_pickle=True)
    A = sp.csr_matrix((d["adj_data"], d["adj_indices"], d["adj_indptr"]), shape=d["adj_shape"]).tocoo()
    X = sp.csr_matrix((d["attr_data"], d["attr_indices"], d["attr_indptr"]), shape=d["attr_shape"])
    return _row_normalize(np.asarray(X.todense(), dtype=np.float32)), d["labels"], np.array([A.row, A.col])

def _load_actor(root):
    ff = fetch(_URLS["actor_feat"], os.path.join(root, "actor", "feat.txt"))
    fe = fetch(_URLS["actor_edge"], os.path.join(root, "actor", "edges.txt"))
    rows = [l.strip().split("\t") for l in open(ff).read().splitlines()[1:]]
    n = len(rows)
    x = np.zeros((n, 932), dtype=np.float32)
    y = np.zeros(n, dtype=np.int64)
    for r in rows:
        i = int(r[0])
        x[i, [int(t) for t in r[1].split(",")]] = 1
        y[i] = int(r[2])
    e = np.loadtxt(fe, skiprows=1, dtype=np.int64).T
    return _row_normalize(x), y, e

def _load_hetero(name, root):
    fp = fetch(_URLS["hetero"].format(name=name), os.path.join(root, "hetero", f"{name}.npz"))
    d = np.load(fp)
    return d["node_features"].astype(np.float32), d["node_labels"].astype(np.int64), d["edges"].T

def syn_compat(regime, K, a):
    """Row-stochastic compatibility B for the three Prop. 1 regimes (uniform class prior)."""
    J = np.ones((K, K)) / K
    if regime == "homo":
        return a * np.eye(K) + (1 - a) * J
    if regime == "perm":  # symmetric derangement: pairs (0,1),(2,3),...
        P = np.zeros((K, K))
        for i in range(0, K, 2):
            P[i, i + 1] = P[i + 1, i] = 1
        return a * P + (1 - a) * J
    h = (1 - a) / K  # "rand": same edge homophily as perm, structureless off-diagonal
    return h * np.eye(K) + (1 - h) / (K - 1) * (np.ones((K, K)) - np.eye(K))

def make_synthetic(regime, seed, cfg):
    rng = np.random.default_rng(10_000 + seed)
    n, K, d = cfg["syn_n"], cfg["syn_k"], cfg["syn_dim"]
    y = rng.integers(0, K, n)
    mu = rng.standard_normal((K, d))
    mu = cfg["syn_signal"] * mu / np.linalg.norm(mu, axis=1, keepdims=True) * np.sqrt(2)
    x = (mu[y] + rng.standard_normal((n, d))).astype(np.float32)
    joint = syn_compat(regime, K, cfg["syn_a"]) / K  # symmetric joint P(Y_u=a, Y_v=b)
    m = n * cfg["syn_deg"] // 2
    pairs = rng.choice(K * K, size=m, p=joint.ravel() / joint.sum())
    a_cls, b_cls = pairs // K, pairs % K
    members = [np.flatnonzero(y == c) for c in range(K)]
    src = np.array([rng.choice(members[c]) for c in a_cls])
    dst = np.array([rng.choice(members[c]) for c in b_cls])
    return x, y, np.array([src, dst])

def load_dataset(name, cfg, seed=0):
    root = cfg["data_dir"]
    if name.startswith("syn_"):
        x, y, e = make_synthetic(name[4:], seed, cfg)
    elif name in ("cora", "citeseer", "pubmed"):
        x, y, e = _load_planetoid(name, root)
    elif name in ("photo", "computers"):
        x, y, e = _load_amazon(name, root)
    elif name == "actor":
        x, y, e = _load_actor(root)
    else:
        x, y, e = _load_hetero(name, root)
    y = np.asarray(y, dtype=np.int64)
    n = len(y)
    e = np.asarray(e, dtype=np.int64)
    e = e[:, e[0] != e[1]]  # drop self loops
    e = np.concatenate([e, e[::-1]], 1)  # undirected
    key = np.unique(e[0] * n + e[1])  # dedupe, sorted by (src, dst)
    e = np.stack([key // n, key % n])
    return {"name": name, "x": x, "y": y, "edges": e, "K": int(y.max() + 1)}

In [ ]:
# ---- 5. GRAPH UTILITIES + LABEL-STRUCTURE STATISTICS -------------------------------------
def build_graph(ds, device):
    x = torch.tensor(ds["x"], device=device)
    y = torch.tensor(ds["y"], device=device)
    n = len(ds["y"])
    src = torch.tensor(ds["edges"][0], device=device)
    dst = torch.tensor(ds["edges"][1], device=device)
    key = src * n + dst  # already sorted and unique
    rev = torch.searchsorted(key, dst * n + src)  # index of the reverse edge
    deg = torch.zeros(n, device=device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
    # GCN: D^-1/2 (A+I) D^-1/2 ; SAGE: D^-1 A (mean of neighbours)
    loop = torch.arange(n, device=device)
    s2, d2 = torch.cat([src, loop]), torch.cat([dst, loop])
    dinv = (deg + 1).rsqrt()
    A_gcn = torch.sparse_coo_tensor(torch.stack([d2, s2]), dinv[d2] * dinv[s2], (n, n)).coalesce()
    A_mean = torch.sparse_coo_tensor(
        torch.stack([dst, src]), 1.0 / deg.clamp(min=1)[dst], (n, n)
    ).coalesce()
    return {"x": x, "y": y, "src": src, "dst": dst, "rev": rev, "n": n, "K": ds["K"],
            "A_gcn": A_gcn, "A_mean": A_mean, "deg": deg}

def mutual_info(M):
    """I(Y_u;Y_v) in nats for a nonnegative symmetric count matrix M."""
    P = M / M.sum()
    pa, pb = P.sum(1, keepdims=True), P.sum(0, keepdims=True)
    nz = P > 0
    return float((P[nz] * np.log(P[nz] / (pa @ pb)[nz])).sum())

def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())

def label_stats(y, src, dst, K):
    """Descriptive stats from ALL labels (dataset property, never used for prediction)."""
    M = np.zeros((K, K))
    np.add.at(M, (y[src], y[dst]), 1)
    pbar = M.sum(1) / M.sum()  # degree-weighted class distribution
    h_edge = float(np.trace(M) / M.sum())
    h_adj = (h_edge - (pbar**2).sum()) / (1 - (pbar**2).sum())
    I_e = mutual_info(M)
    dm = M.sum(1)
    S = M / np.sqrt(np.outer(dm, dm) + 1e-12)
    ev = np.sort(np.linalg.eigvalsh(S))[::-1]
    return {"h_edge": h_edge, "h_adj": float(h_adj), "I_edge": I_e,
            "LI": I_e / max(entropy(pbar), 1e-12),
            "lam2": float(ev[1]) if K > 1 else 0.0, "lam_min": float(ev[-1]),
            "lam_abs": float(max(abs(ev[1]), abs(ev[-1]))) if K > 1 else 0.0}

In [ ]:
# ---- 6. NEURAL PART: MLP / GCN / SAGE with pre-norm residual blocks, + label-trick input ----
nn = torch.nn
F = torch.nn.functional

class GNN(nn.Module):
    def __init__(self, kind, din, dh, K, L, dropout, extra_in=0):
        super().__init__()
        self.kind, self.L, self.p = kind, L, dropout
        self.inp = nn.Linear(din + extra_in, dh)
        self.self_w = nn.ModuleList([nn.Linear(dh, dh) for _ in range(L)])
        self.nbr_w = nn.ModuleList([nn.Linear(dh, dh, bias=False) for _ in range(L)])
        self.norms = nn.ModuleList([nn.LayerNorm(dh) for _ in range(L + 1)])
        self.out = nn.Linear(dh, K)

    def forward(self, x, G):
        h = F.dropout(F.relu(self.inp(x)), self.p, self.training)
        for l in range(self.L):
            z = self.norms[l](h)
            if self.kind == "mlp":
                m = self.self_w[l](z)
            elif self.kind == "gcn":
                m = self.self_w[l](torch.sparse.mm(G["A_gcn"], z))
            else:  # sage: separate ego / neighbour weights (heterophily-capable)
                m = self.self_w[l](z) + self.nbr_w[l](torch.sparse.mm(G["A_mean"], z))
            h = h + F.dropout(F.relu(m), self.p, self.training)
        return self.out(self.norms[-1](h))

def label_inputs(G, idx):
    """Label-trick input block: one-hot label + indicator for nodes in idx, zeros elsewhere."""
    Z = torch.zeros(G["n"], G["K"] + 1, device=G["x"].device)
    Z[idx, G["y"][idx]] = 1.0
    Z[idx, G["K"]] = 1.0
    return torch.cat([G["x"], Z], 1)

def train_model(kind, L, G, tr, va, seed, cfg, soft=None, label_trick=False):
    """Full-batch training, early stopping on validation NLL (the theory is about log-loss).
    soft = (idx, probs): extra soft-label targets (symbolic -> neural feedback, EM round).
    Returns the model at its best epoch."""
    torch.manual_seed(seed)
    K = G["K"]
    model = GNN(kind, G["x"].shape[1], cfg["hidden"], K, L, cfg["dropout"],
                K + 1 if label_trick else 0).to(G["x"].device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    x_eval = label_inputs(G, tr) if label_trick else G["x"]
    best, best_state, bad = float("inf"), None, 0
    gen = torch.Generator(device="cpu").manual_seed(seed)
    for _ep in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        if label_trick:  # half the train labels are inputs, the other half are targets
            perm = tr[torch.randperm(len(tr), generator=gen).to(tr.device)]
            half = len(perm) // 2
            out = model(label_inputs(G, perm[:half]), G)
            loss = F.cross_entropy(out[perm[half:]], G["y"][perm[half:]])
        else:
            out = model(G["x"], G)
            loss = F.cross_entropy(out[tr], G["y"][tr])
        if soft is not None:
            s_idx, s_p = soft
            loss = loss + cfg["em_lambda"] * -(s_p * F.log_softmax(out[s_idx], 1)).sum(1).mean()
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            vl = F.cross_entropy(model(x_eval, G)[va], G["y"][va]).item()
        if vl < best - 1e-5:
            best, bad = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad > cfg["patience"]:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model

In [ ]:
# ---- 7. SYMBOLIC PART: compatibility "rules", Bayes evidence layer, fractional loopy BP -----
F_ = torch.nn.functional

def calibrate_T(logits, y, va):
    """Scalar temperature on validation NLL (makes Q_v calibrated, as Prop. 2 assumes)."""
    Ts = torch.exp(torch.linspace(math.log(0.2), math.log(5.0), 41, device=logits.device))
    nll = torch.stack([F_.cross_entropy(logits[va] / t, y[va]) for t in Ts])
    return float(Ts[nll.argmin()])

def compat_pmi(G, tr, q, alpha=1.0):
    """Symmetric soft co-occurrence counts on edges with >=1 observed endpoint:
    hard one-hot on observed nodes, calibrated GNN beliefs q on the other end.
    Returns PMI matrix Psi[a,b] = log P(a,b) / (P(a)P(b)) and the count matrix."""
    n, K = G["n"], G["K"]
    obs = torch.zeros(n, dtype=torch.bool, device=q.device)
    obs[tr] = True
    W = q.clone()
    W[tr] = F_.one_hot(G["y"][tr], K).float()
    keep = obs[G["src"]]  # directed edges leaving an observed node (both directions present)
    M = W[G["src"][keep]].T @ W[G["dst"][keep]]
    M = 0.5 * (M + M.T) + alpha / K  # Dirichlet smoothing toward independence
    P = M / M.sum()
    Psi = torch.log(P) - torch.log(P.sum(1, keepdim=True)) - torch.log(P.sum(0, keepdim=True))
    return Psi, M

def bayes_evidence(logphi, G, ev_idx, Psi, beta):
    """Exact naive-Bayes update with observed neighbour labels (the evidence channel):
    log p(c) = log q_v(c) + beta * sum_{u in E, u~v} Psi[y_u, c]."""
    obs = torch.zeros(G["n"], dtype=torch.bool, device=logphi.device)
    obs[ev_idx] = True
    keep = obs[G["src"]]
    add = torch.zeros_like(logphi).index_add_(0, G["dst"][keep], Psi[G["y"][G["src"][keep]]])
    return F_.log_softmax(logphi + beta * add, 1)

def loopy_bp(logphi, G, Psi, beta, iters, damp, max_bytes=2**28):
    """Sum-product loopy BP in log space on pairwise potentials exp(beta*Psi).
    Edge-chunked so the E x K x K intermediate never exceeds max_bytes."""
    src, dst, rev, n = G["src"], G["dst"], G["rev"], G["n"]
    E, K = len(src), logphi.shape[1]
    bPsi = beta * Psi
    m = torch.zeros(E, K, device=logphi.device)
    step = max(1, max_bytes // (4 * K * K))
    for _ in range(iters):
        agg = torch.zeros(n, K, device=logphi.device).index_add_(0, dst, m)
        cav = logphi[src] + agg[src] - m[rev]
        new = torch.empty_like(m)
        for s in range(0, E, step):
            c = cav[s:s + step]
            new[s:s + step] = torch.logsumexp(c[:, :, None] + bPsi[None], 1)
        new = new - torch.logsumexp(new, 1, keepdim=True)
        m = damp * m + (1 - damp) * new
    agg = torch.zeros(n, K, device=logphi.device).index_add_(0, dst, m)
    return F_.log_softmax(logphi + agg, 1)

def clamp_unary(logphi, G, ev_idx, eps=1e-3):
    out = logphi.clone()
    K = logphi.shape[1]
    oh = F_.one_hot(G["y"][ev_idx], K).float()
    out[ev_idx] = torch.log(oh * (1 - eps) + eps / K)
    return out

def blank_unary(logphi, idx):
    out = logphi.clone()
    out[idx] = -math.log(logphi.shape[1])  # uniform: removes the label channel
    return out

def fit_beta(make_logp, y, va, betas):
    """Pick the potential strength beta on validation NLL; beta=0 recovers the base model."""
    best = (float("inf"), None, None)
    for b in betas:
        lp = make_logp(b)
        v = float(-lp[va, y[va]].mean())
        if v < best[0]:
            best = (v, b, lp)
    return best[1], best[2]

def correct_and_smooth(logphi, G, tr, y, va, alphas=(0.5, 0.8, 0.95), iters=50):
    """Correct & Smooth (Huang et al., 2021), the standard label-propagation post-processor:
    propagate train residuals (autoscaled), then propagate clamped soft labels; alphas fit on val NLL."""
    n, K = logphi.shape
    A = G["A_gcn"]
    p = logphi.exp()
    oh = F_.one_hot(y[tr], K).float()
    best = (float("inf"), None, None)
    for a1 in alphas:
        E0 = torch.zeros_like(p)
        E0[tr] = oh - p[tr]
        E = E0.clone()
        for _ in range(iters):
            E = (1 - a1) * E0 + a1 * torch.sparse.mm(A, E)
            E[tr] = E0[tr]
        sigma = E0[tr].abs().sum(1).mean()
        scale = sigma / E.abs().sum(1, keepdim=True).clamp(min=1e-12)
        Z = (p + scale * E).clamp(min=0)
        Z[tr] = oh
        for a2 in alphas:
            H = Z.clone()
            for _ in range(iters):
                H = (1 - a2) * Z + a2 * torch.sparse.mm(A, H)
                H[tr] = oh
            H = H.clamp(min=1e-8)
            lp = torch.log(H / H.sum(1, keepdim=True))
            T = calibrate_T(lp, y, va)
            lp = F_.log_softmax(lp / T, 1)
            v = float(-lp[va, y[va]].mean())
            if v < best[0]:
                best = (v, (a1, a2), lp)
    return best[1], best[2]

def rec_fit(logphi, G, ev_idx, Psi, y, va, steps=300):
    """Residual Evidence Combiner: the Bayes combiner the absorption theory asks for, with K^2 + 4 parameters
    fit on validation NLL. log p(c) = a*log Q_v(c) + g_v * n_v^(-gamma) * sum_{u in E, u~v} M[y_u, c], where the
    gate g_v = sigmoid(w0 + w1 * H(Q_v)) lets evidence flow only where the network is still uncertain about v
    (Theorem 0: gain <= H(Y_v | sigma)) and n_v^(-gamma) undoes double counting of correlated neighbours.
    Init: a = 1, M = Psi, gate open, gamma = 0, i.e. exactly the S1 Bayes layer at beta = 1."""
    K = logphi.shape[1]
    dev = logphi.device

    def _agg(ev):
        obs = torch.zeros(G["n"], dtype=torch.bool, device=dev)
        obs[ev] = True
        keep = obs[G["src"]]
        onehot_sum = torch.zeros(G["n"], K, device=dev).index_add_(
            0, G["dst"][keep], F_.one_hot(y[G["src"][keep]], K).float())
        return onehot_sum, onehot_sum.sum(1, keepdim=True)

    H = -(logphi.exp() * logphi).sum(1, keepdim=True)

    def _apply(params, ev):
        a, M, w0, w1, gam = params
        cnt, n_v = _agg(ev)
        gate = torch.sigmoid(w0 + w1 * H)
        ev_term = (cnt @ M) * n_v.clamp(min=1).pow(-gam)
        return F_.log_softmax(a * logphi + gate * ev_term, 1)

    with torch.enable_grad():
        a = torch.ones((), device=dev, requires_grad=True)
        M = Psi.detach().clone().requires_grad_(True)
        w0 = torch.full((), 2.0, device=dev, requires_grad=True)
        w1 = torch.zeros((), device=dev, requires_grad=True)
        gam = torch.zeros((), device=dev, requires_grad=True)
        params = [a, M, w0, w1, gam]
        opt = torch.optim.Adam(params, lr=0.05)
        for _ in range(steps):
            opt.zero_grad()
            lp = _apply(params, ev_idx)
            loss = -lp[va, y[va]].mean() + 1e-3 * (M - Psi).pow(2).mean()
            loss.backward()
            opt.step()
    fixed = [t.detach() for t in params]
    return (lambda ev: _apply(fixed, ev)), {"a": float(fixed[0]), "w0": float(fixed[2]),
                                             "w1": float(fixed[3]), "gamma": float(fixed[4])}

In [ ]:
# ---- 8. METRICS ------------------------------------------------------------------------------
def metrics(logp, y, idx, binary):
    lp, yy = logp[idx], y[idx]
    p = lp.exp()
    conf, pred = p.max(1)
    acc = (pred == yy).float()
    out = {"nll": float(-lp[torch.arange(len(yy), device=yy.device), yy].mean()),
           "acc": float(acc.mean())}
    bins = torch.clamp((conf * 15).long(), max=14)  # 15-bin ECE on top-1 confidence
    ece = torch.zeros((), device=lp.device)
    for b in range(15):
        sel = bins == b
        if sel.any():
            ece = ece + sel.float().mean() * (conf[sel].mean() - acc[sel].mean()).abs()
    out["ece"] = float(ece)
    if binary:  # ROC-AUC via ranks (ties broken arbitrarily; fine for continuous scores)
        s = p[:, 1]
        r = torch.empty_like(s)
        r[s.argsort()] = torch.arange(1, len(s) + 1, device=s.device, dtype=s.dtype)
        npos = float(yy.sum())
        nneg = len(yy) - npos
        out["auc"] = float((r[yy == 1].sum() - npos * (npos + 1) / 2) / max(npos * nneg, 1))
    return out

In [ ]:
# ---- 9. MEMORY PLAN (arithmetic before allocation) ------------------------------------------
# Public sizes: nodes, undirected edges, classes, feature dim. Directed edges = 2x.
_SIZES = {
    "cora": (2708, 5278, 7, 1433), "citeseer": (3327, 4552, 6, 3703), "pubmed": (19717, 44324, 3, 500),
    "photo": (7650, 119081, 8, 745), "computers": (13752, 245861, 10, 767), "actor": (7600, 26659, 5, 932),
    "roman_empire": (22662, 32927, 18, 300), "amazon_ratings": (24492, 93050, 5, 300),
    "minesweeper": (10000, 39402, 2, 7), "tolokers": (11758, 519000, 2, 10), "questions": (48921, 153540, 2, 301),
    "syn_homo": (CFG["syn_n"], CFG["syn_n"] * CFG["syn_deg"] // 2, CFG["syn_k"], CFG["syn_dim"]),
}
_rows, _worst = [], 0.0
for _name, (_n, _e, _k, _d) in _SIZES.items():
    _feat = _n * _d * 4
    _lt = _n * (_d + _k + 1) * 4  # label-trick input copy
    _act = 3 * 4 * _n * CFG["hidden"] * 4 * 4  # ~4 tensors/layer, 3 layers, fwd+bwd+grad, x4 slack
    _bp = 2 * _e * _k * 4 * 6 + min(2 * _e * _k * _k * 4, 2**28)  # messages/cavities + chunked ExKxK
    _gpu = (_feat + _lt + _act + _bp) / 2**30
    _ram = (_feat * 3) / 2**30  # numpy load + dense conversion transient
    _worst = max(_worst, _gpu)
    _rows.append(f"| {_name} | {_n:,} | {2*_e:,} | {_k} | {_d} | {_ram:.2f} | {_gpu:.2f} |")
mo.md(
    "| dataset | nodes | directed edges | K | dim | peak RAM GB (load) | peak VRAM GB (est.) |\n"
    "|---|---|---|---|---|---|---|\n" + "\n".join(_rows)
    + f"\n\n**Worst-case VRAM ≈ {_worst:.1f} GB vs budget {VRAM_CAP_GB:.0f} GB; RAM stays under ~3 GB vs soft "
    f"{RAM_SOFT_GB:.0f} GB.** One dataset is resident at a time; everything is freed between units. "
    "COMFORTABLE on both budgets, so no tiling beyond the edge-chunked BP is needed."
)

In [ ]:
run_btn = mo.ui.run_button(label="Run experiments")
run_btn

In [ ]:
# ---- 10. ONE EXPERIMENT UNIT = (dataset, label rate, seed) ----------------------------------
F2 = torch.nn.functional

def make_split(y, rate, seed):
    rng = np.random.default_rng(seed)
    tr, va, te, rest = [], [], [], []
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c))
        n_tr = max(1, int(round(rate * len(idx))))
        n_va = max(1, int(round(0.25 * len(idx))))
        n_te = max(1, int(round(0.25 * len(idx))))
        tr += list(idx[:n_tr])
        va += list(idx[n_tr:n_tr + n_va])
        te += list(idx[n_tr + n_va:n_tr + n_va + n_te])
        rest += list(idx[n_tr + n_va + n_te:])
    return [np.sort(np.array(a, dtype=np.int64)) for a in (tr, va, te, rest)]

def run_unit(name, rate, seed, device):
    ds = load_dataset(name, CFG, seed)
    G = build_graph(ds, device)
    K, y = G["K"], G["y"]
    binary = K == 2
    tr_np, va_np, te_np, _ = make_split(ds["y"], rate, seed)
    tr, va, te = (torch.tensor(a, device=device) for a in (tr_np, va_np, te_np))
    rows, ev_rows = [], []
    info = {"dataset": name, "rate": rate, "seed": seed, "n": G["n"], "E": int(len(G["src"])), "K": K}
    info.update(label_stats(ds["y"], ds["edges"][0], ds["edges"][1], K))
    prior = torch.bincount(y[tr], minlength=K).float() + 1
    prior = prior / prior.sum()
    H_prior_val = float(-torch.log(prior[y[va]]).mean())  # CE of the class prior on val
    info["H_prior_val"] = H_prior_val
    betas, it, dp = CFG["betas"], CFG["bp_iters"], CFG["bp_damp"]

    def row(model, variant, logp, beta=None, T=None):
        r = {"dataset": name, "rate": rate, "seed": seed, "model": model, "variant": variant,
             "beta": beta, "T": T, "val_nll": float(-logp[va, y[va]].mean())}
        r.update(metrics(logp, y, te, binary))
        rows.append(r)

    def symbolic_variants(tag, logits):
        T = calibrate_T(logits, y, va)
        logphi = F2.log_softmax(logits / T, 1)
        Psi, M = compat_pmi(G, tr, logphi.exp())
        row(tag, "base", logphi, 0.0, T)
        # S1: evidence channel, 1 hop, exact Bayes with observed labels
        b1, lp1 = fit_beta(lambda b: bayes_evidence(logphi, G, tr, Psi, b), y, va, betas)
        row(tag, "s1_evidence", lp1, b1, T)
        # S2: reasoning-only BP over predicted beliefs (observed nodes blanked to uniform)
        lphi_b = blank_unary(logphi, tr)
        b2, lp2 = fit_beta(lambda b: loopy_bp(lphi_b, G, Psi, b, it, dp), y, va, betas)
        row(tag, "s2_reason", lp2, b2, T)
        row(tag, "s2_reason_naive", loopy_bp(lphi_b, G, Psi, 1.0, it, dp), 1.0, T)
        # S3: evidence clamped + multi-hop BP (full collective inference)
        lphi_c = clamp_unary(logphi, G, tr)
        b3, lp3 = fit_beta(lambda b: loopy_bp(lphi_c, G, Psi, b, it, dp), y, va, betas)
        row(tag, "s3_full", lp3, b3, T)
        row(tag, "s3_full_naive", loopy_bp(lphi_c, G, Psi, 1.0, it, dp), 1.0, T)
        # Correct & Smooth: the strongest standard label-propagation baseline (also an evidence channel)
        _ab, lpcs = correct_and_smooth(logphi, G, tr, y, va)
        row(tag, "cs", lpcs, None, T)
        # REC: the absorption-aware Bayes combiner derived from Theorem 0 (fit on val, scored on test)
        rec, _rp = rec_fit(logphi, G, tr, Psi, y, va)
        row(tag, "rec", rec(tr), None, T)
        return {"T": T, "logphi": logphi, "Psi": Psi, "M": M, "b1": b1, "b3": b3, "lp3": lp3, "rec": rec}

    outs = {}
    for kind, L in CFG["models"]:
        tag = f"{kind}{L}"
        model = train_model(kind, L, G, tr, va, seed, CFG)
        with torch.no_grad():
            outs[tag] = symbolic_variants(tag, model(G["x"], G))
        del model
    sage2 = outs["sage2"]

    with torch.no_grad():
        # Prop. 2 predictor, computed BEFORE fitting any layer: label-only Bayes information on val
        Psi, M = sage2["Psi"], sage2["M"]
        lp_prior = torch.log(prior)[None].expand(G["n"], K).contiguous()
        _, lp_lab = fit_beta(lambda b: bayes_evidence(lp_prior, G, tr, Psi, b), y, va, [1.0])
        info["info_labels_val"] = H_prior_val - float(-lp_lab[va, y[va]].mean())
        Mn = M.cpu().numpy()
        P = Mn / Mn.sum()
        info["I_edge_train"] = float((P * np.log(P / np.outer(P.sum(1), P.sum(0)))).sum())
        info["LI_train"] = info["I_edge_train"] / max(entropy(P.sum(1)), 1e-12)

        # Bidirectional round: symbolic posteriors (S3 of sage2) become soft targets for a new GNN
        others = torch.tensor(np.setdiff1d(np.arange(G["n"]), tr_np), device=device)
        soft = (others, sage2["lp3"][others].exp())
    em_model = train_model("sage", 2, G, tr, va, seed + 100, CFG, soft=soft)
    with torch.no_grad():
        symbolic_variants("sage2_em", em_model(G["x"], G))
    del em_model
    for r in rows:
        if r["variant"] == "base":
            info[f"ce_val_{r['model']}"] = r["val_nll"]
    lt_model = train_model("sage", 2, G, tr, va, seed, CFG, label_trick=True)

    with torch.no_grad():
        # E4: evidence shift. Reveal a growing share of a test "pool"; score on the other half.
        rng = np.random.default_rng(seed + 7)
        perm = rng.permutation(te_np)
        pool, ev_eval = perm[: len(perm) // 2], torch.tensor(np.sort(perm[len(perm) // 2:]), device=device)
        lt_logits0 = lt_model(label_inputs(G, tr), G)
        T_lt = calibrate_T(lt_logits0, y, va)
        Psi_lt, _ = compat_pmi(G, tr, F2.log_softmax(lt_logits0 / T_lt, 1).exp())
        b_lt, _ = fit_beta(
            lambda b: loopy_bp(clamp_unary(F2.log_softmax(lt_logits0 / T_lt, 1), G, tr), G, Psi_lt, b, it, dp),
            y, va, betas)
        for rho in CFG["evidence_rhos"]:
            ev = torch.tensor(np.sort(np.concatenate([tr_np, pool[: int(round(rho * len(pool)))]])), device=device)
            lphi = sage2["logphi"]
            variants = {
                "sage2_base": lphi,
                "sage2_s1": bayes_evidence(lphi, G, ev, Psi, sage2["b1"]),
                "sage2_s3": loopy_bp(clamp_unary(lphi, G, ev), G, Psi, sage2["b3"], it, dp),
                "sage2_rec": sage2["rec"](ev),  # parameters frozen at rho = 0: tests extrapolation
            }
            lt_lp = F2.log_softmax(lt_model(label_inputs(G, ev), G) / T_lt, 1)
            variants["lt_base"] = lt_lp
            variants["lt_s3"] = loopy_bp(clamp_unary(lt_lp, G, ev), G, Psi_lt, b_lt, it, dp)
            for vname, lp in variants.items():
                r = {"dataset": name, "rate": rate, "seed": seed, "rho": rho, "method": vname}
                r.update(metrics(lp, y, ev_eval, binary))
                ev_rows.append(r)
        # H6: the absorption bound. One BP round = the 1-hop predictor that already sees every neighbour's
        # belief Q_u. Revealing the pool labels can then add at most sum_{u in pool, u~v} H(Q_u) nats to v
        # (I(Y_v; Y_P | Q_v, Q_N) <= H(Y_P | Q_N) <= sum_u H(Y_u | Q_u)); pool beliefs are out-of-sample.
        bound_rows = []
        ev_all = torch.tensor(np.sort(np.concatenate([tr_np, pool])), device=device)
        is_pool = torch.zeros(G["n"], dtype=torch.bool, device=device)
        is_pool[torch.tensor(pool, device=device)] = True
        for tag, o in outs.items():
            lphi = o["logphi"]
            Hq = -(lphi.exp() * lphi).sum(1)
            keep = is_pool[G["src"]]
            bound = torch.zeros(G["n"], device=device).index_add_(0, G["dst"][keep], Hq[G["src"][keep]])
            b1h, _ = fit_beta(lambda b: loopy_bp(clamp_unary(lphi, G, tr), G, o["Psi"], b, 1, 0.0), y, va, betas)
            lp0 = loopy_bp(clamp_unary(lphi, G, tr), G, o["Psi"], b1h, 1, 0.0)
            lp1 = loopy_bp(clamp_unary(lphi, G, ev_all), G, o["Psi"], b1h, 1, 0.0)
            idx = ev_eval
            delta = (lp1[idx, y[idx]] - lp0[idx, y[idx]]).cpu().numpy()  # nats gained per node
            bnd = bound[idx].cpu().numpy()
            # lower side: [I(Y_v; Y_P) - I(Y_P; sigma)]_+ estimated pointwise on each eval node.
            # I(Y_v; Y_P): label-only Bayes from the revealed pool neighbours vs the class prior (a CE-based
            # estimate, itself a lower bound on MI); I(Y_P; sigma) ~ sum_u [log Q_u(y_u) - log pi(y_u)].
            lp_lab = bayes_evidence(torch.log(prior)[None].expand(G["n"], -1).contiguous(), G,
                                    torch.tensor(pool, device=device), o["Psi"], 1.0)
            pmi_lab = (lp_lab[idx, y[idx]] - torch.log(prior)[y[idx]])
            pmi_q = lphi[torch.arange(G["n"], device=device), y] - torch.log(prior)[y]
            absorbed = torch.zeros(G["n"], device=device).index_add_(0, G["dst"][keep], pmi_q[G["src"][keep]])
            low = (pmi_lab - absorbed[idx]).cpu().numpy()
            zero = bnd < 1e-6
            edges_q = np.quantile(bnd[~zero], [0, 0.25, 0.5, 0.75, 1.0]) if (~zero).sum() >= 8 else None
            bins = [("zero", zero)]
            if edges_q is not None:
                for q in range(4):
                    lo, hi = edges_q[q], edges_q[q + 1]
                    bins.append((f"q{q + 1}", (~zero) & (bnd >= lo) & ((bnd <= hi) if q == 3 else (bnd < hi))))
            for bname, sel in bins:
                if sel.sum() > 0:
                    bound_rows.append({"dataset": name, "rate": rate, "seed": seed, "model": tag, "bin": bname,
                                       "n": int(sel.sum()), "beta": b1h, "mean_bound": float(bnd[sel].mean()),
                                       "mean_gain": float(delta[sel].mean()),
                                       "mean_lower": float(np.maximum(low[sel].mean(), 0.0))})
    del lt_model, G, ds
    return {"rows": rows, "evidence": ev_rows, "info": info, "bound": bound_rows}

In [ ]:
# ---- 11. HEAVY STAGE: resumable loop over units, one atomic JSON checkpoint per unit -------
mo.stop(not (run_btn.value or os.environ.get("RUN_HEAVY") == "1"),
        mo.md("Press **Run experiments** to start (finished units are loaded from disk)."))
if torch is None:
    raise RuntimeError("torch is required")
_t0 = time.time()
_units = [(d, r, s) for d in CFG["datasets"] for r in CFG["label_rates"] for s in CFG["seeds"]]
_log = os.path.join(CFG["out_dir"], "progress.log")
UNIT_RESULTS, _skipped, _failed = [], 0, []
for _d, _r, _s in mo.status.progress_bar(_units, title="units"):
    _path = os.path.join(CFG["out_dir"], "units", f"{CFG_HASH}_{_d}_{_r}_{_s}.json")
    if os.path.exists(_path):
        with open(_path) as _f:
            UNIT_RESULTS.append(json.load(_f))
        _skipped += 1
        continue
    if (time.time() - _t0) / 3600 > CFG["deadline_h"]:
        break  # leave the rest for a resumed session
    _u0 = time.time()
    try:
        _res = run_unit(_d, _r, _s, DEVICE)
    except Exception as _e:  # one bad dataset (e.g. download) must not kill the sweep
        _failed.append((_d, _r, _s, repr(_e)[:300]))
        gpu_free()
        continue
    _res["info"]["seconds"] = time.time() - _u0
    atomic_write(_path, lambda p, _x=_res: open(p, "w").write(json.dumps(_x)))
    UNIT_RESULTS.append(_res)
    with open(_log, "a") as _f:
        _f.write(f"{time.strftime('%H:%M:%S')} {_d} r={_r} s={_s} {time.time()-_u0:.1f}s "
                 f"rss={rss_gb():.2f}GB vram={vram_report()}\n")
    gpu_free()
trim_ram()
mo.md(f"**{len(UNIT_RESULTS)} / {len(_units)} units ready** ({_skipped} resumed from disk), "
      f"failed: `{_failed}`, RAM {rss_gb():.2f} GB, VRAM `{vram_report()}`, "
      f"{(time.time()-_t0)/60:.1f} min")

In [ ]:
# ---- 12. TIDY TABLES -------------------------------------------------------------------------
import pandas as pd

ROWS = pd.DataFrame([r for u in UNIT_RESULTS for r in u["rows"]])
EVID = pd.DataFrame([r for u in UNIT_RESULTS for r in u["evidence"]])
BOUND = pd.DataFrame([r for u in UNIT_RESULTS for r in u.get("bound", [])])
INFO = pd.DataFrame([u["info"] for u in UNIT_RESULTS])
_base = ROWS[ROWS.variant == "base"].set_index(["dataset", "rate", "seed", "model"])
GAINS = ROWS.join(_base[["nll", "acc", "ece"]], on=["dataset", "rate", "seed", "model"], rsuffix="_base")
GAINS["gain_nll"] = GAINS["nll_base"] - GAINS["nll"]  # nats per test node; >0 = better
GAINS["gain_acc"] = GAINS["acc"] - GAINS["acc_base"]
GAINS["d_ece"] = GAINS["ece"] - GAINS["ece_base"]
STATS = INFO.groupby("dataset")[["n", "E", "K", "h_edge", "h_adj", "LI", "I_edge", "lam2", "lam_min", "lam_abs"]].first()
STATS.round(3)

In [ ]:
# ---- 12b. HEADLINE WITH UNCERTAINTY: mean gain, 95% t-interval over seeds, and two-sided p vs 0 -----
from scipy import stats as _st

def _ci(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return float(x.mean()), float("nan"), float("nan")
    h = _st.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    p = _st.ttest_1samp(x, 0.0).pvalue if x.std() > 0 else float("nan")
    return float(x.mean()), float(h), float(p)

_sel = GAINS[~GAINS.dataset.str.startswith("syn_") & GAINS.model.isin(["mlp2", "gcn2", "sage2"])
             & GAINS.variant.isin(["s1_evidence", "s3_full", "s3_full_naive", "cs", "rec"])]
EQ_MARGIN = 0.01  # nats/node: below ~0.5% of a typical cross-entropy, i.e. practically nothing

def _tost(x, d=EQ_MARGIN):
    """Two one-sided t-tests: small p means the mean gain is provably inside (-d, +d)."""
    x = np.asarray(x, dtype=float)
    if len(x) < 2 or x.std(ddof=1) == 0:
        return float("nan")
    se = x.std(ddof=1) / np.sqrt(len(x))
    p_lo = 1 - _st.t.cdf((x.mean() + d) / se, len(x) - 1)
    p_hi = _st.t.cdf((x.mean() - d) / se, len(x) - 1)
    return float(max(p_lo, p_hi))

_rows = []
for (_d, _r, _m, _v), _g in _sel.groupby(["dataset", "rate", "model", "variant"]):
    _mu, _h, _p = _ci(_g["gain_nll"])
    _rows.append({"dataset": _d, "rate": _r, "model": _m, "variant": _v, "gain": round(_mu, 4),
                  "ci95": round(_h, 4), "p": round(_p, 4), "p_equiv_0": round(_tost(_g["gain_nll"]), 4),
                  "n_seeds": len(_g)})
import pandas as _pd

HEADLINE = _pd.DataFrame(_rows)
# pooled equivalence over dataset-rate means (each dataset-rate is one observation)
_pool = (HEADLINE.groupby(["model", "variant"])["gain"]
         .agg(mean="mean", n="size", p_equiv_0=lambda g: _tost(g)).reset_index())
POOLED_EQ = _pool
mo.vstack([mo.md("### Headline: test NLL gain (nats/node) with 95% CI over seeds, and TOST equivalence to 0 "
                 f"within ±{EQ_MARGIN} nats"), HEADLINE,
           mo.md("Pooled over dataset-rates (small `p_equiv_0` = the gain is provably negligible)"), POOLED_EQ])

In [ ]:
_t = (GAINS[GAINS.variant != "base"]
      .groupby(["dataset", "rate", "model", "variant"])["gain_nll"].mean()
      .unstack("variant").round(4))
mo.vstack([mo.md("### Test NLL gain over the calibrated base (nats/node, mean over seeds)"), _t])

In [ ]:
# ---- 13. H1 / Prop. 1: sign of heterophily is irrelevant (synthetic triad) -------------------
_K, _a = CFG["syn_k"], CFG["syn_a"]
_th = []
for _reg in ["homo", "perm", "rand"]:
    _B = syn_compat(_reg, _K, _a) / _K
    _th.append({"regime": _reg, "h_edge_theory": float(np.trace(_B)), "I_edge_theory": mutual_info(_B)})
_th = pd.DataFrame(_th).set_index("regime")
_syn = GAINS[GAINS.dataset.str.startswith("syn_") & GAINS.variant.isin(["base", "s1_evidence", "s3_full"])]
_syn = _syn.assign(regime=_syn.dataset.str[4:])
H1_TABLE = (_syn[_syn.variant != "base"].groupby(["model", "rate", "variant", "regime"])["gain_nll"]
            .mean().unstack("regime").round(4))
H1_BASE = (_syn[_syn.variant == "base"].groupby(["model", "rate", "regime"])["acc"].mean()
           .unstack("regime").round(3))
mo.vstack([mo.md("### H1: synthetic triad. Theory: `perm` and `homo` have identical $I_e$, `perm` and `rand` identical $h$"),
           _th.round(4), mo.md("Test NLL gain of the symbolic layer"), H1_TABLE,
           mo.md("Base accuracy (GCN cannot use `perm`; SAGE can)"), H1_BASE])

In [ ]:
# ---- 14. H2 / Prop. 2: the ceiling, and which statistic predicts the gain ---------------------
_real = GAINS[~GAINS.dataset.str.startswith("syn_") & (GAINS.variant == "s1_evidence")]
_j = _real.merge(INFO, on=["dataset", "rate", "seed"])
_j["ceiling"] = _j.apply(lambda r: min(r[f"ce_val_{r['model']}"], max(r["info_labels_val"], 0.0)), axis=1)
_j["under_ceiling"] = _j["gain_nll"] <= _j["ceiling"] + 0.01
H2_UNITS = _j
_agg = _j[_j.model == "sage2"].groupby(["dataset", "rate"])[
    ["gain_nll", "ceiling", "info_labels_val", "LI", "LI_train", "h_edge", "h_adj", "lam_abs"]].mean()
_rho = {}
for _rate, _g in _agg.groupby(level="rate"):
    for _c in ["ceiling", "info_labels_val", "LI", "LI_train", "h_edge", "h_adj", "lam_abs"]:
        _rho[(_rate, _c)] = _g["gain_nll"].corr(_g[_c], method="spearman")
H2_SPEARMAN = pd.Series(_rho).unstack(0).round(3)
mo.vstack([mo.md(f"### H2: fraction of (dataset, rate, seed, model) units under the ceiling: "
                 f"**{_j['under_ceiling'].mean():.1%}**"),
           mo.md("Spearman correlation of SAGE-2 evidence gain with each statistic, across real datasets"),
           H2_SPEARMAN, _agg.round(4)])

In [ ]:
# ---- 14b. H5 ABSORPTION: how much of the symbolic layer's value does message passing already own? --
# The MLP sees no graph, so its S3 gain is what the symbolic layer is worth when nothing else uses the
# graph. Absorption = 1 - gain(GNN + S3) / gain(MLP + S3): the share of that value a GNN already has.
_r = GAINS[~GAINS.dataset.str.startswith("syn_") & (GAINS.variant == "s3_full")]
_p = _r.pivot_table(index=["dataset", "rate", "seed"], columns="model", values="gain_nll")
_abs = {m: 1 - _p[m] / _p["mlp2"].where(_p["mlp2"] > 0.01) for m in ["gcn2", "sage2"] if m in _p}
ABSORB = _p[["mlp2"]].rename(columns={"mlp2": "gain_mlp_s3"}).assign(
    **{f"absorb_{m}": v for m, v in _abs.items()})
ABSORB = ABSORB.join(INFO.set_index(["dataset", "rate", "seed"])[["LI"]])
_naive = (GAINS[~GAINS.dataset.str.startswith("syn_") & GAINS.variant.isin(["s3_full", "s3_full_naive"])
                & GAINS.model.isin(["gcn2", "sage2"])]
          .groupby(["dataset", "rate", "variant"])["gain_nll"].mean().unstack("variant"))
mo.vstack([mo.md("### H5: absorption of the symbolic layer by message passing (1 = fully redundant)"),
           ABSORB.groupby(["dataset", "rate"]).mean().round(3),
           mo.md("Fitted strength vs the common choice beta = 1 (test NLL gain, GCN-2/SAGE-2 mean)"),
           _naive.round(3)])

In [ ]:
# ---- 14c. H6 ABSORPTION BOUND: gain from revealed labels vs the neighbours' residual uncertainty --------
_b = BOUND[~BOUND.dataset.str.startswith("syn_")] if len(BOUND) else BOUND
H6_TABLE = (_b.groupby(["model", "bin"])[["mean_bound", "mean_gain"]].mean().round(4)
            if len(_b) else _b)
mo.vstack([mo.md("### H6: revealed-label gain (nats) vs the bound $\\sum_u H(Q_u)$, nodes binned by bound "
                 "(mean over real datasets). Theory: gain <= bound in every bin; zero bound => zero gain; "
                 "message passing lowers $H(Q_u)$, which is *why* it absorbs the layer."), H6_TABLE])

In [ ]:
# ---- 15. H3 / Prop. 3: reasoning-only vs evidence channel, as GNN depth grows ----------------
_real = GAINS[~GAINS.dataset.str.startswith("syn_") & GAINS.model.isin(["sage1", "sage2", "sage3"])]
H3_TABLE = (_real[_real.variant.isin(["s1_evidence", "s2_reason", "s2_reason_naive", "s3_full"])]
            .groupby(["rate", "model", "variant"])["gain_nll"].mean().unstack("variant").round(4))
H3_ECE = (_real[_real.variant.isin(["s2_reason_naive", "s3_full_naive", "s2_reason"])]
          .groupby(["rate", "variant"])["d_ece"].mean().unstack("variant").round(4))
mo.vstack([mo.md("### H3: mean test NLL gain over real datasets, by SAGE depth"), H3_TABLE,
           mo.md("Change in ECE vs base (positive = worse calibration)"), H3_ECE])

In [ ]:
# ---- 16. H4: conditional queries under evidence shift -----------------------------------------
H4_TABLE = (EVID[~EVID.dataset.str.startswith("syn_")].groupby(["rate", "method", "rho"])["nll"].mean()
            .unstack("rho").round(4))
mo.vstack([mo.md("### H4: test NLL on held-out nodes as more test labels are revealed (mean over real datasets)"),
           H4_TABLE])

In [ ]:
# ---- 17. FIGURES -------------------------------------------------------------------------------
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

_fig, _ax = plt.subplots(2, 2, figsize=(12, 9))
_s = GAINS[GAINS.dataset.str.startswith("syn_") & (GAINS.variant == "s1_evidence")]
_p = _s.groupby(["model", "dataset"])["gain_nll"].mean().unstack("dataset")
_p.plot.bar(ax=_ax[0, 0])
_ax[0, 0].set_title("H1 synthetic: evidence gain (nats)")
_u = H2_UNITS[H2_UNITS.model == "sage2"]
_ax[0, 1].scatter(_u["ceiling"], _u["gain_nll"], s=12, c=_u["rate"].map({0.05: "C0", 0.5: "C1"}))
_m = max(_u["ceiling"].max(), _u["gain_nll"].max(), 1e-3)
_ax[0, 1].plot([0, _m], [0, _m], "k--", lw=1)
_ax[0, 1].set_xlabel("ceiling min(CE_val, label-only info)")
_ax[0, 1].set_ylabel("observed gain")
_ax[0, 1].set_title("H2 ceiling (blue 5%, orange 50% labels)")
_r = GAINS[~GAINS.dataset.str.startswith("syn_") & GAINS.model.isin(["sage1", "sage2", "sage3"])]
_r.groupby(["model", "variant"])["gain_nll"].mean().unstack("variant")[
    ["s1_evidence", "s2_reason", "s3_full"]].plot(marker="o", ax=_ax[1, 0])
_ax[1, 0].set_title("H3 gain vs depth")
_e = EVID[~EVID.dataset.str.startswith("syn_")].groupby(["method", "rho"])["nll"].mean().unstack("method")
_e.plot(marker="o", ax=_ax[1, 1])
_ax[1, 1].set_title("H4 NLL vs revealed test evidence")
_fig.tight_layout()
FIG_PATH = os.path.join(CFG["out_dir"], "summary.png")
_fig.savefig(FIG_PATH, dpi=120)
plt.close(_fig)
_fig

In [ ]:
# ---- 18. PRE-REGISTERED VERDICTS -------------------------------------------------------------
_v = []
# H1a (theorem): gain(homo) ~ gain(perm) for models with separate ego/neighbour weights.
# H1b (information ordering, no GNN redundancy): for the MLP base, gain(rand) << gain(homo), gain(perm).
try:
    _ok_a, _ok_b, _show = [], [], {}
    for _mdl in ["mlp2", "sage2"]:
        _g = H1_TABLE.xs((_mdl, "s1_evidence"), level=("model", "variant"))
        _show[_mdl] = _g.round(4).to_dict("index")
        for _, _row in _g.iterrows():
            _hi = max(_row["homo"], _row["perm"], 1e-6)
            _ok_a.append(abs(_row["homo"] - _row["perm"]) <= 0.25 * _hi + 0.005)
            if _mdl == "mlp2":
                _ok_b.append(_row["rand"] <= 0.5 * min(_row["homo"], _row["perm"]) + 0.005)
    _v.append(("H1a homo and perm give the same symbolic gain (MLP-2, SAGE-2)", all(_ok_a), _show))
    _v.append(("H1b at equal homophily, rand gains far less than perm (MLP-2)", all(_ok_b), ""))
except KeyError:
    _v.append(("H1", None, "no synthetic results"))
# H2: under-ceiling rate, and LI-type statistic beats homophily as a predictor
_under = float(H2_UNITS["under_ceiling"].mean())
_li = H2_SPEARMAN.loc[["LI_train", "info_labels_val"]].abs().max().mean()
_ho = H2_SPEARMAN.loc[["h_edge", "h_adj"]].abs().max().mean()
_v.append(("H2 gain below computable ceiling in >=90% of units", _under >= 0.9, f"{_under:.1%}"))
_v.append(("H2b label informativeness predicts gain better than homophily",
           bool(_li > _ho) if np.isfinite(_li) and np.isfinite(_ho) else None,
           f"|rho| LI-type {_li:.2f} vs homophily {_ho:.2f}"))
# H3: reasoning-only gain small relative to evidence gain, and shrinking with depth
_t = H3_TABLE.groupby(level="model").mean()
_small = bool((_t["s2_reason"] <= 0.2 * _t["s1_evidence"].clip(lower=1e-6) + 0.002).loc[["sage2", "sage3"]].all())
_shrink = bool(_t.loc["sage1", "s2_reason"] >= _t.loc["sage3", "s2_reason"])
_v.append(("H3 reasoning-only gain <= 20% of evidence gain (depth>=2)", _small, _t.round(4).to_dict()))
_v.append(("H3b reasoning-only gain shrinks with depth", _shrink, ""))
# H5: on real graphs with informative labels (LI > 0.1), GNNs already own most of the layer's value
_ab = ABSORB[(ABSORB["LI"] > 0.1) & ABSORB["gain_mlp_s3"].gt(0.01)]
_ab_med = float(_ab[[c for c in _ab.columns if c.startswith("absorb_")]].median().mean()) if len(_ab) else float("nan")
_naive = GAINS[~GAINS.dataset.str.startswith("syn_") & (GAINS.variant == "s3_full_naive")
               & GAINS.model.isin(["gcn2", "sage2"])]["gain_nll"].mean()
_v.append(("H5 GNNs absorb >=80% of the symbolic layer's value (median, LI>0.1)",
           bool(_ab_med >= 0.8) if np.isfinite(_ab_med) else None, f"median absorption {_ab_med:.2f}"))
_v.append(("H5b the usual beta=1 BP hurts a calibrated GNN (mean NLL gain < 0)", bool(_naive < 0),
           f"{_naive:+.3f} nats/node"))
# H6: the absorption bound holds bin by bin, and gain tracks the bound
_bb = BOUND[~BOUND.dataset.str.startswith("syn_") & (BOUND.n >= 20)] if len(BOUND) else BOUND
if len(_bb):
    _hold = float((_bb["mean_gain"] <= _bb["mean_bound"] + 1e-3).mean())
    _sp = float(_bb["mean_bound"].corr(_bb["mean_gain"], method="spearman"))
    _v.append(("H6 revealed-label gain <= sum of neighbour entropies (share of bins, n>=20)", _hold >= 0.95, f"{_hold:.1%}"))
    _v.append(("H6b gain rises with the bound (Spearman over bins > 0.5)", _sp > 0.5, f"rho = {_sp:.2f}"))
    if "mean_lower" in _bb:
        _lo_ok = float((_bb["mean_gain"] >= _bb["mean_lower"] - 1e-3).mean())
        _nontriv = _bb[_bb["mean_lower"] > 0.01]
        _lo_nt = float((_nontriv["mean_gain"] >= _nontriv["mean_lower"] - 1e-3).mean()) if len(_nontriv) else float("nan")
        _v.append(("H6c gain >= estimated lower bound (all bins; bins where the bound is non-trivial)",
                   bool(_lo_ok >= 0.9 and (np.isnan(_lo_nt) or _lo_nt >= 0.8)),
                   f"{_lo_ok:.1%} of bins; {_lo_nt:.1%} of {len(_nontriv)} non-trivial bins"))
# H9: on GNN bases the symbolic layers are provably negligible (pooled TOST, margin 0.01 nats)
_eq = POOLED_EQ[POOLED_EQ.model.isin(["gcn2", "sage2"]) & POOLED_EQ.variant.isin(["s1_evidence", "s3_full", "cs"])]
_v.append(("H9 symbolic layers on GCN/SAGE are equivalent to zero gain (pooled TOST p < 0.05, +-0.01 nats)",
           bool((_eq["p_equiv_0"] < 0.05).all()) if len(_eq) else None,
           f"{_eq.set_index(['model', 'variant'])['p_equiv_0'].round(4).to_dict()}"))
# H10: the theory-derived combiner is at least as good as the best classical layer, and safe
_rg = GAINS[~GAINS.dataset.str.startswith("syn_") & GAINS.variant.isin(["rec", "s3_full", "cs"])]
if "rec" in set(_rg.variant):
    _rp = _rg.groupby(["dataset", "rate", "model", "variant"])["gain_nll"].mean().unstack("variant")
    _best = _rp[["s3_full", "cs"]].max(1)
    _win = float((_rp["rec"] >= _best - 0.002).mean())
    _safe = float((_rp["rec"] >= -0.005).mean())
    _v.append(("H10 REC >= best of {BP, C&S} (share of dataset-rate-models, tol 0.002)", _win >= 0.6, f"{_win:.0%}"))
    _v.append(("H10b REC never hurts (gain >= -0.005 nats in >= 95% of cases)", _safe >= 0.95, f"{_safe:.0%}"))
# H4: symbolic layer adds on top of the neural label-trick when evidence is shifted
_e = EVID[~EVID.dataset.str.startswith("syn_")]
_hi = _e[_e.rho == _e.rho.max()].groupby(["dataset", "rate", "method"])["nll"].mean().unstack("method")
_win = float((_hi["lt_s3"] < _hi["lt_base"]).mean())
_mono = _e[_e.method == "sage2_s3"].groupby(["dataset", "rate", "rho"])["nll"].mean().unstack("rho")
_mono_ok = float((np.diff(_mono.values, axis=1) <= 1e-3).all(1).mean())
_v.append(("H4 symbolic layer improves the label-trick GNN at max evidence shift (share of dataset-rates)", _win >= 0.5, f"{_win:.0%}"))
_v.append(("H4b S3 NLL falls monotonically as evidence is revealed (share)", _mono_ok >= 0.8, f"{_mono_ok:.0%}"))
VERDICTS = _v
mo.md("### Verdicts\n\n| claim | holds? | evidence |\n|---|---|---|\n" + "\n".join(
    f"| {a} | {'**yes**' if b else ('no' if b is not None else 'n/a')} | `{c}` |" for a, b, c in _v))

In [ ]:
# ---- 19. WHERE RESULTS LIVE ----------------------------------------------------------------
_out = CFG["out_dir"]
atomic_write(os.path.join(_out, "rows.csv"), lambda p: GAINS.to_csv(p, index=False))
atomic_write(os.path.join(_out, "evidence.csv"), lambda p: EVID.to_csv(p, index=False))
atomic_write(os.path.join(_out, "info.csv"), lambda p: INFO.to_csv(p, index=False))
atomic_write(os.path.join(_out, "verdicts.json"),
             lambda p: open(p, "w").write(json.dumps([[a, b, str(c)] for a, b, c in VERDICTS], indent=1)))
mo.md(f"Results written to `{os.path.abspath(_out)}`: `rows.csv`, `evidence.csv`, `info.csv`, "
      f"`verdicts.json`, `{os.path.basename(FIG_PATH)}`, the graph-level files (`graph_rows.csv`, `wl_analysis.csv`, `graph_verdicts.json`), and per-unit checkpoints in `units/`. "
      "molab files may not survive the session: download this folder if you want to keep it.")

## Part II: graph level. Symbolic rules a GNN can compute are absorbed; rules beyond 1-WL are not

Graph classification has no neighbouring labels to observe, so the evidence channel is gone. What remains of
the thesis is **Prop. 4 (expressivity sufficiency):** let $h_G$ be the readout of a maximally expressive
sum-aggregation MPNN with $t$ layers (GIN; equivalent to $t$ rounds of 1-WL, Xu et al. 2019). Take any symbolic
graph rule $\varphi(G)$ that is a function of the 1-WL colour multiset after $t$ rounds, such as node and edge
counts, degree histograms or node-type counts. Then $\varphi=f(h_G)$ for an injective $h$, so
$I(Y;\varphi\mid h_G)=0$, and appending $\varphi$ can add nothing to a Bayes-optimal head. Rules that are *not*
1-WL invariant, such as triangle and 4-cycle counts, connected components, cyclomatic number and clustering,
separate graph pairs that GIN maps to the same $h_G$. They are the only rules whose conditional information
$I(Y;\varphi\mid h_G)$ can be positive. Under finite data a GNN is not Bayes-optimal, so measured absorption of
WL rules below 1 is an *estimation* gap. The node-level synthetic results show the same thing at low label rates.

**Test (H7).** Real TU benchmarks (MUTAG, PROTEINS, NCI1, IMDB-BINARY), stratified 10-fold CV. Bases: GIN and a
graph-blind DeepSets (sum of node MLPs, no edges). Each base runs with no rules, WL rules, beyond-WL rules, or
both. Absorption of a rule family is $1-\text{gain}_{\text{GIN}}/\text{gain}_{\text{DeepSets}}$ in test NLL.
The naive expressivity view predicts that WL rules are absorbed and beyond-WL rules are not. The absorption
principle predicts something sharper: wherever 1-WL already separates the dataset's graphs (H8a), beyond-WL
rules have no representational room ($H(S\mid\sigma)=0$), so they are absorbed just as fully. Any gain must be
statistical and should vanish with data (H8b). DeepSets, which cannot see edges, should still benefit (H7c).

In [ ]:
# ---- G1. TU graph-classification data (GIN authors' release on GitHub raw) + symbolic rule features ----
import zipfile as _zipfile
import urllib.request as _ur
import scipy.sparse as _sp
from scipy.sparse.csgraph import connected_components as _cc

_GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

def _fetch_gin_zip(root):
    path = os.path.join(root, "gin", "dataset.zip")
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        for _try in range(5):  # GitHub raw occasionally resets long downloads
            try:
                with _ur.urlopen(_GIN_ZIP, timeout=300) as r:
                    blob = r.read()
                break
            except Exception:
                if _try == 4:
                    raise
                time.sleep(2 ** (_try + 1))
        with open(path + ".part", "wb") as f:
            f.write(blob)
        os.replace(path + ".part", path)
    return path

def rule_features(A, deg, tags, n_tags):
    """Two families of symbolic graph rules for one graph (A: scipy csr, symmetric, 0/1)."""
    n = A.shape[0]
    m = A.nnz // 2
    dh = np.bincount(np.minimum(deg, 10), minlength=11)
    wl = np.concatenate([[n, m], dh, np.bincount(tags, minlength=n_tags)])  # 1-WL computable
    A2 = A @ A
    tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2  # triangles through each node
    tri = tri_node.sum() / 3
    A4tr = float((A2.multiply(A2.T)).sum())  # trace(A^4) = sum_ij (A^2)_ij (A^2)_ji
    c4 = (A4tr - 2 * float((deg**2).sum()) + 2 * m) / 8  # verified against brute force
    comps = _cc(A, directed=False)[0] if n > 0 else 0
    pairs = deg * (deg - 1) / 2
    clust = float(np.mean(np.divide(tri_node, pairs, out=np.zeros(n), where=pairs > 0))) if n else 0.0
    beyond = np.array([tri, c4, comps, m - n + comps, clust, (tri_node > 0).sum()])  # not 1-WL invariant
    return np.log1p(np.maximum(wl, 0)).astype(np.float32), np.log1p(np.maximum(beyond, 0)).astype(np.float32)

def load_tu(name, cfg):
    z = _zipfile.ZipFile(_fetch_gin_zip(cfg["data_dir"]))
    lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
    ptr, n_graphs = 1, int(lines[0])
    graphs, tagset = [], {}
    for _g in range(n_graphs):
        n, label = map(int, lines[ptr].split())
        ptr += 1
        tags, src, dst = [], [], []
        for i in range(n):
            row = lines[ptr].split()
            ptr += 1
            tags.append(tagset.setdefault(int(row[0]), len(tagset)))
            k = int(row[1])
            for j in row[2:2 + k]:
                src.append(i)
                dst.append(int(j))
        graphs.append((n, label, np.array(tags), np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)))
    labels = sorted({g[1] for g in graphs})
    lab_map = {l: i for i, l in enumerate(labels)}
    degree_as_tag = len(tagset) == 1  # social graphs (IMDB): use degree, the standard choice
    n_feat = 51 if degree_as_tag else len(tagset)
    xs, ys, gid, srcs, dsts, WL, BW = [], [], [], [], [], [], []
    off = 0
    for gi, (n, label, tags, s, d) in enumerate(graphs):
        A = _sp.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        A.setdiag(0)
        A.eliminate_zeros()
        deg = np.asarray(A.sum(1)).ravel().astype(np.int64)
        t = np.minimum(deg, 50) if degree_as_tag else tags
        x = np.zeros((n, n_feat), dtype=np.float32)
        x[np.arange(n), t] = 1
        w, b = rule_features(A, deg, t, n_feat)
        coo = A.tocoo()
        xs.append(x)
        srcs.append(coo.row + off)
        dsts.append(coo.col + off)
        gid.append(np.full(n, gi))
        ys.append(lab_map[label])
        WL.append(w)
        BW.append(b)
        off += n
    return {"name": name, "x": np.concatenate(xs), "src": np.concatenate(srcs), "dst": np.concatenate(dsts),
            "gid": np.concatenate(gid), "y": np.array(ys), "wl": np.stack(WL), "beyond": np.stack(BW),
            "K": len(labels), "n_graphs": n_graphs}

def wl_analysis(d, rounds=4, seed=0):
    """1-WL colour refinement on every graph at once (multiset hashing with random 64-bit weights; the
    collision probability is ~N^2 / 2^64, negligible). For each round t reports how finely the WL partition
    splits the dataset, the empirical H(Y | WL_t) (floor for any MPNN of depth t), and H(rule | WL_t) for each
    beyond-WL rule: the *representational* room left for that rule (Prop. 4 / sandwich upper bound)."""
    rng = np.random.default_rng(seed)
    N, G = len(d["gid"]), d["n_graphs"]
    c = d["x"].argmax(1).astype(np.int64)
    src, dst, gid = d["src"], d["dst"], d["gid"]
    sizes = np.bincount(gid, minlength=G)

    def _H(keys):
        _, cnt = np.unique(keys, return_counts=True, axis=0)
        p = cnt / cnt.sum()
        return float(-(p * np.log(p)).sum())

    out = []
    for t in range(rounds + 1):
        if t > 0:
            w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
            agg = np.zeros(N, dtype=np.uint64)
            np.add.at(agg, dst, w[c[src]])
            key = np.stack([c.astype(np.uint64), agg], 1)
            c = np.unique(key, axis=0, return_inverse=True)[1].ravel().astype(np.int64)
        w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
        gsig = np.zeros(G, dtype=np.uint64)
        np.add.at(gsig, gid, w[c])
        cls = np.unique(np.stack([gsig, sizes.astype(np.uint64)], 1), axis=0, return_inverse=True)[1].ravel()
        n_cls = int(cls.max() + 1)
        single = np.bincount(cls)[cls] == 1
        H_cls = _H(cls[:, None])
        row = {"dataset": d["name"], "t": t, "wl_classes": n_cls, "frac_singleton": float(single.mean()),
               "H_y": _H(d["y"][:, None]),
               "H_y_given_wl": _H(np.stack([cls, d["y"]], 1)) - H_cls,
               "acc_ceiling_wl": float(sum(np.bincount(d["y"][cls == k]).max() for k in range(n_cls)) / G)}
        beyond_names = ["triangles", "c4", "components", "cyclomatic", "clustering", "tri_nodes"]
        for j, nm in enumerate(beyond_names):
            v = np.round(d["beyond"][:, j], 6)
            row[f"H_{nm}"] = _H(v[:, None])
            row[f"H_{nm}_given_wl"] = _H(np.stack([cls.astype(float), v], 1)) - H_cls
            # plug-in H(.|WL) is trivially 0 on singleton classes, so also report the direct check: the share
            # of graphs whose WL class is shared AND contains >= 2 distinct values of the rule
            pairs = np.unique(np.stack([cls.astype(float), v], 1), axis=0)
            n_vals = np.bincount(pairs[:, 0].astype(np.int64), minlength=n_cls)
            row[f"ambig_{nm}"] = float(((n_vals[cls] > 1) & ~single).mean())
        out.append(row)
    return out

def strat_folds(y, k, seed):
    rng = np.random.default_rng(seed)
    fold = np.zeros(len(y), dtype=np.int64)
    for c in np.unique(y):
        idx = rng.permutation(np.flatnonzero(y == c))
        fold[idx] = np.arange(len(idx)) % k
    return fold

In [ ]:
# ---- G2. GIN / DeepSets with optional symbolic rule features at the readout ---------------------
_nn, _F = torch.nn, torch.nn.functional

class GraphNet(_nn.Module):
    """Standard GIN (Xu et al. 2019): eps = 0 sum aggregation, 2-layer MLP + BatchNorm per layer, and a linear
    readout of the sum-pooled features of every layer, summed. Symbolic rules enter through their own small MLP
    added to the logits. BatchNorm statistics come from all nodes in the full batch: label-free, feature-only."""

    def __init__(self, kind, din, dphi, dh, K, L, p):
        super().__init__()
        self.kind, self.L, self.p = kind, L, p
        self.inp = _nn.Linear(din, dh)
        self.mlps = _nn.ModuleList([_nn.Sequential(
            _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU(),
            _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU()) for _ in range(L)])
        self.read = _nn.ModuleList([_nn.Linear(dh, K) for _ in range(L + 1)])
        self.phi = (_nn.Sequential(_nn.Linear(dphi, dh), _nn.ReLU(), _nn.Linear(dh, K)) if dphi else None)

    def _pool(self, D, h):
        return torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h)

    def forward(self, D, phi):
        h = self.inp(D["x"])
        out = self.read[0](_F.dropout(self._pool(D, h), self.p, self.training))
        for l in range(self.L):
            if self.kind == "gin":  # sum aggregation, eps = 0: as expressive as 1-WL
                z = h + torch.sparse.mm(D["A"], h)  # row v sums h over neighbours of v
            else:  # deepsets: no edges at all (graph-blind reference)
                z = h
            h = self.mlps[l](z)
            out = out + self.read[l + 1](_F.dropout(self._pool(D, h), self.p, self.training))
        if phi is not None:
            out = out + self.phi(phi)
        return out

def train_graphnet(kind, D, phi, tr, va, seed, cfg):
    torch.manual_seed(seed)
    model = GraphNet(kind, D["x"].shape[1], 0 if phi is None else phi.shape[1], cfg["graph_hidden"],
                     D["K"], cfg["graph_layers"], cfg["graph_dropout"]).to(D["x"].device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["graph_lr"])
    sched = torch.optim.lr_scheduler.StepLR(opt, 50, 0.5)  # GIN's schedule
    best, state, bad = float("inf"), None, 0
    for _ep in range(cfg["graph_epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        loss = _F.cross_entropy(model(D, phi)[tr], D["y"][tr])
        loss.backward()
        opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vl = _F.cross_entropy(model(D, phi)[va], D["y"][va]).item()
        if state is None or vl < best - 1e-5:
            best, bad = vl, 0
            state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad > cfg["graph_patience"]:
                break
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        return model(D, phi)

In [ ]:
# ---- G3. one graph unit = (dataset, fold) -------------------------------------------------------
_F3 = torch.nn.functional

def run_graph_unit(name, fold, device):
    d = load_tu(name, CFG)
    D = {"x": torch.tensor(d["x"], device=device), "src": torch.tensor(d["src"], device=device),
         "dst": torch.tensor(d["dst"], device=device), "gid": torch.tensor(d["gid"], device=device),
         "y": torch.tensor(d["y"], device=device), "G": d["n_graphs"], "K": d["K"]}
    D["A"] = torch.sparse_coo_tensor(torch.stack([D["dst"], D["src"]]),
                                     torch.ones(len(d["src"]), device=device), (len(d["gid"]),) * 2).coalesce()
    folds = strat_folds(d["y"], CFG["graph_folds"], 0)
    te_np = np.flatnonzero(folds == fold)
    rest = np.flatnonzero(folds != fold)
    rng = np.random.default_rng(fold)
    rest = rng.permutation(rest)
    n_va = max(2, len(rest) // 9)
    va_np, tr_np = np.sort(rest[:n_va]), np.sort(rest[n_va:])
    tr, va, te = (torch.tensor(a, device=device) for a in (tr_np, va_np, te_np))
    phis = {}
    for fam in ["wl", "beyond"]:
        mu, sd = d[fam][tr_np].mean(0), d[fam][tr_np].std(0) + 1e-6  # train statistics only
        # clip: a rule that is (near-)constant on the training fold must not explode on a test graph
        phis[fam] = torch.tensor(np.clip((d[fam] - mu) / np.maximum(sd, 1e-2), -5, 5), device=device,
                                 dtype=torch.float32)
    phis["both"] = torch.cat([phis["wl"], phis["beyond"]], 1)
    rows = []
    for frac in CFG["graph_fracs"]:
        # learning curve: a stratified subsample of the training fold (val / test unchanged)
        if frac < 1.0:
            sub = []
            for c in np.unique(d["y"][tr_np]):
                idx = tr_np[d["y"][tr_np] == c]
                sub += list(rng.permutation(idx)[: max(2, int(round(frac * len(idx))))])
            tr_f = torch.tensor(np.sort(np.array(sub)), device=device)
        else:
            tr_f = tr
        kinds = ["deepsets", "gin"] if frac == 1.0 else ["gin"]
        fams = ["none", "wl", "beyond", "both"] if frac == 1.0 else ["none", "wl", "beyond"]
        for kind in kinds:
            for fam in fams:
                logits = train_graphnet(kind, D, None if fam == "none" else phis[fam], tr_f, va, fold, CFG)
                T = calibrate_T(logits, D["y"], va)
                # test sets of 18-400 graphs: one confident mistake can dominate the mean NLL, so score a
                # 2% uniform mixture (a proper rule, log-loss capped at log(50K)); accuracy is reported as well
                lp = torch.log(0.98 * _F3.softmax(logits / T, 1) + 0.02 / D["K"])
                r = {"dataset": name, "fold": fold, "frac": frac, "base": kind, "rules": fam, "T": T}
                r.update(metrics(lp, D["y"], te, False))
                rows.append(r)
    return rows

# The Absorption Principle
### When can symbolic knowledge help a graph neural network? An information-theoretic answer, tested on real graphs

**Thesis.** Let $\sigma$ be everything a GNN architecture can compute from its input (its receptive field
times its expressive power), and let $S$ be any symbolic or relational channel: observed labels, logic rules,
a compatibility matrix, motif counts, KG facts. The most any neuro-symbolic combination can add, in expected
log-loss over the best neural predictor, is **exactly**
$$\Delta^*(S)=I(Y;S\mid\sigma)=\underbrace{I(Y;S)}_{\text{symbolic value}}-\underbrace{I(S;\sigma)}_{\text{absorbed}}+\underbrace{I(S;\sigma\mid Y)}_{\text{synergy}},$$
and it is **sandwiched** by quantities measurable before any symbolic module is built:
$$\big[I(Y;S)-I(S;\sigma)\big]_+\;\le\;\Delta^*(S)\;\le\;\min\{H(S\mid\sigma),\,H(Y\mid\sigma)\},$$
with gap exactly $H(S\mid Y)$, so the bounds coincide whenever $S$ is determined by $Y$. Every nat the network
learns about $S$ removes up to one nat of symbolic value: a conservation law between the neural and symbolic channels.

**Consequences tested here, all on public data** (11 node-classification graphs, 7 TU graph-classification sets;
the three synthetic graphs are controls for Prop. 1 only):

| level | what $\sigma$ cannot reach | prediction | tests |
|---|---|---|---|
| node | observed labels; features beyond $L$ hops | a compatibility/BP layer is largely absorbed by message passing; what survives is capped by neighbours' residual entropy | H1–H6 |
| node | — | homophily's sign is irrelevant; label informativeness governs | H1, H2b |
| graph | non-1-WL-invariant structure | 1-WL-computable rules are absorbed; on real benchmarks 1-WL already fixes cycle counts, so their gains can only be *statistical* and must shrink with data | H7, H8 |

**Decomposition (conjecture tested by H8b and the low-label node runs).** Observed gain = representational part
$\Delta^*$ (computable in advance; zero for expressible knowledge) + statistical part (inductive bias; vanishes
as training data grows). If this holds, a neuro-symbolic method should be judged by the representational part
alone, and measuring absorption becomes the first step of designing one.

Run order: check the hardware and memory plan, then press **Run experiments** (or run headless with
`RUN_HEAVY=1 python evidence_not_reasoning.py`). Every unit is checkpointed, so a restart resumes.

In [ ]:
# ==== G4. graph-level heavy stage (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g4_graph_heavy():
    """G4: train GIN / DeepSets on every (dataset, fold); checkpoints to disk."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo
    import torch
    VRAM_CAP_GB = 70.0
    try:
        import psutil
    except ImportError:
        psutil = None

    def rss_gb():
        if psutil is not None:
            return psutil.Process().memory_info().rss / 2**30
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
        return float("nan")

    def gpu_free():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

    def vram_report():
        if not torch.cuda.is_available():
            return {}
        return {"allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
                "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}

    if torch.cuda.is_available():
        _total = torch.cuda.mem_get_info()[1] / 2**30
        torch.cuda.set_per_process_memory_fraction(min(1.0, min(VRAM_CAP_GB, 0.95 * _total) / _total))
        torch.backends.cuda.matmul.allow_tf32 = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    F_ = torch.nn.functional

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows
    _RUN_THIS_CELL = True  # set False to keep this cell from running

    # ---- shared helpers (copied from the full notebook) ----
    def calibrate_T(logits, y, va):
        """Scalar temperature on validation NLL (makes Q_v calibrated, as Prop. 2 assumes)."""
        Ts = torch.exp(torch.linspace(math.log(0.2), math.log(5.0), 41, device=logits.device))
        nll = torch.stack([F_.cross_entropy(logits[va] / t, y[va]) for t in Ts])
        return float(Ts[nll.argmin()])

    def metrics(logp, y, idx, binary):
        lp, yy = logp[idx], y[idx]
        p = lp.exp()
        conf, pred = p.max(1)
        acc = (pred == yy).float()
        out = {"nll": float(-lp[torch.arange(len(yy), device=yy.device), yy].mean()),
               "acc": float(acc.mean())}
        bins = torch.clamp((conf * 15).long(), max=14)  # 15-bin ECE on top-1 confidence
        ece = torch.zeros((), device=lp.device)
        for b in range(15):
            sel = bins == b
            if sel.any():
                ece = ece + sel.float().mean() * (conf[sel].mean() - acc[sel].mean()).abs()
        out["ece"] = float(ece)
        if binary:  # ROC-AUC via ranks (ties broken arbitrarily; fine for continuous scores)
            s = p[:, 1]
            r = torch.empty_like(s)
            r[s.argsort()] = torch.arange(1, len(s) + 1, device=s.device, dtype=s.dtype)
            npos = float(yy.sum())
            nneg = len(yy) - npos
            out["auc"] = float((r[yy == 1].sum() - npos * (npos + 1) / 2) / max(npos * nneg, 1))
        return out

    # ---- G1. TU graph-classification data (GIN authors' release on GitHub raw) + symbolic rule features ----
    import zipfile as zipfile
    import urllib.request as urlreq
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def _fetch_gin_zip(root):
        path = os.path.join(root, "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for _try in range(5):  # GitHub raw occasionally resets long downloads
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if _try == 4:
                        raise
                    time.sleep(2 ** (_try + 1))
            with open(path + ".part", "wb") as f:
                f.write(blob)
            os.replace(path + ".part", path)
        return path

    def rule_features(A, deg, tags, n_tags):
        """Two families of symbolic graph rules for one graph (A: scipy csr, symmetric, 0/1)."""
        n = A.shape[0]
        m = A.nnz // 2
        dh = np.bincount(np.minimum(deg, 10), minlength=11)
        wl = np.concatenate([[n, m], dh, np.bincount(tags, minlength=n_tags)])  # 1-WL computable
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2  # triangles through each node
        tri = tri_node.sum() / 3
        A4tr = float((A2.multiply(A2.T)).sum())  # trace(A^4) = sum_ij (A^2)_ij (A^2)_ji
        c4 = (A4tr - 2 * float((deg**2).sum()) + 2 * m) / 8  # verified against brute force
        comps = conn_comp(A, directed=False)[0] if n > 0 else 0
        pairs = deg * (deg - 1) / 2
        clust = float(np.mean(np.divide(tri_node, pairs, out=np.zeros(n), where=pairs > 0))) if n else 0.0
        beyond = np.array([tri, c4, comps, m - n + comps, clust, (tri_node > 0).sum()])  # not 1-WL invariant
        return np.log1p(np.maximum(wl, 0)).astype(np.float32), np.log1p(np.maximum(beyond, 0)).astype(np.float32)

    def load_tu(name, cfg):
        z = zipfile.ZipFile(_fetch_gin_zip(cfg["data_dir"]))
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, n_graphs = 1, int(lines[0])
        graphs, tagset = [], {}
        for _g in range(n_graphs):
            n, label = map(int, lines[ptr].split())
            ptr += 1
            tags, src, dst = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                k = int(row[1])
                for j in row[2:2 + k]:
                    src.append(i)
                    dst.append(int(j))
            graphs.append((n, label, np.array(tags), np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)))
        labels = sorted({g[1] for g in graphs})
        lab_map = {l: i for i, l in enumerate(labels)}
        degree_as_tag = len(tagset) == 1  # social graphs (IMDB): use degree, the standard choice
        n_feat = 51 if degree_as_tag else len(tagset)
        xs, ys, gid, srcs, dsts, WL, BW = [], [], [], [], [], [], []
        off = 0
        for gi, (n, label, tags, s, d) in enumerate(graphs):
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            deg = np.asarray(A.sum(1)).ravel().astype(np.int64)
            t = np.minimum(deg, 50) if degree_as_tag else tags
            x = np.zeros((n, n_feat), dtype=np.float32)
            x[np.arange(n), t] = 1
            w, b = rule_features(A, deg, t, n_feat)
            coo = A.tocoo()
            xs.append(x)
            srcs.append(coo.row + off)
            dsts.append(coo.col + off)
            gid.append(np.full(n, gi))
            ys.append(lab_map[label])
            WL.append(w)
            BW.append(b)
            off += n
        return {"name": name, "x": np.concatenate(xs), "src": np.concatenate(srcs), "dst": np.concatenate(dsts),
                "gid": np.concatenate(gid), "y": np.array(ys), "wl": np.stack(WL), "beyond": np.stack(BW),
                "K": len(labels), "n_graphs": n_graphs}

    def wl_analysis(d, rounds=4, seed=0):
        """1-WL colour refinement on every graph at once (multiset hashing with random 64-bit weights; the
        collision probability is ~N^2 / 2^64, negligible). For each round t reports how finely the WL partition
        splits the dataset, the empirical H(Y | WL_t) (floor for any MPNN of depth t), and H(rule | WL_t) for each
        beyond-WL rule: the *representational* room left for that rule (Prop. 4 / sandwich upper bound)."""
        rng = np.random.default_rng(seed)
        N, G = len(d["gid"]), d["n_graphs"]
        c = d["x"].argmax(1).astype(np.int64)
        src, dst, gid = d["src"], d["dst"], d["gid"]
        sizes = np.bincount(gid, minlength=G)

        def _H(keys):
            _, cnt = np.unique(keys, return_counts=True, axis=0)
            p = cnt / cnt.sum()
            return float(-(p * np.log(p)).sum())

        out = []
        for t in range(rounds + 1):
            if t > 0:
                w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
                agg = np.zeros(N, dtype=np.uint64)
                np.add.at(agg, dst, w[c[src]])
                key = np.stack([c.astype(np.uint64), agg], 1)
                c = np.unique(key, axis=0, return_inverse=True)[1].ravel().astype(np.int64)
            w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
            gsig = np.zeros(G, dtype=np.uint64)
            np.add.at(gsig, gid, w[c])
            cls = np.unique(np.stack([gsig, sizes.astype(np.uint64)], 1), axis=0, return_inverse=True)[1].ravel()
            n_cls = int(cls.max() + 1)
            single = np.bincount(cls)[cls] == 1
            H_cls = _H(cls[:, None])
            row = {"dataset": d["name"], "t": t, "wl_classes": n_cls, "frac_singleton": float(single.mean()),
                   "H_y": _H(d["y"][:, None]),
                   "H_y_given_wl": _H(np.stack([cls, d["y"]], 1)) - H_cls,
                   "acc_ceiling_wl": float(sum(np.bincount(d["y"][cls == k]).max() for k in range(n_cls)) / G)}
            beyond_names = ["triangles", "c4", "components", "cyclomatic", "clustering", "tri_nodes"]
            for j, nm in enumerate(beyond_names):
                v = np.round(d["beyond"][:, j], 6)
                row[f"H_{nm}"] = _H(v[:, None])
                row[f"H_{nm}_given_wl"] = _H(np.stack([cls.astype(float), v], 1)) - H_cls
                # plug-in H(.|WL) is trivially 0 on singleton classes, so also report the direct check: the share
                # of graphs whose WL class is shared AND contains >= 2 distinct values of the rule
                pairs = np.unique(np.stack([cls.astype(float), v], 1), axis=0)
                n_vals = np.bincount(pairs[:, 0].astype(np.int64), minlength=n_cls)
                row[f"ambig_{nm}"] = float(((n_vals[cls] > 1) & ~single).mean())
            out.append(row)
        return out

    def strat_folds(y, k, seed):
        rng = np.random.default_rng(seed)
        fold = np.zeros(len(y), dtype=np.int64)
        for c in np.unique(y):
            idx = rng.permutation(np.flatnonzero(y == c))
            fold[idx] = np.arange(len(idx)) % k
        return fold

    # ---- G2. GIN / DeepSets with optional symbolic rule features at the readout ---------------------
    _nn, _F = torch.nn, torch.nn.functional

    class GraphNet(_nn.Module):
        """Standard GIN (Xu et al. 2019): eps = 0 sum aggregation, 2-layer MLP + BatchNorm per layer, and a linear
        readout of the sum-pooled features of every layer, summed. Symbolic rules enter through their own small MLP
        added to the logits. BatchNorm statistics come from all nodes in the full batch: label-free, feature-only."""

        def __init__(self, kind, din, dphi, dh, K, L, p):
            super().__init__()
            self.kind, self.L, self.p = kind, L, p
            self.inp = _nn.Linear(din, dh)
            self.mlps = _nn.ModuleList([_nn.Sequential(
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU(),
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU()) for _ in range(L)])
            self.read = _nn.ModuleList([_nn.Linear(dh, K) for _ in range(L + 1)])
            self.phi = (_nn.Sequential(_nn.Linear(dphi, dh), _nn.ReLU(), _nn.Linear(dh, K)) if dphi else None)

        def _pool(self, D, h):
            return torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h)

        def forward(self, D, phi):
            h = self.inp(D["x"])
            out = self.read[0](_F.dropout(self._pool(D, h), self.p, self.training))
            for l in range(self.L):
                if self.kind == "gin":  # sum aggregation, eps = 0: as expressive as 1-WL
                    z = h + torch.sparse.mm(D["A"], h)  # row v sums h over neighbours of v
                else:  # deepsets: no edges at all (graph-blind reference)
                    z = h
                h = self.mlps[l](z)
                out = out + self.read[l + 1](_F.dropout(self._pool(D, h), self.p, self.training))
            if phi is not None:
                out = out + self.phi(phi)
            return out

    def train_graphnet(kind, D, phi, tr, va, seed, cfg):
        torch.manual_seed(seed)
        model = GraphNet(kind, D["x"].shape[1], 0 if phi is None else phi.shape[1], cfg["graph_hidden"],
                         D["K"], cfg["graph_layers"], cfg["graph_dropout"]).to(D["x"].device)
        opt = torch.optim.Adam(model.parameters(), lr=cfg["graph_lr"])
        sched = torch.optim.lr_scheduler.StepLR(opt, 50, 0.5)  # GIN's schedule
        best, state, bad = float("inf"), None, 0
        for _ep in range(cfg["graph_epochs"]):
            model.train()
            opt.zero_grad(set_to_none=True)
            loss = _F.cross_entropy(model(D, phi)[tr], D["y"][tr])
            loss.backward()
            opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                vl = _F.cross_entropy(model(D, phi)[va], D["y"][va]).item()
            if state is None or vl < best - 1e-5:
                best, bad = vl, 0
                state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > cfg["graph_patience"]:
                    break
        model.load_state_dict(state)
        model.eval()
        with torch.no_grad():
            return model(D, phi)

    # ---- G3. one graph unit = (dataset, fold) -------------------------------------------------------
    _F3 = torch.nn.functional

    def run_graph_unit(name, fold, device):
        d = load_tu(name, CFG)
        D = {"x": torch.tensor(d["x"], device=device), "src": torch.tensor(d["src"], device=device),
             "dst": torch.tensor(d["dst"], device=device), "gid": torch.tensor(d["gid"], device=device),
             "y": torch.tensor(d["y"], device=device), "G": d["n_graphs"], "K": d["K"]}
        D["A"] = torch.sparse_coo_tensor(torch.stack([D["dst"], D["src"]]),
                                         torch.ones(len(d["src"]), device=device), (len(d["gid"]),) * 2).coalesce()
        folds = strat_folds(d["y"], CFG["graph_folds"], 0)
        te_np = np.flatnonzero(folds == fold)
        rest = np.flatnonzero(folds != fold)
        rng = np.random.default_rng(fold)
        rest = rng.permutation(rest)
        n_va = max(2, len(rest) // 9)
        va_np, tr_np = np.sort(rest[:n_va]), np.sort(rest[n_va:])
        tr, va, te = (torch.tensor(a, device=device) for a in (tr_np, va_np, te_np))
        phis = {}
        for fam in ["wl", "beyond"]:
            mu, sd = d[fam][tr_np].mean(0), d[fam][tr_np].std(0) + 1e-6  # train statistics only
            # clip: a rule that is (near-)constant on the training fold must not explode on a test graph
            phis[fam] = torch.tensor(np.clip((d[fam] - mu) / np.maximum(sd, 1e-2), -5, 5), device=device,
                                     dtype=torch.float32)
        phis["both"] = torch.cat([phis["wl"], phis["beyond"]], 1)
        rows = []
        for frac in CFG["graph_fracs"]:
            # learning curve: a stratified subsample of the training fold (val / test unchanged)
            if frac < 1.0:
                sub = []
                for c in np.unique(d["y"][tr_np]):
                    idx = tr_np[d["y"][tr_np] == c]
                    sub += list(rng.permutation(idx)[: max(2, int(round(frac * len(idx))))])
                tr_f = torch.tensor(np.sort(np.array(sub)), device=device)
            else:
                tr_f = tr
            kinds = ["deepsets", "gin"] if frac == 1.0 else ["gin"]
            fams = ["none", "wl", "beyond", "both"] if frac == 1.0 else ["none", "wl", "beyond"]
            for kind in kinds:
                for fam in fams:
                    logits = train_graphnet(kind, D, None if fam == "none" else phis[fam], tr_f, va, fold, CFG)
                    T = calibrate_T(logits, D["y"], va)
                    # test sets of 18-400 graphs: one confident mistake can dominate the mean NLL, so score a
                    # 2% uniform mixture (a proper rule, log-loss capped at log(50K)); accuracy is reported as well
                    lp = torch.log(0.98 * _F3.softmax(logits / T, 1) + 0.02 / D["K"])
                    r = {"dataset": name, "fold": fold, "frac": frac, "base": kind, "rules": fam, "T": T}
                    r.update(metrics(lp, D["y"], te, False))
                    rows.append(r)
        return rows

    # ---- G4. graph-level heavy stage: resumable, one checkpoint per (dataset, fold) ------------------
    # Any failure is shown below with the step that failed, the full traceback, RAM / VRAM at that moment and
    # a hint; the same text goes to graph_failures.log and to the console. Finished units are always kept.
    import sys as _sys
    import traceback as _tb

    mo.stop(not _RUN_THIS_CELL, mo.md("skipped: set _RUN_THIS_CELL = True at the top of this cell"))

    def _hint(msg):
        m = msg.lower()
        if "out of memory" in m or "cudaerrormemoryallocation" in m:
            return "GPU out of memory: drop this dataset from graph_datasets or lower graph_hidden."
        if "memoryerror" in m or "cannot allocate" in m:
            return "system RAM (28 GB cap) exhausted while building this dataset."
        if "device-side assert" in m or "illegal memory access" in m:
            return "CUDA context is broken (bad index on GPU): restart the kernel; finished units are kept."
        if any(k in m for k in ("urlopen", "http", "timed out", "connection", "badzipfile", "remote end")):
            return ("download of dataset.zip failed: rerun, or delete the half-written file in "
                    f"{os.path.join(CFG['data_dir'], 'gin')} and rerun.")
        if "keyerror" in m and "dataset/" in m:
            return "this dataset name is not in the GIN zip: check spelling in graph_datasets."
        if "nameerror" in m or ("keyerror" in m and "graph_" in m):
            return "an older copy of another cell is still loaded: re-download the notebook and restart."
        return "see the traceback: the last frame is the line that raised."

    def _diagnose(name, fold):
        """Re-run the unit one step at a time so the failing step is named exactly."""
        step = "1. download + parse the TU file (load_tu)"
        try:
            d = load_tu(name, CFG)
            step = f"2. copy to {DEVICE} ({len(d['gid'])} nodes, {len(d['src'])} directed edges)"
            src = torch.tensor(d["src"], device=DEVICE)
            dst = torch.tensor(d["dst"], device=DEVICE)
            torch.tensor(d["x"], device=DEVICE)
            step = "3. build the sparse adjacency"
            A = torch.sparse_coo_tensor(torch.stack([dst, src]), torch.ones(len(d["src"]), device=DEVICE),
                                        (len(d["gid"]),) * 2).coalesce()
            del A, src, dst
            step = "4. the full unit (training GIN / DeepSets, calibration, metrics)"
            run_graph_unit(name, fold, DEVICE)
            return step, None  # passed on retry: the first failure was transient
        except Exception:
            return step, _tb.format_exc()

    _t0 = time.time()
    GRAPH_ROWS, _gfail = [], []
    _gunits = [(d_, f_) for d_ in CFG["graph_datasets"] for f_ in range(CFG["graph_folds"])]
    _log = os.path.join(CFG["out_dir"], "graph_failures.log")
    _dead_ds = set()  # a dataset whose step 1-3 fails will fail on every fold: skip the rest
    try:
        for _d, _f in mo.status.progress_bar(_gunits, title="graph units"):
            _path = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{_d}_{_f}.json")
            if os.path.exists(_path):
                with open(_path) as _fh:
                    GRAPH_ROWS += json.load(_fh)
                continue
            if _d in _dead_ds or (time.time() - _t0) / 3600 > CFG["deadline_h"]:
                continue
            _e = None
            try:
                _rows = run_graph_unit(_d, _f, DEVICE)
            except Exception as _err:
                _e, _first = _err, _tb.format_exc()
            if _e is not None:  # diagnose outside the except block, so tracebacks are not chained
                _mem = f"RAM {rss_gb():.1f} GB, VRAM {vram_report()}"
                try:
                    gpu_free()
                    _step, _again = _diagnose(_d, _f)
                except Exception:
                    _step, _again = "CUDA cleanup", _tb.format_exc()
                _rec = {"dataset": _d, "fold": _f, "error": repr(_e)[:300], "step": _step, "memory": _mem,
                        "hint": _hint(_first), "traceback": _again or _first,
                        "retry": "failed again" if _again else "passed on retry (transient)"}
                _gfail.append(_rec)
                _txt = (f"--- {_d} fold={_f} | step: {_step} | {_mem} | {_rec['retry']}\n"
                        f"hint: {_rec['hint']}\n{_rec['traceback']}\n")
                with open(_log, "a") as _fh:
                    _fh.write(_txt)
                print(_txt, file=_sys.stderr, flush=True)
                if _again and not _step.startswith("4."):
                    _dead_ds.add(_d)
                if "CUDA context" in _rec["hint"] or _step == "CUDA cleanup":
                    break
                continue
            atomic_write(_path, lambda p, _x=_rows: open(p, "w").write(json.dumps(_x)))
            GRAPH_ROWS += _rows
            with open(os.path.join(CFG["out_dir"], "progress.log"), "a") as _fh:
                _fh.write(f"{time.strftime('%H:%M:%S')} graph {_d} fold={_f} r=g\n")
            gpu_free()
    except Exception:  # an error outside a unit (checkpoint write, progress bar, ...) is shown, not raised
        _gfail.append({"dataset": "-", "fold": -1, "error": "outside a unit", "step": "loop / checkpoint write",
                       "memory": f"RAM {rss_gb():.1f} GB", "hint": _hint(_tb.format_exc()),
                       "traceback": _tb.format_exc(), "retry": "-"})
        print(_gfail[-1]["traceback"], file=_sys.stderr, flush=True)

    _done = len({(r["dataset"], r["fold"]) for r in GRAPH_ROWS})
    _head = mo.md(f"**graph units ready: {_done} / {len(_gunits)}**" +
                  (f", **{len(_gfail)} failed** (also in `{_log}`)" if _gfail else ", no failures"))
    _items = {f"{r['dataset']} fold {r['fold']}: {r['error'][:90]}":
              mo.md(f"- **failed at:** {r['step']}\n- **retry:** {r['retry']}\n- **memory:** {r['memory']}\n"
                    f"- **hint:** {r['hint']}\n\n```text\n{r['traceback']}\n```") for r in _gfail}
    return mo.vstack([_head, mo.accordion(_items)] if _gfail else [_head])


_g4_graph_heavy()

In [ ]:
# ==== G5. H7 graph-level absorption (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g5_h7_absorption():
    """G5: H7 tables and verdicts, read from G4's checkpoints on disk."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows

    GRAPH_ROWS = load_graph_rows()  # read from disk: no dependency on the G4 cell's variables

    # ---- G5. H7: absorption of WL-computable vs beyond-WL symbolic rules ------------------------------
    _g = pd.DataFrame(GRAPH_ROWS)
    if len(_g):
        _b = _g[_g.rules == "none"].set_index(["dataset", "fold", "frac", "base"])[["nll", "acc"]]
        _g = _g.join(_b, on=["dataset", "fold", "frac", "base"], rsuffix="_none")
        _g["gain_nll"] = _g["nll_none"] - _g["nll"]
        _g["gain_acc"] = _g["acc"] - _g["acc_none"]
        GRAPH_GAINS = _g.copy()
        _g = _g[_g.frac == 1.0]
        H7_GAINS = _g.groupby(["dataset", "base", "rules"])[["acc", "gain_nll", "gain_acc"]].mean()
        _p = _g[_g.rules != "none"].groupby(["dataset", "rules", "base"])["gain_nll"].mean().unstack("base")
        _p["absorption"] = 1 - _p["gin"] / _p["deepsets"].where(_p["deepsets"] > 0.01)
        _pa = _g[_g.rules != "none"].groupby(["dataset", "rules", "base"])["gain_acc"].mean().unstack("base")
        _p["acc_gain_deepsets"], _p["acc_gain_gin"] = _pa["deepsets"], _pa["gin"]
        _p["absorption_acc"] = 1 - _pa["gin"] / _pa["deepsets"].where(_pa["deepsets"] > 0.005)
        H7_ABSORB = _p
        atomic_write(os.path.join(CFG["out_dir"], "graph_rows.csv"), lambda p: _g.to_csv(p, index=False))
        # absorption on smoothed NLL and on accuracy; a family counts as absorbed when GIN keeps <= 20% of the
        # gain DeepSets gets (or DeepSets gains nothing and GIN gains nothing either)
        def _absorbed(fam):
            _x = _p.xs(fam, level="rules")
            _ok_nll = (_x["absorption"] >= 0.8) | (_x["deepsets"].le(0.01) & _x["gin"].le(0.01))
            _ok_acc = (_x["absorption_acc"] >= 0.8) | (_x["acc_gain_deepsets"].le(0.005) & _x["acc_gain_gin"].le(0.005))
            return _ok_nll & _ok_acc, _x
        _okw, _xw = _absorbed("wl")
        _okb, _xb = _absorbed("beyond")
        H7_VERDICTS = [
            ("H7a WL-computable rules are absorbed by GIN (share of datasets)", bool(_okw.mean() >= 0.5),
             f"{_okw.to_dict()}"),
            ("H7b beyond-WL rules are ALSO absorbed (the non-obvious prediction, since H8a: 1-WL already fixes them)",
             bool(_okb.mean() >= 0.5), f"{_okb.to_dict()}"),
            ("H7c rules help the graph-blind DeepSets (they carry real value; share of datasets, accuracy)",
             bool((_xb["acc_gain_deepsets"] > 0).mean() >= 0.5), f"{_xb['acc_gain_deepsets'].round(4).to_dict()}"),
        ]
    else:
        H7_GAINS = H7_ABSORB = GRAPH_GAINS = pd.DataFrame()
        H7_VERDICTS = []
    atomic_write(os.path.join(CFG["out_dir"], "h7_verdicts.json"), lambda p: open(p, "w").write(
        json.dumps([[a, b, str(c)] for a, b, c in H7_VERDICTS], indent=1)))
    return mo.vstack([mo.md("### H7: graph-level absorption (NLL gain of adding rules; absorption = 1 - GIN gain / DeepSets gain)"),
               H7_ABSORB.round(4) if len(H7_ABSORB) else mo.md("no graph results"),
               mo.md("Mean test accuracy and gains by base and rule family"), H7_GAINS.round(4),
               mo.md("\n".join(f"- **{a}**: {'yes' if b else ('no' if b is not None else 'n/a')} `{c}`"
                               for a, b, c in H7_VERDICTS))])


_g5_h7_absorption()

In [ ]:
# ==== G6. H8 WL partition and learning curves (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g6_h8_wl():
    """G6: H8 WL analysis and learning curves; reads checkpoints and h7_verdicts.json."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows
    _RUN_THIS_CELL = True  # set False to keep this cell from running

    # ---- G1. TU graph-classification data (GIN authors' release on GitHub raw) + symbolic rule features ----
    import zipfile as zipfile
    import urllib.request as urlreq
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def _fetch_gin_zip(root):
        path = os.path.join(root, "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for _try in range(5):  # GitHub raw occasionally resets long downloads
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if _try == 4:
                        raise
                    time.sleep(2 ** (_try + 1))
            with open(path + ".part", "wb") as f:
                f.write(blob)
            os.replace(path + ".part", path)
        return path

    def rule_features(A, deg, tags, n_tags):
        """Two families of symbolic graph rules for one graph (A: scipy csr, symmetric, 0/1)."""
        n = A.shape[0]
        m = A.nnz // 2
        dh = np.bincount(np.minimum(deg, 10), minlength=11)
        wl = np.concatenate([[n, m], dh, np.bincount(tags, minlength=n_tags)])  # 1-WL computable
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2  # triangles through each node
        tri = tri_node.sum() / 3
        A4tr = float((A2.multiply(A2.T)).sum())  # trace(A^4) = sum_ij (A^2)_ij (A^2)_ji
        c4 = (A4tr - 2 * float((deg**2).sum()) + 2 * m) / 8  # verified against brute force
        comps = conn_comp(A, directed=False)[0] if n > 0 else 0
        pairs = deg * (deg - 1) / 2
        clust = float(np.mean(np.divide(tri_node, pairs, out=np.zeros(n), where=pairs > 0))) if n else 0.0
        beyond = np.array([tri, c4, comps, m - n + comps, clust, (tri_node > 0).sum()])  # not 1-WL invariant
        return np.log1p(np.maximum(wl, 0)).astype(np.float32), np.log1p(np.maximum(beyond, 0)).astype(np.float32)

    def load_tu(name, cfg):
        z = zipfile.ZipFile(_fetch_gin_zip(cfg["data_dir"]))
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, n_graphs = 1, int(lines[0])
        graphs, tagset = [], {}
        for _g in range(n_graphs):
            n, label = map(int, lines[ptr].split())
            ptr += 1
            tags, src, dst = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                k = int(row[1])
                for j in row[2:2 + k]:
                    src.append(i)
                    dst.append(int(j))
            graphs.append((n, label, np.array(tags), np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)))
        labels = sorted({g[1] for g in graphs})
        lab_map = {l: i for i, l in enumerate(labels)}
        degree_as_tag = len(tagset) == 1  # social graphs (IMDB): use degree, the standard choice
        n_feat = 51 if degree_as_tag else len(tagset)
        xs, ys, gid, srcs, dsts, WL, BW = [], [], [], [], [], [], []
        off = 0
        for gi, (n, label, tags, s, d) in enumerate(graphs):
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            deg = np.asarray(A.sum(1)).ravel().astype(np.int64)
            t = np.minimum(deg, 50) if degree_as_tag else tags
            x = np.zeros((n, n_feat), dtype=np.float32)
            x[np.arange(n), t] = 1
            w, b = rule_features(A, deg, t, n_feat)
            coo = A.tocoo()
            xs.append(x)
            srcs.append(coo.row + off)
            dsts.append(coo.col + off)
            gid.append(np.full(n, gi))
            ys.append(lab_map[label])
            WL.append(w)
            BW.append(b)
            off += n
        return {"name": name, "x": np.concatenate(xs), "src": np.concatenate(srcs), "dst": np.concatenate(dsts),
                "gid": np.concatenate(gid), "y": np.array(ys), "wl": np.stack(WL), "beyond": np.stack(BW),
                "K": len(labels), "n_graphs": n_graphs}

    def wl_analysis(d, rounds=4, seed=0):
        """1-WL colour refinement on every graph at once (multiset hashing with random 64-bit weights; the
        collision probability is ~N^2 / 2^64, negligible). For each round t reports how finely the WL partition
        splits the dataset, the empirical H(Y | WL_t) (floor for any MPNN of depth t), and H(rule | WL_t) for each
        beyond-WL rule: the *representational* room left for that rule (Prop. 4 / sandwich upper bound)."""
        rng = np.random.default_rng(seed)
        N, G = len(d["gid"]), d["n_graphs"]
        c = d["x"].argmax(1).astype(np.int64)
        src, dst, gid = d["src"], d["dst"], d["gid"]
        sizes = np.bincount(gid, minlength=G)

        def _H(keys):
            _, cnt = np.unique(keys, return_counts=True, axis=0)
            p = cnt / cnt.sum()
            return float(-(p * np.log(p)).sum())

        out = []
        for t in range(rounds + 1):
            if t > 0:
                w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
                agg = np.zeros(N, dtype=np.uint64)
                np.add.at(agg, dst, w[c[src]])
                key = np.stack([c.astype(np.uint64), agg], 1)
                c = np.unique(key, axis=0, return_inverse=True)[1].ravel().astype(np.int64)
            w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
            gsig = np.zeros(G, dtype=np.uint64)
            np.add.at(gsig, gid, w[c])
            cls = np.unique(np.stack([gsig, sizes.astype(np.uint64)], 1), axis=0, return_inverse=True)[1].ravel()
            n_cls = int(cls.max() + 1)
            single = np.bincount(cls)[cls] == 1
            H_cls = _H(cls[:, None])
            row = {"dataset": d["name"], "t": t, "wl_classes": n_cls, "frac_singleton": float(single.mean()),
                   "H_y": _H(d["y"][:, None]),
                   "H_y_given_wl": _H(np.stack([cls, d["y"]], 1)) - H_cls,
                   "acc_ceiling_wl": float(sum(np.bincount(d["y"][cls == k]).max() for k in range(n_cls)) / G)}
            beyond_names = ["triangles", "c4", "components", "cyclomatic", "clustering", "tri_nodes"]
            for j, nm in enumerate(beyond_names):
                v = np.round(d["beyond"][:, j], 6)
                row[f"H_{nm}"] = _H(v[:, None])
                row[f"H_{nm}_given_wl"] = _H(np.stack([cls.astype(float), v], 1)) - H_cls
                # plug-in H(.|WL) is trivially 0 on singleton classes, so also report the direct check: the share
                # of graphs whose WL class is shared AND contains >= 2 distinct values of the rule
                pairs = np.unique(np.stack([cls.astype(float), v], 1), axis=0)
                n_vals = np.bincount(pairs[:, 0].astype(np.int64), minlength=n_cls)
                row[f"ambig_{nm}"] = float(((n_vals[cls] > 1) & ~single).mean())
            out.append(row)
        return out

    def strat_folds(y, k, seed):
        rng = np.random.default_rng(seed)
        fold = np.zeros(len(y), dtype=np.int64)
        for c in np.unique(y):
            idx = rng.permutation(np.flatnonzero(y == c))
            fold[idx] = np.arange(len(idx)) % k
        return fold

    GRAPH_ROWS = load_graph_rows()
    _g6 = pd.DataFrame(GRAPH_ROWS)
    if len(_g6):
        _b6 = _g6[_g6.rules == "none"].set_index(["dataset", "fold", "frac", "base"])[["nll", "acc"]]
        _g6 = _g6.join(_b6, on=["dataset", "fold", "frac", "base"], rsuffix="_none")
        _g6["gain_nll"] = _g6["nll_none"] - _g6["nll"]
        _g6["gain_acc"] = _g6["acc"] - _g6["acc_none"]
    GRAPH_GAINS = _g6
    _h7p = os.path.join(CFG["out_dir"], "h7_verdicts.json")
    H7_VERDICTS = [tuple(v) for v in json.load(open(_h7p))] if os.path.exists(_h7p) else []

    # ---- G6. H8: representational vs statistical value of beyond-WL rules --------------------------------
    # If 1-WL already separates (almost) every graph, H(rule | WL) ~ 0: the rule has no representational value
    # left (sandwich upper bound), so any gain it shows must be statistical, and must shrink as data grows.
    mo.stop(not _RUN_THIS_CELL, mo.md("skipped: set _RUN_THIS_CELL = True at the top of this cell"))
    _wl = []
    for _d in CFG["graph_datasets"]:
        try:
            _wl += wl_analysis(load_tu(_d, CFG), CFG["graph_wl_rounds"])
        except Exception as _e:
            _wl.append({"dataset": _d, "error": repr(_e)[:200]})
    WL_TABLE = pd.DataFrame(_wl)
    atomic_write(os.path.join(CFG["out_dir"], "wl_analysis.csv"), lambda p: WL_TABLE.to_csv(p, index=False))
    _hcols = [c for c in WL_TABLE.columns if c.endswith("_given_wl") and c != "H_y_given_wl"]
    _last = WL_TABLE[WL_TABLE.t == WL_TABLE.t.max()] if "t" in WL_TABLE else WL_TABLE
    H8_VERDICTS = []
    if len(GRAPH_GAINS):
        _lc = (GRAPH_GAINS[(GRAPH_GAINS.base == "gin") & (GRAPH_GAINS.rules != "none")]
               .groupby(["dataset", "rules", "frac"])["gain_nll"].mean().unstack("frac"))
        LC_TABLE = _lc
        _fr = sorted(GRAPH_GAINS.frac.unique())
        if len(_fr) > 1 and "beyond" in _lc.index.get_level_values("rules"):
            _bw = _lc.xs("beyond", level="rules")
            _shrink = float((_bw[_fr[0]] > _bw[_fr[-1]]).mean())
            H8_VERDICTS.append(("H8b beyond-WL gain on GIN shrinks with training size (share of datasets)",
                                _shrink >= 0.5, f"{_bw.round(4).to_dict()}"))
    else:
        LC_TABLE = pd.DataFrame()
    _acols = [c for c in WL_TABLE.columns if c.startswith("ambig_")]
    if len(_acols) and len(_last):
        _amb = _last.set_index("dataset")[_acols].max(axis=1)
        H8_VERDICTS.insert(0, ("H8a 1-WL leaves no representational room for beyond-WL rules "
                               "(<= 1% of graphs share a WL class with a graph of different rule value)",
                               bool((_amb <= 0.01).all()), f"{_amb.round(4).to_dict()}"))
    atomic_write(os.path.join(CFG["out_dir"], "graph_verdicts.json"), lambda p: open(p, "w").write(
        json.dumps([[a, b, str(c)] for a, b, c in list(H7_VERDICTS) + H8_VERDICTS], indent=1)))
    return mo.vstack([mo.md("### H8: WL partition of each dataset, and the room it leaves for beyond-WL rules"),
               WL_TABLE.round(4),
               mo.md("Learning curves: GIN test NLL gain from rules at each training fraction"),
               LC_TABLE.round(4) if len(LC_TABLE) else mo.md("no graph results"),
               mo.md("\n".join(f"- **{a}**: {'yes' if b else 'no'} `{c}`" for a, b, c in H8_VERDICTS))])


_g6_h8_wl()

In [ ]:
def _proofs_md():
    """Proofs (markdown only)."""
    import marimo as mo

    return mo.md(
        r"""
    ## Proofs (short, complete)

    **Theorem 0 (absorption identity and sandwich).** Let $\mathcal F$ be the architecture and $\sigma=\sigma(f(O):f\in\mathcal F)$
    the information it can compute from the observable input $O$. For any channel $S$, the best log-loss from $\sigma$ is
    $H(Y\mid\sigma)$, and from $(\sigma,S)$ it is $H(Y\mid\sigma,S)$ (Gibbs), so the maximal gain is
    $\Delta^*=I(Y;S\mid\sigma)$, attained by the Bayes combiner. Expanding $I(Y;S,\sigma)$ both ways gives the
    interaction identity $I(Y;S\mid\sigma)=I(Y;S)-I(S;\sigma)+I(S;\sigma\mid Y)$. The lower bound drops the
    non-negative synergy term. The upper bounds are $I(Y;S\mid\sigma)\le H(S\mid\sigma)$ and $\le H(Y\mid\sigma)$.
    Upper minus lower $=H(S\mid\sigma)-I(Y;S)+I(S;\sigma)=H(S)-I(Y;S)=H(S\mid Y)$ (using the first upper bound).
    So the bounds coincide iff $H(S\mid Y)=0$. $\square$

    **Corollary 0.1 (expressible knowledge is worthless).** If $S$ is $\sigma$-measurable, $H(S\mid\sigma)=0$, so $\Delta^*=0$.
    For a $t$-layer GIN with injective aggregation, $\sigma$ equals the $\sigma$-algebra of the 1-WL colour histogram
    after $t$ rounds (Xu et al. 2019; Morris et al. 2019). Every rule that factors through it (node/edge counts,
    degree histograms, type counts) has zero representational value. This is Prop. 4.

    **Corollary 0.2 (node-level evidence).** For $S=Y_{\mathcal P\cap N(v)}$,
    $\Delta^*\le H(Y_{\mathcal P}\mid\sigma)\le\sum_u H(Y_u\mid Q_u)$, which H6 checks bin by bin. Props. 2–3 are
    the special cases where $S$ is observed labels or features beyond $L$ hops.

    **What is proved vs conjectured.** Theorem 0, its corollaries and Props. 1–4 are proved (above and below). That
    measured gains split into this representational part plus a statistical part that vanishes with data is a
    *conjecture*, tested by H8b and by the label-rate contrast at node level. Whether the principle reshapes how
    neuro-symbolic graph methods are built is an empirical claim this notebook can support or refute, not prove.

    **Setup.** Labels $Y$, features $X$, graph $G$. Base GNN output $Q_v$ (a calibrated distribution over $K$
    classes). Observed set $\mathcal L$. Edge joint $P_{ab}=\Pr(Y_u{=}a,Y_v{=}b)$ over a uniformly random edge,
    symmetric; row-stochastic compatibility $B=\mathrm{diag}(\pi)^{-1}P$; edge homophily $h=\operatorname{tr}P$;
    edge information $I_e=I(Y_u;Y_v)$.

    **Prop. 1 (relabelling invariance; homophily is only a floor).**
    (a) For any permutation $\sigma$ of classes, the joint $P^\sigma_{ab}=P_{a\sigma(b)}$ (relabel one endpoint)
    has $I(P^\sigma)=I(P)$, since mutual information is invariant under a bijection of one variable. The naive-Bayes
    evidence posterior under $P^\sigma$ given neighbour labels $y_u$ equals the posterior under $P$ given $\sigma(y_u)$,
    and $\sigma$ is a bijection, so the full distribution of posteriors, hence the expected log-loss of the evidence
    layer, is identical. Edge homophily changes from $\operatorname{tr}P$ to $\sum_a P_{a\sigma(a)}$, which for a
    derangement can be anything down to $\approx 0$.
    (b) Fix uniform $\pi$ and $h$. Averaging $B$ over conjugations $\Pi_\tau B\Pi_\tau^\top$ by all permutations $\tau$
    gives $B_r=h I+\tfrac{1-h}{K-1}(J-I)$. $I$ is convex in $B$ for a fixed input marginal and invariant under
    conjugation, so $I(B_r)\le I(B)$: the structureless matrix **minimises** information at given $h$, with
    $I_{\min}(h)=\log K - \big[-h\log h-(1-h)\log\tfrac{1-h}{K-1}\big]$, which is $0$ at $h=1/K$. A derangement at the
    same $h$ reaches $I=\log K$ when $a\to 1$. So $h$ bounds $I_e$ from below only. $\square$

    **Prop. 2 (ceiling).** For any predictor $f$ of $Y_v$ from $(Q_v, Y_{\mathcal L\cap N(v)})$,
    $\mathbb E[-\log f(Y_v)]\ge H(Y_v\mid Q_v,Y_{\mathcal L\cap N(v)})$ (Gibbs). A calibrated base attains
    $H(Y_v\mid Q_v)$. Hence gain $\le I(Y_v;Y_{\mathcal L\cap N(v)}\mid Q_v)\le H(Y_v\mid Q_v)$, and if the neighbour
    labels are conditionally independent given $(Y_v,Q_v)$, the chain rule gives
    $I(Y_v;Y_{\mathcal L\cap N}\mid Q_v)\le\sum_u I(Y_v;Y_u\mid Q_v)$. The notebook's *computable* ceiling replaces the
    conditional term by the unconditional label-only information measured on validation; that substitution is a
    **heuristic** (conditioning on $Q_v$ can raise MI), which is exactly why H2 is tested rather than assumed. $\square$

    **Prop. 3 (sufficiency kills reasoning-only gain).** Let $R_L(v)$ be the $L$-hop receptive field and suppose
    $Q_v=p(Y_v\mid X_{R_L(v)})$. Then $Y_v\perp X_{R_L(v)}\mid Q_v$ (a posterior is a sufficient statistic). $T$ rounds
    of BP over predicted beliefs, with no observed labels, compute a function of $X_{R_{L+T}(v)}$, so their gain is at
    most $I(Y_v; X_{R_{L+T}(v)}\mid X_{R_L(v)})$, the information in the feature shell beyond $L$ hops, which shrinks
    as $L$ grows whenever dependence decays with distance. Observed labels are *not* functions of $X$, so the evidence
    channel $I(Y_v;Y_{\mathcal L\cap N}\mid X_{R_L})$ survives any depth. Naive BP ($\beta=1$) also double counts
    the shared receptive fields of neighbours, so it should hurt calibration (ECE up). $\square$

    **What would falsify the thesis.** H1 fails if `perm` gains much less than `homo` for SAGE; H3 fails if
    reasoning-only BP gives gains comparable to evidence BP at depth $\ge 2$; H4 fails if the symbolic layer adds
    nothing on top of a label-trick GNN even when evidence density is far outside training.
    """
    )


_proofs_md()

## Proofs (short, complete)

**Theorem 0 (absorption identity and sandwich).** Let $\mathcal F$ be the architecture and $\sigma=\sigma(f(O):f\in\mathcal F)$
the information it can compute from the observable input $O$. For any channel $S$, the best log-loss from $\sigma$ is
$H(Y\mid\sigma)$, and from $(\sigma,S)$ it is $H(Y\mid\sigma,S)$ (Gibbs), so the maximal gain is
$\Delta^*=I(Y;S\mid\sigma)$, attained by the Bayes combiner. Expanding $I(Y;S,\sigma)$ both ways gives the
interaction identity $I(Y;S\mid\sigma)=I(Y;S)-I(S;\sigma)+I(S;\sigma\mid Y)$. The lower bound drops the
non-negative synergy term. The upper bounds are $I(Y;S\mid\sigma)\le H(S\mid\sigma)$ and $\le H(Y\mid\sigma)$.
Upper minus lower $=H(S\mid\sigma)-I(Y;S)+I(S;\sigma)=H(S)-I(Y;S)=H(S\mid Y)$ (using the first upper bound).
So the bounds coincide iff $H(S\mid Y)=0$. $\square$

**Corollary 0.1 (expressible knowledge is worthless).** If $S$ is $\sigma$-measurable, $H(S\mid\sigma)=0$, so $\Delta^*=0$.
For a $t$-layer GIN with injective aggregation, $\sigma$ equals the $\sigma$-algebra of the 1-WL colour histogram
after $t$ rounds (Xu et al. 2019; Morris et al. 2019). Every rule that factors through it (node/edge counts,
degree histograms, type counts) has zero representational value. This is Prop. 4.

**Corollary 0.2 (node-level evidence).** For $S=Y_{\mathcal P\cap N(v)}$,
$\Delta^*\le H(Y_{\mathcal P}\mid\sigma)\le\sum_u H(Y_u\mid Q_u)$, which H6 checks bin by bin. Props. 2–3 are
the special cases where $S$ is observed labels or features beyond $L$ hops.

**What is proved vs conjectured.** Theorem 0, its corollaries and Props. 1–4 are proved (above and below). That
measured gains split into this representational part plus a statistical part that vanishes with data is a
*conjecture*, tested by H8b and by the label-rate contrast at node level. Whether the principle reshapes how
neuro-symbolic graph methods are built is an empirical claim this notebook can support or refute, not prove.

**Setup.** Labels $Y$, features $X$, graph $G$. Base GNN output $Q_v$ (a calibrated distribution over $K$
classes). Observed set $\mathcal L$. Edge joint $P_{ab}=\Pr(Y_u{=}a,Y_v{=}b)$ over a uniformly random edge,
symmetric; row-stochastic compatibility $B=\mathrm{diag}(\pi)^{-1}P$; edge homophily $h=\operatorname{tr}P$;
edge information $I_e=I(Y_u;Y_v)$.

**Prop. 1 (relabelling invariance; homophily is only a floor).**
(a) For any permutation $\sigma$ of classes, the joint $P^\sigma_{ab}=P_{a\sigma(b)}$ (relabel one endpoint)
has $I(P^\sigma)=I(P)$, since mutual information is invariant under a bijection of one variable. The naive-Bayes
evidence posterior under $P^\sigma$ given neighbour labels $y_u$ equals the posterior under $P$ given $\sigma(y_u)$,
and $\sigma$ is a bijection, so the full distribution of posteriors, hence the expected log-loss of the evidence
layer, is identical. Edge homophily changes from $\operatorname{tr}P$ to $\sum_a P_{a\sigma(a)}$, which for a
derangement can be anything down to $\approx 0$.
(b) Fix uniform $\pi$ and $h$. Averaging $B$ over conjugations $\Pi_\tau B\Pi_\tau^\top$ by all permutations $\tau$
gives $B_r=h I+\tfrac{1-h}{K-1}(J-I)$. $I$ is convex in $B$ for a fixed input marginal and invariant under
conjugation, so $I(B_r)\le I(B)$: the structureless matrix **minimises** information at given $h$, with
$I_{\min}(h)=\log K - \big[-h\log h-(1-h)\log\tfrac{1-h}{K-1}\big]$, which is $0$ at $h=1/K$. A derangement at the
same $h$ reaches $I=\log K$ when $a\to 1$. So $h$ bounds $I_e$ from below only. $\square$

**Prop. 2 (ceiling).** For any predictor $f$ of $Y_v$ from $(Q_v, Y_{\mathcal L\cap N(v)})$,
$\mathbb E[-\log f(Y_v)]\ge H(Y_v\mid Q_v,Y_{\mathcal L\cap N(v)})$ (Gibbs). A calibrated base attains
$H(Y_v\mid Q_v)$. Hence gain $\le I(Y_v;Y_{\mathcal L\cap N(v)}\mid Q_v)\le H(Y_v\mid Q_v)$, and if the neighbour
labels are conditionally independent given $(Y_v,Q_v)$, the chain rule gives
$I(Y_v;Y_{\mathcal L\cap N}\mid Q_v)\le\sum_u I(Y_v;Y_u\mid Q_v)$. The notebook's *computable* ceiling replaces the
conditional term by the unconditional label-only information measured on validation; that substitution is a
**heuristic** (conditioning on $Q_v$ can raise MI), which is exactly why H2 is tested rather than assumed. $\square$

**Prop. 3 (sufficiency kills reasoning-only gain).** Let $R_L(v)$ be the $L$-hop receptive field and suppose
$Q_v=p(Y_v\mid X_{R_L(v)})$. Then $Y_v\perp X_{R_L(v)}\mid Q_v$ (a posterior is a sufficient statistic). $T$ rounds
of BP over predicted beliefs, with no observed labels, compute a function of $X_{R_{L+T}(v)}$, so their gain is at
most $I(Y_v; X_{R_{L+T}(v)}\mid X_{R_L(v)})$, the information in the feature shell beyond $L$ hops, which shrinks
as $L$ grows whenever dependence decays with distance. Observed labels are *not* functions of $X$, so the evidence
channel $I(Y_v;Y_{\mathcal L\cap N}\mid X_{R_L})$ survives any depth. Naive BP ($\beta=1$) also double counts
the shared receptive fields of neighbours, so it should hurt calibration (ECE up). $\square$

**What would falsify the thesis.** H1 fails if `perm` gains much less than `homo` for SAGE; H3 fails if
reasoning-only BP gives gains comparable to evidence BP at depth $\ge 2$; H4 fails if the symbolic layer adds
nothing on top of a label-trick GNN even when evidence density is far outside training.

In [ ]:
# ==== G7. readout test (GIN linear vs MLP readout) (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g7_readout_test():
    """G7: GIN with linear vs MLP readout, with and without rules; checkpoints to disk."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo
    import torch
    VRAM_CAP_GB = 70.0
    try:
        import psutil
    except ImportError:
        psutil = None

    def rss_gb():
        if psutil is not None:
            return psutil.Process().memory_info().rss / 2**30
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
        return float("nan")

    def gpu_free():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

    def vram_report():
        if not torch.cuda.is_available():
            return {}
        return {"allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
                "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}

    if torch.cuda.is_available():
        _total = torch.cuda.mem_get_info()[1] / 2**30
        torch.cuda.set_per_process_memory_fraction(min(1.0, min(VRAM_CAP_GB, 0.95 * _total) / _total))
        torch.backends.cuda.matmul.allow_tf32 = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    F_ = torch.nn.functional

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows
    _RUN_THIS_CELL = True  # set False to keep this cell from running

    # ---- shared helpers (copied from the full notebook) ----
    def calibrate_T(logits, y, va):
        """Scalar temperature on validation NLL (makes Q_v calibrated, as Prop. 2 assumes)."""
        Ts = torch.exp(torch.linspace(math.log(0.2), math.log(5.0), 41, device=logits.device))
        nll = torch.stack([F_.cross_entropy(logits[va] / t, y[va]) for t in Ts])
        return float(Ts[nll.argmin()])

    def metrics(logp, y, idx, binary):
        lp, yy = logp[idx], y[idx]
        p = lp.exp()
        conf, pred = p.max(1)
        acc = (pred == yy).float()
        out = {"nll": float(-lp[torch.arange(len(yy), device=yy.device), yy].mean()),
               "acc": float(acc.mean())}
        bins = torch.clamp((conf * 15).long(), max=14)  # 15-bin ECE on top-1 confidence
        ece = torch.zeros((), device=lp.device)
        for b in range(15):
            sel = bins == b
            if sel.any():
                ece = ece + sel.float().mean() * (conf[sel].mean() - acc[sel].mean()).abs()
        out["ece"] = float(ece)
        if binary:  # ROC-AUC via ranks (ties broken arbitrarily; fine for continuous scores)
            s = p[:, 1]
            r = torch.empty_like(s)
            r[s.argsort()] = torch.arange(1, len(s) + 1, device=s.device, dtype=s.dtype)
            npos = float(yy.sum())
            nneg = len(yy) - npos
            out["auc"] = float((r[yy == 1].sum() - npos * (npos + 1) / 2) / max(npos * nneg, 1))
        return out

    # ---- G1. TU graph-classification data (GIN authors' release on GitHub raw) + symbolic rule features ----
    import zipfile as zipfile
    import urllib.request as urlreq
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def _fetch_gin_zip(root):
        path = os.path.join(root, "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for _try in range(5):  # GitHub raw occasionally resets long downloads
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if _try == 4:
                        raise
                    time.sleep(2 ** (_try + 1))
            with open(path + ".part", "wb") as f:
                f.write(blob)
            os.replace(path + ".part", path)
        return path

    def rule_features(A, deg, tags, n_tags):
        """Two families of symbolic graph rules for one graph (A: scipy csr, symmetric, 0/1)."""
        n = A.shape[0]
        m = A.nnz // 2
        dh = np.bincount(np.minimum(deg, 10), minlength=11)
        wl = np.concatenate([[n, m], dh, np.bincount(tags, minlength=n_tags)])  # 1-WL computable
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2  # triangles through each node
        tri = tri_node.sum() / 3
        A4tr = float((A2.multiply(A2.T)).sum())  # trace(A^4) = sum_ij (A^2)_ij (A^2)_ji
        c4 = (A4tr - 2 * float((deg**2).sum()) + 2 * m) / 8  # verified against brute force
        comps = conn_comp(A, directed=False)[0] if n > 0 else 0
        pairs = deg * (deg - 1) / 2
        clust = float(np.mean(np.divide(tri_node, pairs, out=np.zeros(n), where=pairs > 0))) if n else 0.0
        beyond = np.array([tri, c4, comps, m - n + comps, clust, (tri_node > 0).sum()])  # not 1-WL invariant
        return np.log1p(np.maximum(wl, 0)).astype(np.float32), np.log1p(np.maximum(beyond, 0)).astype(np.float32)

    def load_tu(name, cfg):
        z = zipfile.ZipFile(_fetch_gin_zip(cfg["data_dir"]))
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, n_graphs = 1, int(lines[0])
        graphs, tagset = [], {}
        for _g in range(n_graphs):
            n, label = map(int, lines[ptr].split())
            ptr += 1
            tags, src, dst = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                k = int(row[1])
                for j in row[2:2 + k]:
                    src.append(i)
                    dst.append(int(j))
            graphs.append((n, label, np.array(tags), np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)))
        labels = sorted({g[1] for g in graphs})
        lab_map = {l: i for i, l in enumerate(labels)}
        degree_as_tag = len(tagset) == 1  # social graphs (IMDB): use degree, the standard choice
        n_feat = 51 if degree_as_tag else len(tagset)
        xs, ys, gid, srcs, dsts, WL, BW = [], [], [], [], [], [], []
        off = 0
        for gi, (n, label, tags, s, d) in enumerate(graphs):
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            deg = np.asarray(A.sum(1)).ravel().astype(np.int64)
            t = np.minimum(deg, 50) if degree_as_tag else tags
            x = np.zeros((n, n_feat), dtype=np.float32)
            x[np.arange(n), t] = 1
            w, b = rule_features(A, deg, t, n_feat)
            coo = A.tocoo()
            xs.append(x)
            srcs.append(coo.row + off)
            dsts.append(coo.col + off)
            gid.append(np.full(n, gi))
            ys.append(lab_map[label])
            WL.append(w)
            BW.append(b)
            off += n
        return {"name": name, "x": np.concatenate(xs), "src": np.concatenate(srcs), "dst": np.concatenate(dsts),
                "gid": np.concatenate(gid), "y": np.array(ys), "wl": np.stack(WL), "beyond": np.stack(BW),
                "K": len(labels), "n_graphs": n_graphs}

    def wl_analysis(d, rounds=4, seed=0):
        """1-WL colour refinement on every graph at once (multiset hashing with random 64-bit weights; the
        collision probability is ~N^2 / 2^64, negligible). For each round t reports how finely the WL partition
        splits the dataset, the empirical H(Y | WL_t) (floor for any MPNN of depth t), and H(rule | WL_t) for each
        beyond-WL rule: the *representational* room left for that rule (Prop. 4 / sandwich upper bound)."""
        rng = np.random.default_rng(seed)
        N, G = len(d["gid"]), d["n_graphs"]
        c = d["x"].argmax(1).astype(np.int64)
        src, dst, gid = d["src"], d["dst"], d["gid"]
        sizes = np.bincount(gid, minlength=G)

        def _H(keys):
            _, cnt = np.unique(keys, return_counts=True, axis=0)
            p = cnt / cnt.sum()
            return float(-(p * np.log(p)).sum())

        out = []
        for t in range(rounds + 1):
            if t > 0:
                w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
                agg = np.zeros(N, dtype=np.uint64)
                np.add.at(agg, dst, w[c[src]])
                key = np.stack([c.astype(np.uint64), agg], 1)
                c = np.unique(key, axis=0, return_inverse=True)[1].ravel().astype(np.int64)
            w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
            gsig = np.zeros(G, dtype=np.uint64)
            np.add.at(gsig, gid, w[c])
            cls = np.unique(np.stack([gsig, sizes.astype(np.uint64)], 1), axis=0, return_inverse=True)[1].ravel()
            n_cls = int(cls.max() + 1)
            single = np.bincount(cls)[cls] == 1
            H_cls = _H(cls[:, None])
            row = {"dataset": d["name"], "t": t, "wl_classes": n_cls, "frac_singleton": float(single.mean()),
                   "H_y": _H(d["y"][:, None]),
                   "H_y_given_wl": _H(np.stack([cls, d["y"]], 1)) - H_cls,
                   "acc_ceiling_wl": float(sum(np.bincount(d["y"][cls == k]).max() for k in range(n_cls)) / G)}
            beyond_names = ["triangles", "c4", "components", "cyclomatic", "clustering", "tri_nodes"]
            for j, nm in enumerate(beyond_names):
                v = np.round(d["beyond"][:, j], 6)
                row[f"H_{nm}"] = _H(v[:, None])
                row[f"H_{nm}_given_wl"] = _H(np.stack([cls.astype(float), v], 1)) - H_cls
                # plug-in H(.|WL) is trivially 0 on singleton classes, so also report the direct check: the share
                # of graphs whose WL class is shared AND contains >= 2 distinct values of the rule
                pairs = np.unique(np.stack([cls.astype(float), v], 1), axis=0)
                n_vals = np.bincount(pairs[:, 0].astype(np.int64), minlength=n_cls)
                row[f"ambig_{nm}"] = float(((n_vals[cls] > 1) & ~single).mean())
            out.append(row)
        return out

    def strat_folds(y, k, seed):
        rng = np.random.default_rng(seed)
        fold = np.zeros(len(y), dtype=np.int64)
        for c in np.unique(y):
            idx = rng.permutation(np.flatnonzero(y == c))
            fold[idx] = np.arange(len(idx)) % k
        return fold

    # ---- G7. readout test: is GIN's missing "absorption" a readout limit? ---------------------------------
    # GIN's standard readout is LINEAR in the sum-pooled features, so it cannot form nonlinear functions of pooled
    # counts (log n, degree ratios, ...) even though 1-WL makes those counts available. Prediction (refined theory):
    # with an MLP readout the gain from WL-computable rules collapses toward 0, while beyond-WL rules keep theirs.
    _nn, _F7 = torch.nn, torch.nn.functional
    READOUT_CONFIGS = [("gin", "linear"), ("gin", "mlp"), ("deepsets", "mlp")]
    READOUT_RULES = ["none", "wl", "beyond"]
    RKEY = dict(GKEY, readout_configs=READOUT_CONFIGS, readout_rules=READOUT_RULES, rv=1)
    READOUT_HASH = hashlib.sha1(json.dumps(RKEY, sort_keys=True).encode()).hexdigest()[:10]

    class GraphNetR(_nn.Module):
        """GIN (eps = 0, sum aggregation, BN MLPs) or DeepSets, with either the standard per-layer LINEAR readout of
        sum pools, or an MLP readout on the concatenated sum pools of all layers (BatchNorm first, so raw count
        scales are handled). Rules enter through their own MLP added to the logits, as in G4."""

        def __init__(self, kind, readout, din, dphi, dh, K, L, p):
            super().__init__()
            self.kind, self.readout, self.L, self.p = kind, readout, L, p
            self.inp = _nn.Linear(din, dh)
            self.mlps = _nn.ModuleList([_nn.Sequential(
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU(),
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU()) for _ in range(L)])
            if readout == "linear":
                self.read = _nn.ModuleList([_nn.Linear(dh, K) for _ in range(L + 1)])
            else:
                self.read = _nn.Sequential(_nn.BatchNorm1d((L + 1) * dh), _nn.Linear((L + 1) * dh, dh), _nn.ReLU(),
                                           _nn.Dropout(p), _nn.Linear(dh, dh), _nn.ReLU(), _nn.Linear(dh, K))
            self.phi = (_nn.Sequential(_nn.Linear(dphi, dh), _nn.ReLU(), _nn.Linear(dh, K)) if dphi else None)

        def forward(self, D, phi):
            h = self.inp(D["x"])
            pools = [torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h)]
            for l in range(self.L):
                z = h + torch.sparse.mm(D["A"], h) if self.kind == "gin" else h
                h = self.mlps[l](z)
                pools.append(torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h))
            if self.readout == "linear":
                out = sum(r(_F7.dropout(q, self.p, self.training)) for r, q in zip(self.read, pools))
            else:
                out = self.read(torch.cat(pools, 1))
            if phi is not None:
                out = out + self.phi(phi)
            return out

    def train_r(kind, readout, D, phi, tr, va, seed, cfg):
        torch.manual_seed(seed)
        model = GraphNetR(kind, readout, D["x"].shape[1], 0 if phi is None else phi.shape[1], cfg["graph_hidden"],
                          D["K"], cfg["graph_layers"], cfg["graph_dropout"]).to(D["x"].device)
        opt = torch.optim.Adam(model.parameters(), lr=cfg["graph_lr"])
        sched = torch.optim.lr_scheduler.StepLR(opt, 50, 0.5)
        best, state, bad = float("inf"), None, 0
        for _ep in range(cfg["graph_epochs"]):
            model.train()
            opt.zero_grad(set_to_none=True)
            _F7.cross_entropy(model(D, phi)[tr], D["y"][tr]).backward()
            opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                vl = _F7.cross_entropy(model(D, phi)[va], D["y"][va]).item()
            if state is None or vl < best - 1e-5:
                best, bad = vl, 0
                state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > cfg["graph_patience"]:
                    break
        model.load_state_dict(state)
        model.eval()
        with torch.no_grad():
            return model(D, phi)

    def run_readout_unit(name, fold, device):
        """Same data, folds, validation split and rule standardisation as G3/G4, so results pair with G4's."""
        d = load_tu(name, CFG)
        D = {"x": torch.tensor(d["x"], device=device), "src": torch.tensor(d["src"], device=device),
             "dst": torch.tensor(d["dst"], device=device), "gid": torch.tensor(d["gid"], device=device),
             "y": torch.tensor(d["y"], device=device), "G": d["n_graphs"], "K": d["K"]}
        D["A"] = torch.sparse_coo_tensor(torch.stack([D["dst"], D["src"]]), torch.ones(len(d["src"]), device=device),
                                         (len(d["gid"]),) * 2).coalesce()
        folds = strat_folds(d["y"], CFG["graph_folds"], 0)
        te_np = np.flatnonzero(folds == fold)
        rest = np.random.default_rng(fold).permutation(np.flatnonzero(folds != fold))
        n_va = max(2, len(rest) // 9)
        va_np, tr_np = np.sort(rest[:n_va]), np.sort(rest[n_va:])
        tr, va, te = (torch.tensor(a, device=device) for a in (tr_np, va_np, te_np))
        phis = {}
        for fam in ["wl", "beyond"]:
            mu, sd = d[fam][tr_np].mean(0), d[fam][tr_np].std(0) + 1e-6
            phis[fam] = torch.tensor(np.clip((d[fam] - mu) / np.maximum(sd, 1e-2), -5, 5), device=device,
                                     dtype=torch.float32)
        rows = []
        for kind, readout in READOUT_CONFIGS:
            for fam in READOUT_RULES:
                logits = train_r(kind, readout, D, None if fam == "none" else phis[fam], tr, va, fold, CFG)
                T = calibrate_T(logits, D["y"], va)
                lp = torch.log(0.98 * _F7.softmax(logits / T, 1) + 0.02 / D["K"])
                r = {"dataset": name, "fold": fold, "base": f"{kind}_{readout}", "rules": fam, "T": T}
                r.update(metrics(lp, D["y"], te, False))
                rows.append(r)
        return rows

    import sys
    import traceback
    mo.stop(not _RUN_THIS_CELL, mo.md("skipped: set _RUN_THIS_CELL = True at the top of this cell"))
    _t0 = time.time()
    _units = [(d_, f_) for d_ in CFG["graph_datasets"] for f_ in range(CFG["graph_folds"])]
    _fail, _n_done = [], 0
    _log = os.path.join(CFG["out_dir"], "readout_failures.log")
    for _d, _f in mo.status.progress_bar(_units, title="readout units"):
        _path = os.path.join(CFG["out_dir"], "units", f"R{READOUT_HASH}_{_d}_{_f}.json")
        if os.path.exists(_path):
            _n_done += 1
            continue
        if (time.time() - _t0) / 3600 > CFG["deadline_h"]:
            break
        _err = None
        try:
            _rows = run_readout_unit(_d, _f, DEVICE)
        except Exception as _e:
            _err = (repr(_e)[:300], traceback.format_exc())
        if _err is not None:
            _fail.append((_d, _f, _err[0]))
            _txt = f"--- {_d} fold={_f} | RAM {rss_gb():.1f} GB | VRAM {vram_report()}\n{_err[1]}\n"
            with open(_log, "a") as _fh:
                _fh.write(_txt)
            print(_txt, file=sys.stderr, flush=True)
            try:
                gpu_free()
            except Exception:
                _fail.append(("ALL", -1, "CUDA context lost: restart the kernel (finished units are kept)"))
                break
            continue
        atomic_write(_path, lambda p, _x=_rows: open(p, "w").write(json.dumps(_x)))
        _n_done += 1
        with open(os.path.join(CFG["out_dir"], "progress.log"), "a") as _fh:
            _fh.write(f"{time.strftime('%H:%M:%S')} readout {_d} fold={_f}\n")
        gpu_free()
    return mo.vstack([mo.md(f"**readout units ready: {_n_done} / {len(_units)}**" +
                     (f", **{len(_fail)} failed** (full tracebacks in `{_log}`)" if _fail else ", no failures")),
               mo.accordion({f"{a} fold {b}": mo.md(f"`{c}`") for a, b, c in _fail}) if _fail else mo.md("")])


_g7_readout_test()

In [ ]:
# ==== G8. fold-level CIs and H12 verdicts (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g8_fold_cis():
    """G8: fold-level CIs for G4 and G7 results, and the H12 verdicts; reads checkpoints from disk."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows

    # ---- G8. fold-level statistics for G4 and G7, and the H12 verdicts ------------------------------------
    # Every gain is paired within a fold (same split, same seed), so its 95% CI is a t-interval over the folds.
    from scipy import stats as st
    READOUT_CONFIGS = [("gin", "linear"), ("gin", "mlp"), ("deepsets", "mlp")]
    READOUT_RULES = ["none", "wl", "beyond"]
    RKEY = dict(GKEY, readout_configs=READOUT_CONFIGS, readout_rules=READOUT_RULES, rv=1)
    READOUT_HASH = hashlib.sha1(json.dumps(RKEY, sort_keys=True).encode()).hexdigest()[:10]

    def load_units(prefix, h):
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"{prefix}{h}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return pd.DataFrame(rows)

    def ci(x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = len(x)
        if n < 2:
            return pd.Series({"mean": x.mean() if n else np.nan, "lo": np.nan, "hi": np.nan, "n": n, "p": np.nan})
        m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
        tq = st.t.ppf(0.975, n - 1)
        p = float(st.ttest_1samp(x, 0.0).pvalue) if se > 0 else (0.0 if m != 0 else 1.0)
        return pd.Series({"mean": m, "lo": m - tq * se, "hi": m + tq * se, "n": n, "p": p})

    def paired_gains(df, keys):
        """gain = metric(no rules) - metric(rules) for NLL, metric(rules) - metric(no rules) for accuracy."""
        base = df[df.rules == "none"].set_index(keys)[["nll", "acc"]]
        out = df[df.rules != "none"].join(base, on=keys, rsuffix="_none")
        out["gain_nll"] = out["nll_none"] - out["nll"]
        out["gain_acc"] = out["acc"] - out["acc_none"]
        return out

    _G = load_units("G", GRAPH_HASH)
    _R = load_units("R", READOUT_HASH)
    _tabs = []

    # (1) G4 at full data: rule gains for DeepSets and GIN, and the absorbed part (DeepSets gain - GIN gain)
    if len(_G):
        _g1 = paired_gains(_G[_G.frac == 1.0], ["dataset", "fold", "frac", "base"])
        CI_G4 = _g1.groupby(["dataset", "rules", "base"])["gain_nll"].apply(ci).unstack()
        _w = _g1.pivot_table(index=["dataset", "rules", "fold"], columns="base", values="gain_nll")
        ABSORBED_G4 = (_w["deepsets"] - _w["gin"]).groupby(["dataset", "rules"]).apply(ci).unstack()
        _lc = paired_gains(_G[_G.base == "gin"], ["dataset", "fold", "frac", "base"])
        LC_CI = _lc[_lc.rules == "beyond"].groupby(["dataset", "frac"])["gain_nll"].apply(ci).unstack()
        _tabs += [mo.md("#### G4 rule gains (test NLL, nats) with 95% CI over folds"), CI_G4.round(4),
                  mo.md("#### Absorbed part = DeepSets gain - GIN gain (> 0 means message passing absorbed some)"),
                  ABSORBED_G4.round(4),
                  mo.md("#### Learning curve: GIN gain from beyond-WL rules at each training fraction"), LC_CI.round(4)]
    else:
        CI_G4 = ABSORBED_G4 = LC_CI = pd.DataFrame()
        _tabs.append(mo.md("no G4 checkpoints found"))

    # (2) G7: does an MLP readout absorb the WL-computable rules?
    H12 = []
    if len(_R):
        _r = paired_gains(_R, ["dataset", "fold", "base"])
        CI_G7 = _r.groupby(["dataset", "rules", "base"])[["gain_nll", "gain_acc"]].agg(
            lambda s: tuple(ci(s)[["mean", "lo", "hi"]].round(4)))
        _wn = _r.pivot_table(index=["dataset", "rules", "fold"], columns="base", values="gain_nll")
        DROP_G7 = (_wn["gin_linear"] - _wn["gin_mlp"]).groupby(["dataset", "rules"]).apply(ci).unstack()
        _base_nll = _R[_R.rules == "none"].groupby(["dataset", "base"])[["nll", "acc"]].mean().unstack("base")
        _tabs += [mo.md("#### G7 rule gains by readout: (mean, CI low, CI high) over folds"), CI_G7,
                  mo.md("#### Gain removed by the MLP readout = gain(linear) - gain(MLP), 95% CI"), DROP_G7.round(4),
                  mo.md("#### No-rule baselines: does the MLP readout itself fix what the rules fixed?"), _base_nll.round(4)]
        _wl = _r[(_r.rules == "wl")].groupby(["dataset", "base"])["gain_nll"].apply(ci).unstack()
        _bw = _r[(_r.rules == "beyond")].groupby(["dataset", "base"])["gain_nll"].apply(ci).unstack()
        _ds = sorted(_R.dataset.unique())
        # datasets where WL rules help the linear-readout GIN significantly: the cases the test is about
        _sig = [d for d in _ds if _wl.loc[(d, "gin_linear"), "lo"] > 0] if ("gin_linear" in _R.base.unique()) else []
        _coll = {d: bool(_wl.loc[(d, "gin_mlp"), "lo"] <= 0 or _wl.loc[(d, "gin_mlp"), "mean"] <=
                         0.25 * _wl.loc[(d, "gin_linear"), "mean"]) for d in _sig}
        H12.append(("H12a an MLP readout absorbs WL-computable rules (WL gain on GIN-MLP is no longer significant, or "
                    "<= 25% of GIN-linear's, on datasets where GIN-linear gains significantly)",
                    bool(len(_coll) and np.mean(list(_coll.values())) >= 0.5) if _coll else None,
                    f"significant on {_sig}; collapsed: {_coll}"))
        _keep = {d: bool(_bw.loc[(d, "gin_mlp"), "mean"] >= 0.5 * _bw.loc[(d, "gin_linear"), "mean"])
                 for d in _ds if _bw.loc[(d, "gin_linear"), "lo"] > 0}
        H12.append(("H12b beyond-WL rules keep >= 50% of their gain under the MLP readout (where significant before)",
                    bool(np.mean(list(_keep.values())) >= 0.5) if _keep else None, f"{_keep}"))
        _pool = _wn["gin_linear"] - _wn["gin_mlp"]
        _pw = ci(_pool.xs("wl", level="rules").groupby("dataset").mean())
        _pb = ci(_pool.xs("beyond", level="rules").groupby("dataset").mean())
        H12.append(("H12c across datasets the MLP readout removes more of the WL-rule gain than of the beyond-WL gain",
                    bool(_pw["mean"] > _pb["mean"]), f"removed WL {_pw[['mean', 'lo', 'hi']].round(4).to_dict()}, "
                    f"removed beyond {_pb[['mean', 'lo', 'hi']].round(4).to_dict()}"))
    else:
        CI_G7 = DROP_G7 = pd.DataFrame()
        _tabs.append(mo.md("no G7 (readout) checkpoints found yet: run G7 first"))

    for _name, _tab in [("ci_g4.csv", CI_G4), ("absorbed_g4.csv", ABSORBED_G4), ("learning_curve_ci.csv", LC_CI),
                        ("readout_gains.csv", CI_G7), ("readout_drop.csv", DROP_G7)]:
        if len(_tab):
            atomic_write(os.path.join(CFG["out_dir"], _name), lambda p, _t=_tab: _t.to_csv(p))
    atomic_write(os.path.join(CFG["out_dir"], "h12_verdicts.json"),
                 lambda p: open(p, "w").write(json.dumps([[a, b, str(c)] for a, b, c in H12], indent=1)))
    return mo.vstack([mo.md("### G8: fold-level confidence intervals and the readout test (H12)")] + _tabs +
              [mo.md("\n".join(f"- **{a}**: {'yes' if b else ('no' if b is not None else 'n/a')} `{c}`"
                               for a, b, c in H12))])


_g8_fold_cis()

In [ ]:
# ==== G9. gain ledger runs (readout, log coordinates, size-only and shuffled controls) (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g9_ledger_runs():
    """G9: GIN with linear, MLP and log-MLP readouts x rules {none, wl, beyond, size, shuffle}; checkpoints to disk."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo
    import torch
    VRAM_CAP_GB = 70.0
    try:
        import psutil
    except ImportError:
        psutil = None

    def rss_gb():
        if psutil is not None:
            return psutil.Process().memory_info().rss / 2**30
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 2**20
        return float("nan")

    def gpu_free():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

    def vram_report():
        if not torch.cuda.is_available():
            return {}
        return {"allocated_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
                "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)}

    if torch.cuda.is_available():
        _total = torch.cuda.mem_get_info()[1] / 2**30
        torch.cuda.set_per_process_memory_fraction(min(1.0, min(VRAM_CAP_GB, 0.95 * _total) / _total))
        torch.backends.cuda.matmul.allow_tf32 = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    F_ = torch.nn.functional

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows
    _RUN_THIS_CELL = True  # set False to keep this cell from running

    # ---- shared helpers (copied from the full notebook) ----
    def calibrate_T(logits, y, va):
        """Scalar temperature on validation NLL (makes Q_v calibrated, as Prop. 2 assumes)."""
        Ts = torch.exp(torch.linspace(math.log(0.2), math.log(5.0), 41, device=logits.device))
        nll = torch.stack([F_.cross_entropy(logits[va] / t, y[va]) for t in Ts])
        return float(Ts[nll.argmin()])

    def metrics(logp, y, idx, binary):
        lp, yy = logp[idx], y[idx]
        p = lp.exp()
        conf, pred = p.max(1)
        acc = (pred == yy).float()
        out = {"nll": float(-lp[torch.arange(len(yy), device=yy.device), yy].mean()),
               "acc": float(acc.mean())}
        bins = torch.clamp((conf * 15).long(), max=14)  # 15-bin ECE on top-1 confidence
        ece = torch.zeros((), device=lp.device)
        for b in range(15):
            sel = bins == b
            if sel.any():
                ece = ece + sel.float().mean() * (conf[sel].mean() - acc[sel].mean()).abs()
        out["ece"] = float(ece)
        if binary:  # ROC-AUC via ranks (ties broken arbitrarily; fine for continuous scores)
            s = p[:, 1]
            r = torch.empty_like(s)
            r[s.argsort()] = torch.arange(1, len(s) + 1, device=s.device, dtype=s.dtype)
            npos = float(yy.sum())
            nneg = len(yy) - npos
            out["auc"] = float((r[yy == 1].sum() - npos * (npos + 1) / 2) / max(npos * nneg, 1))
        return out

    # ---- G1. TU graph-classification data (GIN authors' release on GitHub raw) + symbolic rule features ----
    import zipfile as zipfile
    import urllib.request as urlreq
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def _fetch_gin_zip(root):
        path = os.path.join(root, "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for _try in range(5):  # GitHub raw occasionally resets long downloads
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if _try == 4:
                        raise
                    time.sleep(2 ** (_try + 1))
            with open(path + ".part", "wb") as f:
                f.write(blob)
            os.replace(path + ".part", path)
        return path

    def rule_features(A, deg, tags, n_tags):
        """Two families of symbolic graph rules for one graph (A: scipy csr, symmetric, 0/1)."""
        n = A.shape[0]
        m = A.nnz // 2
        dh = np.bincount(np.minimum(deg, 10), minlength=11)
        wl = np.concatenate([[n, m], dh, np.bincount(tags, minlength=n_tags)])  # 1-WL computable
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2  # triangles through each node
        tri = tri_node.sum() / 3
        A4tr = float((A2.multiply(A2.T)).sum())  # trace(A^4) = sum_ij (A^2)_ij (A^2)_ji
        c4 = (A4tr - 2 * float((deg**2).sum()) + 2 * m) / 8  # verified against brute force
        comps = conn_comp(A, directed=False)[0] if n > 0 else 0
        pairs = deg * (deg - 1) / 2
        clust = float(np.mean(np.divide(tri_node, pairs, out=np.zeros(n), where=pairs > 0))) if n else 0.0
        beyond = np.array([tri, c4, comps, m - n + comps, clust, (tri_node > 0).sum()])  # not 1-WL invariant
        return np.log1p(np.maximum(wl, 0)).astype(np.float32), np.log1p(np.maximum(beyond, 0)).astype(np.float32)

    def load_tu(name, cfg):
        z = zipfile.ZipFile(_fetch_gin_zip(cfg["data_dir"]))
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, n_graphs = 1, int(lines[0])
        graphs, tagset = [], {}
        for _g in range(n_graphs):
            n, label = map(int, lines[ptr].split())
            ptr += 1
            tags, src, dst = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                k = int(row[1])
                for j in row[2:2 + k]:
                    src.append(i)
                    dst.append(int(j))
            graphs.append((n, label, np.array(tags), np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)))
        labels = sorted({g[1] for g in graphs})
        lab_map = {l: i for i, l in enumerate(labels)}
        degree_as_tag = len(tagset) == 1  # social graphs (IMDB): use degree, the standard choice
        n_feat = 51 if degree_as_tag else len(tagset)
        xs, ys, gid, srcs, dsts, WL, BW = [], [], [], [], [], [], []
        off = 0
        for gi, (n, label, tags, s, d) in enumerate(graphs):
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            deg = np.asarray(A.sum(1)).ravel().astype(np.int64)
            t = np.minimum(deg, 50) if degree_as_tag else tags
            x = np.zeros((n, n_feat), dtype=np.float32)
            x[np.arange(n), t] = 1
            w, b = rule_features(A, deg, t, n_feat)
            coo = A.tocoo()
            xs.append(x)
            srcs.append(coo.row + off)
            dsts.append(coo.col + off)
            gid.append(np.full(n, gi))
            ys.append(lab_map[label])
            WL.append(w)
            BW.append(b)
            off += n
        return {"name": name, "x": np.concatenate(xs), "src": np.concatenate(srcs), "dst": np.concatenate(dsts),
                "gid": np.concatenate(gid), "y": np.array(ys), "wl": np.stack(WL), "beyond": np.stack(BW),
                "K": len(labels), "n_graphs": n_graphs}

    def wl_analysis(d, rounds=4, seed=0):
        """1-WL colour refinement on every graph at once (multiset hashing with random 64-bit weights; the
        collision probability is ~N^2 / 2^64, negligible). For each round t reports how finely the WL partition
        splits the dataset, the empirical H(Y | WL_t) (floor for any MPNN of depth t), and H(rule | WL_t) for each
        beyond-WL rule: the *representational* room left for that rule (Prop. 4 / sandwich upper bound)."""
        rng = np.random.default_rng(seed)
        N, G = len(d["gid"]), d["n_graphs"]
        c = d["x"].argmax(1).astype(np.int64)
        src, dst, gid = d["src"], d["dst"], d["gid"]
        sizes = np.bincount(gid, minlength=G)

        def _H(keys):
            _, cnt = np.unique(keys, return_counts=True, axis=0)
            p = cnt / cnt.sum()
            return float(-(p * np.log(p)).sum())

        out = []
        for t in range(rounds + 1):
            if t > 0:
                w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
                agg = np.zeros(N, dtype=np.uint64)
                np.add.at(agg, dst, w[c[src]])
                key = np.stack([c.astype(np.uint64), agg], 1)
                c = np.unique(key, axis=0, return_inverse=True)[1].ravel().astype(np.int64)
            w = rng.integers(1, 2**63 - 1, size=c.max() + 1, dtype=np.uint64)
            gsig = np.zeros(G, dtype=np.uint64)
            np.add.at(gsig, gid, w[c])
            cls = np.unique(np.stack([gsig, sizes.astype(np.uint64)], 1), axis=0, return_inverse=True)[1].ravel()
            n_cls = int(cls.max() + 1)
            single = np.bincount(cls)[cls] == 1
            H_cls = _H(cls[:, None])
            row = {"dataset": d["name"], "t": t, "wl_classes": n_cls, "frac_singleton": float(single.mean()),
                   "H_y": _H(d["y"][:, None]),
                   "H_y_given_wl": _H(np.stack([cls, d["y"]], 1)) - H_cls,
                   "acc_ceiling_wl": float(sum(np.bincount(d["y"][cls == k]).max() for k in range(n_cls)) / G)}
            beyond_names = ["triangles", "c4", "components", "cyclomatic", "clustering", "tri_nodes"]
            for j, nm in enumerate(beyond_names):
                v = np.round(d["beyond"][:, j], 6)
                row[f"H_{nm}"] = _H(v[:, None])
                row[f"H_{nm}_given_wl"] = _H(np.stack([cls.astype(float), v], 1)) - H_cls
                # plug-in H(.|WL) is trivially 0 on singleton classes, so also report the direct check: the share
                # of graphs whose WL class is shared AND contains >= 2 distinct values of the rule
                pairs = np.unique(np.stack([cls.astype(float), v], 1), axis=0)
                n_vals = np.bincount(pairs[:, 0].astype(np.int64), minlength=n_cls)
                row[f"ambig_{nm}"] = float(((n_vals[cls] > 1) & ~single).mean())
            out.append(row)
        return out

    def strat_folds(y, k, seed):
        rng = np.random.default_rng(seed)
        fold = np.zeros(len(y), dtype=np.int64)
        for c in np.unique(y):
            idx = rng.permutation(np.flatnonzero(y == c))
            fold[idx] = np.arange(len(idx)) % k
        return fold

    # ---- G9. the gain ledger: readout vs coordinates vs knowledge -----------------------------------------
    # Any gain of a rule S that 1-WL already determines (H8a) is not representational. It can only come from
    #   (i) the readout (linear vs MLP), (ii) coordinates: the MLP sees raw counts, the rules arrive in log scale, or
    #   (iii) a genuine prior/statistical effect. Controls: "size" = log n, log m only; "shuffle" = WL rules permuted
    #   across graphs (same marginal, no information: measures the effect of the extra branch itself).
    # GIN's standard readout is LINEAR in the sum-pooled features, so it cannot form nonlinear functions of pooled
    # counts (log n, degree ratios, ...) even though 1-WL makes those counts available. Prediction (refined theory):
    # with an MLP readout the gain from WL-computable rules collapses toward 0, while beyond-WL rules keep theirs.
    _nn, _F7 = torch.nn, torch.nn.functional
    READOUT_CONFIGS = [("gin", "linear"), ("gin", "mlp"), ("gin", "logmlp")]
    READOUT_RULES = ["none", "wl", "beyond", "size", "shuffle"]
    RKEY = dict(GKEY, readout_configs=READOUT_CONFIGS, readout_rules=READOUT_RULES, rv=1, ledger=1)
    READOUT_HASH = hashlib.sha1(json.dumps(RKEY, sort_keys=True).encode()).hexdigest()[:10]

    class GraphNetR(_nn.Module):
        """GIN (eps = 0, sum aggregation, BN MLPs) or DeepSets, with either the standard per-layer LINEAR readout of
        sum pools, or an MLP readout on the concatenated sum pools of all layers (BatchNorm first, so raw count
        scales are handled). Rules enter through their own MLP added to the logits, as in G4."""

        def __init__(self, kind, readout, din, dphi, dh, K, L, p):
            super().__init__()
            self.kind, self.readout, self.L, self.p = kind, readout, L, p
            self.inp = _nn.Linear(din, dh)
            self.mlps = _nn.ModuleList([_nn.Sequential(
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU(),
                _nn.Linear(dh, dh), _nn.BatchNorm1d(dh), _nn.ReLU()) for _ in range(L)])
            if readout == "linear":
                self.read = _nn.ModuleList([_nn.Linear(dh, K) for _ in range(L + 1)])
            else:
                self.read = _nn.Sequential(_nn.BatchNorm1d((L + 1) * dh), _nn.Linear((L + 1) * dh, dh), _nn.ReLU(),
                                           _nn.Dropout(p), _nn.Linear(dh, dh), _nn.ReLU(), _nn.Linear(dh, K))
            self.phi = (_nn.Sequential(_nn.Linear(dphi, dh), _nn.ReLU(), _nn.Linear(dh, K)) if dphi else None)

        def forward(self, D, phi):
            h = self.inp(D["x"])
            pools = [torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h)]
            for l in range(self.L):
                z = h + torch.sparse.mm(D["A"], h) if self.kind == "gin" else h
                h = self.mlps[l](z)
                pools.append(torch.zeros(D["G"], h.shape[1], device=h.device).index_add_(0, D["gid"], h))
            if self.readout == "linear":
                out = sum(r(_F7.dropout(q, self.p, self.training)) for r, q in zip(self.read, pools))
            elif self.readout == "logmlp":  # same MLP, pooled counts in signed log coordinates
                q = torch.cat(pools, 1)
                out = self.read(torch.sign(q) * torch.log1p(q.abs()))
            else:
                out = self.read(torch.cat(pools, 1))
            if phi is not None:
                out = out + self.phi(phi)
            return out

    def train_r(kind, readout, D, phi, tr, va, seed, cfg):
        torch.manual_seed(seed)
        model = GraphNetR(kind, readout, D["x"].shape[1], 0 if phi is None else phi.shape[1], cfg["graph_hidden"],
                          D["K"], cfg["graph_layers"], cfg["graph_dropout"]).to(D["x"].device)
        opt = torch.optim.Adam(model.parameters(), lr=cfg["graph_lr"])
        sched = torch.optim.lr_scheduler.StepLR(opt, 50, 0.5)
        best, state, bad = float("inf"), None, 0
        for _ep in range(cfg["graph_epochs"]):
            model.train()
            opt.zero_grad(set_to_none=True)
            _F7.cross_entropy(model(D, phi)[tr], D["y"][tr]).backward()
            opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                vl = _F7.cross_entropy(model(D, phi)[va], D["y"][va]).item()
            if state is None or vl < best - 1e-5:
                best, bad = vl, 0
                state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad > cfg["graph_patience"]:
                    break
        model.load_state_dict(state)
        model.eval()
        with torch.no_grad():
            return model(D, phi)

    def run_readout_unit(name, fold, device):
        """Same data, folds, validation split and rule standardisation as G3/G4, so results pair with G4's."""
        d = load_tu(name, CFG)
        D = {"x": torch.tensor(d["x"], device=device), "src": torch.tensor(d["src"], device=device),
             "dst": torch.tensor(d["dst"], device=device), "gid": torch.tensor(d["gid"], device=device),
             "y": torch.tensor(d["y"], device=device), "G": d["n_graphs"], "K": d["K"]}
        D["A"] = torch.sparse_coo_tensor(torch.stack([D["dst"], D["src"]]), torch.ones(len(d["src"]), device=device),
                                         (len(d["gid"]),) * 2).coalesce()
        folds = strat_folds(d["y"], CFG["graph_folds"], 0)
        te_np = np.flatnonzero(folds == fold)
        rest = np.random.default_rng(fold).permutation(np.flatnonzero(folds != fold))
        n_va = max(2, len(rest) // 9)
        va_np, tr_np = np.sort(rest[:n_va]), np.sort(rest[n_va:])
        tr, va, te = (torch.tensor(a, device=device) for a in (tr_np, va_np, te_np))
        phis = {}
        d["size"] = d["wl"][:, :2]  # log1p(n), log1p(m): the first two WL rules
        for fam in ["wl", "beyond", "size"]:
            mu, sd = d[fam][tr_np].mean(0), d[fam][tr_np].std(0) + 1e-6
            phis[fam] = torch.tensor(np.clip((d[fam] - mu) / np.maximum(sd, 1e-2), -5, 5), device=device,
                                     dtype=torch.float32)
        _perm = np.random.default_rng(10_000 + fold).permutation(d["n_graphs"])
        phis["shuffle"] = phis["wl"][torch.tensor(_perm, device=device)]
        rows = []
        for kind, readout in READOUT_CONFIGS:
            for fam in READOUT_RULES:
                logits = train_r(kind, readout, D, None if fam == "none" else phis[fam], tr, va, fold, CFG)
                T = calibrate_T(logits, D["y"], va)
                lp = torch.log(0.98 * _F7.softmax(logits / T, 1) + 0.02 / D["K"])
                r = {"dataset": name, "fold": fold, "base": f"{kind}_{readout}", "rules": fam, "T": T}
                r.update(metrics(lp, D["y"], te, False))
                rows.append(r)
        return rows

    import sys
    import traceback
    mo.stop(not _RUN_THIS_CELL, mo.md("skipped: set _RUN_THIS_CELL = True at the top of this cell"))
    _t0 = time.time()
    _units = [(d_, f_) for d_ in CFG["graph_datasets"] for f_ in range(CFG["graph_folds"])]
    _fail, _n_done = [], 0
    _log = os.path.join(CFG["out_dir"], "ledger_failures.log")
    for _d, _f in mo.status.progress_bar(_units, title="ledger units"):
        _path = os.path.join(CFG["out_dir"], "units", f"L{READOUT_HASH}_{_d}_{_f}.json")
        if os.path.exists(_path):
            _n_done += 1
            continue
        if (time.time() - _t0) / 3600 > CFG["deadline_h"]:
            break
        _err = None
        try:
            _rows = run_readout_unit(_d, _f, DEVICE)
        except Exception as _e:
            _err = (repr(_e)[:300], traceback.format_exc())
        if _err is not None:
            _fail.append((_d, _f, _err[0]))
            _txt = f"--- {_d} fold={_f} | RAM {rss_gb():.1f} GB | VRAM {vram_report()}\n{_err[1]}\n"
            with open(_log, "a") as _fh:
                _fh.write(_txt)
            print(_txt, file=sys.stderr, flush=True)
            try:
                gpu_free()
            except Exception:
                _fail.append(("ALL", -1, "CUDA context lost: restart the kernel (finished units are kept)"))
                break
            continue
        atomic_write(_path, lambda p, _x=_rows: open(p, "w").write(json.dumps(_x)))
        _n_done += 1
        with open(os.path.join(CFG["out_dir"], "progress.log"), "a") as _fh:
            _fh.write(f"{time.strftime('%H:%M:%S')} ledger {_d} fold={_f}\n")
        gpu_free()
    return mo.vstack([mo.md(f"**ledger units ready: {_n_done} / {len(_units)}**" +
                     (f", **{len(_fail)} failed** (full tracebacks in `{_log}`)" if _fail else ", no failures")),
               mo.accordion({f"{a} fold {b}": mo.md(f"`{c}`") for a, b, c in _fail}) if _fail else mo.md("")])


_g9_ledger_runs()

In [ ]:
# ==== G10. gain ledger with CIs and H13 verdicts (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _g10_ledger():
    """G10: gain ledger (readout / coordinate / residual parts) with fold CIs, and the H13 verdicts."""
    import os, json, time, math, hashlib, gc
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "deadline_h": 11.0,
        "graph_datasets": ["MUTAG"] if SMOKE else ["MUTAG", "PTC", "PROTEINS", "NCI1", "IMDBBINARY", "IMDBMULTI", "COLLAB"],
        "graph_folds": 2 if SMOKE else 10,
        "graph_hidden": 32 if SMOKE else 64,
        "graph_layers": 4,
        "graph_lr": 1e-2,
        "graph_dropout": 0.5,
        "graph_epochs": 30 if SMOKE else 350,
        "graph_patience": 15 if SMOKE else 100,
        "graph_fracs": [0.25, 1.0] if SMOKE else [0.25, 0.5, 1.0],
        "graph_wl_rounds": 4,
    }
    # same key as the full notebook, so checkpoints already on disk are reused
    GKEY = {k: v for k, v in CFG.items() if k.startswith("graph_") and k != "graph_datasets"}
    GKEY.update(v=3)
    GRAPH_HASH = hashlib.sha1(json.dumps(GKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, writer):
        base, ext = os.path.splitext(path)
        tmp = f"{base}.tmp{ext}"
        writer(tmp)
        os.replace(tmp, path)

    def load_graph_rows():
        """Every finished (dataset, fold) checkpoint written by the G4 cell."""
        rows = []
        for d_ in CFG["graph_datasets"]:
            for f_ in range(CFG["graph_folds"]):
                p_ = os.path.join(CFG["out_dir"], "units", f"G{GRAPH_HASH}_{d_}_{f_}.json")
                if os.path.exists(p_):
                    with open(p_) as fh_:
                        rows += json.load(fh_)
        return rows

    # ---- G10. the gain ledger with 95% CIs, and the H13 verdicts -----------------------------------------
    # Telescoping identity, per dataset, rule family and fold (all terms paired on the same split and seed):
    #   gain(GIN-linear) = [gain(linear) - gain(MLP)]       readout part
    #                    + [gain(MLP) - gain(log-MLP)]      coordinate part
    #                    + gain(log-MLP)                    residual (prior / statistical part)
    # and the representational part is I(Y;S|WL) = 0 here (H8a). A term whose CI excludes 0 is a real contribution.
    from scipy import stats as st
    READOUT_CONFIGS = [("gin", "linear"), ("gin", "mlp"), ("gin", "logmlp")]
    READOUT_RULES = ["none", "wl", "beyond", "size", "shuffle"]
    RKEY = dict(GKEY, readout_configs=READOUT_CONFIGS, readout_rules=READOUT_RULES, rv=1, ledger=1)
    READOUT_HASH = hashlib.sha1(json.dumps(RKEY, sort_keys=True).encode()).hexdigest()[:10]

    rows = []
    for d_ in CFG["graph_datasets"]:
        for f_ in range(CFG["graph_folds"]):
            p_ = os.path.join(CFG["out_dir"], "units", f"L{READOUT_HASH}_{d_}_{f_}.json")
            if os.path.exists(p_):
                with open(p_) as fh_:
                    rows += json.load(fh_)
    L = pd.DataFrame(rows)

    def ci(x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        n = len(x)
        if n < 2:
            return {"mean": float(x.mean()) if n else np.nan, "lo": np.nan, "hi": np.nan, "n": n}
        m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
        tq = st.t.ppf(0.975, n - 1)
        return {"mean": float(m), "lo": float(m - tq * se), "hi": float(m + tq * se), "n": n}

    def fmt(c):
        star = "*" if np.isfinite(c["lo"]) and (c["lo"] > 0 or c["hi"] < 0) else ""
        return f"{c['mean']:+.4f} [{c['lo']:+.4f}, {c['hi']:+.4f}]{star}"

    H13, ledger_rows, tabs = [], [], []
    if len(L):
        base = L[L.rules == "none"].set_index(["dataset", "fold", "base"])[["nll", "acc"]]
        g = L[L.rules != "none"].join(base, on=["dataset", "fold", "base"], rsuffix="_none")
        g["gain"] = g["nll_none"] - g["nll"]
        g["gain_acc"] = g["acc"] - g["acc_none"]
        W = g.pivot_table(index=["dataset", "rules", "fold"], columns="base", values="gain")
        WA = g.pivot_table(index=["dataset", "rules", "fold"], columns="base", values="gain_acc")
        for (ds, fam), w in W.groupby(level=["dataset", "rules"]):
            terms = {"observed (GIN-linear)": w["gin_linear"], "readout part": w["gin_linear"] - w["gin_mlp"],
                     "coordinate part": w["gin_mlp"] - w["gin_logmlp"], "residual (log-MLP)": w["gin_logmlp"]}
            r = {"dataset": ds, "rules": fam}
            for k, v in terms.items():
                c = ci(v.values)
                r[k] = fmt(c)
                r[k + " mean"], r[k + " lo"], r[k + " hi"] = c["mean"], c["lo"], c["hi"]
            r["acc gain (GIN-linear)"] = fmt(ci(WA.loc[(ds, fam), "gin_linear"].values))
            r["acc gain (log-MLP)"] = fmt(ci(WA.loc[(ds, fam), "gin_logmlp"].values))
            ledger_rows.append(r)
        LEDGER = pd.DataFrame(ledger_rows)
        show = ["dataset", "rules", "observed (GIN-linear)", "readout part", "coordinate part", "residual (log-MLP)",
                "acc gain (GIN-linear)", "acc gain (log-MLP)"]
        base_tab = L[L.rules == "none"].groupby(["dataset", "base"])[["nll", "acc"]].mean().unstack("base")
        base_tab.columns = [f"{a} {b}" for a, b in base_tab.columns]  # flat columns for marimo's table
        tabs += [mo.md("#### Gain ledger (test NLL in nats; mean [95% CI]; * = CI excludes 0)"),
                 mo.ui.table(LEDGER[show], selection=None, page_size=40),
                 mo.md("#### No-rule baselines per readout (does a better readout alone close the gap?)"),
                 mo.ui.table(base_tab.round(4).reset_index(), selection=None)]

        def get(ds, fam, col):
            x = LEDGER[(LEDGER.dataset == ds) & (LEDGER.rules == fam)]
            return float(x[col].iloc[0]) if len(x) else np.nan

        sig_wl = [ds for ds in LEDGER.dataset.unique() if get(ds, "wl", "observed (GIN-linear) lo") > 0]
        # H13a: coordinates. Where the MLP readout did NOT absorb the WL rules, does the log readout?
        left = {ds: get(ds, "wl", "residual (log-MLP) mean") for ds in sig_wl}
        gone = {ds: bool(get(ds, "wl", "residual (log-MLP) lo") <= 0 or
                         left[ds] <= 0.25 * get(ds, "wl", "observed (GIN-linear) mean")) for ds in sig_wl}
        H13.append(("H13a readout + log coordinates absorb the WL-rule gain (residual not significant or <= 25% of the "
                    "observed gain, on datasets where WL rules help GIN-linear significantly)",
                    bool(np.mean(list(gone.values())) >= 0.5) if gone else None,
                    f"significant on {sig_wl}; absorbed: {gone}"))
        # H13b: size suffices. log n and log m alone carry most of the WL-rule gain
        share = {ds: round(get(ds, "size", "observed (GIN-linear) mean") / get(ds, "wl", "observed (GIN-linear) mean"), 3)
                 for ds in sig_wl}
        H13.append(("H13b graph size (log n, log m) alone gives >= 70% of the WL-rule gain on GIN-linear",
                    bool(np.median(list(share.values())) >= 0.7) if share else None, f"share of WL gain from size: {share}"))
        # H13c: placebo. Shuffled rules (no information) give no gain, so gains are not an extra-branch artefact
        plc = {ds: (round(get(ds, "shuffle", "observed (GIN-linear) lo"), 4), round(get(ds, "shuffle", "observed (GIN-linear) hi"), 4))
               for ds in LEDGER.dataset.unique()}
        H13.append(("H13c placebo: shuffled WL rules give no significant gain on any dataset (CI includes 0 or is < 0)",
                    bool(all(not (lo > 0) for lo, hi in plc.values())), f"shuffle CI per dataset: {plc}"))
        # H13d: what survives everything is the only candidate for real symbolic knowledge
        res = {(ds, fam): fmt({"mean": get(ds, fam, "residual (log-MLP) mean"), "lo": get(ds, fam, "residual (log-MLP) lo"),
                               "hi": get(ds, fam, "residual (log-MLP) hi")})
               for ds in LEDGER.dataset.unique() for fam in ["wl", "beyond"] if get(ds, fam, "residual (log-MLP) lo") > 0}
        H13.append(("H13d no rule family keeps a significant gain once readout and coordinates are fixed "
                    "(the residual is the only place symbolic knowledge can live)", not res, f"significant residuals: {res}"))
        atomic_write(os.path.join(CFG["out_dir"], "ledger.csv"), lambda p: LEDGER.to_csv(p, index=False))
    else:
        tabs.append(mo.md("no G9 (ledger) checkpoints found yet: run G9 first"))
    atomic_write(os.path.join(CFG["out_dir"], "h13_verdicts.json"),
                 lambda p: open(p, "w").write(json.dumps([[a, b, str(c)] for a, b, c in H13], indent=1)))
    return mo.vstack([mo.md("### G10: where does a symbolic rule's gain come from? (gain ledger, H13)")] + tabs +
              [mo.md("\n".join(f"- **{a}**: {'yes' if b else ('no' if b is not None else 'n/a')} `{c}`" for a, b, c in H13))])


_g10_ledger()

In [ ]:
# ==== L1. learnability law: a rule is worth only the label information in the part of it the network cannot learn (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _l1_learnability_sweep():
    """L1: rule library x {GIN, GINE} on ZINC, NCI1, PROTEINS. Phase 1 trains the plain network and runs the
    learnability probes, phase 2 trains every rule (plus the probe-selected top-2 and all rules) as one batched network.
    One checkpoint per (dataset, family); safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["MUTAG", "ZINC"] if SMOKE else ["ZINC", "NCI1", "PROTEINS"],
        "seeds": 1 if SMOKE else 3,
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,
        "probe_epochs": 6 if SMOKE else 120,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1)
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": len(tagset), "n_bond": 0}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate(nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_tu(name)
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            rows = [rule_library(g, D["n_types"], D["n_bond"]) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        if D["n_bond"] == 0:
            RULES_D = [r for r in RULES if r != "bond_hist"]
        else:
            RULES_D = list(RULES)
        D["rules"] = RULES_D
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            out.append({"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                        "idx": torch.tensor(sel, device=DEVICE)})
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            self.emb = [torch.stack([torch.randn(D["n_types"], dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(max(D["n_bond"], 1), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                msg = h[B["src"]]
                if self.edge:
                    msg = F_.relu(msg + self.bemb[0][:, B["bond"], li].transpose(0, 1))
                z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or CFG["epochs"]
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            with torch.no_grad():
                vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            Ts = torch.zeros(len(D["graphs"]), E, (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            return {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seeds"]))
        rules = D["rules"]
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rows, probes = [], []
        # phase 1a: the plain network (no rules), all seeds as members
        out0, _, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        base = [test_metric(D, out0[:, e], te_idx) for e in range(len(seeds))]
        # phase 1b: label-free a-priori probe: the same architecture trained to regress each rule
        pout, _, _, _ = train_members(D, edge, [r for r in rules], [0] * len(rules), target="probe", epochs=CFG["probe_epochs"])
        # phase 1c: frozen-representation probe (does the trained plain network already contain the rule?) and the
        # residual-information score I(Y; S | T) estimated on the validation graphs with small MLPs
        T = T0[:, 0].float()
        yv = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE).float()
        a, b = torch.tensor(tr, device=DEVICE), torch.tensor(va, device=DEVICE)
        task = "clf" if D["task"] == "clf" else "mae"

        def score(X):
            o = small_fit(X[a], yv[a], X[b], task, D["K"])
            if task == "clf":
                return float(F_.cross_entropy(o, yv[b].long()))
            return float((o.squeeze(-1) - yv[b]).abs().mean())

        s_T = score(T)
        for k, r in enumerate(rules):
            Z = torch.tensor(D["Z"][r], device=DEVICE)
            zt = Z[torch.tensor(te_idx, device=DEVICE)]
            sst = ((zt - zt.mean(0)) ** 2).sum().clamp_min(1e-6)  # pooled over the rule's columns (rare bins can't dominate)
            r2_prior = 1 - ((pout[:, k, :Z.shape[1]] - zt) ** 2).sum() / sst
            zhat = small_fit(T[a], Z[a], T[torch.tensor(te_idx, device=DEVICE)], "mse", 1)
            r2_T = 1 - ((zhat - zt) ** 2).sum() / sst
            # exact-recovery error of the raw count (Fano bound on H(S | T)) for count rules
            fano = None
            if r in COUNT_RULES:
                raw = torch.tensor(D["raw"][r], device=DEVICE, dtype=torch.float32)
                Xc = np.log1p(np.maximum(D["raw"][r], 0))
                mu, sd = Xc[tr].mean(0), Xc[tr].std(0)
                keep = sd > 1e-8
                rec = torch.expm1(zhat * torch.tensor(sd[keep], device=DEVICE) + torch.tensor(mu[keep], device=DEVICE))
                truth = raw[torch.tensor(te_idx, device=DEVICE)][:, torch.tensor(keep, device=DEVICE)]
                fano = 0.0
                for c in range(truth.shape[1]):
                    pe = float((torch.round(rec[:, c] * 2) / 2 != truth[:, c]).float().mean())  # bond counts are halves
                    nv = len(np.unique(D["raw"][r][:, np.flatnonzero(keep)[c]]))
                    h = 0.0 if pe in (0.0, 1.0) else -pe * math.log(pe) - (1 - pe) * math.log(1 - pe)
                    fano += h + pe * math.log(max(nv - 1, 1))
            probes.append({"rule": r, "r2_prior": float(r2_prior), "r2_T": float(r2_T),
                           "resid_score": s_T - score(torch.cat([T, Z], 1)), "fano_bound_nats": fano})
        # pre-registered selection: the two rules with the largest residual-information score (no rule model seen yet)
        top2 = [p["rule"] for p in sorted(probes, key=lambda p: -p["resid_score"]) if p["rule"] != "noise"][:2]
        sel = combo(D, top2)
        allr = combo(D, [r for r in rules if r != "noise"])
        # phase 2: every rule, the selected pair and all rules, x seeds, as one batched network
        member_rules = [r for r in rules + [sel, allr] for _ in seeds]
        member_seeds = [s for _ in rules + [sel, allr] for s in seeds]
        out1, _, _, _ = train_members(D, edge, member_rules, member_seeds)
        for e, (r, s) in enumerate(zip(member_rules, member_seeds)):
            m = test_metric(D, out1[:, e], te_idx)
            rows.append({"rule": "top2:" + sel if r == sel else ("all" if r == allr else r), "seed": s, **m})
        for s, m in zip(seeds, base):
            rows.append({"rule": "none", "seed": s, **m})
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows,
                "probes": probes, "top2": top2}

    # ---------------------------------------------------------------- driver
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [(d, edge) for d in CFG["datasets"] for edge in ([False, True] if d == "ZINC" else [False])]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="L1 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"L{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "L1_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        done += 1
        line = f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, top-2 by score: {res['top2']}"
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**L1 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `L1_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_l1_learnability_sweep()

In [ ]:
# ==== L2. learnability law: verdicts (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _l2_learnability_verdicts():
    """L2: reads the L1 checkpoints and tests L-a..L-f. Gains are 'none minus rule' (MAE on ZINC, NLL nats on TU),
    so positive = the rule helps. Prints the table and writes learnability_rows.csv + learnability_verdicts.json."""
    import os, json, hashlib, math
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match L1 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["MUTAG", "ZINC"] if SMOKE else ["ZINC", "NCI1", "PROTEINS"],
        "seeds": 1 if SMOKE else 3,
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,
        "probe_epochs": 6 if SMOKE else 120,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1)
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]

    units = []
    for d in CFG["datasets"]:
        for fam in (["gin", "gine"] if d == "ZINC" else ["gin"]):
            p = os.path.join(CFG["out_dir"], "units", f"L{L_HASH}_{d}_{fam}.json")
            if os.path.exists(p):
                with open(p) as fh:
                    units.append(json.load(fh))
    mo.stop(not units, mo.md(f"no L1 checkpoints with hash `{L_HASH}` yet: run L1 first"))

    rng = np.random.default_rng(0)

    def spearman(a, b):
        a, b = np.asarray(a, float), np.asarray(b, float)
        ra, rb = pd.Series(a).rank().values, pd.Series(b).rank().values
        return float(np.corrcoef(ra, rb)[0, 1]) if len(a) > 2 and ra.std() > 0 and rb.std() > 0 else float("nan")

    rows = []
    for u in units:
        metric = "mae" if u["task"] == "reg" else "nll"
        R = pd.DataFrame(u["rows"])
        base = R[R.rule == "none"].set_index("seed")[metric]
        probes = {p["rule"]: p for p in u["probes"]}
        for rule, g in R[R.rule != "none"].groupby("rule"):
            gains = (base.loc[g.seed].values - g[metric].values)  # paired by seed
            boots = [rng.choice(gains, len(gains)).mean() for _ in range(2000)] if len(gains) > 1 else [gains.mean()]
            key = rule.split(":")[0] if rule.startswith("top2") else rule
            p = probes.get(key, {})
            rows.append({"dataset": u["dataset"], "family": u["family"], "metric": metric, "rule": rule,
                         "base": float(base.mean()), "rule_value": float(g[metric].mean()),
                         "gain": float(gains.mean()), "gain_lo": float(np.percentile(boots, 2.5)),
                         "gain_hi": float(np.percentile(boots, 97.5)), "rel_gain_pct": float(100 * gains.mean() / base.mean()),
                         "r2_prior": p.get("r2_prior"), "r2_T": p.get("r2_T"), "resid_score": p.get("resid_score"),
                         "fano_bound_nats": p.get("fano_bound_nats")})
    T = pd.DataFrame(rows)
    single = T[~T.rule.str.startswith("top2") & (T.rule != "all")]

    V = {}
    # L-a: rules the trained network already carries (r2_T >= 0.95) are worth ~nothing (|gain| < 2% of the base)
    learn = single[(single.r2_T >= 0.95) & (single.rule != "noise")]
    V["L-a learnable rules are absorbed"] = {
        "holds": bool(len(learn) > 0 and (learn.rel_gain_pct.abs() < 2).mean() >= 0.8),
        "detail": f"{int((learn.rel_gain_pct.abs() < 2).sum())}/{len(learn)} learnable rules within 2% of the base"}
    # L-b: the residual-information score (computed before any rule model) ranks the rules' real gains
    per = []
    for (d, f), g in single[single.rule != "noise"].groupby(["dataset", "family"]):
        per.append((d, f, spearman(g.resid_score, g.gain), spearman(1 - g.r2_T.clip(-1, 1), g.gain),
                    spearman(1 - g.r2_prior.clip(-1, 1), g.gain)))
    rho = np.nanmean([x[2] for x in per]) if per else float("nan")
    V["L-b residual score ranks the gains"] = {
        "holds": bool(rho >= 0.6),
        "detail": "; ".join(f"{d}/{f}: rho(score)={a:.2f}, rho(1-R2_T)={b:.2f}, rho(1-R2_prior)={c:.2f}" for d, f, a, b, c in per)}
    # L-c: the same rule's value depends on the family: bond types help GIN (cannot see them) but not GINE (can)
    bz = single[(single.dataset == "ZINC") & (single.rule == "bond_hist")].set_index("family")
    if {"gin", "gine"} <= set(bz.index):
        V["L-c value is a (rule, family) property"] = {
            "holds": bool(bz.loc["gin", "gain"] > max(0.0, bz.loc["gine", "gain"]) and bz.loc["gin", "gain_lo"] > 0),
            "detail": f"bond_hist gain GIN {bz.loc['gin', 'gain']:.3f} vs GINE {bz.loc['gine', 'gain']:.3f} MAE "
                      f"(R2_T {bz.loc['gin', 'r2_T']:.2f} vs {bz.loc['gine', 'r2_T']:.2f})"}
    # L-d: the pre-registered top-2 pick gets >= 90% of the all-rules gain
    sel = []
    for (d, f), g in T.groupby(["dataset", "family"]):
        t2, al = g[g.rule.str.startswith("top2")], g[g.rule == "all"]
        if len(t2) and len(al):
            sel.append((d, f, t2.rule.iloc[0][5:], float(t2.gain.iloc[0]), float(al.gain.iloc[0])))
    V["L-d probe-selected pair matches all rules"] = {
        "holds": bool(sel and np.mean([a >= 0.9 * b if b > 0 else a >= b for *_, a, b in sel]) >= 0.75),
        "detail": "; ".join(f"{d}/{f} top2={r}: {a:.3f} vs all {b:.3f}" for d, f, r, a, b in sel)}
    # L-e: the noise placebo never helps
    nz = single[single.rule == "noise"]
    V["L-e placebo never helps"] = {"holds": bool((nz.gain_lo <= 0).all()),
                                    "detail": "; ".join(f"{a}/{b}: {c:.3f} [{lo:.3f}, {hi:.3f}]" for a, b, c, lo, hi in
                                                        nz[["dataset", "family", "gain", "gain_lo", "gain_hi"]].values)}
    # L-f: the Fano certificate (from the frozen representation) upper-bounds the measured gain in nats
    fb = single[(single.metric == "nll") & single.fano_bound_nats.notna()]
    if len(fb):
        V["L-f Fano certificate bounds the gain"] = {"holds": bool((fb.gain <= fb.fano_bound_nats + 1e-3).mean() >= 0.95),
                                                     "detail": f"{int((fb.gain <= fb.fano_bound_nats + 1e-3).sum())}/{len(fb)} rules within the bound"}
    best = single[(single.dataset == "ZINC") & (single.rule != "noise")].sort_values("rel_gain_pct", ascending=False).head(3)
    headline = "; ".join(f"{f} + {r}: MAE {b:.3f} -> {v:.3f} ({p:.0f}% lower)" for f, r, b, v, p in
                         best[["family", "rule", "base", "rule_value", "rel_gain_pct"]].values)

    T.to_csv(os.path.join(CFG["out_dir"], "learnability_rows.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "learnability_verdicts.json"), "w") as fh:
        json.dump(V, fh, indent=1)
    show = T[["dataset", "family", "rule", "base", "gain", "gain_lo", "gain_hi", "rel_gain_pct", "r2_prior", "r2_T",
              "resid_score", "fano_bound_nats"]].round(4)
    return mo.vstack([
        mo.md("### L2 learnability verdicts\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())
              + (f"\n\n**Largest ZINC gains:** {headline}" if headline else "")),
        mo.ui.table(show, page_size=40, selection=None),
    ])


_l2_learnability_verdicts()

In [ ]:
# ==== HF. push every artifact and data file to Hugging Face (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _hf_push():
    """HF: upload nsgnn_runs/, nsgnn_data/ and top-level artifacts to the Hugging Face repo. Safe to re-run."""
    import os, re, sys, glob, time, subprocess
    import marimo as mo

    # ---- hardcoded on request; HF_TOKEN in the environment wins if set ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    REPO_TYPE = "model"
    ROOT = os.getcwd()  # the folder the notebook runs in (where nsgnn_runs/ and nsgnn_data/ live)

    try:
        import huggingface_hub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
        import huggingface_hub
    from huggingface_hub import HfApi

    # what gets pushed: every run output, every downloaded/derived dataset, and top-level artifacts
    ALLOW = [
        "nsgnn_runs/**", "nsgnn_runs_smoke/**", "nsgnn_data/**",
        "*.py", "*.csv", "*.json", "*.jsonl", "*.md", "*.txt", "*.log",
        "*.png", "*.pdf", "*.svg", "*.npz", "*.npy", "*.pt", "*.zip", "*.parquet",
    ]
    JUNK = ["**/*.tmp", "**/*.tmp.*", "**/__pycache__/**", "**/.ipynb_checkpoints/**", ".*", ".*/**", "**/.*/**"]

    # files that contain an HF token are skipped: HF scans uploads and revokes leaked tokens,
    # and this cell (and the notebook it lives in) contains one
    TOKEN_RE = re.compile(rb"hf_[A-Za-z0-9]{30,}")
    tok = HF_TOKEN.encode()

    def has_token(path, chunk=16 << 20):
        tail = b""
        with open(path, "rb") as fh:
            while True:
                buf = fh.read(chunk)
                if not buf:
                    return False
                win = tail + buf
                if tok in win or TOKEN_RE.search(win):
                    return True
                tail = win[-64:]

    from fnmatch import fnmatch
    candidates, leaked = [], []
    for dirpath, dirnames, filenames in os.walk(ROOT):
        dirnames[:] = [d for d in dirnames if not d.startswith(".") and d != "__pycache__"]
        for fn in filenames:
            rel = os.path.relpath(os.path.join(dirpath, fn), ROOT).replace(os.sep, "/")
            if not any(fnmatch(rel, p) for p in ALLOW) or any(fnmatch(rel, p) for p in JUNK):
                continue
            (leaked if has_token(os.path.join(ROOT, rel)) else candidates).append(rel)

    total_mb = sum(os.path.getsize(os.path.join(ROOT, r)) for r in candidates) / 2**20
    print(f"[HF] {len(candidates)} files ({total_mb:.1f} MB) to sync, {len(leaked)} skipped for containing a token")
    for r in leaked:
        print(f"[HF]   skipped (token inside): {r}")

    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type=REPO_TYPE, private=True, exist_ok=True)

    # upload_folder only sends files whose content changed since the last commit,
    # so re-running adds new outputs without duplicating anything; nothing remote is deleted
    t0 = time.time()
    commit = None
    if candidates:
        commit = api.upload_folder(
            repo_id=HF_REPO,
            repo_type=REPO_TYPE,
            folder_path=ROOT,
            allow_patterns=ALLOW,
            ignore_patterns=JUNK + [glob.escape(r) for r in leaked],
            commit_message=f"sync notebook artifacts ({len(candidates)} files, {time.strftime('%Y-%m-%d %H:%M')})",
        )
    url = f"https://huggingface.co/{HF_REPO}"
    print(f"[HF] done in {time.time() - t0:.0f}s -> {url}")
    skipped_md = "".join(f"\n- `{r}`" for r in leaked) or " none"
    return mo.md(
        f"**Hugging Face sync:** [{HF_REPO}]({url}), {len(candidates)} files ({total_mb:.1f} MB) checked; "
        f"only changed files are uploaded.\n\n"
        f"Commit: {getattr(commit, 'commit_url', commit) or 'nothing to upload'}\n\n"
        f"Skipped because they contain a token:{skipped_md}"
    )


_hf_push()

In [ ]:
# ==== D1. rule discovery: let the network's own residual pick the missing symbolic rule from a ring-candidate space (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _d1_rule_discovery():
    """D1: train the plain GIN/GINE once, then greedily pick the ring statistics whose unlearnable part explains the
    network's residual, then retrain with the discovered rules vs the hand rule, a random placebo and the whole pool.
    ZINC (GIN, GINE) and ogbg-molhiv (GINE, scaffold split). One checkpoint per (dataset, family); safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 60,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D1")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0])})
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate(nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D1] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"]) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        if D["n_bond"] == 0:
            RULES_D = [r for r in RULES if r != "bond_hist"]
        else:
            RULES_D = list(RULES)
        D["rules"] = RULES_D
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            out.append({"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                        "idx": torch.tensor(sel, device=DEVICE)})
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            self.emb = [torch.stack([torch.randn(D["n_types"], dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(max(D["n_bond"], 1), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                msg = h[B["src"]]
                if self.edge:
                    msg = F_.relu(msg + self.bemb[0][:, B["bond"], li].transpose(0, 1))
                z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or (CFG["epochs_molhiv"] if D["name"] == "ogbg-molhiv" else CFG["epochs"])
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D1] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            Ts = torch.zeros(len(D["graphs"]), E, (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def build_candidates(D):
        cache = os.path.join(CFG["data_dir"], f"D1_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D1] building the ring candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [candidate_library(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.log1p(np.maximum(X, 0))
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def discover(D, T, Z, names, kmax):
        """Greedy residual pursuit. r = label minus what the trained plain network's representation T predicts;
        C_res = candidate minus what T predicts of it (both cross-fitted on the training graphs). Each step picks the
        candidate whose unlearnable part best explains the remaining residual (Theorem L: value = I(Y; S | T))."""
        tr = np.flatnonzero(D["split"] == 0)
        rng = np.random.default_rng(0)
        fold = rng.permutation(np.arange(len(tr)) % 2)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        yhat, zhat = torch.zeros_like(y), torch.zeros_like(Zt)
        task = "clf" if D["task"] == "clf" else "mae"
        for k in range(2):
            a = torch.tensor(np.flatnonzero(fold != k), device=DEVICE)
            b = torch.tensor(np.flatnonzero(fold == k), device=DEVICE)
            o = small_fit(Tt[a], y[a], Tt[b], task, D["K"])
            yhat[b] = F_.softmax(o, -1)[:, 1] if task == "clf" else o.squeeze(-1)
            zhat[b] = small_fit(Tt[a], Zt[a], Tt[b], "mse", 1)
        r = y - yhat
        Cres = Zt - zhat
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        sel, steps, res = [], [], r - r.mean()
        for _ in range(kmax):
            score = (Cc * res[:, None]).mean(0).abs() / (res.std() + 1e-9)  # covariance with the unlearnable part
            score[sel] = -1
            j = int(score.argmax())
            sel.append(j)
            X = Cc[:, sel]
            beta = torch.linalg.lstsq(X, (r - r.mean())[:, None]).solution
            res = (r - r.mean()) - (X @ beta)[:, 0]
            steps.append({"feature": names[j], "score": float(score[j]), "r2_from_T": float(r2[j]),
                          "residual_std_after": float(res.std())})
        return sel, steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seeds"][D["name"]]))
        Zc, names = build_candidates(D)
        # phase 1: the plain network, all seeds as members; its representation drives discovery
        out0, val0, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        sel, steps = discover(D, T0[:, 0].float(), Zc, names, CFG["kmax"])
        rng = np.random.default_rng(1)
        rand = rng.choice([j for j in range(len(names)) if j not in sel], size=min(8, len(names) - len(sel)), replace=False)
        for key, cols in [("disc1", sel[:1]), ("disc3", sel[:3]), ("disc8", sel[:8]), ("rand8", list(rand)),
                          ("allcand", list(range(len(names))))]:
            D["Z"][key] = Zc[:, cols]
        configs = ["cycles", "disc1", "disc3", "disc8", "rand8", "allcand"]
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule from L1, the discovered sets, a random-candidate placebo and the whole pool
        out1, val1, _, _ = train_members(D, edge, member_rules, member_seeds)
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows,
                "discovered": steps, "n_candidates": len(names), "random8": [names[j] for j in rand]}

    # ---------------------------------------------------------------- driver
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("ZINC", False), ("ZINC", True), ("ogbg-molhiv", True)] if not SMOKE else [("ZINC", True), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D1 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"D{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D1_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, discovered: "
                + "; ".join(s["feature"] for s in res["discovered"][:5]))
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D1 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D1_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d1_rule_discovery()

In [ ]:
# ==== D2. rule discovery: verdicts (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _d2_discovery_verdicts():
    """D2: reads the D1 checkpoints and tests D-a..D-e (fixed before the run). Gain = improvement over the plain
    network (MAE lower is better on ZINC, ROC-AUC higher is better on molhiv), paired by seed, bootstrap CIs."""
    import os, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D1 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 60,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D1")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    units = []
    for d, fam in [("ZINC", "gin"), ("ZINC", "gine"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"D{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units.append(json.load(fh))
    mo.stop(not units, mo.md(f"no D1 checkpoints with hash `{L_HASH}` yet: run D1 first"))
    rng = np.random.default_rng(0)

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    table, V, picked = [], {}, {}
    for u in units:
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        base = R[R.config == "none"].set_index("seed")
        disc = [c for c in ["disc1", "disc3", "disc8"] if c in set(R.config)]
        # the discovered set is chosen by validation loss only (never by test)
        best_k = min(disc, key=lambda c: R[R.config == c].val.mean())
        picked[(u["dataset"], u["family"])] = best_k
        for c, g in R[R.config != "none"].groupby("config"):
            g = g.set_index("seed")
            gain = sign * (g[metric] - base.loc[g.index, metric]).values
            m, lo, hi = ci(gain)
            table.append({"dataset": u["dataset"], "family": u["family"], "config": c, "picked_by_val": c == best_k,
                          metric: float(g[metric].mean()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi})
    T = pd.DataFrame(table)

    def paired(u, a, b):
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci(sign * (A - B.loc[A.index]).values)

    # D-a: on ZINC the top-3 discovered features include the large-ring term of penalised logP (rings longer than 6)
    zs = [u for u in units if u["dataset"] == "ZINC"]
    def big_ring(f):
        return f in ("max_ring_len", "ring_excess_over_6", "rings_gt6") or (f.startswith("ring") and f[4:].split(":")[0].isdigit()
                                                                           and int(f[4:].split(":")[0]) > 6)
    hits = [(u["family"], [s["feature"] for s in u["discovered"][:3]]) for u in zs]
    V["D-a rediscovers the hidden large-ring term on ZINC"] = {
        "holds": bool(zs and any(any(big_ring(f) for f in fs) for _, fs in hits)),
        "detail": "; ".join(f"{fam}: {fs}" for fam, fs in hits)}
    # D-b: the discovered set beats the hand-made cycle rule from L1 (ZINC)
    det, ok = [], []
    for u in zs:
        k = picked[(u["dataset"], u["family"])]
        m, lo, hi = paired(u, k, "cycles")
        ok.append(lo > 0)
        det.append(f"{u['family']} {k} vs cycles: {m:+.4f} MAE better [{lo:+.4f}, {hi:+.4f}]")
    V["D-b discovery beats the hand rule (ZINC)"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    # D-c: selection matters: the picked set beats the random-8 placebo and the whole candidate pool
    det, ok = [], []
    for u in units:
        k = picked[(u["dataset"], u["family"])]
        for other in ["rand8", "allcand"]:
            m, lo, hi = paired(u, k, other)
            ok.append(m > 0)
            det.append(f"{u['dataset']}/{u['family']} {k} vs {other}: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]")
    V["D-c selection beats random and dump-everything"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    # D-d: out of sample, molhiv (scaffold split): the picked set improves test ROC-AUC over the plain GINE
    mh = T[(T.dataset == "ogbg-molhiv") & T.picked_by_val]
    if len(mh):
        r = mh.iloc[0]
        V["D-d discovery helps out of sample (molhiv AUC)"] = {
            "holds": bool(r.lo > 0), "detail": f"{r.config}: AUC {r.base:.4f} -> {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]"}
    # D-e: discovery skips what the network already knows: no selected feature has R2 >= 0.9 from T
    sel = [(u["dataset"], u["family"], s["feature"], s["r2_from_T"]) for u in units for s in u["discovered"]]
    V["D-e learnable candidates are never picked"] = {
        "holds": bool(all(r2 < 0.9 for *_, r2 in sel)),
        "detail": f"{sum(r2 >= 0.9 for *_, r2 in sel)}/{len(sel)} picked features have R2_T >= 0.9"}

    disc = pd.DataFrame([{"dataset": u["dataset"], "family": u["family"], "step": i + 1, **s}
                         for u in units for i, s in enumerate(u["discovered"])])
    T.to_csv(os.path.join(CFG["out_dir"], "discovery_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "discovery_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "discovery_verdicts.json"), "w") as fh:
        json.dump(V, fh, indent=1)
    return mo.vstack([
        mo.md("### D2 discovery verdicts\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Discovered features, in the order picked**"),
        mo.ui.table(disc.round(4), page_size=30, selection=None),
    ])


_d2_discovery_verdicts()

In [ ]:
# ==== D3. certify the discovery: permutation null, a second discovery round, and molhiv with validation-AUC selection (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _d3_certify_discovery():
    """D3: rerun D1's discovery with fresh seeds (independent replication), attach a 1000-permutation null to every
    pick, run a second discovery round on the network trained WITH the first pick (does the signal drop to noise?),
    and rerun molhiv for 100 epochs with checkpoints chosen by validation ROC-AUC. Safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 100,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 100,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D3")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0])})
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate(nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D3] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"]) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        if D["n_bond"] == 0:
            RULES_D = [r for r in RULES if r != "bond_hist"]
        else:
            RULES_D = list(RULES)
        D["rules"] = RULES_D
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            out.append({"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                        "idx": torch.tensor(sel, device=DEVICE)})
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            self.emb = [torch.stack([torch.randn(D["n_types"], dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(max(D["n_bond"], 1), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                msg = h[B["src"]]
                if self.edge:
                    msg = F_.relu(msg + self.bemb[0][:, B["bond"], li].transpose(0, 1))
                z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or (CFG["epochs_molhiv"] if D["name"] == "ogbg-molhiv" else CFG["epochs"])
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D3] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"] == "ogbg-molhiv" and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            Ts = torch.zeros(len(D["graphs"]), E, (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def build_candidates(D):
        cache = os.path.join(CFG["data_dir"], f"D1_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D3] building the ring candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [candidate_library(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.log1p(np.maximum(X, 0))
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def discover(D, T, Z, names, kmax, exclude=()):
        """Greedy residual pursuit. r = label minus what the trained plain network's representation T predicts;
        C_res = candidate minus what T predicts of it (both cross-fitted on the training graphs). Each step picks the
        candidate whose unlearnable part best explains the remaining residual (Theorem L: value = I(Y; S | T))."""
        tr = np.flatnonzero(D["split"] == 0)
        rng = np.random.default_rng(0)
        fold = rng.permutation(np.arange(len(tr)) % 2)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        yhat, zhat = torch.zeros_like(y), torch.zeros_like(Zt)
        task = "clf" if D["task"] == "clf" else "mae"
        for k in range(2):
            a = torch.tensor(np.flatnonzero(fold != k), device=DEVICE)
            b = torch.tensor(np.flatnonzero(fold == k), device=DEVICE)
            o = small_fit(Tt[a], y[a], Tt[b], task, D["K"])
            yhat[b] = F_.softmax(o, -1)[:, 1] if task == "clf" else o.squeeze(-1)
            zhat[b] = small_fit(Tt[a], Zt[a], Tt[b], "mse", 1)
        r = y - yhat
        Cres = Zt - zhat
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        sel, steps, res = [], [], r - r.mean()
        gperm = torch.Generator(device="cpu").manual_seed(123)
        for _ in range(kmax):
            score = (Cc * res[:, None]).mean(0).abs() / (res.std() + 1e-9)  # covariance with the unlearnable part
            score[sel + list(exclude)] = -1
            j = int(score.argmax())
            # permutation null for the max score: shuffle the residual against the candidates
            null = []
            for b0 in range(0, CFG["n_perm"], 100):
                nb = min(100, CFG["n_perm"] - b0)
                P = torch.argsort(torch.rand(len(res), nb, generator=gperm), 0).to(DEVICE)
                ns = (Cc.T @ res[P]).abs() / len(res) / (res.std() + 1e-9)  # [J, nb]
                ns[sel + list(exclude)] = -1
                null.append(ns.max(0).values)
            null = torch.cat(null)
            pval = float((1 + (null >= score[j]).sum()) / (1 + len(null)))
            sel.append(j)
            X = Cc[:, sel]
            beta = torch.linalg.lstsq(X, (r - r.mean())[:, None]).solution
            res = (r - r.mean()) - (X @ beta)[:, 0]
            steps.append({"feature": names[j], "score": float(score[j]), "r2_from_T": float(r2[j]),
                          "residual_std_after": float(res.std()), "perm_p": pval, "null_q95": float(null.quantile(0.95))})
        return sel, steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"][D["name"]]))
        Zc, names = build_candidates(D)
        # phase 1: the plain network, all seeds as members; its representation drives discovery
        out0, val0, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        sel, steps = discover(D, T0[:, 0].float(), Zc, names, CFG["kmax"])
        rng = np.random.default_rng(1)
        rand = rng.choice([j for j in range(len(names)) if j not in sel], size=min(8, len(names) - len(sel)), replace=False)
        for key, cols in [("disc1", sel[:1]), ("disc3", sel[:3]), ("disc8", sel[:8]), ("rand8", list(rand)),
                          ("allcand", list(range(len(names))))]:
            D["Z"][key] = Zc[:, cols]
        configs = ["cycles", "disc1", "disc3", "disc8", "rand8", "allcand"]
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule from L1, the discovered sets, a random-candidate placebo and the whole pool
        out1, val1, T1, _ = train_members(D, edge, member_rules, member_seeds)
        # round 2: discovery again, on the representation of the network trained WITH the first pick (plus the pick
        # itself as an input). If the first pick paid the debt, what is left should look like the permutation null.
        e1 = member_rules.index("disc1")
        T_aug = torch.cat([T1[:, e1].float(), torch.tensor(Zc[:, sel[:1]], device=DEVICE)], 1)
        _, steps2 = discover(D, T_aug, Zc, names, 3, exclude=sel[:1])
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows,
                "discovered": steps, "round2": steps2, "n_candidates": len(names), "random8": [names[j] for j in rand]}

    # ---------------------------------------------------------------- driver
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("ZINC", False), ("ZINC", True), ("ogbg-molhiv", True)] if not SMOKE else [("ZINC", True), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D3 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"E{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D3_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, discovered: "
                + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["discovered"][:3])
                + " | round 2: " + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["round2"][:2]))
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D3 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D3_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d3_certify_discovery()

In [ ]:
# ==== D4. certified discovery: verdicts (self-contained: all names live inside one function, so it can be pasted anywhere) ====
def _d4_certified_verdicts():
    """D4: reads the D3 checkpoints and tests R-a..R-e (fixed before the run). Gain = improvement over the plain
    network (MAE lower is better on ZINC, ROC-AUC higher on molhiv), paired by seed, bootstrap CIs."""
    import os, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D3 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 100,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 100,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D3")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    units = []
    for d, fam in [("ZINC", "gin"), ("ZINC", "gine"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"E{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units.append(json.load(fh))
    mo.stop(not units, mo.md(f"no D3 checkpoints with hash `{L_HASH}` yet: run D3 first"))
    rng = np.random.default_rng(0)

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    def big_ring(f):
        return f in ("max_ring_len", "ring_excess_over_6", "rings_gt6") or (
            f.startswith("ring") and f[4:].split(":")[0].isdigit() and int(f[4:].split(":")[0]) > 6)

    table, picked = [], {}
    for u in units:
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        base = R[R.config == "none"].set_index("seed")
        best_k = min(["disc1", "disc3", "disc8"], key=lambda c: R[R.config == c].val.mean())  # validation only
        picked[(u["dataset"], u["family"])] = best_k
        for c, g in R[R.config != "none"].groupby("config"):
            g = g.set_index("seed")
            m, lo, hi = ci(sign * (g[metric] - base.loc[g.index, metric]).values)
            table.append({"dataset": u["dataset"], "family": u["family"], "config": c, "picked_by_val": c == best_k,
                          metric: float(g[metric].mean()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi})
    T = pd.DataFrame(table)

    def paired(u, a, b):
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci(sign * (A - B.loc[A.index]).values)

    V = {}
    zs = [u for u in units if u["dataset"] == "ZINC"]
    first = [(u["family"], u["discovered"][0]) for u in zs]
    V["R-a fresh seeds rediscover the large-ring term, p <= 0.005"] = {
        "holds": bool(zs and all(big_ring(s["feature"]) and s["perm_p"] <= 0.005 for _, s in first)),
        "detail": "; ".join(f"{f}: {s['feature']} score {s['score']:.3f} vs null q95 {s['null_q95']:.3f}, p={s['perm_p']:.4f}" for f, s in first)}
    pd_ = [(u["family"], u["discovered"][0]["score"], u["round2"][0]) for u in zs]
    V["R-b the first pick pays the debt down (round-2 top <= 25% of round-1 top)"] = {
        "holds": bool(zs and all(s2["score"] <= 0.25 * s1 for _, s1, s2 in pd_)),
        "detail": "; ".join(f"{f}: round 1 {s1:.3f} -> round 2 {s2['score']:.3f} ({s2['feature']})" for f, s1, s2 in pd_)}
    V["R-c certificate: nothing significant left after round 1 (round-2 p > 0.01)"] = {
        "holds": bool(zs and all(s2["perm_p"] > 0.01 for _, _, s2 in pd_)),
        "detail": "; ".join(f"{f}: round-2 top {s2['feature']} p={s2['perm_p']:.4f}" for f, _, s2 in pd_)}
    det, ok = [], []
    for u in zs:
        k = picked[(u["dataset"], u["family"])]
        m, lo, hi = paired(u, k, "cycles")
        ok.append(lo > 0)
        det.append(f"{u['family']} {k} vs cycles: {m:+.4f} MAE better [{lo:+.4f}, {hi:+.4f}]")
    V["R-d replication: discovery beats the hand rule (ZINC)"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    mh = [u for u in units if u["dataset"] == "ogbg-molhiv"]
    if mh:
        r = T[(T.dataset == "ogbg-molhiv") & T.picked_by_val].iloc[0]
        V["R-e molhiv: baseline >= 0.755 under val-AUC selection and discovery still helps"] = {
            "holds": bool(r.base >= 0.755 and r.lo > 0),
            "detail": f"plain GINE {r.base:.4f}; {r.config} {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]; "
                      f"picks: {[s['feature'] for s in mh[0]['discovered'][:3]]}"}
    disc = pd.DataFrame([{"dataset": u["dataset"], "family": u["family"], "round": rd, "step": i + 1, **s}
                         for u in units for rd, key in [(1, "discovered"), (2, "round2")] for i, s in enumerate(u[key])])
    T.to_csv(os.path.join(CFG["out_dir"], "certified_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "certified_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "certified_verdicts.json"), "w") as fh:
        json.dump(V, fh, indent=1)
    return mo.vstack([
        mo.md("### D4 certified-discovery verdicts\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Picked features, both rounds, with permutation p-values**"),
        mo.ui.table(disc.round(4), page_size=40, selection=None),
    ])


_d4_certified_verdicts()

In [ ]:
# ==== HF check. compare local results with the Hugging Face repo and upload anything missing or changed (self-contained) ====
def _hf_check_and_sync():
    """Lists every local result file, compares name and size with the Hub repo, shows what is missing per stage,
    and uploads only the missing or changed files. Safe to re-run; nothing remote is deleted."""
    import os, re, sys, subprocess, collections
    import marimo as mo

    UPLOAD = True  # set False to only report
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"  # same token as in HF_push_cell.py
    REPO_TYPE = "model"
    try:
        from huggingface_hub import HfApi
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
        from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type=REPO_TYPE, private=True, exist_ok=True)

    def remote_files():
        return {f.path: f.size for f in api.list_repo_tree(HF_REPO, repo_type=REPO_TYPE, recursive=True)
                if getattr(f, "size", None) is not None}

    remote = remote_files()
    tok = re.compile(rb"hf_[A-Za-z0-9]{30,}")
    local, skipped = {}, []

    def consider(p):
        with open(p, "rb") as fh:
            if tok.search(fh.read()):
                skipped.append(p)
                return
        local[p] = os.path.getsize(p)

    for root in ["nsgnn_runs", "nsgnn_runs_smoke"]:
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if not d.startswith(".")]
            for f in fn:
                if ".tmp" not in f and not f.endswith(".part"):
                    consider(os.path.join(dp, f).replace(os.sep, "/"))
    for f in os.listdir("."):
        if os.path.isfile(f) and f.endswith((".py", ".csv", ".json", ".md", ".log", ".png", ".pdf")):
            consider(f)
    todo = sorted(p for p, s in local.items() if remote.get(p) != s)

    # checkpoint units per prefix: G = G4, R = G7, L = G9/L1, S = G11, M = L3, D = D1, E = D3
    stage, stage_todo = collections.Counter(), collections.Counter()
    for p in local:
        if "/units/" in p:
            k = os.path.basename(p)[0]
            stage[k] += 1
            stage_todo[k] += p in todo
    rows = "\n".join(f"| {k} | {stage[k]} | {stage[k] - stage_todo[k]} | {stage_todo[k]} |" for k in sorted(stage))
    msg = (f"**Local files:** {len(local)}, **already on the Hub:** {len(local) - len(todo)}, "
           f"**missing or changed:** {len(todo)} (remote repo holds {len(remote)} files)\n\n"
           f"| unit prefix | local | on Hub | to upload |\n|---|---|---|---|\n{rows}")
    if todo and UPLOAD:
        api.upload_folder(repo_id=HF_REPO, repo_type=REPO_TYPE, folder_path=".", allow_patterns=todo,
                          commit_message=f"sync {len(todo)} missing or changed result files")
        after = remote_files()
        still = [p for p in todo if after.get(p) != local[p]]
        msg += f"\n\n**Uploaded {len(todo) - len(still)} files; still missing after upload: {len(still)}**"
        msg += "".join(f"\n- `{p}`" for p in still[:30])
    elif todo:
        msg += "\n\nMissing:" + "".join(f"\n- `{p}`" for p in todo[:50])
    else:
        msg += "\n\n**Everything local is already on the Hub.**"
    if skipped:
        msg += "\n\nSkipped because they contain a token: " + ", ".join(f"`{p}`" for p in skipped)
    return mo.md(msg)


_hf_check_and_sync()

In [ ]:
# ==== HF restore. download nsgnn_runs/ (and nsgnn_data caches) back from the Hugging Face repo after a reset (self-contained) ====
def _hf_restore():
    """Pulls every result file from the Hub into this folder, so finished cells skip their saved units on re-run."""
    import os, sys, subprocess, collections
    import marimo as mo

    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"  # same token as in HF_push_cell.py
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
        from huggingface_hub import snapshot_download
    print("cwd:", os.getcwd(), "| before:", os.listdir("."))
    snapshot_download(HF_REPO, repo_type="model", token=HF_TOKEN, local_dir=".",
                      allow_patterns=["nsgnn_runs/**", "nsgnn_runs_smoke/**", "nsgnn_data/**", "*.csv", "*.json"])
    units = [f for f in os.listdir("nsgnn_runs/units")] if os.path.isdir("nsgnn_runs/units") else []
    c = collections.Counter(f[0] for f in units)
    return mo.md(f"**Restored {len(units)} checkpoint units** into `{os.getcwd()}/nsgnn_runs/units`: "
                 + ", ".join(f"{k}: {v}" for k, v in sorted(c.items())))


_hf_restore()

In [ ]:
# ==== D1. rule discovery: let the network's own residual pick the missing symbolic rule from a ring-candidate space (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d1_rule_discovery():
    """D1: train the plain GIN/GINE once, then greedily pick the ring statistics whose unlearnable part explains the
    network's residual, then retrain with the discovered rules vs the hand rule, a random placebo and the whole pool.
    ZINC (GIN, GINE) and ogbg-molhiv (GINE, scaffold split). One checkpoint per (dataset, family); safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 60,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D1")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0])})
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate(nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D1] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"]) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        if D["n_bond"] == 0:
            RULES_D = [r for r in RULES if r != "bond_hist"]
        else:
            RULES_D = list(RULES)
        D["rules"] = RULES_D
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            out.append({"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                        "idx": torch.tensor(sel, device=DEVICE)})
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            self.emb = [torch.stack([torch.randn(D["n_types"], dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(max(D["n_bond"], 1), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                msg = h[B["src"]]
                if self.edge:
                    msg = F_.relu(msg + self.bemb[0][:, B["bond"], li].transpose(0, 1))
                z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or (CFG["epochs_molhiv"] if D["name"] == "ogbg-molhiv" else CFG["epochs"])
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D1] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            Ts = torch.zeros(len(D["graphs"]), E, (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def build_candidates(D):
        cache = os.path.join(CFG["data_dir"], f"D1_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D1] building the ring candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [candidate_library(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.log1p(np.maximum(X, 0))
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def discover(D, T, Z, names, kmax):
        """Greedy residual pursuit. r = label minus what the trained plain network's representation T predicts;
        C_res = candidate minus what T predicts of it (both cross-fitted on the training graphs). Each step picks the
        candidate whose unlearnable part best explains the remaining residual (Theorem L: value = I(Y; S | T))."""
        tr = np.flatnonzero(D["split"] == 0)
        rng = np.random.default_rng(0)
        fold = rng.permutation(np.arange(len(tr)) % 2)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        yhat, zhat = torch.zeros_like(y), torch.zeros_like(Zt)
        task = "clf" if D["task"] == "clf" else "mae"
        for k in range(2):
            a = torch.tensor(np.flatnonzero(fold != k), device=DEVICE)
            b = torch.tensor(np.flatnonzero(fold == k), device=DEVICE)
            o = small_fit(Tt[a], y[a], Tt[b], task, D["K"])
            yhat[b] = F_.softmax(o, -1)[:, 1] if task == "clf" else o.squeeze(-1)
            zhat[b] = small_fit(Tt[a], Zt[a], Tt[b], "mse", 1)
        r = y - yhat
        Cres = Zt - zhat
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        sel, steps, res = [], [], r - r.mean()
        for _ in range(kmax):
            score = (Cc * res[:, None]).mean(0).abs() / (res.std() + 1e-9)  # covariance with the unlearnable part
            score[sel] = -1
            j = int(score.argmax())
            sel.append(j)
            X = Cc[:, sel]
            beta = torch.linalg.lstsq(X, (r - r.mean())[:, None]).solution
            res = (r - r.mean()) - (X @ beta)[:, 0]
            steps.append({"feature": names[j], "score": float(score[j]), "r2_from_T": float(r2[j]),
                          "residual_std_after": float(res.std())})
        return sel, steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seeds"][D["name"]]))
        Zc, names = build_candidates(D)
        # phase 1: the plain network, all seeds as members; its representation drives discovery
        out0, val0, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        sel, steps = discover(D, T0[:, 0].float(), Zc, names, CFG["kmax"])
        rng = np.random.default_rng(1)
        rand = rng.choice([j for j in range(len(names)) if j not in sel], size=min(8, len(names) - len(sel)), replace=False)
        for key, cols in [("disc1", sel[:1]), ("disc3", sel[:3]), ("disc8", sel[:8]), ("rand8", list(rand)),
                          ("allcand", list(range(len(names))))]:
            D["Z"][key] = Zc[:, cols]
        configs = ["cycles", "disc1", "disc3", "disc8", "rand8", "allcand"]
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule from L1, the discovered sets, a random-candidate placebo and the whole pool
        out1, val1, _, _ = train_members(D, edge, member_rules, member_seeds)
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows,
                "discovered": steps, "n_candidates": len(names), "random8": [names[j] for j in rand]}

    # ---------------------------------------------------------------- driver
    # restore this stage's finished units and the feature caches, so nothing is recomputed after a reset
    hf_pull([f"{CFG['out_dir']}/units/D{L_HASH}_*", f"{CFG['data_dir']}/L1_*_full.npz", f"{CFG['data_dir']}/D1_*_full_cand.json"])
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("ZINC", False), ("ZINC", True), ("ogbg-molhiv", True)] if not SMOKE else [("ZINC", True), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D1 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"D{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D1_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"L1_{d}_full.npz"), os.path.join(CFG["data_dir"], f"D1_{d}_full_cand.json")],
                f"D1 unit {d} {'gine' if edge else 'gin'}")
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, discovered: "
                + "; ".join(s["feature"] for s in res["discovered"][:5]))
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D1 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D1_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d1_rule_discovery()

In [ ]:
# ==== D2. rule discovery: verdicts (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d2_discovery_verdicts():
    """D2: reads the D1 checkpoints and tests D-a..D-e (fixed before the run). Gain = improvement over the plain
    network (MAE lower is better on ZINC, ROC-AUC higher is better on molhiv), paired by seed, bootstrap CIs."""
    import os, sys, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D1 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 60,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D1")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)
    hf_pull([f"{CFG['out_dir']}/units/D{L_HASH}_*"])
    units = []
    for d, fam in [("ZINC", "gin"), ("ZINC", "gine"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"D{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units.append(json.load(fh))
    mo.stop(not units, mo.md(f"no D1 checkpoints with hash `{L_HASH}` yet: run D1 first"))
    rng = np.random.default_rng(0)

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    table, V, picked = [], {}, {}
    for u in units:
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        base = R[R.config == "none"].set_index("seed")
        disc = [c for c in ["disc1", "disc3", "disc8"] if c in set(R.config)]
        # the discovered set is chosen by validation loss only (never by test)
        best_k = min(disc, key=lambda c: R[R.config == c].val.mean())
        picked[(u["dataset"], u["family"])] = best_k
        for c, g in R[R.config != "none"].groupby("config"):
            g = g.set_index("seed")
            gain = sign * (g[metric] - base.loc[g.index, metric]).values
            m, lo, hi = ci(gain)
            table.append({"dataset": u["dataset"], "family": u["family"], "config": c, "picked_by_val": c == best_k,
                          metric: float(g[metric].mean()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi})
    T = pd.DataFrame(table)

    def paired(u, a, b):
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci(sign * (A - B.loc[A.index]).values)

    # D-a: on ZINC the top-3 discovered features include the large-ring term of penalised logP (rings longer than 6)
    zs = [u for u in units if u["dataset"] == "ZINC"]
    def big_ring(f):
        return f in ("max_ring_len", "ring_excess_over_6", "rings_gt6") or (f.startswith("ring") and f[4:].split(":")[0].isdigit()
                                                                           and int(f[4:].split(":")[0]) > 6)
    hits = [(u["family"], [s["feature"] for s in u["discovered"][:3]]) for u in zs]
    V["D-a rediscovers the hidden large-ring term on ZINC"] = {
        "holds": bool(zs and any(any(big_ring(f) for f in fs) for _, fs in hits)),
        "detail": "; ".join(f"{fam}: {fs}" for fam, fs in hits)}
    # D-b: the discovered set beats the hand-made cycle rule from L1 (ZINC)
    det, ok = [], []
    for u in zs:
        k = picked[(u["dataset"], u["family"])]
        m, lo, hi = paired(u, k, "cycles")
        ok.append(lo > 0)
        det.append(f"{u['family']} {k} vs cycles: {m:+.4f} MAE better [{lo:+.4f}, {hi:+.4f}]")
    V["D-b discovery beats the hand rule (ZINC)"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    # D-c: selection matters: the picked set beats the random-8 placebo and the whole candidate pool
    det, ok = [], []
    for u in units:
        k = picked[(u["dataset"], u["family"])]
        for other in ["rand8", "allcand"]:
            m, lo, hi = paired(u, k, other)
            ok.append(m > 0)
            det.append(f"{u['dataset']}/{u['family']} {k} vs {other}: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]")
    V["D-c selection beats random and dump-everything"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    # D-d: out of sample, molhiv (scaffold split): the picked set improves test ROC-AUC over the plain GINE
    mh = T[(T.dataset == "ogbg-molhiv") & T.picked_by_val]
    if len(mh):
        r = mh.iloc[0]
        V["D-d discovery helps out of sample (molhiv AUC)"] = {
            "holds": bool(r.lo > 0), "detail": f"{r.config}: AUC {r.base:.4f} -> {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]"}
    # D-e: discovery skips what the network already knows: no selected feature has R2 >= 0.9 from T
    sel = [(u["dataset"], u["family"], s["feature"], s["r2_from_T"]) for u in units for s in u["discovered"]]
    V["D-e learnable candidates are never picked"] = {
        "holds": bool(all(r2 < 0.9 for *_, r2 in sel)),
        "detail": f"{sum(r2 >= 0.9 for *_, r2 in sel)}/{len(sel)} picked features have R2_T >= 0.9"}

    disc = pd.DataFrame([{"dataset": u["dataset"], "family": u["family"], "step": i + 1, **s}
                         for u in units for i, s in enumerate(u["discovered"])])
    T.to_csv(os.path.join(CFG["out_dir"], "discovery_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "discovery_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "discovery_verdicts.json"), "w") as fh:
        json.dump(V, fh, indent=1)
    hf_push([os.path.join(CFG["out_dir"], f) for f in ["discovery_rows.csv", "discovery_features.csv", "discovery_verdicts.json"]],
            "D2 verdicts")
    return mo.vstack([
        mo.md("### D2 discovery verdicts\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Discovered features, in the order picked**"),
        mo.ui.table(disc.round(4), page_size=30, selection=None),
    ])


_d2_discovery_verdicts()

In [ ]:
# ==== D3. certify the discovery: permutation null, a second discovery round, and molhiv with validation-AUC selection (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d3_certify_discovery():
    """D3: rerun D1's discovery with fresh seeds (independent replication), attach a 1000-permutation null to every
    pick, run a second discovery round on the network trained WITH the first pick (does the signal drop to noise?),
    and rerun molhiv for 100 epochs with checkpoints chosen by validation ROC-AUC. Safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 100,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 100,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D3")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0])})
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate(nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D3] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"]) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        if D["n_bond"] == 0:
            RULES_D = [r for r in RULES if r != "bond_hist"]
        else:
            RULES_D = list(RULES)
        D["rules"] = RULES_D
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            out.append({"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                        "idx": torch.tensor(sel, device=DEVICE)})
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            self.emb = [torch.stack([torch.randn(D["n_types"], dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(max(D["n_bond"], 1), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                msg = h[B["src"]]
                if self.edge:
                    msg = F_.relu(msg + self.bemb[0][:, B["bond"], li].transpose(0, 1))
                z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or (CFG["epochs_molhiv"] if D["name"] == "ogbg-molhiv" else CFG["epochs"])
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D3] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"] == "ogbg-molhiv" and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            Ts = torch.zeros(len(D["graphs"]), E, (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def build_candidates(D):
        cache = os.path.join(CFG["data_dir"], f"D1_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D3] building the ring candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [candidate_library(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.log1p(np.maximum(X, 0))
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def discover(D, T, Z, names, kmax, exclude=()):
        """Greedy residual pursuit. r = label minus what the trained plain network's representation T predicts;
        C_res = candidate minus what T predicts of it (both cross-fitted on the training graphs). Each step picks the
        candidate whose unlearnable part best explains the remaining residual (Theorem L: value = I(Y; S | T))."""
        tr = np.flatnonzero(D["split"] == 0)
        rng = np.random.default_rng(0)
        fold = rng.permutation(np.arange(len(tr)) % 2)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        yhat, zhat = torch.zeros_like(y), torch.zeros_like(Zt)
        task = "clf" if D["task"] == "clf" else "mae"
        for k in range(2):
            a = torch.tensor(np.flatnonzero(fold != k), device=DEVICE)
            b = torch.tensor(np.flatnonzero(fold == k), device=DEVICE)
            o = small_fit(Tt[a], y[a], Tt[b], task, D["K"])
            yhat[b] = F_.softmax(o, -1)[:, 1] if task == "clf" else o.squeeze(-1)
            zhat[b] = small_fit(Tt[a], Zt[a], Tt[b], "mse", 1)
        r = y - yhat
        Cres = Zt - zhat
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        sel, steps, res = [], [], r - r.mean()
        gperm = torch.Generator(device="cpu").manual_seed(123)
        for _ in range(kmax):
            score = (Cc * res[:, None]).mean(0).abs() / (res.std() + 1e-9)  # covariance with the unlearnable part
            score[sel + list(exclude)] = -1
            j = int(score.argmax())
            # permutation null for the max score: shuffle the residual against the candidates
            null = []
            for b0 in range(0, CFG["n_perm"], 100):
                nb = min(100, CFG["n_perm"] - b0)
                P = torch.argsort(torch.rand(len(res), nb, generator=gperm), 0).to(DEVICE)
                ns = (Cc.T @ res[P]).abs() / len(res) / (res.std() + 1e-9)  # [J, nb]
                ns[sel + list(exclude)] = -1
                null.append(ns.max(0).values)
            null = torch.cat(null)
            pval = float((1 + (null >= score[j]).sum()) / (1 + len(null)))
            sel.append(j)
            X = Cc[:, sel]
            beta = torch.linalg.lstsq(X, (r - r.mean())[:, None]).solution
            res = (r - r.mean()) - (X @ beta)[:, 0]
            steps.append({"feature": names[j], "score": float(score[j]), "r2_from_T": float(r2[j]),
                          "residual_std_after": float(res.std()), "perm_p": pval, "null_q95": float(null.quantile(0.95))})
        return sel, steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"][D["name"]]))
        Zc, names = build_candidates(D)
        # phase 1: the plain network, all seeds as members; its representation drives discovery
        out0, val0, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        sel, steps = discover(D, T0[:, 0].float(), Zc, names, CFG["kmax"])
        rng = np.random.default_rng(1)
        rand = rng.choice([j for j in range(len(names)) if j not in sel], size=min(8, len(names) - len(sel)), replace=False)
        for key, cols in [("disc1", sel[:1]), ("disc3", sel[:3]), ("disc8", sel[:8]), ("rand8", list(rand)),
                          ("allcand", list(range(len(names))))]:
            D["Z"][key] = Zc[:, cols]
        configs = ["cycles", "disc1", "disc3", "disc8", "rand8", "allcand"]
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule from L1, the discovered sets, a random-candidate placebo and the whole pool
        out1, val1, T1, _ = train_members(D, edge, member_rules, member_seeds)
        # round 2: discovery again, on the representation of the network trained WITH the first pick (plus the pick
        # itself as an input). If the first pick paid the debt, what is left should look like the permutation null.
        e1 = member_rules.index("disc1")
        T_aug = torch.cat([T1[:, e1].float(), torch.tensor(Zc[:, sel[:1]], device=DEVICE)], 1)
        _, steps2 = discover(D, T_aug, Zc, names, 3, exclude=sel[:1])
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows,
                "discovered": steps, "round2": steps2, "n_candidates": len(names), "random8": [names[j] for j in rand]}

    # ---------------------------------------------------------------- driver
    # restore this stage's finished units and the feature caches, so nothing is recomputed after a reset
    hf_pull([f"{CFG['out_dir']}/units/E{L_HASH}_*", f"{CFG['data_dir']}/L1_*_full.npz", f"{CFG['data_dir']}/D1_*_full_cand.json"])
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("ZINC", False), ("ZINC", True), ("ogbg-molhiv", True)] if not SMOKE else [("ZINC", True), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D3 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"E{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D3_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"L1_{d}_full.npz"), os.path.join(CFG["data_dir"], f"D1_{d}_full_cand.json")],
                f"D3 unit {d} {'gine' if edge else 'gin'}")
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, discovered: "
                + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["discovered"][:3])
                + " | round 2: " + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["round2"][:2]))
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D3 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D3_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d3_certify_discovery()

In [ ]:
# ==== D4. certified discovery: verdicts (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d4_certified_verdicts():
    """D4: reads the D3 checkpoints and tests R-a..R-e (fixed before the run). Gain = improvement over the plain
    network (MAE lower is better on ZINC, ROC-AUC higher on molhiv), paired by seed, bootstrap CIs."""
    import os, sys, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D3 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ZINC", "ogbg-molhiv"],
        "seeds": {"ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 5},
        "hidden": 32 if SMOKE else 128,
        "layers": 4,
        "epochs": 6 if SMOKE else 200,  # ZINC; molhiv uses epochs_molhiv
        "epochs_molhiv": 6 if SMOKE else 100,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 100,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "batch": 128,
        "lr": 1e-3,
        "dropout": 0.0,
        "max_cycle": 8,
    }
    LKEY = dict(CFG, v=1, stage="D3")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)
    hf_pull([f"{CFG['out_dir']}/units/E{L_HASH}_*"])
    units = []
    for d, fam in [("ZINC", "gin"), ("ZINC", "gine"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"E{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units.append(json.load(fh))
    mo.stop(not units, mo.md(f"no D3 checkpoints with hash `{L_HASH}` yet: run D3 first"))
    rng = np.random.default_rng(0)

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    def big_ring(f):
        return f in ("max_ring_len", "ring_excess_over_6", "rings_gt6") or (
            f.startswith("ring") and f[4:].split(":")[0].isdigit() and int(f[4:].split(":")[0]) > 6)

    table, picked = [], {}
    for u in units:
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        base = R[R.config == "none"].set_index("seed")
        best_k = min(["disc1", "disc3", "disc8"], key=lambda c: R[R.config == c].val.mean())  # validation only
        picked[(u["dataset"], u["family"])] = best_k
        for c, g in R[R.config != "none"].groupby("config"):
            g = g.set_index("seed")
            m, lo, hi = ci(sign * (g[metric] - base.loc[g.index, metric]).values)
            table.append({"dataset": u["dataset"], "family": u["family"], "config": c, "picked_by_val": c == best_k,
                          metric: float(g[metric].mean()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi})
    T = pd.DataFrame(table)

    def paired(u, a, b):
        R = pd.DataFrame(u["rows"])
        metric, sign = ("mae", -1.0) if u["task"] == "reg" else ("auc", 1.0)
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci(sign * (A - B.loc[A.index]).values)

    V = {}
    zs = [u for u in units if u["dataset"] == "ZINC"]
    first = [(u["family"], u["discovered"][0]) for u in zs]
    V["R-a fresh seeds rediscover the large-ring term, p <= 0.005"] = {
        "holds": bool(zs and all(big_ring(s["feature"]) and s["perm_p"] <= 0.005 for _, s in first)),
        "detail": "; ".join(f"{f}: {s['feature']} score {s['score']:.3f} vs null q95 {s['null_q95']:.3f}, p={s['perm_p']:.4f}" for f, s in first)}
    pd_ = [(u["family"], u["discovered"][0]["score"], u["round2"][0]) for u in zs]
    V["R-b the first pick pays the debt down (round-2 top <= 25% of round-1 top)"] = {
        "holds": bool(zs and all(s2["score"] <= 0.25 * s1 for _, s1, s2 in pd_)),
        "detail": "; ".join(f"{f}: round 1 {s1:.3f} -> round 2 {s2['score']:.3f} ({s2['feature']})" for f, s1, s2 in pd_)}
    V["R-c certificate: nothing significant left after round 1 (round-2 p > 0.01)"] = {
        "holds": bool(zs and all(s2["perm_p"] > 0.01 for _, _, s2 in pd_)),
        "detail": "; ".join(f"{f}: round-2 top {s2['feature']} p={s2['perm_p']:.4f}" for f, _, s2 in pd_)}
    det, ok = [], []
    for u in zs:
        k = picked[(u["dataset"], u["family"])]
        m, lo, hi = paired(u, k, "cycles")
        ok.append(lo > 0)
        det.append(f"{u['family']} {k} vs cycles: {m:+.4f} MAE better [{lo:+.4f}, {hi:+.4f}]")
    V["R-d replication: discovery beats the hand rule (ZINC)"] = {"holds": bool(ok and all(ok)), "detail": "; ".join(det)}
    mh = [u for u in units if u["dataset"] == "ogbg-molhiv"]
    if mh:
        r = T[(T.dataset == "ogbg-molhiv") & T.picked_by_val].iloc[0]
        V["R-e molhiv: baseline >= 0.755 under val-AUC selection and discovery still helps"] = {
            "holds": bool(r.base >= 0.755 and r.lo > 0),
            "detail": f"plain GINE {r.base:.4f}; {r.config} {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]; "
                      f"picks: {[s['feature'] for s in mh[0]['discovered'][:3]]}"}
    disc = pd.DataFrame([{"dataset": u["dataset"], "family": u["family"], "round": rd, "step": i + 1, **s}
                         for u in units for rd, key in [(1, "discovered"), (2, "round2")] for i, s in enumerate(u[key])])
    T.to_csv(os.path.join(CFG["out_dir"], "certified_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "certified_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "certified_verdicts.json"), "w") as fh:
        json.dump(V, fh, indent=1)
    hf_push([os.path.join(CFG["out_dir"], f) for f in ["certified_rows.csv", "certified_features.csv", "certified_verdicts.json"]],
            "D4 verdicts")
    return mo.vstack([
        mo.md("### D4 certified-discovery verdicts\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Picked features, both rounds, with permutation p-values**"),
        mo.ui.table(disc.round(4), page_size=40, selection=None),
    ])


_d4_certified_verdicts()

In [ ]:
# ==== D5. discovery beyond ZINC: a fair molhiv test (all OGB features) and a non-chemistry domain (COLLAB) (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d5_discovery_beyond_zinc():
    """D5: the D3 protocol (permutation null on every pick, round-2 certificate, validation-only selection) on
    (1) ogbg-molhiv with all 9 atom and 3 bond features (OGB AtomEncoder/BondEncoder style), dropout 0.5, 8 seeds,
    checkpoints chosen by validation ROC-AUC, so the plain GINE is at the published level; and
    (2) COLLAB (collaboration ego-networks, 3 classes, no chemistry), GIN with degree tags, 5 seeds, where the
    candidates are generic graph statistics instead of rings. COLLAB runs first (fast). Safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["COLLAB", "ogbg-molhiv"],
        "seeds": {"COLLAB": 2 if SMOKE else 5, "ogbg-molhiv": 2 if SMOKE else 8},
        "hidden_by": {"COLLAB": 32 if SMOKE else 64, "ogbg-molhiv": 32 if SMOKE else 256},
        "batch_by": {"COLLAB": 32, "ogbg-molhiv": 128},
        "dropout_by": {"COLLAB": 0.0, "ogbg-molhiv": 0.5},  # OGB's GIN baseline uses dropout 0.5 after every layer
        "epochs_by": {"COLLAB": 6 if SMOKE else 150, "ogbg-molhiv": 6 if SMOKE else 80},
        "layers": 4,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 200,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
    }
    LKEY = dict(CFG, v=1, stage="D5")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)
    CFG.update(hidden=CFG["hidden_by"]["COLLAB"], batch=32, dropout=0.0)  # set per unit by the driver (not hashed)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        if SMOKE:
            raw = [raw[i] for i in np.random.default_rng(0).permutation(len(raw))[:300]]
        deg_tags = len(tagset) == 1  # social graphs carry no node labels: use the degree (capped) as the tag, as GIN does
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            if deg_tags:
                tags = np.minimum(np.asarray(A.sum(1)).ravel(), CFG["max_deg_tag"]).astype(np.int64)
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": CFG["max_deg_tag"] + 1 if deg_tags else len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            # t / bond (atom and bond type) drive the rule and candidate libraries, as in D1/D3 (same caches);
            # xf / bf (all 9 atom and 3 bond features) feed the network
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0]),
                           "xf": data.x.numpy().astype(np.int64).reshape(-1, data.x.shape[1]),
                           "bf": data.edge_attr.numpy().astype(np.int64).reshape(-1, data.edge_attr.shape[1])})
        try:  # OGB's fixed vocabularies (what AtomEncoder / BondEncoder use)
            from ogb.utils.features import get_atom_feature_dims, get_bond_feature_dims
            x_dims, b_dims = list(get_atom_feature_dims()), list(get_bond_feature_dims())
        except Exception:
            x_dims = list(np.max([g["xf"].max(0) for g in graphs], 0) + 1)
            b_dims = list(np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0) + 1)
        x_dims = [int(max(a, b + 1)) for a, b in zip(x_dims, np.max([g["xf"].max(0) for g in graphs], 0))]
        b_dims = [int(max(a, b + 1)) for a, b in zip(b_dims, np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0))]
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5, "x_dims": x_dims, "b_dims": b_dims}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond, skip_cycles=False):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate([] if skip_cycles else nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        social = name in ("COLLAB", "IMDB-BINARY", "IMDB-MULTI", "REDDIT-BINARY")
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D5] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"], social) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        drop = ({"bond_hist"} if D["n_bond"] == 0 else set()) | ({"cycles", "ring_atoms"} if social else set())
        RULES_D = [r for r in RULES if r not in drop]
        D["rules"] = RULES_D
        D["social"] = social
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            b = {"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                 "idx": torch.tensor(sel, device=DEVICE)}
            if "xf" in gs[0]:
                b["xf"] = torch.tensor(np.concatenate([g["xf"] for g in gs]), device=DEVICE)
                b["bf"] = torch.tensor(np.concatenate([g["bf"] for g in gs]), device=DEVICE)
            # without edge features the GIN sum is one sparse product (no per-edge message tensor on dense social graphs)
            b["A"] = torch.sparse_coo_tensor(torch.stack([dst, src]), torch.ones(len(src), device=DEVICE),
                                             (b["N"], b["N"])).coalesce()
            out.append(b)
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            # one embedding table per atom (bond) feature column, summed, as OGB's AtomEncoder (BondEncoder)
            self.xd = D.get("x_dims", [D["n_types"]])
            self.bd = D.get("b_dims", [max(D["n_bond"], 1)])
            self.xo, self.bo = np.cumsum([0] + self.xd)[:-1].tolist(), np.cumsum([0] + self.bd)[:-1].tolist()
            self.emb = [torch.stack([torch.randn(sum(self.xd), dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(sum(self.bd), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            if "xf" in B:
                h = sum(self.emb[0][:, B["xf"][:, c] + o] for c, o in enumerate(self.xo)).transpose(0, 1)  # [N, E, dh]
            else:
                h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                if self.edge:
                    if "bf" in B:
                        be = sum(self.bemb[0][:, B["bf"][:, c] + o, li] for c, o in enumerate(self.bo))
                    else:
                        be = self.bemb[0][:, B["bond"], li]
                    msg = F_.relu(h[B["src"]] + be.transpose(0, 1))
                    z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                else:
                    z = h + torch.sparse.mm(B["A"], h.reshape(B["N"], -1)).view_as(h)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                if li < self.L - 1:
                    h = F_.dropout(h, CFG["dropout"], training)
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None, t_keep=(0,)):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule."""
        epochs = epochs or CFG["epochs_by"][D["name"]]
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                loss_fn(out, B["idx"]).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D5] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"] == "ogbg-molhiv" and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            keep = torch.tensor(list(t_keep), device=DEVICE)
            Ts = torch.zeros(len(D["graphs"]), len(keep), (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1][:, keep]
        return best_out, best, Ts, te

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def social_candidates(g):
        """Generic graph statistics for graphs without chemistry: size, density, degree shape, triangles and
        clustering, cores, cliques (collaboration ego-networks are unions of per-paper cliques), components, spectrum,
        assortativity. Many are not 1-WL invariant (triangles, cliques, cores, spectrum)."""
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz / 2
        tri = np.asarray((A @ A).multiply(A).sum(1)).ravel() / 2
        pairs = deg * (deg - 1) / 2
        nc, lab = conn_comp(A, directed=False)
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        core = np.array(list(nx.core_number(nxg).values())) if m else np.zeros(1)
        cl_max, cl_n = 0, 0
        for k_, c in enumerate(nx.find_cliques(nxg)):
            cl_max, cl_n = max(cl_max, len(c)), cl_n + 1
            if k_ > 20000:
                break
        ev = np.linalg.eigvalsh(A.toarray()) if n > 1 else np.zeros(1)
        if m:
            du, dv = deg[s], deg[d]
            sd_ = du.std()
            assort = float(((du - du.mean()) * (dv - dv.mean())).mean() / sd_ ** 2) if sd_ > 1e-9 else 0.0
        else:
            assort = 0.0
        q = np.quantile(deg, [0.1, 0.25, 0.5, 0.75, 0.9]) if n else np.zeros(5)
        fixed = {"n_nodes": float(n), "n_edges": float(m), "mean_degree": float(deg.mean()) if n else 0.0,
                 "density": float(2 * m / max(n * (n - 1), 1)), "deg_std": float(deg.std()) if n else 0.0,
                 "deg_max": float(deg.max()) if n else 0.0, "frac_deg1": float((deg == 1).mean()) if n else 0.0,
                 **{f"deg_q{int(100 * p)}": float(v) for p, v in zip([0.1, 0.25, 0.5, 0.75, 0.9], q)},
                 "triangles": float(tri.sum() / 3), "transitivity": float(tri.sum() / max(pairs.sum(), 1)),
                 "avg_clustering": float(np.where(pairs > 0, tri / np.maximum(pairs, 1), 0).mean()) if n else 0.0,
                 "max_core": float(core.max()), "mean_core": float(core.mean()),
                 "max_clique": float(cl_max), "n_max_cliques": float(cl_n),
                 "n_components": float(nc), "largest_comp_frac": float(np.bincount(lab).max() / max(n, 1)),
                 "lambda_max": float(ev[-1]), "lambda_min": float(-ev[0]), "spectral_gap": float(ev[-1] - ev[-2]) if n > 1 else 0.0,
                 "assortativity": assort}
        return fixed, {}

    def build_candidates(D):
        social = D["social"]
        tag = "D5" if social else "D1"  # molhiv reuses the D1/D3 ring-candidate cache
        cache = os.path.join(CFG["data_dir"], f"{tag}_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D5] building the {'graph-statistics' if social else 'ring'} candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [(social_candidates if social else candidate_library)(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.sign(X) * np.log1p(np.abs(X))  # signed log (assortativity, lambda_min can be negative)
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def discover(D, T, Z, names, kmax, exclude=()):
        """Greedy residual pursuit. r = label minus what the trained plain network's representation T predicts;
        C_res = candidate minus what T predicts of it (both cross-fitted on the training graphs). Each step picks the
        candidate whose unlearnable part best explains the remaining residual (Theorem L: value = I(Y; S | T))."""
        tr = np.flatnonzero(D["split"] == 0)
        rng = np.random.default_rng(0)
        fold = rng.permutation(np.arange(len(tr)) % 2)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        task = "clf" if D["task"] == "clf" else "mae"
        # the residual has one column per class beyond the first: y - p for binary (as in D3), onehot - softmax for K > 2
        Kr = 1 if task != "clf" or D["K"] == 2 else D["K"]
        yhat, zhat = torch.zeros(len(tr), Kr, device=DEVICE), torch.zeros_like(Zt)
        for k in range(2):
            a = torch.tensor(np.flatnonzero(fold != k), device=DEVICE)
            b = torch.tensor(np.flatnonzero(fold == k), device=DEVICE)
            o = small_fit(Tt[a], y[a], Tt[b], task, D["K"])
            if task != "clf":
                yhat[b] = o.reshape(len(b), 1)
            else:
                yhat[b] = F_.softmax(o, -1)[:, 1:2] if Kr == 1 else F_.softmax(o, -1)
            zhat[b] = small_fit(Tt[a], Zt[a], Tt[b], "mse", 1)
        Y = (y[:, None] if Kr == 1 else F_.one_hot(y.long(), Kr).float())
        r = Y - yhat  # [n, Kr]
        Cres = Zt - zhat
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        r0 = r - r.mean(0)
        sel, steps, res = [], [], r0
        gperm = torch.Generator(device="cpu").manual_seed(123)
        rsd = lambda R: float(R.pow(2).sum(1).mean().sqrt()) + 1e-9  # = std for one column
        for _ in range(kmax):
            score = (Cc.T @ res).norm(dim=1) / len(res) / rsd(res)  # covariance with the unlearnable part
            score[sel + list(exclude)] = -1
            j = int(score.argmax())
            # permutation null for the max score: shuffle the residual rows against the candidates
            null = []
            for b0 in range(0, CFG["n_perm"], 100):
                nb = min(100, CFG["n_perm"] - b0)
                P = torch.argsort(torch.rand(len(res), nb, generator=gperm), 0).to(DEVICE)
                ns = torch.einsum("nj,nbk->jbk", Cc, res[P]).norm(dim=2) / len(res) / rsd(res)  # [J, nb]
                ns[sel + list(exclude)] = -1
                null.append(ns.max(0).values)
            null = torch.cat(null)
            pval = float((1 + (null >= score[j]).sum()) / (1 + len(null)))
            sel.append(j)
            X = Cc[:, sel]
            beta = torch.linalg.lstsq(X, r0).solution
            res = r0 - X @ beta
            steps.append({"feature": names[j], "score": float(score[j]), "r2_from_T": float(r2[j]),
                          "residual_std_after": rsd(res), "perm_p": pval, "null_q95": float(null.quantile(0.95))})
        return sel, steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"][D["name"]]))
        Zc, names = build_candidates(D)
        # phase 1: the plain network, all seeds as members; its representation drives discovery
        out0, val0, T0, te_idx = train_members(D, edge, ["none"] * len(seeds), seeds)
        sel, steps = discover(D, T0[:, 0].float(), Zc, names, CFG["kmax"])
        rng = np.random.default_rng(1)
        rand = rng.choice([j for j in range(len(names)) if j not in sel], size=min(8, len(names) - len(sel)), replace=False)
        for key, cols in [("disc1", sel[:1]), ("disc3", sel[:3]), ("disc8", sel[:8]), ("rand8", list(rand)),
                          ("allcand", list(range(len(names))))]:
            D["Z"][key] = Zc[:, cols]
        hand = "cycles" if "cycles" in D["rules"] else "size"  # the hand-made rule: ring counts (molecules), size (social)
        configs = [hand, "disc1", "disc3", "disc8", "rand8", "allcand"]
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule from L1, the discovered sets, a random-candidate placebo and the whole pool
        e1 = member_rules.index("disc1")
        out1, val1, T1, _ = train_members(D, edge, member_rules, member_seeds, t_keep=(e1,))
        # round 2: discovery again, on the representation of the network trained WITH the first pick (plus the pick
        # itself as an input). If the first pick paid the debt, what is left should look like the permutation null.
        T_aug = torch.cat([T1[:, 0].float(), torch.tensor(Zc[:, sel[:1]], device=DEVICE)], 1)
        _, steps2 = discover(D, T_aug, Zc, names, 3, exclude=sel[:1])
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows, "hand": hand,
                "x_dims": D.get("x_dims"), "hidden": CFG["hidden"], "dropout": CFG["dropout"],
                "discovered": steps, "round2": steps2, "n_candidates": len(names), "random8": [names[j] for j in rand]}

    # ---------------------------------------------------------------- driver
    # restore this stage's finished units and the feature caches, so nothing is recomputed after a reset
    hf_pull([f"{CFG['out_dir']}/units/F{L_HASH}_*", f"{CFG['data_dir']}/L1_ogbg-molhiv_full.npz", f"{CFG['data_dir']}/L1_COLLAB_full.npz",
             f"{CFG['data_dir']}/D1_ogbg-molhiv_full_cand.json", f"{CFG['data_dir']}/D5_COLLAB_full_cand.json"])
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("COLLAB", False), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D5 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"F{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        CFG.update(hidden=CFG["hidden_by"][d], batch=CFG["batch_by"][d], dropout=CFG["dropout_by"][d])
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D5_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"L1_{d}_full.npz"), os.path.join(CFG["data_dir"], f"D1_{d}_full_cand.json"),
                 os.path.join(CFG["data_dir"], f"D5_{d}_full_cand.json")],
                f"D5 unit {d} {'gine' if edge else 'gin'}")
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, discovered: "
                + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["discovered"][:3])
                + " | round 2: " + "; ".join(f"{s['feature']} (p={s['perm_p']:.3f})" for s in res["round2"][:2]))
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D5 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D5_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d5_discovery_beyond_zinc()

In [ ]:
# ==== D6. discovery beyond ZINC: verdicts (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d6_beyond_zinc_verdicts():
    """D6: reads the D5 checkpoints and tests F-a..F-c (molhiv, all OGB features) and C-a..C-c (COLLAB), fixed before
    the run. Gain = improvement over the plain network (ROC-AUC on molhiv, accuracy on COLLAB, higher is better),
    paired by seed, bootstrap CIs. The discovered set (1, 3 or 8 features) is chosen by validation only."""
    import os, sys, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D5 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["COLLAB", "ogbg-molhiv"],
        "seeds": {"COLLAB": 2 if SMOKE else 5, "ogbg-molhiv": 2 if SMOKE else 8},
        "hidden_by": {"COLLAB": 32 if SMOKE else 64, "ogbg-molhiv": 32 if SMOKE else 256},
        "batch_by": {"COLLAB": 32, "ogbg-molhiv": 128},
        "dropout_by": {"COLLAB": 0.0, "ogbg-molhiv": 0.5},  # OGB's GIN baseline uses dropout 0.5 after every layer
        "epochs_by": {"COLLAB": 6 if SMOKE else 150, "ogbg-molhiv": 6 if SMOKE else 80},
        "layers": 4,
        "n_perm": 50 if SMOKE else 1000,
        "seed_offset": 200,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
    }
    LKEY = dict(CFG, v=1, stage="D5")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)
    hf_pull([f"{CFG['out_dir']}/units/F{L_HASH}_*"])
    units = {}
    for d, fam in [("COLLAB", "gin"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"F{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units[d] = json.load(fh)
    mo.stop(not units, mo.md(f"no D5 checkpoints with hash `{L_HASH}` yet: run D5 first"))
    rng = np.random.default_rng(0)
    METRIC = {"ogbg-molhiv": "auc", "COLLAB": "acc"}

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    def paired(u, a, b, metric):
        R = pd.DataFrame(u["rows"])
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci((A - B.loc[A.index]).values)

    table, picked = [], {}
    for d, u in units.items():
        R = pd.DataFrame(u["rows"])
        metric = METRIC[d]
        base = R[R.config == "none"].set_index("seed")
        picked[d] = min(["disc1", "disc3", "disc8"], key=lambda c: R[R.config == c].val.mean())  # validation only
        for c, g in R.groupby("config"):
            g = g.set_index("seed")
            m, lo, hi = ci((g[metric] - base.loc[g.index, metric]).values) if c != "none" else (0.0, 0.0, 0.0)
            row = {"dataset": d, "config": c, "picked_by_val": c == picked[d], metric: float(g[metric].mean()),
                   f"{metric}_sd": float(g[metric].std()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi}
            if "nll" in g:
                row["nll"] = float(g["nll"].mean())
            table.append(row)
    T = pd.DataFrame(table)

    def size_coord(f):
        return f in ("n_nodes", "n_edges", "mean_degree", "density", "L1:walks[0]") or f.startswith("L1:size[")

    V, notes = {}, []
    if "ogbg-molhiv" in units:
        u = units["ogbg-molhiv"]
        k = picked["ogbg-molhiv"]
        r = T[(T.dataset == "ogbg-molhiv") & (T.config == k)].iloc[0]
        b = T[(T.dataset == "ogbg-molhiv") & (T.config == "none")].iloc[0]
        V["F-a molhiv: the plain GINE with all OGB features reaches test ROC-AUC >= 0.755"] = {
            "holds": bool(b.auc >= 0.755), "detail": f"plain GINE {b.auc:.4f} +- {b.auc_sd:.4f} over {len(u['rows']) // 7} seeds "
                                                   f"(published OGB GIN 0.7558 +- 0.0140, GIN-virtual 0.7707 +- 0.0149)"}
        V["F-b molhiv: the discovered set (picked by validation) improves test ROC-AUC, CI above 0"] = {
            "holds": bool(r.lo > 0), "detail": f"{k}: {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]; "
                                              f"picks {[s['feature'] for s in u['discovered'][:3]]}"}
        s1, s2 = u["discovered"][0], u["round2"][0]
        V["F-c molhiv: first pick significant (p <= 0.005) and round 2 certifies (p > 0.01)"] = {
            "holds": bool(s1["perm_p"] <= 0.005 and s2["perm_p"] > 0.01),
            "detail": f"round 1 {s1['feature']} score {s1['score']:.3f} vs null q95 {s1['null_q95']:.3f}, p={s1['perm_p']:.4f}; "
                      f"round 2 {s2['feature']} score {s2['score']:.3f}, p={s2['perm_p']:.4f}"}
        for other in [u.get("hand", "cycles"), "rand8", "allcand"]:
            m, lo, hi = paired(u, k, other, "auc")
            notes.append(f"molhiv {k} vs {other}: {m:+.4f} AUC [{lo:+.4f}, {hi:+.4f}]")
    if "COLLAB" in units:
        u = units["COLLAB"]
        k = picked["COLLAB"]
        r = T[(T.dataset == "COLLAB") & (T.config == k)].iloc[0]
        b = T[(T.dataset == "COLLAB") & (T.config == "none")].iloc[0]
        top3 = [s["feature"] for s in u["discovered"][:3]]
        V["C-a COLLAB: the top-3 discovered features include a graph-size coordinate"] = {
            "holds": bool(any(size_coord(f) for f in top3)),
            "detail": "; ".join(f"{s['feature']} (p={s['perm_p']:.4f}, R2 from T {s['r2_from_T']:.2f})" for s in u["discovered"][:3])}
        V["C-b COLLAB: the discovered set (picked by validation) improves test accuracy, CI above 0"] = {
            "holds": bool(r.lo > 0), "detail": f"plain GIN {b.acc:.4f} +- {b.acc_sd:.4f} (GIN paper, 10-fold CV: 0.802 +- 0.019); "
                                              f"{k}: {r.acc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]"}
        s2 = u["round2"][0]
        V["C-c COLLAB certificate: nothing significant left after round 1 (round-2 p > 0.01)"] = {
            "holds": bool(s2["perm_p"] > 0.01), "detail": f"round 2 top {s2['feature']} score {s2['score']:.3f}, p={s2['perm_p']:.4f}"}
        for other in [u.get("hand", "size"), "rand8", "allcand"]:
            m, lo, hi = paired(u, k, other, "acc")
            notes.append(f"COLLAB {k} vs {other}: {m:+.4f} accuracy [{lo:+.4f}, {hi:+.4f}]")
    disc = pd.DataFrame([{"dataset": d, "round": rd, "step": i + 1, **s}
                         for d, u in units.items() for rd, key in [(1, "discovered"), (2, "round2")] for i, s in enumerate(u[key])])
    T.to_csv(os.path.join(CFG["out_dir"], "beyond_zinc_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "beyond_zinc_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "beyond_zinc_verdicts.json"), "w") as fh:
        json.dump({"verdicts": V, "comparisons": notes}, fh, indent=1)
    hf_push([os.path.join(CFG["out_dir"], f) for f in ["beyond_zinc_rows.csv", "beyond_zinc_features.csv", "beyond_zinc_verdicts.json"]],
            "D6 verdicts")
    return mo.vstack([
        mo.md("### D6 verdicts: discovery beyond ZINC\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Discovered set against the hand rule, random-8 and the whole pool** (paired by seed)\n" + "\n".join(f"- {n}" for n in notes)),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Picked features, both rounds, with permutation p-values**"),
        mo.ui.table(disc.round(4), page_size=40, selection=None),
    ])


_d6_beyond_zinc_verdicts()

In [ ]:
# ==== D7. group-level discovery from out-of-fold errors, on ZINC, molhiv and COLLAB (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d7_block_discovery():
    """D7: discovery v2. Two changes from D3/D5, each aimed at a D6 failure:
    (1) out-of-fold debt: fold networks trained on half of the training graphs are scored on the other half, so the
        residual is what the network fails to GENERALISE (I(Y;S|T) on unseen graphs), not what it fails to fit;
    (2) rule families as blocks: every candidate column and every family (all cycle counts, all ring sizes, the
        spectrum, ...) is a block; a block's statistic is the residual energy in its span, standardised by its own
        permutation null, and the pick's p-value is Westfall-Young max-T adjusted over all blocks.
    Fresh seeds (offset 300). ZINC GINE, ogbg-molhiv GINE (all OGB features) and COLLAB GIN. Safe to re-run."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["COLLAB", "ZINC", "ogbg-molhiv"],
        "seeds": {"COLLAB": 2 if SMOKE else 5, "ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 8},
        "fold_seeds": 2,  # seeds of the half-data fold networks (their out-of-fold outputs are averaged)
        "hidden_by": {"COLLAB": 32 if SMOKE else 64, "ZINC": 32 if SMOKE else 128, "ogbg-molhiv": 32 if SMOKE else 256},
        "batch_by": {"COLLAB": 32, "ZINC": 128, "ogbg-molhiv": 128},
        "dropout_by": {"COLLAB": 0.0, "ZINC": 0.0, "ogbg-molhiv": 0.5},  # OGB's GIN baseline uses dropout 0.5 after every layer
        "epochs_by": {"COLLAB": 6 if SMOKE else 150, "ZINC": 6 if SMOKE else 200, "ogbg-molhiv": 6 if SMOKE else 80},
        "layers": 4,
        "n_perm": 50 if SMOKE else 1000,
        "n_blocks": 3,
        "seed_offset": 300,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
    }
    LKEY = dict(CFG, v=1, stage="D7")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)
    CFG.update(hidden=CFG["hidden_by"]["COLLAB"], batch=32, dropout=0.0)  # set per unit by the driver (not hashed)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        if SMOKE:
            raw = [raw[i] for i in np.random.default_rng(0).permutation(len(raw))[:300]]
        deg_tags = len(tagset) == 1  # social graphs carry no node labels: use the degree (capped) as the tag, as GIN does
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            if deg_tags:
                tags = np.minimum(np.asarray(A.sum(1)).ravel(), CFG["max_deg_tag"]).astype(np.int64)
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": CFG["max_deg_tag"] + 1 if deg_tags else len(tagset), "n_bond": 0}

    def load_molhiv():
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name="ogbg-molhiv", root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            # t / bond (atom and bond type) drive the rule and candidate libraries, as in D1/D3 (same caches);
            # xf / bf (all 9 atom and 3 bond features) feed the network
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0]),
                           "xf": data.x.numpy().astype(np.int64).reshape(-1, data.x.shape[1]),
                           "bf": data.edge_attr.numpy().astype(np.int64).reshape(-1, data.edge_attr.shape[1])})
        try:  # OGB's fixed vocabularies (what AtomEncoder / BondEncoder use)
            from ogb.utils.features import get_atom_feature_dims, get_bond_feature_dims
            x_dims, b_dims = list(get_atom_feature_dims()), list(get_bond_feature_dims())
        except Exception:
            x_dims = list(np.max([g["xf"].max(0) for g in graphs], 0) + 1)
            b_dims = list(np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0) + 1)
        x_dims = [int(max(a, b + 1)) for a, b in zip(x_dims, np.max([g["xf"].max(0) for g in graphs], 0))]
        b_dims = [int(max(a, b + 1)) for a, b in zip(b_dims, np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0))]
        return {"name": "ogbg-molhiv", "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5, "x_dims": x_dims, "b_dims": b_dims}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond, skip_cycles=False):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate([] if skip_cycles else nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_molhiv() if name == "ogbg-molhiv" else load_tu(name)
        social = name in ("COLLAB", "IMDB-BINARY", "IMDB-MULTI", "REDDIT-BINARY")
        if os.path.exists(cache):
            z = np.load(cache)
            raw = {r: z[r] for r in RULES}
        else:
            print(f"[D7] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"], social) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        drop = ({"bond_hist"} if D["n_bond"] == 0 else set()) | ({"cycles", "ring_atoms"} if social else set())
        RULES_D = [r for r in RULES if r not in drop]
        D["rules"] = RULES_D
        D["social"] = social
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            b = {"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                 "idx": torch.tensor(sel, device=DEVICE)}
            if "xf" in gs[0]:
                b["xf"] = torch.tensor(np.concatenate([g["xf"] for g in gs]), device=DEVICE)
                b["bf"] = torch.tensor(np.concatenate([g["bf"] for g in gs]), device=DEVICE)
            # without edge features the GIN sum is one sparse product (no per-edge message tensor on dense social graphs)
            b["A"] = torch.sparse_coo_tensor(torch.stack([dst, src]), torch.ones(len(src), device=DEVICE),
                                             (b["N"], b["N"])).coalesce()
            out.append(b)
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            # one embedding table per atom (bond) feature column, summed, as OGB's AtomEncoder (BondEncoder)
            self.xd = D.get("x_dims", [D["n_types"]])
            self.bd = D.get("b_dims", [max(D["n_bond"], 1)])
            self.xo, self.bo = np.cumsum([0] + self.xd)[:-1].tolist(), np.cumsum([0] + self.bd)[:-1].tolist()
            self.emb = [torch.stack([torch.randn(sum(self.xd), dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(sum(self.bd), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            if "xf" in B:
                h = sum(self.emb[0][:, B["xf"][:, c] + o] for c, o in enumerate(self.xo)).transpose(0, 1)  # [N, E, dh]
            else:
                h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                if self.edge:
                    if "bf" in B:
                        be = sum(self.bemb[0][:, B["bf"][:, c] + o, li] for c, o in enumerate(self.bo))
                    else:
                        be = self.bemb[0][:, B["bond"], li]
                    msg = F_.relu(h[B["src"]] + be.transpose(0, 1))
                    z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                else:
                    z = h + torch.sparse.mm(B["A"], h.reshape(B["N"], -1)).view_as(h)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                if li < self.L - 1:
                    h = F_.dropout(h, CFG["dropout"], training)
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None, t_keep=(0,), fit_mask=None, keep_idx=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule.
        fit_mask [n_graphs, E]: member e takes its loss only on training graphs with fit_mask[:, e] (fold networks);
        keep_idx: graphs whose outputs are frozen at each member's best validation epoch (out-of-fold predictions)."""
        epochs = epochs or CFG["epochs_by"][D["name"]]
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        FM = torch.tensor(fit_mask, device=DEVICE, dtype=torch.float32) if fit_mask is not None else None
        Bkp = make_batches(D, keep_idx, 1024, None) if keep_idx is not None else []
        best_keep = torch.zeros(len(keep_idx) if keep_idx is not None else 0, E, D["K"] if D["task"] == "clf" else 1, device=DEVICE)
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                if FM is None:
                    loss_fn(out, B["idx"]).sum().backward()
                else:
                    m = FM[B["idx"]]
                    ((loss_fn(out, B["idx"], False) * m).sum(0) / m.sum(0).clamp_min(1)).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D7] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"] == "ogbg-molhiv" and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
                    if Bkp:
                        outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bkp])
                        best_keep[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            keep = torch.tensor(list(t_keep), device=DEVICE)
            Ts = torch.zeros(len(D["graphs"]), len(keep), (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1][:, keep]
        return best_out, best, Ts, te, best_keep

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def social_candidates(g):
        """Generic graph statistics for graphs without chemistry: size, density, degree shape, triangles and
        clustering, cores, cliques (collaboration ego-networks are unions of per-paper cliques), components, spectrum,
        assortativity. Many are not 1-WL invariant (triangles, cliques, cores, spectrum)."""
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz / 2
        tri = np.asarray((A @ A).multiply(A).sum(1)).ravel() / 2
        pairs = deg * (deg - 1) / 2
        nc, lab = conn_comp(A, directed=False)
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        core = np.array(list(nx.core_number(nxg).values())) if m else np.zeros(1)
        cl_max, cl_n = 0, 0
        for k_, c in enumerate(nx.find_cliques(nxg)):
            cl_max, cl_n = max(cl_max, len(c)), cl_n + 1
            if k_ > 20000:
                break
        ev = np.linalg.eigvalsh(A.toarray()) if n > 1 else np.zeros(1)
        if m:
            du, dv = deg[s], deg[d]
            sd_ = du.std()
            assort = float(((du - du.mean()) * (dv - dv.mean())).mean() / sd_ ** 2) if sd_ > 1e-9 else 0.0
        else:
            assort = 0.0
        q = np.quantile(deg, [0.1, 0.25, 0.5, 0.75, 0.9]) if n else np.zeros(5)
        fixed = {"n_nodes": float(n), "n_edges": float(m), "mean_degree": float(deg.mean()) if n else 0.0,
                 "density": float(2 * m / max(n * (n - 1), 1)), "deg_std": float(deg.std()) if n else 0.0,
                 "deg_max": float(deg.max()) if n else 0.0, "frac_deg1": float((deg == 1).mean()) if n else 0.0,
                 **{f"deg_q{int(100 * p)}": float(v) for p, v in zip([0.1, 0.25, 0.5, 0.75, 0.9], q)},
                 "triangles": float(tri.sum() / 3), "transitivity": float(tri.sum() / max(pairs.sum(), 1)),
                 "avg_clustering": float(np.where(pairs > 0, tri / np.maximum(pairs, 1), 0).mean()) if n else 0.0,
                 "max_core": float(core.max()), "mean_core": float(core.mean()),
                 "max_clique": float(cl_max), "n_max_cliques": float(cl_n),
                 "n_components": float(nc), "largest_comp_frac": float(np.bincount(lab).max() / max(n, 1)),
                 "lambda_max": float(ev[-1]), "lambda_min": float(-ev[0]), "spectral_gap": float(ev[-1] - ev[-2]) if n > 1 else 0.0,
                 "assortativity": assort}
        return fixed, {}

    def build_candidates(D):
        social = D["social"]
        tag = "D5" if social else "D1"  # molhiv reuses the D1/D3 ring-candidate cache
        cache = os.path.join(CFG["data_dir"], f"{tag}_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D7] building the {'graph-statistics' if social else 'ring'} candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [(social_candidates if social else candidate_library)(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.sign(X) * np.log1p(np.abs(X))  # signed log (assortativity, lambda_min can be negative)
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def crossfit(T, Zt, task="mse", K=1, y=None):
        """2-fold cross-fitted prediction from T (of Zt, or of labels y when task='clf'/'mae')."""
        fold = torch.tensor(np.random.default_rng(0).permutation(np.arange(len(T)) % 2), device=DEVICE)
        tgt = Zt if y is None else y
        out = None
        for k in range(2):
            a, b = torch.nonzero(fold != k)[:, 0], torch.nonzero(fold == k)[:, 0]
            o = small_fit(T[a], tgt[a], T[b], task, K)
            if out is None:
                out = torch.zeros(len(T), o.shape[1], device=DEVICE)
            out[b] = o.reshape(len(b), -1)
        return out

    def residual_target(D, y):
        """Residual coordinates: y for regression and binary tasks, one-hot for K > 2 classes."""
        Kr = 1 if D["task"] != "clf" or D["K"] == 2 else D["K"]
        return (y[:, None] if Kr == 1 else F_.one_hot(y.long(), Kr).float()), Kr

    def insample_inputs(D, T, Z):
        """D3/D5 inputs: residual of a probe fitted on the plain network's own training graphs (fit debt)."""
        tr = np.flatnonzero(D["split"] == 0)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        Y, Kr = residual_target(D, y)
        if D["task"] == "clf":
            p = F_.softmax(crossfit(Tt, None, "clf", D["K"], y), -1)
            yhat = p[:, 1:2] if Kr == 1 else p
        else:
            yhat = crossfit(Tt, None, "mae", 1, y)
        return Y - yhat, Zt - crossfit(Tt, Zt), Zt

    def oof_inputs(D, out_keep, Tf, fold_tr, Z, extra=None):
        """Out-of-fold inputs: out_keep [n_tr, F, K] best-epoch outputs of the F fold members on every training
        graph, Tf [n_tr, d] the representation from the member that did NOT train on that graph. The residual is
        label minus the averaged out-of-fold prediction; the candidate's unlearnable part is fitted within each fold
        (the two fold networks have different coordinates)."""
        tr = np.flatnonzero(D["split"] == 0)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Y, Kr = residual_target(D, y)
        Zt = torch.tensor(Z[tr], device=DEVICE)
        if D["task"] == "clf":
            p = F_.softmax(out_keep, -1).mean(1)
            yhat = p[:, 1:2] if Kr == 1 else p
        else:
            yhat = out_keep[..., 0].mean(1, keepdim=True)
        Cres = torch.zeros_like(Zt)
        for k in range(2):
            b = torch.tensor(np.flatnonzero(fold_tr == k), device=DEVICE)
            Tk = Tf[b] if extra is None else torch.cat([Tf[b], torch.tensor(extra[tr][fold_tr == k], device=DEVICE)], 1)
            Cres[b] = Zt[b] - crossfit(Tk, Zt[b])
        return Y - yhat, Cres, Zt

    def make_blocks(names):
        """Every candidate column is a block, and so is every family of columns (an L1 rule, all ring sizes, ...)."""
        famof = {"n_rings": "ring_summary", "max_ring_len": "ring_summary", "ring_excess_over_6": "ring_summary",
                 "rings_gt6": "ring_summary", "atoms_in_2plus_rings": "ring_fusion", "ring_atoms_frac": "ring_fusion",
                 "fused_pairs": "ring_fusion", "spiro_pairs": "ring_fusion", "ring_systems": "ring_fusion",
                 "max_rings_per_system": "ring_fusion", "n_nodes": "size", "n_edges": "size", "mean_degree": "size",
                 "density": "size", "deg_std": "degree_shape", "deg_max": "degree_shape", "frac_deg1": "degree_shape",
                 "triangles": "triangles_clustering", "transitivity": "triangles_clustering",
                 "avg_clustering": "triangles_clustering", "max_core": "cores_cliques", "mean_core": "cores_cliques",
                 "max_clique": "cores_cliques", "n_max_cliques": "cores_cliques", "n_components": "components",
                 "largest_comp_frac": "components", "lambda_max": "spectrum_stats", "lambda_min": "spectrum_stats",
                 "spectral_gap": "spectrum_stats"}
        fam = {}
        for j, n in enumerate(names):
            if n.startswith("L1:"):
                f = n.split("[")[0]
            elif ":atoms" in n:
                f = "ring_atom_patterns"
            elif ":bonds" in n:
                f = "ring_bond_patterns"
            elif n.startswith("ring") and n[4:].isdigit():
                f = "ring_sizes"
            elif n.startswith("deg_q"):
                f = "degree_shape"
            else:
                f = famof.get(n)
            if f:
                fam.setdefault(f, []).append(j)
        return [(n, [j]) for j, n in enumerate(names)] + [(f"family:{f}", js) for f, js in fam.items() if len(js) > 1]

    def orth(X, tol=1e-5):
        U, S, _ = torch.linalg.svd(X, full_matrices=False)
        return U[:, S > tol * max(float(S[0]), 1e-12)] if len(S) else X[:, :0]

    def discover_blocks(R, Cres, Zt, names, blocks, kmax, exclude=()):
        """Greedy block pursuit with a calibrated statistic. For block b with orthonormal basis Q_b of its centred
        unlearnable part, E_b = ||Q_b^T res||^2 / n. Each block's permutation null (shuffling the residual rows) is
        moment-matched to a scaled chi-square, which turns E_b into a calibrated surprise s_b = -log p_b that is
        comparable across block sizes (a raw z favours 1-column blocks, whose null is heavy-tailed). The pick is
        argmax s_b; its p-value is Westfall-Young: how often the max over blocks of the permuted s reaches the
        observed one. After a pick, the residual is projected off the picked span."""
        from scipy.stats import chi2
        n = len(R)
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        Cc = Cc / Cc.norm(dim=0).clamp_min(1e-9)
        Qs, seg = [], []
        for bi, (_, js) in enumerate(blocks):
            Q = orth(Cc[:, js])
            Qs.append(Q)
            seg += [bi] * Q.shape[1]
        Qall, seg = torch.cat(Qs, 1), torch.tensor(seg, device=DEVICE)
        nB = len(blocks)
        r0 = R - R.mean(0)
        S = torch.zeros(n, 0, device=DEVICE)
        res = r0
        gperm = torch.Generator(device="cpu").manual_seed(123)
        sel, steps = list(exclude), []
        for i in exclude:
            S = orth(torch.cat([S, Qs[i] - S @ (S.T @ Qs[i])], 1))
        res = r0 - S @ (S.T @ r0)

        def energy(Rm):  # Rm [n, nb, K] -> [nB, nb]
            G = torch.einsum("nd,nbk->dbk", Qall, Rm).pow(2).sum(-1)
            return torch.zeros(nB, G.shape[1], device=DEVICE).index_add_(0, seg, G) / n

        for _ in range(kmax):
            obs = energy(res[:, None, :])[:, 0]
            null = []
            for b0 in range(0, CFG["n_perm"], 100):
                nb = min(100, CFG["n_perm"] - b0)
                P = torch.argsort(torch.rand(n, nb, generator=gperm), 0).to(DEVICE)
                Rp = res[P]  # [n, nb, K]
                if S.shape[1]:
                    Rp = Rp - torch.einsum("ns,sbk->nbk", S, torch.einsum("ns,nbk->sbk", S, Rp))
                null.append(energy(Rp))
            null = torch.cat(null, 1)  # [nB, n_perm]
            mu, var = null.mean(1).clamp_min(1e-12), null.var(1).clamp_min(1e-24)
            z = (obs - mu) / var.sqrt()
            nu, a = (2 * mu ** 2 / var).clamp(0.2, 1e6).cpu().numpy(), (var / (2 * mu)).cpu().numpy()
            sobs = -chi2.logsf(obs.cpu().numpy() / a, nu)
            snull = -chi2.logsf(null.cpu().numpy() / a[:, None], nu[:, None])
            sobs[sel], snull[sel] = -np.inf, -np.inf
            j = int(np.argmax(sobs))
            p_adj = float((1 + (snull.max(0) >= sobs[j]).sum()) / (1 + null.shape[1]))
            p_raw = float((1 + (null[j] >= obs[j]).sum()) / (1 + null.shape[1]))
            sel.append(j)
            S = orth(torch.cat([S, Qs[j] - S @ (S.T @ Qs[j])], 1))
            res = r0 - S @ (S.T @ r0)
            js = blocks[j][1]
            steps.append({"block": blocks[j][0], "features": [names[c] for c in js], "cols": js, "dim": len(js),
                          "z": float(z[j]), "log10_p_chi2": float(-sobs[j] / np.log(10)), "energy_over_null": float(obs[j] / mu[j].clamp_min(1e-12)),
                          "p_adj": p_adj, "p_raw": p_raw, "r2_from_T": float(r2[js].mean()),
                          "residual_rms_after": float(res.pow(2).sum(1).mean().sqrt())})
        return steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"][D["name"]]))
        fs = seeds[:CFG["fold_seeds"]]
        Zc, names = build_candidates(D)
        blocks = make_blocks(names)
        tr = np.flatnonzero(D["split"] == 0)
        fold = np.full(len(D["graphs"]), -1)
        fold[tr] = np.random.default_rng(0).permutation(np.arange(len(tr)) % 2)
        fold_tr = fold[tr]

        def fold_members(base):  # members (seed s, fold k): train on the training graphs outside fold k
            ms = [(s, k) for s in fs for k in range(2)]
            mask = np.ones((len(D["graphs"]), base + len(ms)), bool)
            for e, (_, k) in enumerate(ms):
                mask[:, base + e] = fold != k
            return ms, mask

        def oof(out_keep, Ts, base, ms):  # pick, for every training graph, the members that did not see it
            ft = torch.tensor(fold_tr, device=DEVICE)
            by_fold = torch.tensor([[base + e for e, (_, k) in enumerate(ms) if k == f] for f in range(2)], device=DEVICE)
            sel = by_fold[ft]  # [n_tr, fold seeds]
            ok = torch.gather(out_keep, 1, sel[:, :, None].expand(-1, -1, out_keep.shape[-1]))
            return ok, Ts[torch.tensor(tr, device=DEVICE), ft]

        # phase 1: plain networks on all training graphs (the baseline) + half-data fold networks
        ms, mask = fold_members(len(seeds))
        rules1 = ["none"] * (len(seeds) + len(ms))
        seeds1 = seeds + [s for s, _ in ms]
        t_keep = [0] + [len(seeds) + e for e, (s, _) in enumerate(ms) if s == fs[0]]  # full member, then fold 0, fold 1
        out0, val0, T0, te_idx, keep0 = train_members(D, edge, rules1, seeds1, fit_mask=mask, keep_idx=tr, t_keep=t_keep)
        ok, Tf = oof(keep0, T0[:, 1:], len(seeds), ms)
        R, Cres, Zt = oof_inputs(D, ok, Tf.float(), fold_tr, Zc)
        steps = discover_blocks(R, Cres, Zt, names, blocks, CFG["n_blocks"])
        Ri, Ci, Zi = insample_inputs(D, T0[:, 0].float(), Zc)  # the D3/D5 statistic, for comparison
        steps_in = discover_blocks(Ri, Ci, Zi, names, blocks, 1)
        cols = []
        cfg_cols = {}
        for k, s in enumerate(steps):
            cols += [c for c in s["cols"] if c not in cols]
            cfg_cols[f"blk{k + 1}"] = list(cols)
        rng = np.random.default_rng(1)
        rest = [j for j in range(len(names)) if j not in cols]
        cfg_cols["randcols"] = list(rng.choice(rest, size=min(len(cols), len(rest)), replace=False))  # matched in size
        cfg_cols["allcand"] = list(range(len(names)))
        for key, cs in cfg_cols.items():
            D["Z"][key] = Zc[:, cs]
        hand = "cycles" if "cycles" in D["rules"] else "size"  # the hand-made rule: ring counts (molecules), size (social)
        configs = [hand] + list(cfg_cols)
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule, the discovered blocks, a size-matched random set, the whole pool, and fold networks
        # given the first block (round 2: is anything left to generalise once the first block is supplied?)
        base = len(member_rules)
        ms2, mask2 = fold_members(base)
        member_rules += ["blk1"] * len(ms2)
        member_seeds += [s for s, _ in ms2]
        t_keep2 = [base + e for e, (s, _) in enumerate(ms2) if s == fs[0]]
        out1, val1, T1, _, keep1 = train_members(D, edge, member_rules, member_seeds, fit_mask=mask2, keep_idx=tr, t_keep=t_keep2)
        ok2, Tf2 = oof(keep1, T1, base, ms2)
        R2, Cres2, Zt2 = oof_inputs(D, ok2, Tf2.float(), fold_tr, Zc, extra=Zc[:, cfg_cols["blk1"]])
        first = [bi for bi, b in enumerate(blocks) if b[0] == steps[0]["block"]]
        steps2 = discover_blocks(R2, Cres2, Zt2, names, blocks, 2, exclude=first)
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules[:base], member_seeds[:base]))]
        half = [{"config": f"half_fold{k}", "seed": s, "val": float(val0[len(seeds) + e]),
                 **test_metric(D, out0[:, len(seeds) + e], te_idx)} for e, (s, k) in enumerate(ms)]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows, "half_data_rows": half,
                "hand": hand, "hidden": CFG["hidden"], "dropout": CFG["dropout"], "n_candidates": len(names), "n_blocks": len(blocks),
                "discovered": steps, "discovered_insample": steps_in, "round2": steps2,
                "config_features": {k: [names[j] for j in v] for k, v in cfg_cols.items() if k != "allcand"}}

    # ---------------------------------------------------------------- driver
    # restore this stage's finished units and the feature caches, so nothing is recomputed after a reset
    hf_pull([f"{CFG['out_dir']}/units/K{L_HASH}_*"] + [f"{CFG['data_dir']}/L1_{d}_full.npz" for d in CFG["datasets"]]
            + [f"{CFG['data_dir']}/D1_ZINC_full_cand.json", f"{CFG['data_dir']}/D1_ogbg-molhiv_full_cand.json",
               f"{CFG['data_dir']}/D5_COLLAB_full_cand.json", f"{CFG['data_dir']}/ZINC/*"])
    t0 = time.time()
    lines, fails, done = [], [], 0
    units = [("COLLAB", False), ("ZINC", True), ("ogbg-molhiv", True)]
    D, cur = None, None
    for d, edge in mo.status.progress_bar(units, title="D7 units (dataset x family)"):
        path = os.path.join(CFG["out_dir"], "units", f"K{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            done += 1
            continue
        ts = time.time()
        CFG.update(hidden=CFG["hidden_by"][d], batch=CFG["batch_by"][d], dropout=CFG["dropout_by"][d])
        try:
            if cur != d:
                D = None
                gc.collect()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D7_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"L1_{d}_full.npz"), os.path.join(CFG["data_dir"], f"D1_{d}_full_cand.json"),
                 os.path.join(CFG["data_dir"], f"D5_{d}_full_cand.json")],
                f"D7 unit {d} {'gine' if edge else 'gin'}")
        done += 1
        line = (f"{time.strftime('%H:%M:%S')} {d} {'gine' if edge else 'gin'}: {time.time() - ts:.0f}s, blocks: "
                + "; ".join(f"{s['block']} (z={s['z']:.1f}, p_adj={s['p_adj']:.3f})" for s in res["discovered"])
                + " | round 2: " + "; ".join(f"{s['block']} (p_adj={s['p_adj']:.3f})" for s in res["round2"])
                + f" | in-sample pick: {res['discovered_insample'][0]['block']} (z={res['discovered_insample'][0]['z']:.1f})")
        lines.append(line)
        print(line, flush=True)
    return mo.vstack([
        mo.md(f"**D7 units ready: {done} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
              + (f", **{len(fails)} failed** (tracebacks in `D7_failures.log`)" if fails else ", no failures")),
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` edge={b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
    ])


_d7_block_discovery()

In [ ]:
# ==== D8. group-level discovery from out-of-fold errors: verdicts (self-contained: all names live inside one function; restores from and syncs to Hugging Face) ====
def _d8_block_discovery_verdicts():
    """D8: reads the D7 checkpoints and tests K-a..K-f, fixed before the run. Gain = improvement over the plain
    network (MAE on ZINC: lower is better; ROC-AUC on molhiv and accuracy on COLLAB: higher is better), paired by
    seed, bootstrap CIs. The discovered set (first 1, 2 or 3 blocks) is chosen by validation only."""
    import os, sys, json, hashlib
    import numpy as np
    import pandas as pd
    import marimo as mo

    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {  # must match D7 exactly (it sets the checkpoint hash)
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["COLLAB", "ZINC", "ogbg-molhiv"],
        "seeds": {"COLLAB": 2 if SMOKE else 5, "ZINC": 2 if SMOKE else 3, "ogbg-molhiv": 2 if SMOKE else 8},
        "fold_seeds": 2,  # seeds of the half-data fold networks (their out-of-fold outputs are averaged)
        "hidden_by": {"COLLAB": 32 if SMOKE else 64, "ZINC": 32 if SMOKE else 128, "ogbg-molhiv": 32 if SMOKE else 256},
        "batch_by": {"COLLAB": 32, "ZINC": 128, "ogbg-molhiv": 128},
        "dropout_by": {"COLLAB": 0.0, "ZINC": 0.0, "ogbg-molhiv": 0.5},  # OGB's GIN baseline uses dropout 0.5 after every layer
        "epochs_by": {"COLLAB": 6 if SMOKE else 150, "ZINC": 6 if SMOKE else 200, "ogbg-molhiv": 6 if SMOKE else 80},
        "layers": 4,
        "n_perm": 50 if SMOKE else 1000,
        "n_blocks": 3,
        "seed_offset": 300,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
    }
    LKEY = dict(CFG, v=1, stage="D7")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)
    hf_pull([f"{CFG['out_dir']}/units/K{L_HASH}_*"])
    units = {}
    for d, fam in [("COLLAB", "gin"), ("ZINC", "gine"), ("ogbg-molhiv", "gine")]:
        p = os.path.join(CFG["out_dir"], "units", f"K{L_HASH}_{d}_{fam}.json")
        if os.path.exists(p):
            with open(p) as fh:
                units[d] = json.load(fh)
    mo.stop(not units, mo.md(f"no D7 checkpoints with hash `{L_HASH}` yet: run D7 first"))
    rng = np.random.default_rng(0)
    METRIC = {"ZINC": ("mae", -1.0), "ogbg-molhiv": ("auc", 1.0), "COLLAB": ("acc", 1.0)}

    def ci(x):
        x = np.asarray(x, float)
        b = [rng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    def paired(d, a, b):
        R = pd.DataFrame(units[d]["rows"])
        metric, sign = METRIC[d]
        A, B = R[R.config == a].set_index("seed")[metric], R[R.config == b].set_index("seed")[metric]
        return ci(sign * (A - B.loc[A.index]).values)

    table, picked = [], {}
    for d, u in units.items():
        R = pd.DataFrame(u["rows"])
        metric, sign = METRIC[d]
        base = R[R.config == "none"].set_index("seed")
        blk = [c for c in ["blk1", "blk2", "blk3"] if c in set(R.config)]
        picked[d] = min(blk, key=lambda c: R[R.config == c].val.mean())  # validation only
        for c, g in R.groupby("config"):
            g = g.set_index("seed")
            m, lo, hi = ci(sign * (g[metric] - base.loc[g.index, metric]).values) if c != "none" else (0.0, 0.0, 0.0)
            table.append({"dataset": d, "config": c, "picked_by_val": c == picked[d], metric: float(g[metric].mean()),
                          "sd": float(g[metric].std()), "base": float(base[metric].mean()), "gain": m, "lo": lo, "hi": hi,
                          "features": "; ".join(u["config_features"].get(c, []))[:200]})
    T = pd.DataFrame(table)

    def big_ring(f):
        return f in ("max_ring_len", "ring_excess_over_6", "rings_gt6") or (
            f.startswith("ring") and f[4:].split(":")[0].isdigit() and int(f[4:].split(":")[0]) > 6)

    def row(d, c):
        return T[(T.dataset == d) & (T.config == c)].iloc[0]

    V, notes = {}, []
    if "ZINC" in units:
        s = units["ZINC"]["discovered"][0]
        V["K-a ZINC: the first block contains the hidden large-ring term, Westfall-Young p <= 0.005"] = {
            "holds": bool(any(big_ring(f) for f in s["features"]) and s["p_adj"] <= 0.005),
            "detail": f"{s['block']} ({s['dim']} columns), log10 p {s['log10_p_chi2']:.1f}, p_adj={s['p_adj']:.4f}"}
        k = picked["ZINC"]
        m, lo, hi = paired("ZINC", k, "cycles")
        V["K-b ZINC: the discovered set beats the hand cycle rule"] = {
            "holds": bool(lo > 0), "detail": f"{k} {row('ZINC', k).mae:.4f} vs cycles {row('ZINC', 'cycles').mae:.4f} MAE: "
                                            f"{m:+.4f} better [{lo:+.4f}, {hi:+.4f}]"}
    if "ogbg-molhiv" in units:
        k = picked["ogbg-molhiv"]
        r = row("ogbg-molhiv", k)
        m, lo, hi = paired("ogbg-molhiv", k, "cycles")
        V["K-c molhiv: the discovered set beats the plain GINE (CI above 0) and is not worse than the hand cycle rule"] = {
            "holds": bool(r.lo > 0 and hi >= 0),
            "detail": f"plain {r.base:.4f}; {k} {r.auc:.4f}, gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]; "
                      f"vs cycles {row('ogbg-molhiv', 'cycles').auc:.4f}: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]; "
                      f"blocks {[s['block'] for s in units['ogbg-molhiv']['discovered']]}"}
    if "COLLAB" in units:
        s = units["COLLAB"]["discovered"][0]
        k = picked["COLLAB"]
        r = row("COLLAB", k)
        V["K-d COLLAB: the certificate reports no debt (p_adj > 0.01) and indeed the discovered set's gain CI includes 0"] = {
            "holds": bool(s["p_adj"] > 0.01 and r.lo <= 0 <= r.hi),
            "detail": f"first block {s['block']} p_adj={s['p_adj']:.4f}; plain {r.base:.4f}, {k} {r.acc:.4f}, "
                      f"gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]"}
    r2 = [(d, units[d]["round2"][0]) for d in ("ZINC", "ogbg-molhiv") if d in units]
    V["K-e round-2 certificate (ZINC, molhiv): nothing significant left once the first block is supplied (p_adj > 0.01)"] = {
        "holds": bool(r2 and all(s["p_adj"] > 0.01 for _, s in r2)),
        "detail": "; ".join(f"{d}: {s['block']} p_adj={s['p_adj']:.4f}" for d, s in r2)}
    if "ogbg-molhiv" in units:
        so, si = units["ogbg-molhiv"]["discovered"][0], units["ogbg-molhiv"]["discovered_insample"][0]
        V["K-f molhiv: out-of-fold errors give stronger evidence than in-sample errors (lower log10 p for the first pick)"] = {
            "holds": bool(so["log10_p_chi2"] < si["log10_p_chi2"]),
            "detail": f"out-of-fold {so['block']} log10 p {so['log10_p_chi2']:.1f}; in-sample {si['block']} log10 p {si['log10_p_chi2']:.1f}"}
    # descriptive: does the certificate predict where rules pay?
    for d, u in units.items():
        s = u["discovered"][0]
        k = picked[d]
        r = row(d, k)
        notes.append(f"{d}: first block p_adj={s['p_adj']:.4f} -> {k} gain {r.gain:+.4f} [{r.lo:+.4f}, {r.hi:+.4f}]")
        for other in [u["hand"], "randcols", "allcand"]:
            m, lo, hi = paired(d, k, other)
            notes.append(f"{d} {k} vs {other}: {m:+.4f} [{lo:+.4f}, {hi:+.4f}]")
        H = pd.DataFrame(u["half_data_rows"])
        if len(H):
            metric, _ = METRIC[d]
            notes.append(f"{d} half-data plain networks: {metric} {H[metric].mean():.4f} (full data {row(d, k).base:.4f})")
    disc = pd.DataFrame([{"dataset": d, "round": rd, "step": i + 1, **{k_: v for k_, v in s.items() if k_ not in ("cols",)}}
                         for d, u in units.items()
                         for rd, key in [("oof", "discovered"), ("in-sample", "discovered_insample"), ("round 2", "round2")]
                         for i, s in enumerate(u[key])])
    disc["features"] = disc["features"].map(lambda f: "; ".join(f)[:200])
    T.to_csv(os.path.join(CFG["out_dir"], "block_rows.csv"), index=False)
    disc.to_csv(os.path.join(CFG["out_dir"], "block_features.csv"), index=False)
    with open(os.path.join(CFG["out_dir"], "block_verdicts.json"), "w") as fh:
        json.dump({"verdicts": V, "comparisons": notes}, fh, indent=1)
    hf_push([os.path.join(CFG["out_dir"], f) for f in ["block_rows.csv", "block_features.csv", "block_verdicts.json"]], "D8 verdicts")
    return mo.vstack([
        mo.md("### D8 verdicts: group-level discovery from out-of-fold errors\n" + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Certificate vs gain, and the discovered set against the hand rule, a size-matched random set and the whole pool** (paired by seed)\n" + "\n".join(f"- {n}" for n in notes)),
        mo.md("**Results** (gain > 0 means better than the plain network)"),
        mo.ui.table(T.round(4), page_size=30, selection=None),
        mo.md("**Picked blocks: out-of-fold, in-sample (for comparison) and round 2**"),
        mo.ui.table(disc.round(4), page_size=40, selection=None),
    ])


_d8_block_discovery_verdicts()

In [ ]:
# ==== D9. confirmation on unseen datasets: the frozen D7 discovery method + verdicts, in ONE standalone cell (all imports, downloads, caches and Hugging Face sync are inside; it reads nothing from any other cell) ====
def _d9_frozen_confirmation():
    """D9: the discovery method exactly as in D7 (out-of-fold debt, rule-family blocks, chi-square-calibrated
    Westfall-Young p, validation-only choice of 1-3 blocks), run unchanged on five datasets it has never seen:
    ogbg-molbace, ogbg-molbbbp (scaffold splits, ROC-AUC), PROTEINS, IMDB-BINARY, REDDIT-BINARY (accuracy).
    Network recipes are the D7 ones: molecules use the molhiv recipe, TU graphs the COLLAB recipe. Fresh seeds
    (offset 400), 10 per dataset. The checks Z-a..Z-e at the end were fixed before the run. Safe to re-run: each
    finished dataset is saved locally and on the Hub, and is restored instead of recomputed."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": ["ogbg-molbace", "ogbg-molbbbp", "PROTEINS", "IMDBBINARY", "REDDITBINARY"],
        "seeds": {d: 2 if SMOKE else 10 for d in ["ogbg-molbace", "ogbg-molbbbp", "PROTEINS", "IMDBBINARY", "REDDITBINARY"]},
        "fold_seeds": 2,  # seeds of the half-data fold networks (their out-of-fold outputs are averaged)
        # molecules: the molhiv recipe (OGB-style, dropout 0.5) with batch 32 for these small sets; TU: the COLLAB recipe
        "hidden_by": {"ogbg-molbace": 32 if SMOKE else 256, "ogbg-molbbbp": 32 if SMOKE else 256,
                      "PROTEINS": 32 if SMOKE else 64, "IMDBBINARY": 32 if SMOKE else 64, "REDDITBINARY": 32 if SMOKE else 64},
        "batch_by": {"ogbg-molbace": 32, "ogbg-molbbbp": 32, "PROTEINS": 32, "IMDBBINARY": 32, "REDDITBINARY": 32},
        "dropout_by": {"ogbg-molbace": 0.5, "ogbg-molbbbp": 0.5, "PROTEINS": 0.0, "IMDBBINARY": 0.0, "REDDITBINARY": 0.0},
        "epochs_by": {"ogbg-molbace": 6 if SMOKE else 100, "ogbg-molbbbp": 6 if SMOKE else 100,
                      "PROTEINS": 6 if SMOKE else 150, "IMDBBINARY": 6 if SMOKE else 150, "REDDITBINARY": 6 if SMOKE else 150},
        "layers": 4,
        "n_perm": 50 if SMOKE else 1000,
        "n_blocks": 3,
        "seed_offset": 400,
        "ring_bound": 12,
        "max_patterns": 150,
        "kmax": 8,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
    }
    LKEY = dict(CFG, v=1, stage="D9")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)
    CFG.update(hidden=64, batch=32, dropout=0.0)  # set per unit by the driver (not hashed)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        if SMOKE:
            raw = [raw[i] for i in np.random.default_rng(0).permutation(len(raw))[:300]]
        deg_tags = len(tagset) == 1  # social graphs carry no node labels: use the degree (capped) as the tag, as GIN does
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            if deg_tags:
                tags = np.minimum(np.asarray(A.sum(1)).ravel(), CFG["max_deg_tag"]).astype(np.int64)
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": CFG["max_deg_tag"] + 1 if deg_tags else len(tagset), "n_bond": 0}

    def load_ogb(name):
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name=name, root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            # t / bond (atom and bond type) drive the rule and candidate libraries, as in D1/D3 (same caches);
            # xf / bf (all 9 atom and 3 bond features) feed the network
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0]),
                           "xf": data.x.numpy().astype(np.int64).reshape(-1, data.x.shape[1]),
                           "bf": data.edge_attr.numpy().astype(np.int64).reshape(-1, data.edge_attr.shape[1])})
        try:  # OGB's fixed vocabularies (what AtomEncoder / BondEncoder use)
            from ogb.utils.features import get_atom_feature_dims, get_bond_feature_dims
            x_dims, b_dims = list(get_atom_feature_dims()), list(get_bond_feature_dims())
        except Exception:
            x_dims = list(np.max([g["xf"].max(0) for g in graphs], 0) + 1)
            b_dims = list(np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0) + 1)
        x_dims = [int(max(a, b + 1)) for a, b in zip(x_dims, np.max([g["xf"].max(0) for g in graphs], 0))]
        b_dims = [int(max(a, b + 1)) for a, b in zip(b_dims, np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0))]
        return {"name": name, "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5, "x_dims": x_dims, "b_dims": b_dims}

    def load_zinc():
        try:
            import torch_geometric  # noqa: F401
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"])
        from torch_geometric.datasets import ZINC
        graphs, split = [], []
        for k, sp in enumerate(["train", "val", "test"]):
            ds = ZINC(os.path.join(CFG["data_dir"], "ZINC"), subset=True, split=sp)
            if SMOKE:
                ds = ds[:200]
            for data in ds:
                n = int(data.num_nodes)
                ei = data.edge_index.numpy().astype(np.int64)
                graphs.append({"n": n, "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                               "bond": data.edge_attr.numpy().astype(np.int64) - 1, "y": float(data.y)})
                split.append(k)
        return {"name": "ZINC", "graphs": graphs, "split": np.array(split), "task": "reg", "K": 1,
                "n_types": 28, "n_bond": 3}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond, skip_cycles=False):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate([] if skip_cycles else nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"D9_L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_zinc() if name == "ZINC" else load_ogb(name) if name.startswith("ogbg-") else load_tu(name)
        social = name in ("COLLAB", "IMDBBINARY", "IMDBMULTI", "REDDITBINARY", "REDDITMULTI5K")  # dense or huge: no cycle enumeration
        raw = None
        if os.path.exists(cache):
            z = np.load(cache)
            if all(r in z.files for r in RULES) and len(z[RULES[0]]) == len(D["graphs"]):
                raw = {r: z[r] for r in RULES}
        if raw is None:
            print(f"[D9] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"], social) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        tr = D["split"] == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on the training graphs
            X = np.log1p(np.maximum(raw[r], 0)) if r in COUNT_RULES else raw[r]
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)
        drop = ({"bond_hist"} if D["n_bond"] == 0 else set()) | ({"cycles", "ring_atoms"} if social else set())
        RULES_D = [r for r in RULES if r not in drop]
        D["rules"] = RULES_D
        D["social"] = social
        D["stat_cand"] = not name.startswith("ogbg-") and name != "ZINC"  # generic graph statistics unless molecules
        return D

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            b = {"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                 "idx": torch.tensor(sel, device=DEVICE)}
            if "xf" in gs[0]:
                b["xf"] = torch.tensor(np.concatenate([g["xf"] for g in gs]), device=DEVICE)
                b["bf"] = torch.tensor(np.concatenate([g["bf"] for g in gs]), device=DEVICE)
            # without edge features the GIN sum is one sparse product (no per-edge message tensor on dense social graphs)
            b["A"] = torch.sparse_coo_tensor(torch.stack([dst, src]), torch.ones(len(src), device=DEVICE),
                                             (b["N"], b["N"])).coalesce()
            out.append(b)
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            # one embedding table per atom (bond) feature column, summed, as OGB's AtomEncoder (BondEncoder)
            self.xd = D.get("x_dims", [D["n_types"]])
            self.bd = D.get("b_dims", [max(D["n_bond"], 1)])
            self.xo, self.bo = np.cumsum([0] + self.xd)[:-1].tolist(), np.cumsum([0] + self.bd)[:-1].tolist()
            self.emb = [torch.stack([torch.randn(sum(self.xd), dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(sum(self.bd), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            if "xf" in B:
                h = sum(self.emb[0][:, B["xf"][:, c] + o] for c, o in enumerate(self.xo)).transpose(0, 1)  # [N, E, dh]
            else:
                h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                if self.edge:
                    if "bf" in B:
                        be = sum(self.bemb[0][:, B["bf"][:, c] + o, li] for c, o in enumerate(self.bo))
                    else:
                        be = self.bemb[0][:, B["bond"], li]
                    msg = F_.relu(h[B["src"]] + be.transpose(0, 1))
                    z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                else:
                    z = h + torch.sparse.mm(B["A"], h.reshape(B["N"], -1)).view_as(h)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                if li < self.L - 1:
                    h = F_.dropout(h, CFG["dropout"], training)
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None, t_keep=(0,), fit_mask=None, keep_idx=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule.
        fit_mask [n_graphs, E]: member e takes its loss only on training graphs with fit_mask[:, e] (fold networks);
        keep_idx: graphs whose outputs are frozen at each member's best validation epoch (out-of-fold predictions)."""
        epochs = epochs or CFG["epochs_by"][D["name"]]
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        FM = torch.tensor(fit_mask, device=DEVICE, dtype=torch.float32) if fit_mask is not None else None
        Bkp = make_batches(D, keep_idx, 1024, None) if keep_idx is not None else []
        best_keep = torch.zeros(len(keep_idx) if keep_idx is not None else 0, E, D["K"] if D["task"] == "clf" else 1, device=DEVICE)
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                if FM is None:
                    loss_fn(out, B["idx"]).sum().backward()
                else:
                    m = FM[B["idx"]]
                    ((loss_fn(out, B["idx"], False) * m).sum(0) / m.sum(0).clamp_min(1)).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[D9] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"].startswith("ogbg-") and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
                    if Bkp:
                        outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bkp])
                        best_keep[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            keep = torch.tensor(list(t_keep), device=DEVICE)
            Ts = torch.zeros(len(D["graphs"]), len(keep), (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1][:, keep]
        return best_out, best, Ts, te, best_keep

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- candidate space for rule discovery
    def candidate_library(g):
        """Ring (chordless-cycle) statistics, the chemist's notion of a ring: sizes, atom and bond compositions,
        fusion. Pattern counts are keyed by strings so the top patterns can be chosen on the training graphs."""
        import networkx as nx
        s, d, t = g["src"], g["dst"], g["t"]
        nxg = nx.Graph()
        nxg.add_nodes_from(range(g["n"]))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        bond = {}
        if g["bond"] is not None:
            for a_, b_, k_ in zip(s.tolist(), d.tolist(), g["bond"].tolist()):
                bond[(a_, b_)] = k_
        rings = []
        for k_, c in enumerate(nx.chordless_cycles(nxg, length_bound=CFG["ring_bound"])):
            rings.append(c)
            if k_ > 5000:
                break
        lens = [len(c) for c in rings]
        fixed = {f"ring{k}": float(sum(1 for x in lens if x == k)) for k in range(3, CFG["ring_bound"] + 1)}
        fixed["n_rings"] = float(len(rings))
        fixed["max_ring_len"] = float(max(lens) if lens else 0)
        fixed["ring_excess_over_6"] = float(max(0, (max(lens) if lens else 0) - 6))
        fixed["rings_gt6"] = float(sum(1 for x in lens if x > 6))
        member = np.zeros(g["n"])
        esets = []
        for c in rings:
            member[c] += 1
            esets.append({tuple(sorted((c[i], c[(i + 1) % len(c)]))) for i in range(len(c))})
        fixed["atoms_in_2plus_rings"] = float((member >= 2).sum())
        fixed["ring_atoms_frac"] = float((member > 0).mean()) if g["n"] else 0.0
        fused = spiro = 0
        parent = list(range(len(rings)))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        for i in range(len(rings)):
            for j in range(i + 1, len(rings)):
                if esets[i] & esets[j]:
                    fused += 1
                    parent[find(i)] = find(j)
                elif set(rings[i]) & set(rings[j]):
                    spiro += 1
        roots = [find(i) for i in range(len(rings))]
        sizes = np.bincount(np.unique(roots, return_inverse=True)[1]) if rings else np.zeros(1)
        fixed.update(fused_pairs=float(fused), spiro_pairs=float(spiro), ring_systems=float(len(set(roots))),
                     max_rings_per_system=float(sizes.max()))
        pats = {}
        for c in rings:
            key = f"ring{len(c)}:atoms" + ",".join(map(str, sorted(t[c].tolist())))
            pats[key] = pats.get(key, 0) + 1
            if bond:
                bt = sorted(bond.get((c[i], c[(i + 1) % len(c)]), -1) for i in range(len(c)))
                key = f"ring{len(c)}:bonds" + ",".join(map(str, bt))
                pats[key] = pats.get(key, 0) + 1
        return fixed, pats

    def social_candidates(g):
        """Generic graph statistics for graphs without chemistry: size, density, degree shape, triangles and
        clustering, cores, cliques (collaboration ego-networks are unions of per-paper cliques), components, spectrum,
        assortativity. Many are not 1-WL invariant (triangles, cliques, cores, spectrum)."""
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz / 2
        tri = np.asarray((A @ A).multiply(A).sum(1)).ravel() / 2
        pairs = deg * (deg - 1) / 2
        nc, lab = conn_comp(A, directed=False)
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        core = np.array(list(nx.core_number(nxg).values())) if m else np.zeros(1)
        cl_max, cl_n = 0, 0
        for k_, c in enumerate(nx.find_cliques(nxg)):
            cl_max, cl_n = max(cl_max, len(c)), cl_n + 1
            if k_ > 20000:
                break
        ev = np.linalg.eigvalsh(A.toarray()) if n > 1 else np.zeros(1)
        if m:
            du, dv = deg[s], deg[d]
            sd_ = du.std()
            assort = float(((du - du.mean()) * (dv - dv.mean())).mean() / sd_ ** 2) if sd_ > 1e-9 else 0.0
        else:
            assort = 0.0
        q = np.quantile(deg, [0.1, 0.25, 0.5, 0.75, 0.9]) if n else np.zeros(5)
        fixed = {"n_nodes": float(n), "n_edges": float(m), "mean_degree": float(deg.mean()) if n else 0.0,
                 "density": float(2 * m / max(n * (n - 1), 1)), "deg_std": float(deg.std()) if n else 0.0,
                 "deg_max": float(deg.max()) if n else 0.0, "frac_deg1": float((deg == 1).mean()) if n else 0.0,
                 **{f"deg_q{int(100 * p)}": float(v) for p, v in zip([0.1, 0.25, 0.5, 0.75, 0.9], q)},
                 "triangles": float(tri.sum() / 3), "transitivity": float(tri.sum() / max(pairs.sum(), 1)),
                 "avg_clustering": float(np.where(pairs > 0, tri / np.maximum(pairs, 1), 0).mean()) if n else 0.0,
                 "max_core": float(core.max()), "mean_core": float(core.mean()),
                 "max_clique": float(cl_max), "n_max_cliques": float(cl_n),
                 "n_components": float(nc), "largest_comp_frac": float(np.bincount(lab).max() / max(n, 1)),
                 "lambda_max": float(ev[-1]), "lambda_min": float(-ev[0]), "spectral_gap": float(ev[-1] - ev[-2]) if n > 1 else 0.0,
                 "assortativity": assort}
        return fixed, {}

    def build_candidates(D):
        social = D["stat_cand"]  # ring candidates for molecules, generic graph statistics otherwise
        cache = os.path.join(CFG["data_dir"], f"D9_{D['name']}_{'smoke' if SMOKE else 'full'}_cand.json")
        if os.path.exists(cache):
            with open(cache) as fh:
                rows = json.load(fh)
        else:
            print(f"[D9] building the {'graph-statistics' if social else 'ring'} candidate space for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [(social_candidates if social else candidate_library)(g) for g in D["graphs"]]
            with open(cache, "w") as fh:
                json.dump(rows, fh)
        tr = D["split"] == 0
        names = list(rows[0][0].keys())
        freq = {}
        for (fx, pt), is_tr in zip(rows, tr):
            if is_tr:
                for k in pt:
                    freq[k] = freq.get(k, 0) + 1
        top = [k for k, v in sorted(freq.items(), key=lambda kv: -kv[1]) if v >= 20][: CFG["max_patterns"]]
        X = np.array([[fx[k] for k in names] + [pt.get(k, 0) for k in top] for fx, pt in rows], np.float64)
        names = names + top
        # the L1 library columns join the pool too: discovery must learn to skip what the network already knows
        for r in D["rules"]:
            if r == "noise":
                continue
            X = np.concatenate([X, D["raw"][r].reshape(len(X), -1)], 1)
            names += [f"L1:{r}[{j}]" for j in range(D["raw"][r].reshape(len(X), -1).shape[1])]
        X = np.sign(X) * np.log1p(np.abs(X))  # signed log (assortativity, lambda_min can be negative)
        mu, sd = X[tr].mean(0), X[tr].std(0)
        keep = sd > 1e-8
        Z = ((X[:, keep] - mu[keep]) / sd[keep]).clip(-6, 6).astype(np.float32)
        return Z, [n for n, k in zip(names, keep) if k]

    def crossfit(T, Zt, task="mse", K=1, y=None):
        """2-fold cross-fitted prediction from T (of Zt, or of labels y when task='clf'/'mae')."""
        fold = torch.tensor(np.random.default_rng(0).permutation(np.arange(len(T)) % 2), device=DEVICE)
        tgt = Zt if y is None else y
        out = None
        for k in range(2):
            a, b = torch.nonzero(fold != k)[:, 0], torch.nonzero(fold == k)[:, 0]
            o = small_fit(T[a], tgt[a], T[b], task, K)
            if out is None:
                out = torch.zeros(len(T), o.shape[1], device=DEVICE)
            out[b] = o.reshape(len(b), -1)
        return out

    def residual_target(D, y):
        """Residual coordinates: y for regression and binary tasks, one-hot for K > 2 classes."""
        Kr = 1 if D["task"] != "clf" or D["K"] == 2 else D["K"]
        return (y[:, None] if Kr == 1 else F_.one_hot(y.long(), Kr).float()), Kr

    def insample_inputs(D, T, Z):
        """D3/D5 inputs: residual of a probe fitted on the plain network's own training graphs (fit debt)."""
        tr = np.flatnonzero(D["split"] == 0)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Tt, Zt = T[torch.tensor(tr, device=DEVICE)], torch.tensor(Z[tr], device=DEVICE)
        Y, Kr = residual_target(D, y)
        if D["task"] == "clf":
            p = F_.softmax(crossfit(Tt, None, "clf", D["K"], y), -1)
            yhat = p[:, 1:2] if Kr == 1 else p
        else:
            yhat = crossfit(Tt, None, "mae", 1, y)
        return Y - yhat, Zt - crossfit(Tt, Zt), Zt

    def oof_inputs(D, out_keep, Tf, fold_tr, Z, extra=None):
        """Out-of-fold inputs: out_keep [n_tr, F, K] best-epoch outputs of the F fold members on every training
        graph, Tf [n_tr, d] the representation from the member that did NOT train on that graph. The residual is
        label minus the averaged out-of-fold prediction; the candidate's unlearnable part is fitted within each fold
        (the two fold networks have different coordinates)."""
        tr = np.flatnonzero(D["split"] == 0)
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in tr]), device=DEVICE).float()
        Y, Kr = residual_target(D, y)
        Zt = torch.tensor(Z[tr], device=DEVICE)
        if D["task"] == "clf":
            p = F_.softmax(out_keep, -1).mean(1)
            yhat = p[:, 1:2] if Kr == 1 else p
        else:
            yhat = out_keep[..., 0].mean(1, keepdim=True)
        Cres = torch.zeros_like(Zt)
        for k in range(2):
            b = torch.tensor(np.flatnonzero(fold_tr == k), device=DEVICE)
            Tk = Tf[b] if extra is None else torch.cat([Tf[b], torch.tensor(extra[tr][fold_tr == k], device=DEVICE)], 1)
            Cres[b] = Zt[b] - crossfit(Tk, Zt[b])
        return Y - yhat, Cres, Zt

    def make_blocks(names):
        """Every candidate column is a block, and so is every family of columns (an L1 rule, all ring sizes, ...)."""
        famof = {"n_rings": "ring_summary", "max_ring_len": "ring_summary", "ring_excess_over_6": "ring_summary",
                 "rings_gt6": "ring_summary", "atoms_in_2plus_rings": "ring_fusion", "ring_atoms_frac": "ring_fusion",
                 "fused_pairs": "ring_fusion", "spiro_pairs": "ring_fusion", "ring_systems": "ring_fusion",
                 "max_rings_per_system": "ring_fusion", "n_nodes": "size", "n_edges": "size", "mean_degree": "size",
                 "density": "size", "deg_std": "degree_shape", "deg_max": "degree_shape", "frac_deg1": "degree_shape",
                 "triangles": "triangles_clustering", "transitivity": "triangles_clustering",
                 "avg_clustering": "triangles_clustering", "max_core": "cores_cliques", "mean_core": "cores_cliques",
                 "max_clique": "cores_cliques", "n_max_cliques": "cores_cliques", "n_components": "components",
                 "largest_comp_frac": "components", "lambda_max": "spectrum_stats", "lambda_min": "spectrum_stats",
                 "spectral_gap": "spectrum_stats"}
        fam = {}
        for j, n in enumerate(names):
            if n.startswith("L1:"):
                f = n.split("[")[0]
            elif ":atoms" in n:
                f = "ring_atom_patterns"
            elif ":bonds" in n:
                f = "ring_bond_patterns"
            elif n.startswith("ring") and n[4:].isdigit():
                f = "ring_sizes"
            elif n.startswith("deg_q"):
                f = "degree_shape"
            else:
                f = famof.get(n)
            if f:
                fam.setdefault(f, []).append(j)
        return [(n, [j]) for j, n in enumerate(names)] + [(f"family:{f}", js) for f, js in fam.items() if len(js) > 1]

    def orth(X, tol=1e-5):
        U, S, _ = torch.linalg.svd(X, full_matrices=False)
        return U[:, S > tol * max(float(S[0]), 1e-12)] if len(S) else X[:, :0]

    def discover_blocks(R, Cres, Zt, names, blocks, kmax, exclude=()):
        """Greedy block pursuit with a calibrated statistic. For block b with orthonormal basis Q_b of its centred
        unlearnable part, E_b = ||Q_b^T res||^2 / n. Each block's permutation null (shuffling the residual rows) is
        moment-matched to a scaled chi-square, which turns E_b into a calibrated surprise s_b = -log p_b that is
        comparable across block sizes (a raw z favours 1-column blocks, whose null is heavy-tailed). The pick is
        argmax s_b; its p-value is Westfall-Young: how often the max over blocks of the permuted s reaches the
        observed one. After a pick, the residual is projected off the picked span."""
        from scipy.stats import chi2
        n = len(R)
        r2 = 1 - (Cres ** 2).mean(0) / Zt.var(0, unbiased=False).clamp_min(1e-6)
        Cc = Cres - Cres.mean(0)
        Cc = Cc / Cc.norm(dim=0).clamp_min(1e-9)
        Qs, seg = [], []
        for bi, (_, js) in enumerate(blocks):
            Q = orth(Cc[:, js])
            Qs.append(Q)
            seg += [bi] * Q.shape[1]
        Qall, seg = torch.cat(Qs, 1), torch.tensor(seg, device=DEVICE)
        nB = len(blocks)
        r0 = R - R.mean(0)
        S = torch.zeros(n, 0, device=DEVICE)
        res = r0
        gperm = torch.Generator(device="cpu").manual_seed(123)
        sel, steps = list(exclude), []
        for i in exclude:
            S = orth(torch.cat([S, Qs[i] - S @ (S.T @ Qs[i])], 1))
        res = r0 - S @ (S.T @ r0)

        def energy(Rm):  # Rm [n, nb, K] -> [nB, nb]
            G = torch.einsum("nd,nbk->dbk", Qall, Rm).pow(2).sum(-1)
            return torch.zeros(nB, G.shape[1], device=DEVICE).index_add_(0, seg, G) / n

        for _ in range(kmax):
            obs = energy(res[:, None, :])[:, 0]
            null = []
            for b0 in range(0, CFG["n_perm"], 100):
                nb = min(100, CFG["n_perm"] - b0)
                P = torch.argsort(torch.rand(n, nb, generator=gperm), 0).to(DEVICE)
                Rp = res[P]  # [n, nb, K]
                if S.shape[1]:
                    Rp = Rp - torch.einsum("ns,sbk->nbk", S, torch.einsum("ns,nbk->sbk", S, Rp))
                null.append(energy(Rp))
            null = torch.cat(null, 1)  # [nB, n_perm]
            mu, var = null.mean(1).clamp_min(1e-12), null.var(1).clamp_min(1e-24)
            z = (obs - mu) / var.sqrt()
            nu, a = (2 * mu ** 2 / var).clamp(0.2, 1e6).cpu().numpy(), (var / (2 * mu)).cpu().numpy()
            sobs = -chi2.logsf(obs.cpu().numpy() / a, nu)
            snull = -chi2.logsf(null.cpu().numpy() / a[:, None], nu[:, None])
            sobs[sel], snull[sel] = -np.inf, -np.inf
            j = int(np.argmax(sobs))
            p_adj = float((1 + (snull.max(0) >= sobs[j]).sum()) / (1 + null.shape[1]))
            p_raw = float((1 + (null[j] >= obs[j]).sum()) / (1 + null.shape[1]))
            sel.append(j)
            S = orth(torch.cat([S, Qs[j] - S @ (S.T @ Qs[j])], 1))
            res = r0 - S @ (S.T @ r0)
            js = blocks[j][1]
            steps.append({"block": blocks[j][0], "features": [names[c] for c in js], "cols": js, "dim": len(js),
                          "z": float(z[j]), "log10_p_chi2": float(-sobs[j] / np.log(10)), "energy_over_null": float(obs[j] / mu[j].clamp_min(1e-12)),
                          "p_adj": p_adj, "p_raw": p_raw, "r2_from_T": float(r2[js].mean()),
                          "residual_rms_after": float(res.pow(2).sum(1).mean().sqrt())})
        return steps

    # ---------------------------------------------------------------- one (dataset, family) unit
    def run_unit(D, edge):
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"][D["name"]]))
        fs = seeds[:CFG["fold_seeds"]]
        Zc, names = build_candidates(D)
        blocks = make_blocks(names)
        tr = np.flatnonzero(D["split"] == 0)
        fold = np.full(len(D["graphs"]), -1)
        fold[tr] = np.random.default_rng(0).permutation(np.arange(len(tr)) % 2)
        fold_tr = fold[tr]

        def fold_members(base):  # members (seed s, fold k): train on the training graphs outside fold k
            ms = [(s, k) for s in fs for k in range(2)]
            mask = np.ones((len(D["graphs"]), base + len(ms)), bool)
            for e, (_, k) in enumerate(ms):
                mask[:, base + e] = fold != k
            return ms, mask

        def oof(out_keep, Ts, base, ms):  # pick, for every training graph, the members that did not see it
            ft = torch.tensor(fold_tr, device=DEVICE)
            by_fold = torch.tensor([[base + e for e, (_, k) in enumerate(ms) if k == f] for f in range(2)], device=DEVICE)
            sel = by_fold[ft]  # [n_tr, fold seeds]
            ok = torch.gather(out_keep, 1, sel[:, :, None].expand(-1, -1, out_keep.shape[-1]))
            return ok, Ts[torch.tensor(tr, device=DEVICE), ft]

        # phase 1: plain networks on all training graphs (the baseline) + half-data fold networks
        ms, mask = fold_members(len(seeds))
        rules1 = ["none"] * (len(seeds) + len(ms))
        seeds1 = seeds + [s for s, _ in ms]
        t_keep = [0] + [len(seeds) + e for e, (s, _) in enumerate(ms) if s == fs[0]]  # full member, then fold 0, fold 1
        out0, val0, T0, te_idx, keep0 = train_members(D, edge, rules1, seeds1, fit_mask=mask, keep_idx=tr, t_keep=t_keep)
        ok, Tf = oof(keep0, T0[:, 1:], len(seeds), ms)
        R, Cres, Zt = oof_inputs(D, ok, Tf.float(), fold_tr, Zc)
        steps = discover_blocks(R, Cres, Zt, names, blocks, CFG["n_blocks"])
        Ri, Ci, Zi = insample_inputs(D, T0[:, 0].float(), Zc)  # the D3/D5 statistic, for comparison
        steps_in = discover_blocks(Ri, Ci, Zi, names, blocks, 1)
        cols = []
        cfg_cols = {}
        for k, s in enumerate(steps):
            cols += [c for c in s["cols"] if c not in cols]
            cfg_cols[f"blk{k + 1}"] = list(cols)
        rng = np.random.default_rng(1)
        rest = [j for j in range(len(names)) if j not in cols]
        cfg_cols["randcols"] = list(rng.choice(rest, size=min(len(cols), len(rest)), replace=False))  # matched in size
        cfg_cols["allcand"] = list(range(len(names)))
        for key, cs in cfg_cols.items():
            D["Z"][key] = Zc[:, cs]
        hand = "cycles" if "cycles" in D["rules"] else "size"  # the hand-made rule: ring counts (molecules), size (social)
        configs = [hand] + list(cfg_cols)
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        # phase 2: the hand rule, the discovered blocks, a size-matched random set, the whole pool, and fold networks
        # given the first block (round 2: is anything left to generalise once the first block is supplied?)
        base = len(member_rules)
        ms2, mask2 = fold_members(base)
        member_rules += ["blk1"] * len(ms2)
        member_seeds += [s for s, _ in ms2]
        t_keep2 = [base + e for e, (s, _) in enumerate(ms2) if s == fs[0]]
        out1, val1, T1, _, keep1 = train_members(D, edge, member_rules, member_seeds, fit_mask=mask2, keep_idx=tr, t_keep=t_keep2)
        ok2, Tf2 = oof(keep1, T1, base, ms2)
        R2, Cres2, Zt2 = oof_inputs(D, ok2, Tf2.float(), fold_tr, Zc, extra=Zc[:, cfg_cols["blk1"]])
        first = [bi for bi, b in enumerate(blocks) if b[0] == steps[0]["block"]]
        steps2 = discover_blocks(R2, Cres2, Zt2, names, blocks, 2, exclude=first)
        rows = [{"config": "none", "seed": s, "val": float(val0[e]), **test_metric(D, out0[:, e], te_idx)} for e, s in enumerate(seeds)]
        rows += [{"config": c, "seed": s, "val": float(val1[e]), **test_metric(D, out1[:, e], te_idx)}
                 for e, (c, s) in enumerate(zip(member_rules[:base], member_seeds[:base]))]
        half = [{"config": f"half_fold{k}", "seed": s, "val": float(val0[len(seeds) + e]),
                 **test_metric(D, out0[:, len(seeds) + e], te_idx)} for e, (s, k) in enumerate(ms)]
        return {"dataset": D["name"], "family": "gine" if edge else "gin", "task": D["task"], "rows": rows, "half_data_rows": half,
                "hand": hand, "hidden": CFG["hidden"], "dropout": CFG["dropout"], "n_candidates": len(names), "n_blocks": len(blocks),
                "discovered": steps, "discovered_insample": steps_in, "round2": steps2,
                "config_features": {k: [names[j] for j in v] for k, v in cfg_cols.items() if k != "allcand"}}

    # ---------------------------------------------------------------- driver
    # restore this cell's own finished datasets and feature caches from the Hub (nothing from other cells)
    hf_pull([f"{CFG['out_dir']}/units/Q{L_HASH}_*", f"{CFG['data_dir']}/D9_*_full.npz", f"{CFG['data_dir']}/D9_*_full_cand.json"])
    t0 = time.time()
    lines, fails = [], []
    units = [(d, d.startswith("ogbg-")) for d in CFG["datasets"]]  # GINE (bond features) for molecules, GIN otherwise
    results = {}
    for d, edge in mo.status.progress_bar(units, title="D9 datasets"):
        path = os.path.join(CFG["out_dir"], "units", f"Q{L_HASH}_{d}_{'gine' if edge else 'gin'}.json")
        if os.path.exists(path):
            with open(path) as fh:
                results[d] = json.load(fh)
            continue
        ts = time.time()
        CFG.update(hidden=CFG["hidden_by"][d], batch=CFG["batch_by"][d], dropout=CFG["dropout_by"][d])
        try:
            D = prepare(d)
            res = run_unit(D, edge)
        except Exception as e:
            fails.append((d, edge, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "D9_failures.log"), "a") as fh:
                fh.write(f"--- {d} edge={edge}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        finally:
            D = None
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"D9_L1_{d}_full.npz"), os.path.join(CFG["data_dir"], f"D9_{d}_full_cand.json")],
                f"D9 dataset {d}")
        results[d] = res
        line = (f"{time.strftime('%H:%M:%S')} {d}: {time.time() - ts:.0f}s, blocks: "
                + "; ".join(f"{s['block']} (p_adj={s['p_adj']:.3f})" for s in res["discovered"]))
        lines.append(line)
        print(line, flush=True)

    # ---------------------------------------------------------------- verdicts (fixed before the run)
    import pandas as pd
    head = mo.md(f"**D9 datasets ready: {len(results)} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
                 + (f", **{len(fails)} failed** (tracebacks in `D9_failures.log`)" if fails else ", no failures"))
    if not results:
        return mo.vstack([head, mo.md("\n".join(f"- `{a}`: `{c}`" for a, _, c in fails))])
    brng = np.random.default_rng(0)

    def ci(x):
        x = np.asarray(x, float)
        b = [brng.choice(x, len(x)).mean() for _ in range(4000)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    metric_of = lambda d: "auc" if d.startswith("ogbg-") else "acc"
    table, info = [], {}
    for d, u in results.items():
        R = pd.DataFrame(u["rows"])
        met = metric_of(d)
        base = R[R.config == "none"].set_index("seed")[met]
        k = min(["blk1", "blk2", "blk3"], key=lambda c: R[R.config == c].val.mean())  # validation only

        def diff(a, b):
            A, B = R[R.config == a].set_index("seed")[met], R[R.config == b].set_index("seed")[met]
            return ci((A - B.loc[A.index]).values)
        for c, g in R.groupby("config"):
            g = g.set_index("seed")[met]
            m, lo, hi = ci((g - base.loc[g.index]).values) if c != "none" else (0.0, 0.0, 0.0)
            table.append({"dataset": d, "config": c, "picked_by_val": c == k, met: float(g.mean()), "sd": float(g.std()),
                          "gain": m, "lo": lo, "hi": hi, "features": "; ".join(u["config_features"].get(c, []))[:160]})
        s, si = u["discovered"][0], u["discovered_insample"][0]
        info[d] = {"k": k, "p_adj": s["p_adj"], "gain": diff(k, "none"), "vs_hand": diff(k, u["hand"]),
                   "vs_all": diff(k, "allcand"), "vs_rand": diff(k, "randcols"), "lp_oof": s["log10_p_chi2"],
                   "lp_in": si["log10_p_chi2"], "first": s["block"], "hand": u["hand"], "metric": met}
    T = pd.DataFrame(table)
    n = len(info)
    V = {}
    V["Z-a no harm: on every dataset the discovered set is not significantly worse than the plain network"] = {
        "holds": all(i["gain"][2] >= 0 for i in info.values()),
        "detail": "; ".join(f"{d} {i['k']} {i['gain'][0]:+.4f} [{i['gain'][1]:+.4f}, {i['gain'][2]:+.4f}]" for d, i in info.items())}
    sig = {d: i for d, i in info.items() if i["p_adj"] <= 0.005}
    nul = {d: i for d, i in info.items() if i["p_adj"] > 0.05}
    V["Z-b debt predicts gain: p_adj <= 0.005 -> mean gain > 0; p_adj > 0.05 -> gain CI includes 0"] = {
        "holds": all(i["gain"][0] > 0 for i in sig.values()) and all(i["gain"][1] <= 0 <= i["gain"][2] for i in nul.values()),
        "detail": "; ".join(f"{d} first block {i['first']} p_adj={i['p_adj']:.4f} -> gain {i['gain'][0]:+.4f}" for d, i in info.items())}
    ok = sum(i["vs_hand"][2] >= 0 for i in info.values())
    V[f"Z-c matches the expert: in >= 4 of {n} the discovered set is not significantly worse than the hand rule"] = {
        "holds": ok >= min(4, n), "detail": f"{ok}/{n}: " + "; ".join(
            f"{d} vs {i['hand']} {i['vs_hand'][0]:+.4f} [{i['vs_hand'][1]:+.4f}, {i['vs_hand'][2]:+.4f}]" for d, i in info.items())}
    a_ = sum(i["vs_all"][0] > 0 for i in info.values())
    r_ = sum(i["vs_rand"][0] > 0 for i in info.values())
    V[f"Z-d selection matters: in >= 3 of {n} the discovered set beats the whole pool AND in >= 3 of {n} a size-matched random set"] = {
        "holds": a_ >= min(3, n) and r_ >= min(3, n),
        "detail": f"beats whole pool {a_}/{n}, beats random {r_}/{n}: " + "; ".join(
            f"{d} vs all {i['vs_all'][0]:+.4f}, vs random {i['vs_rand'][0]:+.4f}" for d, i in info.items())}
    e_ = sum(i["lp_oof"] < i["lp_in"] for i in info.values())
    V[f"Z-e out-of-fold evidence beats in-sample evidence (lower log10 p) in >= 4 of {n}"] = {
        "holds": e_ >= min(4, n), "detail": f"{e_}/{n}: " + "; ".join(
            f"{d} {i['lp_oof']:.1f} vs {i['lp_in']:.1f}" for d, i in info.items())}
    disc = pd.DataFrame([{"dataset": d, "round": rd, "step": j + 1, **{k_: v for k_, v in s.items() if k_ not in ("cols",)}}
                         for d, u in results.items()
                         for rd, key in [("oof", "discovered"), ("in-sample", "discovered_insample"), ("round 2", "round2")]
                         for j, s in enumerate(u[key])])
    disc["features"] = disc["features"].map(lambda f: "; ".join(f)[:160])
    disc["log10_p_chi2"] = disc["log10_p_chi2"].clip(lower=-300)
    outs = [os.path.join(CFG["out_dir"], f) for f in ["confirm_rows.csv", "confirm_features.csv", "confirm_verdicts.json"]]
    T.to_csv(outs[0], index=False)
    disc.to_csv(outs[1], index=False)
    with open(outs[2], "w") as fh:
        json.dump({"verdicts": V, "complete": len(results) == len(units), "datasets": list(results)}, fh, indent=1)
    hf_push(outs, "D9 verdicts")
    return mo.vstack([
        head,
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}`: `{c}`" for a, _, c in fails)) if fails else mo.md(""),
        mo.md("### D9 verdicts: the frozen method on unseen datasets\n"
              + ("" if len(results) == len(units) else f"*Partial: {len(results)} of {len(units)} datasets so far; rerun to finish.*\n")
              + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Results** (gain > 0 means better than the plain network; ROC-AUC for molecules, accuracy for TU)"),
        mo.ui.table(T.round(4), page_size=40, selection=None),
        mo.md("**Picked blocks: out-of-fold, in-sample (for comparison) and round 2**"),
        mo.ui.table(disc.round(4), page_size=40, selection=None),
    ])


_d9_frozen_confirmation()

In [ ]:
# ==== T1. the transport test: does symbolic knowledge pay only where learning fails to transport? ONE standalone cell (all imports, downloads, caches, Hugging Face sync and the verdicts are inside; it reads nothing from any other cell) ====
def _t1_transport_test():
    """T1. Claim (the transport principle): a trained network absorbs a rule's information on the distribution it was
    trained on, so the rule's value is I_Q(Y; S | T), the label information the rule adds to the trained network's
    representation T on the distribution Q where the model is used. In distribution (Q = P) this is the learnability
    law; under a shift it adds the information that the network learned but did not transport.

    Design: the same dataset and the same rules under two splits that differ only in shift: a random split (Q = P)
    and a shifted split (scaffold split for OGB molecules; train-small / test-large for TU graphs). Six datasets,
    eleven L1 rules each (including a pure-noise placebo), three seeds, all trained together as batched members.
    Before any rule-augmented result is looked at, one number per (dataset, split, rule) is computed from the plain
    network alone: the cross-validated log-loss improvement when the rule is added to the plain network's own logit on
    the validation graphs (which follow the test split). The checks T-a..T-f at the end were fixed before the run.
    Safe to re-run: each finished (dataset, split) is saved locally and on the Hub and restored instead of recomputed."""
    import os, json, time, math, hashlib, gc, sys, subprocess, zipfile, traceback
    import urllib.request as urlreq
    import numpy as np
    import marimo as mo
    import torch
    import scipy.sparse as sps
    from scipy.sparse.csgraph import connected_components as conn_comp

    RUN_THIS_CELL = True  # set False to keep this cell from running
    mo.stop(not RUN_THIS_CELL, mo.md("skipped: set RUN_THIS_CELL = True at the top of this cell"))
    F_ = torch.nn.functional
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    SMOKE = os.environ.get("SMOKE", "0") == "1"
    DS = ["IMDBBINARY", "PROTEINS", "ogbg-molbace", "ogbg-molbbbp", "NCI1", "ogbg-molhiv"]  # small to large
    CFG = {
        "data_dir": "nsgnn_data",
        "out_dir": "nsgnn_runs_smoke" if SMOKE else "nsgnn_runs",
        "datasets": DS,
        "arms": {d: (["random", "scaffold"] if d.startswith("ogbg-") else ["random", "size"]) for d in DS},
        "seeds": 2 if SMOKE else 3,
        "seed_offset": 500,
        # OGB molecules: the D7/D9 molecule recipe (GINE, all OGB features, dropout 0.5); molhiv at hidden 128 to keep
        # the run near an hour. TU graphs: the COLLAB recipe (GIN, hidden 64, no dropout).
        "hidden_by": {d: 32 if SMOKE else (256 if d in ("ogbg-molbace", "ogbg-molbbbp") else 128 if d == "ogbg-molhiv" else 64) for d in DS},
        "batch_by": {d: 128 if d == "ogbg-molhiv" else 32 for d in DS},
        "dropout_by": {d: 0.5 if d.startswith("ogbg-") else 0.0 for d in DS},
        "epochs_by": {d: 6 if SMOKE else (60 if d == "ogbg-molhiv" else 100 if d.startswith("ogbg-") else 150) for d in DS},
        "layers": 4,
        "lr": 1e-3,
        "max_cycle": 8,
        "max_deg_tag": 50,
        "probe_dims": 8,  # rule columns enter the probe through their top principal components (label-free, train graphs)
        "probe_folds": 5,
        "probe_reps": 3,
        "probe_l2": 1e-1,
        "train_probe_n": 3000,
    }
    LKEY = dict(CFG, v=1, stage="T1")
    LKEY.pop("out_dir")
    L_HASH = hashlib.sha1(json.dumps(LKEY, sort_keys=True).encode()).hexdigest()[:10]
    os.makedirs(os.path.join(CFG["out_dir"], "units"), exist_ok=True)
    os.makedirs(CFG["data_dir"], exist_ok=True)
    CFG.update(hidden=64, batch=32, dropout=0.0)  # set per unit by the driver (not hashed)

    # ---- Hugging Face sync: restore saved results on start, upload each result the moment it is written ----
    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    HF_SYNC = not SMOKE  # smoke runs stay local

    def hf_api():
        try:
            from huggingface_hub import HfApi
        except ImportError:
            import subprocess as _sp
            _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
            from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        return api

    def hf_pull(patterns):
        """Download matching files from the Hub that are missing locally (a kernel reset can wipe the disk)."""
        if not HF_SYNC:
            return 0
        try:
            from fnmatch import fnmatch
            api = hf_api()
            want = [f.path for f in api.list_repo_tree(HF_REPO, repo_type="model", recursive=True)
                    if getattr(f, "size", None) is not None and any(fnmatch(f.path, p) for p in patterns)
                    and not os.path.exists(f.path)]
            from huggingface_hub import hf_hub_download
            for p in want:
                hf_hub_download(HF_REPO, p, repo_type="model", token=HF_TOKEN, local_dir=".")
            if want:
                print(f"[HF] restored {len(want)} files from {HF_REPO}", flush=True)
            return len(want)
        except Exception as e:
            print(f"[HF] restore skipped: {e!r}"[:300], flush=True)
            return 0

    def hf_push(paths, msg):
        """Upload these files now; a failed upload never stops the experiment."""
        if not HF_SYNC:
            return
        paths = [p for p in paths if os.path.exists(p)]
        if not paths:
            return
        try:
            from huggingface_hub import CommitOperationAdd
            hf_api().create_commit(HF_REPO, repo_type="model", commit_message=msg,
                                   operations=[CommitOperationAdd(path_in_repo=p.replace(os.sep, "/"), path_or_fileobj=p) for p in paths])
            print(f"[HF] uploaded {len(paths)} file(s): {msg}", flush=True)
        except Exception as e:
            print(f"[HF] upload failed (kept locally, rerun to retry): {e!r}"[:300], flush=True)

    def atomic_write(path, text):
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            fh.write(text)
        os.replace(tmp, path)

    # ---------------------------------------------------------------- data: one record per graph
    _GIN_ZIP = "https://raw.githubusercontent.com/weihua916/powerful-gnns/master/dataset.zip"

    def fetch_gin_zip():
        path = os.path.join(CFG["data_dir"], "gin", "dataset.zip")
        if not os.path.exists(path):
            os.makedirs(os.path.dirname(path), exist_ok=True)
            for k in range(5):
                try:
                    with urlreq.urlopen(_GIN_ZIP, timeout=300) as r:
                        blob = r.read()
                    break
                except Exception:
                    if k == 4:
                        raise
                    time.sleep(2 ** (k + 1))
            with open(path + ".part", "wb") as fh:
                fh.write(blob)
            os.replace(path + ".part", path)
        return path

    def load_tu(name):
        z = zipfile.ZipFile(fetch_gin_zip())
        lines = z.read(f"dataset/{name}/{name}.txt").decode().split("\n")
        ptr, G = 1, int(lines[0])
        raw, tagset = [], {}
        for _ in range(G):
            n, lab = map(int, lines[ptr].split())
            ptr += 1
            tags, s, d = [], [], []
            for i in range(n):
                row = lines[ptr].split()
                ptr += 1
                tags.append(tagset.setdefault(int(row[0]), len(tagset)))
                for j in row[2:2 + int(row[1])]:
                    s.append(i)
                    d.append(int(j))
            raw.append((n, lab, np.array(tags, np.int64), np.array(s, np.int64), np.array(d, np.int64)))
        if SMOKE:
            raw = [raw[i] for i in np.random.default_rng(0).permutation(len(raw))[:300]]
        deg_tags = len(tagset) == 1  # social graphs carry no node labels: use the degree (capped) as the tag, as GIN does
        labs = {l: i for i, l in enumerate(sorted({r[1] for r in raw}))}
        graphs = []
        for n, lab, tags, s, d in raw:
            A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
            A = ((A + A.T) > 0).astype(np.float64)
            A.setdiag(0)
            A.eliminate_zeros()
            coo = A.tocoo()
            if deg_tags:
                tags = np.minimum(np.asarray(A.sum(1)).ravel(), CFG["max_deg_tag"]).astype(np.int64)
            graphs.append({"n": n, "t": tags, "src": coo.row.astype(np.int64), "dst": coo.col.astype(np.int64),
                           "bond": None, "y": labs[lab]})
        G_ = len(graphs)
        rng = np.random.default_rng(0)
        ys = np.array([g["y"] for g in graphs])
        split = np.zeros(G_, np.int64)  # stratified 80/10/10: 0 train, 1 val, 2 test (fixed across seeds)
        for c in np.unique(ys):
            idx = rng.permutation(np.flatnonzero(ys == c))
            k1, k2 = int(0.8 * len(idx)), int(0.9 * len(idx))
            split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        return {"name": name, "graphs": graphs, "split": split, "task": "clf", "K": len(labs),
                "n_types": CFG["max_deg_tag"] + 1 if deg_tags else len(tagset), "n_bond": 0}

    def load_ogb(name):
        for pkg in ["torch_geometric", "ogb"]:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        from ogb.graphproppred import PygGraphPropPredDataset
        _load = torch.load  # OGB's processed files need weights_only=False on torch >= 2.6
        torch.load = lambda *a, **k: _load(*a, **{**k, "weights_only": False})
        try:
            ds = PygGraphPropPredDataset(name=name, root=os.path.join(CFG["data_dir"], "ogb"))
            sp = ds.get_idx_split()
        finally:
            torch.load = _load
        split = np.zeros(len(ds), np.int64)
        split[sp["valid"].numpy()], split[sp["test"].numpy()] = 1, 2
        keep = np.arange(len(ds))
        if SMOKE:
            keep = np.concatenate([np.flatnonzero(split == k)[:150 if k == 0 else 60] for k in range(3)])
        graphs = []
        for i in keep:
            data = ds[int(i)]
            ei = data.edge_index.numpy().astype(np.int64)
            # t / bond (atom and bond type) drive the rule and candidate libraries, as in D1/D3 (same caches);
            # xf / bf (all 9 atom and 3 bond features) feed the network
            graphs.append({"n": int(data.num_nodes), "t": data.x[:, 0].numpy().astype(np.int64), "src": ei[0], "dst": ei[1],
                           "bond": data.edge_attr[:, 0].numpy().astype(np.int64), "y": int(data.y.view(-1)[0]),
                           "xf": data.x.numpy().astype(np.int64).reshape(-1, data.x.shape[1]),
                           "bf": data.edge_attr.numpy().astype(np.int64).reshape(-1, data.edge_attr.shape[1])})
        try:  # OGB's fixed vocabularies (what AtomEncoder / BondEncoder use)
            from ogb.utils.features import get_atom_feature_dims, get_bond_feature_dims
            x_dims, b_dims = list(get_atom_feature_dims()), list(get_bond_feature_dims())
        except Exception:
            x_dims = list(np.max([g["xf"].max(0) for g in graphs], 0) + 1)
            b_dims = list(np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0) + 1)
        x_dims = [int(max(a, b + 1)) for a, b in zip(x_dims, np.max([g["xf"].max(0) for g in graphs], 0))]
        b_dims = [int(max(a, b + 1)) for a, b in zip(b_dims, np.max([g["bf"].max(0) for g in graphs if len(g["bf"])], 0))]
        return {"name": name, "graphs": graphs, "split": split[keep], "task": "clf", "K": 2,
                "n_types": 119, "n_bond": 5, "x_dims": x_dims, "b_dims": b_dims}

    # ---------------------------------------------------------------- the rule library
    # (WL) = determined by 1-WL colours, so GIN can in principle compute it; (beyond) = not 1-WL invariant;
    # bond_hist is invisible to GIN (no edge features) but visible to GINE; noise is a placebo.
    RULES = ["size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "ring_atoms",
             "spectrum", "bond_hist", "noise"]
    COUNT_RULES = {"size", "degree_hist", "type_hist", "walks", "cyclomatic", "triangles", "cycles", "bond_hist"}

    def rule_library(g, n_types, n_bond, skip_cycles=False):
        import networkx as nx
        n, s, d = g["n"], g["src"], g["dst"]
        A = sps.csr_matrix((np.ones(len(s)), (s, d)), shape=(n, n))
        A = ((A + A.T) > 0).astype(np.float64)
        deg = np.asarray(A.sum(1)).ravel()
        m = A.nnz // 2
        comps = conn_comp(A, directed=False)[0]
        one = np.ones(n)
        w1 = A @ one
        w2 = A @ w1
        w3 = A @ w2
        A2 = A @ A
        tri_node = np.asarray(A2.multiply(A).sum(1)).ravel() / 2
        nxg = nx.Graph()
        nxg.add_nodes_from(range(n))
        nxg.add_edges_from(zip(s[s < d].tolist(), d[s < d].tolist()))
        cyc = np.zeros(CFG["max_cycle"] - 2)
        on_cycle = np.zeros(n, bool)
        for k_, c in enumerate([] if skip_cycles else nx.simple_cycles(nxg, length_bound=CFG["max_cycle"])):
            cyc[len(c) - 3] += 1
            on_cycle[c] = True
            if k_ > 20000:  # guard for unusually dense graphs
                break
        if n > 1:
            dinv = np.where(deg > 0, 1 / np.sqrt(np.maximum(deg, 1)), 0)
            ev = np.sort(np.linalg.eigvalsh((A.multiply(dinv[:, None]).multiply(dinv[None, :])).toarray()))[::-1]
        else:
            ev = np.zeros(1)
        spec = np.zeros(6)
        spec[:min(6, len(ev) - 1)] = ev[1:7]  # top non-trivial eigenvalues of the normalised adjacency
        bh = np.bincount(g["bond"], minlength=n_bond)[:n_bond] / 2 if n_bond and g["bond"] is not None else np.zeros(1)
        noise = np.random.default_rng(int(hashlib.sha1(g["t"].tobytes() + s.tobytes()).hexdigest()[:8], 16)).normal(size=4)
        return {"size": [n, m], "degree_hist": np.bincount(np.minimum(deg, 6).astype(int), minlength=7),
                "type_hist": np.bincount(g["t"], minlength=n_types)[:n_types],
                "walks": [w1.sum(), w2.sum(), w3.sum()], "cyclomatic": [m - n + comps],
                "triangles": [tri_node.sum() / 3, (tri_node > 0).sum()], "cycles": cyc, "ring_atoms": [on_cycle.mean()],
                "spectrum": spec, "bond_hist": bh, "noise": noise}

    def prepare(name):
        cache = os.path.join(CFG["data_dir"], f"T1_L1_{name}_{'smoke' if SMOKE else 'full'}.npz")
        D = load_ogb(name) if name.startswith("ogbg-") else load_tu(name)
        social = name in ("COLLAB", "IMDBBINARY", "IMDBMULTI", "REDDITBINARY", "REDDITMULTI5K")  # dense or huge: no cycle enumeration
        raw = None
        if os.path.exists(cache):
            z = np.load(cache)
            if all(r in z.files for r in RULES) and len(z[RULES[0]]) == len(D["graphs"]):
                raw = {r: z[r] for r in RULES}
        if raw is None:
            print(f"[T1] computing the L1 rule library for {len(D['graphs'])} graphs (cached after the first run)", flush=True)
            rows = [rule_library(g, D["n_types"], D["n_bond"], social) for g in D["graphs"]]
            raw = {r: np.array([np.asarray(x[r], np.float64) for x in rows]) for r in RULES}
            np.savez(cache, **raw)
        D["raw"] = raw
        D["split_native"] = D["split"].copy()
        drop = ({"bond_hist"} if D["n_bond"] == 0 else set()) | ({"cycles", "ring_atoms"} if social else set())
        RULES_D = [r for r in RULES if r not in drop]
        D["rules"] = RULES_D
        D["social"] = social
        return D

    def set_split(D, arm):
        """random: stratified by label (OGB: the scaffold split's sizes; TU: 70/15/15). scaffold: OGB's own split.
        size: within each class the smallest 70% of graphs train, the next 15% validate, the largest 15% test."""
        ys = np.array([g["y"] for g in D["graphs"]])
        ns = np.array([g["n"] for g in D["graphs"]])
        rng = np.random.default_rng(0)
        split = np.zeros(len(ys), np.int64)
        if arm == "scaffold":
            split = D["split_native"].copy()
        else:
            if D["name"].startswith("ogbg-"):
                fr = [np.mean(D["split_native"] == k) for k in range(3)]
            else:
                fr = [0.70, 0.15, 0.15]
            for c in np.unique(ys):
                idx = np.flatnonzero(ys == c)
                idx = rng.permutation(idx) if arm == "random" else idx[np.lexsort((rng.random(len(idx)), ns[idx]))]
                k1, k2 = int(round(fr[0] * len(idx))), int(round((fr[0] + fr[1]) * len(idx)))
                split[idx[k1:k2]], split[idx[k2:]] = 1, 2
        D["split"] = split
        tr = split == 0
        D["Z"] = {}
        for r in RULES:  # counts enter as log1p, everything standardised on this arm's training graphs
            X = np.log1p(np.maximum(D["raw"][r], 0)) if r in COUNT_RULES else D["raw"][r]
            X = X.reshape(len(X), -1)
            mu, sd = X[tr].mean(0), X[tr].std(0)
            keep = sd > 1e-8
            D["Z"][r] = np.clip((X[:, keep] - mu[keep]) / sd[keep], -6, 6).astype(np.float32) if keep.any() \
                else np.zeros((len(X), 1), np.float32)

    # ---------------------------------------------------------------- batching (shared by all members)
    def make_batches(D, idx, bs, rng):
        idx = rng.permutation(idx) if rng is not None else np.asarray(idx)
        out = []
        for k in range(0, len(idx), bs):
            sel = idx[k:k + bs]
            gs = [D["graphs"][i] for i in sel]
            off = np.cumsum([0] + [g["n"] for g in gs])
            t = torch.tensor(np.concatenate([g["t"] for g in gs]), device=DEVICE)
            src = torch.tensor(np.concatenate([g["src"] + o for g, o in zip(gs, off)]), device=DEVICE)
            dst = torch.tensor(np.concatenate([g["dst"] + o for g, o in zip(gs, off)]), device=DEVICE)
            bond = torch.tensor(np.concatenate([g["bond"] for g in gs]), device=DEVICE) if gs[0]["bond"] is not None else None
            gid = torch.tensor(np.concatenate([np.full(g["n"], j) for j, g in enumerate(gs)]), device=DEVICE)
            b = {"t": t, "src": src, "dst": dst, "bond": bond, "gid": gid, "G": len(gs), "N": int(off[-1]),
                 "idx": torch.tensor(sel, device=DEVICE)}
            if "xf" in gs[0]:
                b["xf"] = torch.tensor(np.concatenate([g["xf"] for g in gs]), device=DEVICE)
                b["bf"] = torch.tensor(np.concatenate([g["bf"] for g in gs]), device=DEVICE)
            # without edge features the GIN sum is one sparse product (no per-edge message tensor on dense social graphs)
            b["A"] = torch.sparse_coo_tensor(torch.stack([dst, src]), torch.ones(len(src), device=DEVICE),
                                             (b["N"], b["N"])).coalesce()
            out.append(b)
        return out

    # ---------------------------------------------------------------- batched members (as in G11)
    def uinit(fin, fout, gens):
        Ws, bs = [], []
        for g in gens:
            bound = 1.0 / math.sqrt(fin)
            Ws.append((torch.rand(fin, fout, generator=g) * 2 - 1) * bound)
            bs.append((torch.rand(fout, generator=g) * 2 - 1) * bound)
        return [torch.stack(Ws).to(DEVICE).requires_grad_(), torch.stack(bs).to(DEVICE).requires_grad_()]

    def lin(x, Wb):
        return torch.einsum("nei,eio->neo", x, Wb[0]) + Wb[1]

    class BN:
        def __init__(self, E, C):
            self.g = torch.ones(E, C, device=DEVICE, requires_grad=True)
            self.b = torch.zeros(E, C, device=DEVICE, requires_grad=True)
            self.rm, self.rv = torch.zeros(E, C, device=DEVICE), torch.ones(E, C, device=DEVICE)

        def __call__(self, x, training):
            if training:
                mu, var = x.mean(0), x.var(0, unbiased=False)
                with torch.no_grad():
                    self.rm.mul_(0.9).add_(0.1 * mu)
                    self.rv.mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.rm, self.rv
            return (x - mu) / torch.sqrt(var + 1e-5) * self.g + self.b

    class Members:
        """E independent networks trained as one: each member has its own weights, BN statistics and rule input."""

        def __init__(self, D, edge, E, out_dim, phi_dims, gens):
            dh, L = CFG["hidden"], CFG["layers"]
            self.E, self.edge, self.out_dim, self.L = E, edge, out_dim, L
            # one embedding table per atom (bond) feature column, summed, as OGB's AtomEncoder (BondEncoder)
            self.xd = D.get("x_dims", [D["n_types"]])
            self.bd = D.get("b_dims", [max(D["n_bond"], 1)])
            self.xo, self.bo = np.cumsum([0] + self.xd)[:-1].tolist(), np.cumsum([0] + self.bd)[:-1].tolist()
            self.emb = [torch.stack([torch.randn(sum(self.xd), dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()]
            self.bemb = [torch.stack([torch.randn(sum(self.bd), L, dh, generator=g) * 0.1 for g in gens]).to(DEVICE).requires_grad_()] if edge else []
            self.layers = [(uinit(dh, dh, gens), BN(E, dh), uinit(dh, dh, gens), BN(E, dh)) for _ in range(L)]
            F0 = (L + 1) * dh
            self.rbn = BN(E, F0)
            self.r1, self.r2, self.r3 = uinit(F0, dh, gens), uinit(dh, dh // 2, gens), uinit(dh // 2, out_dim, gens)
            self.P = max(1, max(phi_dims))
            self.p1, self.p2 = uinit(self.P, dh, gens), uinit(dh, out_dim, gens)
            self.on = torch.tensor([1.0 if k > 0 else 0.0 for k in phi_dims], device=DEVICE)
            ps = self.emb + self.bemb + self.r1 + self.r2 + self.r3 + self.p1 + self.p2 + [self.rbn.g, self.rbn.b]
            for (w1, b1, w2, b2) in self.layers:
                ps += w1 + w2 + [b1.g, b1.b, b2.g, b2.b]
            self.params = ps

        def embed(self, B, training):
            E, dh = self.E, CFG["hidden"]
            if "xf" in B:
                h = sum(self.emb[0][:, B["xf"][:, c] + o] for c, o in enumerate(self.xo)).transpose(0, 1)  # [N, E, dh]
            else:
                h = self.emb[0][:, B["t"]].transpose(0, 1)  # [N, E, dh]
            pool = lambda x: torch.zeros(B["G"], E, x.shape[-1], device=DEVICE).index_add_(0, B["gid"], x)
            pools = [pool(h)]
            for li, (w1, b1, w2, b2) in enumerate(self.layers):
                if self.edge:
                    if "bf" in B:
                        be = sum(self.bemb[0][:, B["bf"][:, c] + o, li] for c, o in enumerate(self.bo))
                    else:
                        be = self.bemb[0][:, B["bond"], li]
                    msg = F_.relu(h[B["src"]] + be.transpose(0, 1))
                    z = h + torch.zeros_like(h).index_add_(0, B["dst"], msg)
                else:
                    z = h + torch.sparse.mm(B["A"], h.reshape(B["N"], -1)).view_as(h)
                h = F_.relu(b1(lin(z, w1), training))
                h = F_.relu(b2(lin(h, w2), training))
                if li < self.L - 1:
                    h = F_.dropout(h, CFG["dropout"], training)
                pools.append(pool(h))
            return torch.cat(pools, -1)  # [G, E, (L+1) dh]

        def forward(self, B, PHI, training):
            T = self.embed(B, training)
            q = self.rbn(T, training)
            q = F_.dropout(F_.relu(lin(q, self.r1)), CFG["dropout"], training)
            q = F_.relu(lin(q, self.r2))
            out = lin(q, self.r3)
            return out + self.on[None, :, None] * lin(F_.relu(lin(PHI, self.p1)), self.p2), T

    def phi_tensor(D, member_rules, idx):
        P = max(1, max((D["Z"][r].shape[1] if r != "none" else 0) for r in member_rules))
        out = torch.zeros(len(idx), len(member_rules), P, device=DEVICE)
        for e, r in enumerate(member_rules):
            if r != "none":
                z = torch.tensor(D["Z"][r][idx], device=DEVICE)
                out[:, e, :z.shape[1]] = z
        return out

    def combo(D, rules):
        D["Z"]["+".join(rules)] = np.concatenate([D["Z"][r] for r in rules], 1)
        return "+".join(rules)

    def train_members(D, edge, member_rules, seeds, target="label", epochs=None, t_keep=(0,), fit_mask=None, keep_idx=None):
        """target='label': supervised task; target=<rule>: label-free probe that regresses that rule.
        fit_mask [n_graphs, E]: member e takes its loss only on training graphs with fit_mask[:, e] (fold networks);
        keep_idx: graphs whose outputs are frozen at each member's best validation epoch (out-of-fold predictions)."""
        epochs = epochs or CFG["epochs_by"][D["name"]]
        E = len(member_rules)
        gens = [torch.Generator().manual_seed(1000 * s + 17 * k + (7 if edge else 0)) for k, s in enumerate(seeds)]
        if target == "label":
            out_dim = D["K"] if D["task"] == "clf" else 1
            phi_dims = [D["Z"][r].shape[1] if r != "none" else 0 for r in member_rules]
        else:  # probe members: member k regresses rule member_rules[k]; no rule input
            out_dim = max(D["Z"][r].shape[1] for r in member_rules)
            phi_dims = [0] * E
        net = Members(D, edge, E, out_dim, phi_dims, gens)
        opt = torch.optim.Adam(net.params, lr=CFG["lr"])
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        rng = np.random.default_rng(seeds[0])
        Bva, Bte = make_batches(D, va, 1024, None), make_batches(D, te, 1024, None)
        ally = torch.tensor(np.array([g["y"] for g in D["graphs"]]), device=DEVICE)
        PHI_all = phi_tensor(D, member_rules if target == "label" else ["none"] * E, np.arange(len(D["graphs"])))
        FM = torch.tensor(fit_mask, device=DEVICE, dtype=torch.float32) if fit_mask is not None else None
        Bkp = make_batches(D, keep_idx, 1024, None) if keep_idx is not None else []
        best_keep = torch.zeros(len(keep_idx) if keep_idx is not None else 0, E, D["K"] if D["task"] == "clf" else 1, device=DEVICE)
        if target != "label":
            Y_all = torch.zeros(len(D["graphs"]), E, out_dim, device=DEVICE)
            M_all = torch.zeros(1, E, out_dim, device=DEVICE)
            for e, r in enumerate(member_rules):
                z = torch.tensor(D["Z"][r], device=DEVICE)
                Y_all[:, e, :z.shape[1]] = z
                M_all[0, e, :z.shape[1]] = 1.0

        def loss_fn(out, idx, reduce=True):
            if target == "label":
                yy = ally[idx]
                if D["task"] == "clf":
                    l = F_.cross_entropy(out.reshape(-1, out.shape[-1]), yy.long().repeat_interleave(E), reduction="none").view(-1, E)
                else:
                    l = (out[..., 0] - yy[:, None].float()).abs()
            else:
                l = (((out - Y_all[idx]) ** 2) * M_all).sum(-1) / M_all.sum(-1)
            return l.mean(0) if reduce else l  # per member

        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        best = torch.full((E,), float("inf"), device=DEVICE)
        n_te = len(te)
        best_out = torch.zeros(n_te, E, out_dim, device=DEVICE)
        best_T = None
        for ep in range(epochs):
            for B in make_batches(D, tr, CFG["batch"], rng):
                opt.zero_grad(set_to_none=True)
                out, _ = net.forward(B, PHI_all[B["idx"]], True)
                if FM is None:
                    loss_fn(out, B["idx"]).sum().backward()
                else:
                    m = FM[B["idx"]]
                    ((loss_fn(out, B["idx"], False) * m).sum(0) / m.sum(0).clamp_min(1)).sum().backward()
                opt.step()
            sched.step()
            if ep % 20 == 19 or ep == epochs - 1:
                print(f"[T1] {time.strftime('%H:%M:%S')} {D['name']} {'gine' if edge else 'gin'}: epoch {ep + 1}/{epochs}, "
                      f"{E} members, best val {float(best.min()):.4f}", flush=True)
            with torch.no_grad():
                if D["name"].startswith("ogbg-") and target == "label":  # select by validation ROC-AUC (the OGB protocol)
                    sc = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bva])
                    sc = sc[..., 1] - sc[..., 0]  # [n_va, E]
                    yv = ally[torch.cat([B["idx"] for B in Bva])].float()
                    rk = sc.argsort(0).argsort(0).float() + 1
                    npos = yv.sum()
                    vl = -((rk * yv[:, None]).sum(0) - npos * (npos + 1) / 2) / (npos * (len(yv) - npos)).clamp_min(1)
                else:
                    vl = torch.cat([loss_fn(net.forward(B, PHI_all[B["idx"]], False)[0], B["idx"], False) for B in Bva]).mean(0)
                imp = vl < best
                if bool(imp.any()):
                    best = torch.where(imp, vl, best)
                    outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bte])
                    best_out[:, imp] = outs[:, imp]
                    if Bkp:
                        outs = torch.cat([net.forward(B, PHI_all[B["idx"]], False)[0] for B in Bkp])
                        best_keep[:, imp] = outs[:, imp]
        # final representation of every graph (for the frozen-representation probe)
        with torch.no_grad():
            keep = torch.tensor(list(t_keep), device=DEVICE)
            Ts = torch.zeros(len(D["graphs"]), len(keep), (CFG["layers"] + 1) * CFG["hidden"], device=DEVICE)
            for B in make_batches(D, np.arange(len(D["graphs"])), 1024, None):
                Ts[B["idx"]] = net.forward(B, PHI_all[B["idx"]], False)[1][:, keep]
        return best_out, best, Ts, te, best_keep

    def test_metric(D, out_e, te):
        y = torch.tensor(np.array([D["graphs"][i]["y"] for i in te]), device=DEVICE)
        if D["task"] == "clf":
            lp = F_.log_softmax(out_e, -1)
            out = {"nll": float(-lp[torch.arange(len(y)), y.long()].mean()), "acc": float((lp.argmax(-1) == y).float().mean())}
            if lp.shape[-1] == 2:  # ROC-AUC via ranks (the OGB metric for molhiv)
                sc = lp[:, 1]
                r = torch.empty_like(sc)
                r[sc.argsort()] = torch.arange(1, len(sc) + 1, device=sc.device, dtype=sc.dtype)
                npos = float(y.sum())
                out["auc"] = float((r[y == 1].sum() - npos * (npos + 1) / 2) / max(npos * (len(y) - npos), 1))
            return out
        return {"mae": float((out_e[:, 0] - y).abs().mean())}

    def small_fit(Xtr, ytr, Xte, task, K, steps=300):
        """2-layer MLP on fixed features (frozen-representation probes and residual-information scores)."""
        g = torch.Generator(device="cpu").manual_seed(0)
        mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-4
        Xtr, Xte = ((Xtr - mu) / sd).clamp(-10, 10), ((Xte - mu) / sd).clamp(-10, 10)  # dead units cannot explode
        d_out = K if task == "clf" else ytr.shape[1] if ytr.dim() > 1 else 1
        W1 = (torch.randn(Xtr.shape[1], 64, generator=g) / math.sqrt(Xtr.shape[1])).to(DEVICE).requires_grad_()
        W2 = (torch.randn(64, d_out, generator=g) / 8).to(DEVICE).requires_grad_()
        b1, b2 = torch.zeros(64, device=DEVICE, requires_grad=True), torch.zeros(d_out, device=DEVICE, requires_grad=True)
        opt = torch.optim.Adam([W1, W2, b1, b2], lr=3e-3, weight_decay=1e-4)
        f = lambda X: F_.relu(X @ W1 + b1) @ W2 + b2
        lossf = lambda o, y: F_.cross_entropy(o, y.long()) if task == "clf" else (o.squeeze(-1) - y).abs().mean() \
            if task == "mae" else ((o - y) ** 2).mean()
        stop = torch.randperm(len(Xtr), generator=g)[: max(len(Xtr) // 5, 2)].to(Xtr.device)  # early-stopping split
        fit = torch.ones(len(Xtr), dtype=torch.bool, device=Xtr.device)
        fit[stop] = False
        best, pred = float("inf"), None
        for k in range(steps):
            opt.zero_grad()
            lossf(f(Xtr[fit]), ytr[fit]).backward()
            opt.step()
            if k % 10 == 9:
                with torch.no_grad():
                    vl = float(lossf(f(Xtr[stop]), ytr[stop]))
                    if vl < best:
                        best, pred = vl, f(Xte)
        return pred

    # ---------------------------------------------------------------- the predictor: I_Q(Y; S | T) on validation graphs
    def logreg_cv_nll(X, y, seed0=0):
        """Cross-validated log-loss (nats per graph) of an L2 logistic regression of binary y on X (X may have 0
        columns: intercept only). Stratified folds, repeated."""
        n = len(y)
        X = torch.tensor(X, dtype=torch.float64)
        yt = torch.tensor(y, dtype=torch.float64)
        tot = 0.0
        for rep_ in range(CFG["probe_reps"]):
            g = np.random.default_rng(seed0 + rep_)
            f = np.zeros(n, np.int64)
            for c in (0, 1):
                idx = g.permutation(np.flatnonzero(y == c))
                f[idx] = np.arange(len(idx)) % CFG["probe_folds"]
            nll = 0.0
            for k in range(CFG["probe_folds"]):
                a, b = torch.tensor(f != k), torch.tensor(f == k)
                if int(b.sum()) == 0:
                    continue
                ya = yt[a]
                if X.shape[1] == 0:
                    p = ya.mean().clamp(1e-4, 1 - 1e-4)
                    nll += float(-(yt[b] * torch.log(p) + (1 - yt[b]) * torch.log(1 - p)).sum())
                    continue
                mu, sd = X[a].mean(0), X[a].std(0).clamp_min(1e-6)
                Xa, Xb = (X[a] - mu) / sd, (X[b] - mu) / sd
                w = torch.zeros(X.shape[1], dtype=torch.float64, requires_grad=True)
                p0 = float(ya.mean().clamp(1e-4, 1 - 1e-4))
                b0 = torch.tensor(math.log(p0 / (1 - p0)), dtype=torch.float64, requires_grad=True)
                opt = torch.optim.LBFGS([w, b0], lr=0.5, max_iter=100, line_search_fn="strong_wolfe")

                def closure():
                    opt.zero_grad()
                    loss = F_.binary_cross_entropy_with_logits(Xa @ w + b0, ya) + CFG["probe_l2"] * (w ** 2).sum()
                    loss.backward()
                    return loss
                opt.step(closure)
                with torch.no_grad():
                    nll += float(F_.binary_cross_entropy_with_logits(Xb @ w + b0, yt[b], reduction="sum"))
            tot += nll / n
        return tot / CFG["probe_reps"]

    def rule_proj(D, r, tr):
        """Top principal directions of a rule's standardised columns on this arm's training graphs (label-free)."""
        Z = D["Z"][r]
        if Z.shape[1] <= CFG["probe_dims"]:
            return Z
        U, S_, Vt = np.linalg.svd(Z[tr] - Z[tr].mean(0), full_matrices=False)
        return (Z - Z[tr].mean(0)) @ Vt[:CFG["probe_dims"]].T

    def ridge_r2(X, Y, lam=1.0):
        """2-fold cross-fitted ridge R^2 of Y from X (pooled over Y's columns): how much of the rule T already holds."""
        n = len(X)
        f = np.random.default_rng(0).permutation(np.arange(n) % 2)
        pred = np.zeros_like(Y)
        for k in range(2):
            a, b = f != k, f == k
            mx, my = X[a].mean(0), Y[a].mean(0)
            Xa = X[a] - mx
            Wt = np.linalg.solve(Xa.T @ Xa + lam * np.eye(X.shape[1]), Xa.T @ (Y[a] - my))
            pred[b] = (X[b] - mx) @ Wt + my
        return float(1 - ((Y - pred) ** 2).sum() / max(((Y - Y.mean(0)) ** 2).sum(), 1e-9))

    # ---------------------------------------------------------------- one (dataset, split) unit
    def run_unit(D, edge, arm):
        set_split(D, arm)
        seeds = list(range(CFG["seed_offset"], CFG["seed_offset"] + CFG["seeds"]))
        rules = list(D["rules"])  # the L1 library, the noise placebo included
        configs = ["none"] + rules
        member_rules = [c for c in configs for _ in seeds]
        member_seeds = [s for _ in configs for s in seeds]
        tr, va, te = (np.flatnonzero(D["split"] == k) for k in range(3))
        trs = np.sort(np.random.default_rng(0).choice(tr, size=min(len(tr), CFG["train_probe_n"]), replace=False))
        keep = np.concatenate([va, trs])
        plain = list(range(len(seeds)))  # the plain members come first
        out, val, Ts, te_idx, kp = train_members(D, edge, member_rules, member_seeds, keep_idx=keep, t_keep=plain)
        rows = [{"config": c, "seed": s, "val": float(val[e]), **test_metric(D, out[:, e], te_idx)}
                for e, (c, s) in enumerate(zip(member_rules, member_seeds))]
        # predictors, from the plain networks only (nothing trained with a rule is used here)
        ally = np.array([g["y"] for g in D["graphs"]])
        lg = (kp[..., 1] - kp[..., 0]).double().cpu().numpy()  # plain logit at each member's best validation epoch
        nva = len(va)
        yv, ys_ = ally[va], ally[trs]
        base_v = [logreg_cv_nll(lg[:nva, e:e + 1], yv) for e in plain]
        base_t = [logreg_cv_nll(lg[nva:, e:e + 1], ys_) for e in plain]
        prior_t = logreg_cv_nll(np.zeros((len(trs), 0)), ys_)
        Tp = []
        for e in plain:  # the plain network's representation of the training probe graphs, top 32 directions
            Tt = Ts[torch.tensor(trs, device=DEVICE), e].double().cpu().numpy()
            Tt = (Tt - Tt.mean(0)) / (Tt.std(0) + 1e-6)
            U, S_, Vt = np.linalg.svd(Tt, full_matrices=False)
            Tp.append(Tt @ Vt[:32].T)
        pred = {}
        for r in rules:
            Zr = rule_proj(D, r, tr)
            pred[r] = {
                # primary: label information the rule adds to the plain network's logit on the validation graphs
                "dval": float(np.mean([bv - logreg_cv_nll(np.concatenate([lg[:nva, e:e + 1], Zr[va]], 1), yv)
                                       for bv, e in zip(base_v, plain)])),
                # the same quantity on the network's own training graphs (in-sample, as in D3/D5)
                "dtrain": float(np.mean([bt - logreg_cv_nll(np.concatenate([lg[nva:, e:e + 1], Zr[trs]], 1), ys_)
                                         for bt, e in zip(base_t, plain)])),
                # label information of the rule alone, I(Y; S), on training graphs
                "mi": float(prior_t - logreg_cv_nll(Zr[trs], ys_)),
                # how much of the rule the representation already holds
                "r2_T": float(np.mean([ridge_r2(T_, Zr[trs]) for T_ in Tp])),
                "dims": int(D["Z"][r].shape[1]),
            }
        return {"dataset": D["name"], "arm": arm, "family": "gine" if edge else "gin", "rows": rows, "pred": pred,
                "n": [int(len(tr)), int(len(va)), int(len(te))], "hidden": CFG["hidden"], "dropout": CFG["dropout"],
                "size_ranges": {k: [int(min(D["graphs"][i]["n"] for i in ix)), int(max(D["graphs"][i]["n"] for i in ix))]
                                for k, ix in [("train", tr), ("val", va), ("test", te)]}}

    # ---------------------------------------------------------------- driver
    hf_pull([f"{CFG['out_dir']}/units/T{L_HASH}_*", f"{CFG['data_dir']}/T1_L1_*_full.npz"])  # this cell's own files only
    t0 = time.time()
    lines, fails, results = [], [], {}
    units = [(d, arm) for d in CFG["datasets"] for arm in CFG["arms"][d]]
    D, cur = None, None
    for d, arm in mo.status.progress_bar(units, title="T1 units (dataset x split)"):
        edge = d.startswith("ogbg-")  # GINE with bond features for OGB molecules, GIN otherwise
        path = os.path.join(CFG["out_dir"], "units", f"T{L_HASH}_{d}_{arm}.json")
        if os.path.exists(path):
            with open(path) as fh:
                results[(d, arm)] = json.load(fh)
            continue
        ts = time.time()
        CFG.update(hidden=CFG["hidden_by"][d], batch=CFG["batch_by"][d], dropout=CFG["dropout_by"][d])
        try:
            if cur != d:
                D = None
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
                D = prepare(d)
                cur = d
            res = run_unit(D, edge, arm)
        except Exception as e:
            fails.append((d, arm, repr(e)[:300]))
            with open(os.path.join(CFG["out_dir"], "T1_failures.log"), "a") as fh:
                fh.write(f"--- {d} {arm}\n{traceback.format_exc()}\n")
            print(traceback.format_exc(), file=sys.stderr, flush=True)
            continue
        atomic_write(path, json.dumps(res))
        hf_push([path, os.path.join(CFG["data_dir"], f"T1_L1_{d}_full.npz")], f"T1 unit {d} {arm}")
        results[(d, arm)] = res
        R_ = [r for r in res["rows"] if r["config"] == "none"]
        line = (f"{time.strftime('%H:%M:%S')} {d} {arm}: {time.time() - ts:.0f}s, plain test NLL "
                f"{np.mean([r['nll'] for r in R_]):.4f}, top rules by d_val: "
                + ", ".join(f"{k} {v['dval']:+.4f}" for k, v in sorted(res["pred"].items(), key=lambda kv: -kv[1]["dval"])[:3]))
        lines.append(line)
        print(line, flush=True)
    D = None

    # ---------------------------------------------------------------- verdicts (fixed before the run)
    import pandas as pd
    head = mo.md(f"**T1 units ready: {len(results)} / {len(units)}** in {(time.time() - t0) / 60:.1f} min (hash `{L_HASH}`)"
                 + (f", **{len(fails)} failed** (tracebacks in `T1_failures.log`)" if fails else ", no failures"))
    if not results:
        return mo.vstack([head, mo.md("\n".join(f"- `{a}` {b}: `{c}`" for a, b, c in fails))])
    brng = np.random.default_rng(0)

    def ci(x, B=4000):
        x = np.asarray(x, float)
        b = [brng.choice(x, len(x)).mean() for _ in range(B)] if len(x) > 1 else [x.mean()]
        return float(x.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

    pts = []
    for (d, arm), u in results.items():
        R = pd.DataFrame(u["rows"])
        base = R[R.config == "none"].set_index("seed")
        met = "auc" if d.startswith("ogbg-") else "acc"
        twin = R[R.config == "noise"].set_index("seed")  # same architecture and extra input branch, no information
        for r, pr in u["pred"].items():
            g = R[R.config == r].set_index("seed")
            # realized gain = test log-loss of the noise-input twin minus that of the rule model (nats per graph), so the
            # extra input branch itself is held fixed; for the noise rule it is reported against the plain network
            ref = base if r == "noise" else twin
            gn = (ref.loc[g.index, "nll"] - g["nll"]).values
            m, lo, hi = ci(gn)
            pts.append({"dataset": d, "arm": arm, "shift": arm != "random", "rule": r, **pr, "gain_nll": m, "gain_lo": lo,
                        "gain_hi": hi, "gain_metric": float((g[met] - ref.loc[g.index, met]).mean()), "metric": met,
                        "gain_vs_plain": float((base.loc[g.index, "nll"] - g["nll"]).mean()),
                        "plain_nll": float(base["nll"].mean()), "plain_metric": float(base[met].mean())})
    P = pd.DataFrame(pts)
    real = P[P.rule != "noise"]
    V, notes = {}, []
    # T-a / T-b: the same rule on the same data pays more under shift
    W = real.pivot_table(index=["dataset", "rule"], columns="shift", values="gain_nll")
    W = W.dropna() if {True, False} <= set(W.columns) else W.iloc[0:0]
    if len(W):
        diff = (W[True] - W[False]).values
        m, lo, hi = ci(diff)
        V["T-a transport: a rule's test log-loss gain is larger under the shifted split than under the random split (mean over dataset x rule pairs, CI above 0)"] = {
            "holds": bool(lo > 0), "detail": f"{len(diff)} pairs: shift minus random {m:+.4f} nats [{lo:+.4f}, {hi:+.4f}]; "
                                            f"mean gain random {W[False].mean():+.4f}, shifted {W[True].mean():+.4f}"}
        byd = W.groupby(level=0).mean()
        k_ = int((byd[True] > byd[False]).sum())
        V[f"T-b transport, per dataset: the mean rule gain is larger under shift in >= 4 of {len(byd)} datasets"] = {
            "holds": bool(k_ >= min(4, len(byd))),
            "detail": f"{k_}/{len(byd)}: " + "; ".join(f"{d} random {a:+.4f} vs shifted {b:+.4f}" for d, a, b in zip(byd.index, byd[False], byd[True]))}
    # T-c / T-d: one number from the plain network predicts the gain, and beats the alternatives
    def rho(a, b):
        return float(pd.Series(a).rank().corr(pd.Series(b).rank()))
    Pall, P = P, P[P.rule != "noise"].reset_index(drop=True)  # the placebo is the reference, so it is not a point
    if len(P) >= 6:
        r_main = rho(P.dval.values, P.gain_nll.values)
        V["T-c prediction: Spearman(d_val, realized test log-loss gain) >= 0.4 over all (dataset, split, rule) points"] = {
            "holds": bool(r_main >= 0.4), "detail": f"rho = {r_main:.3f} over {len(P)} points; within-dataset mean rho = "
            + f"{np.nanmean([rho(g.dval.values, g.gain_nll.values) for _, g in P.groupby(['dataset', 'arm']) if len(g) > 3]):.3f}"
            + f"; with the task metric (AUC / accuracy) rho = {rho(P.dval.values, P.gain_metric.values):.3f}"}
        alts = {"in-sample probe (training graphs)": P.dtrain.values, "label information I(Y;S)": P.mi.values,
                "unlearnability 1 - R2(S|T)": 1 - P.r2_T.values}
        det, ok = [], []
        idx = np.arange(len(P))
        for name_, a in alts.items():
            bs = []
            for _ in range(2000):
                j = brng.choice(idx, len(idx))
                bs.append(rho(P.dval.values[j], P.gain_nll.values[j]) - rho(a[j], P.gain_nll.values[j]))
            lo, hi = np.percentile(bs, [2.5, 97.5])
            ok.append(lo > 0)
            det.append(f"{name_}: rho {rho(a, P.gain_nll.values):.3f}, difference [{lo:+.3f}, {hi:+.3f}]")
        V["T-d the shift-aware number beats each alternative predictor (bootstrap CI of the rho difference above 0)"] = {
            "holds": bool(all(ok)), "detail": "; ".join(det)}
    # T-e: absorption in distribution
    ab = real[(~real["shift"]) & (real.r2_T >= 0.9)]
    lo_ = real[(~real["shift"]) & (real.r2_T < 0.5)]
    V["T-e absorption: under the random split, rules the network already holds (R2(S|T) >= 0.9) have median gain within +-0.005 nats"] = {
        "holds": bool(len(ab) > 0 and abs(ab.gain_nll.median()) <= 0.005),
        "detail": f"{len(ab)} rules, median gain {ab.gain_nll.median() if len(ab) else float('nan'):+.4f}; "
                  f"rules with R2 < 0.5: {len(lo_)}, median {lo_.gain_nll.median() if len(lo_) else float('nan'):+.4f}"}
    # T-f: the placebo
    nz = Pall[Pall.rule == "noise"]
    V["T-f placebo: the predictor never recommends the noise rule (its d_val <= 0.01 in every dataset and split)"] = {
        "holds": bool(len(nz) > 0 and all(nz.dval <= 0.01)),
        "detail": "; ".join(f"{d}/{a}: d_val {v:+.4f} (noise branch vs plain: {g:+.4f} [{l:+.4f}, {h:+.4f}] nats)"
                            for d, a, g, l, h, v in zip(nz.dataset, nz.arm, nz.gain_nll, nz.gain_lo, nz.gain_hi, nz.dval))}
    P = Pall
    for (d, arm), g in P[P.rule != "noise"].groupby(["dataset", "arm"]):
        notes.append(f"{d} / {arm}: plain {g.metric.iloc[0]} {g.plain_metric.iloc[0]:.4f}, test NLL {g.plain_nll.iloc[0]:.4f}; "
                     f"best rule by d_val {g.sort_values('dval').rule.iloc[-1]} (realized {g.sort_values('dval').gain_nll.iloc[-1]:+.4f}); "
                     f"best realized {g.sort_values('gain_nll').rule.iloc[-1]} ({g.gain_nll.max():+.4f})")
    outs = [os.path.join(CFG["out_dir"], f) for f in ["transport_points.csv", "transport_verdicts.json"]]
    P.to_csv(outs[0], index=False)
    with open(outs[1], "w") as fh:
        json.dump({"verdicts": V, "notes": notes, "complete": len(results) == len(units),
                   "units": [f"{d}/{a}" for d, a in results]}, fh, indent=1)
    hf_push(outs, "T1 verdicts")
    return mo.vstack([
        head,
        mo.md("```text\n" + "\n".join(lines) + "\n```") if lines else mo.md(""),
        mo.md("\n".join(f"- `{a}` {b}: `{c}`" for a, b, c in fails)) if fails else mo.md(""),
        mo.md("### T1 verdicts: the transport test\n"
              + ("" if len(results) == len(units) else f"*Partial: {len(results)} of {len(units)} units so far; rerun to finish.*\n")
              + "\n".join(f"- **{k}: {'holds' if v['holds'] else 'fails'}**. {v['detail']}" for k, v in V.items())),
        mo.md("**Per dataset and split**\n" + "\n".join(f"- {n_}" for n_ in notes)),
        mo.md("**All points** (gain = test log-loss of the noise-input twin minus the rule model, nats per graph; d_val is computed from the plain network before any rule-trained model is seen)"),
        mo.ui.table(P.round(4), page_size=40, selection=None),
    ])


_t1_transport_test()

In [ ]:
# ==== HF-ALL. save EVERY file of this molab folder to Hugging Face (standalone: own imports, nothing from other cells) ====
def _hf_save_everything():
    """Upload every file under the notebook folder (runs, caches, datasets, notebooks, logs, figures, paper files)
    to the HF repo, plus a manifest with sizes and SHA-256 of each file and a snapshot of the environment.
    Files that contain an HF token are uploaded as a redacted copy (token replaced by "__HF_TOKEN__"), so the
    notebook code itself is saved too without the Hub revoking the token. Only changed files are sent; nothing
    remote is ever deleted. Safe to re-run any time."""
    import os, re, io, sys, json, time, hashlib, platform, subprocess
    from fnmatch import fnmatch
    import marimo as mo

    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    REPO_TYPE = "model"
    ROOT = os.getcwd()                        # the molab folder (nsgnn_runs/, nsgnn_data/, notebooks, ...)
    SNAP_DIR = "_snapshot"                     # manifest + environment land here on the Hub
    MAX_FILE_GB = 45                           # the Hub refuses single files above 50 GB

    try:
        import huggingface_hub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24"])
        import huggingface_hub
    from huggingface_hub import HfApi, CommitOperationAdd

    # only true junk is left out: virtualenvs, interpreter caches, half-written temp files
    SKIP_DIRS = {"__pycache__", ".ipynb_checkpoints", ".git", ".venv", "venv", "node_modules", ".cache", ".Trash-0"}
    SKIP_FILES = ["*.tmp", "*.tmp.*", "*.pyc", "*.swp", ".DS_Store"]

    TOKEN_RE = re.compile(rb"hf_[A-Za-z0-9]{30,}")
    tok = HF_TOKEN.encode()

    def scan(path, chunk=16 << 20):
        """sha256 of the file and whether it contains an HF token"""
        h, tail, hit = hashlib.sha256(), b"", False
        with open(path, "rb") as fh:
            while True:
                buf = fh.read(chunk)
                if not buf:
                    return h.hexdigest(), hit
                h.update(buf)
                win = tail + buf
                if not hit and (tok in win or TOKEN_RE.search(win)):
                    hit = True
                tail = win[-64:]

    t0 = time.time()
    files, redact, too_big, unreadable = [], [], [], []
    manifest = []
    for dirpath, dirnames, filenames in os.walk(ROOT):
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP_DIRS)
        for fn in sorted(filenames):
            full = os.path.join(dirpath, fn)
            rel = os.path.relpath(full, ROOT).replace(os.sep, "/")
            if rel.startswith(SNAP_DIR + "/") or any(fnmatch(fn, p) for p in SKIP_FILES) or os.path.islink(full):
                continue
            try:
                size = os.path.getsize(full)
                if size > MAX_FILE_GB * 2**30:
                    too_big.append(rel)
                    continue
                digest, has_tok = scan(full)
            except OSError as e:
                unreadable.append(f"{rel}: {e}")
                continue
            manifest.append({"path": rel, "bytes": size, "sha256": digest, "redacted": has_tok,
                             "mtime": time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime(os.path.getmtime(full)))})
            (redact if has_tok else files).append(rel)

    total_mb = sum(m["bytes"] for m in manifest) / 2**20
    print(f"[HF-ALL] {len(manifest)} files ({total_mb:.1f} MB): {len(files)} as-is, {len(redact)} redacted, "
          f"{len(too_big)} over {MAX_FILE_GB} GB, {len(unreadable)} unreadable")

    # environment snapshot, so every run can be reproduced later
    try:
        freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=120).stdout
    except Exception as e:
        freeze = f"pip freeze failed: {e!r}"
    env = {"saved_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "python": sys.version,
           "platform": platform.platform(), "root": ROOT, "n_files": len(manifest), "total_mb": round(total_mb, 2)}
    try:
        import torch
        env.update(torch=torch.__version__, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    except Exception:
        pass
    try:
        env["nvidia_smi"] = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30).stdout
    except Exception:
        pass

    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type=REPO_TYPE, private=True, exist_ok=True)

    # 1) the plain files: upload_folder sends only content that changed since the last commit
    commits = []
    if files:
        ignore = [f"{d}/**" for d in SKIP_DIRS] + [f"**/{d}/**" for d in SKIP_DIRS] + [f"**/{p}" for p in SKIP_FILES] \
                 + SKIP_FILES + [re.sub(r"([\[\]*?])", r"[\1]", r) for r in redact + too_big] + [f"{SNAP_DIR}/**"]
        try:
            c = api.upload_folder(repo_id=HF_REPO, repo_type=REPO_TYPE, folder_path=ROOT, ignore_patterns=ignore,
                                  commit_message=f"save everything: {len(files)} files ({time.strftime('%Y-%m-%d %H:%M')})")
            commits.append(getattr(c, "commit_url", str(c)))
        except Exception as e:
            # very large trees: fall back to the resumable multi-commit uploader
            print(f"[HF-ALL] upload_folder failed ({e!r:.200}); retrying with upload_large_folder")
            api.upload_large_folder(repo_id=HF_REPO, repo_type=REPO_TYPE, folder_path=ROOT, ignore_patterns=ignore)
            commits.append("upload_large_folder (multiple commits)")

    # 2) token-bearing files as redacted copies, plus the manifest and environment, in one commit
    ops = []
    for rel in redact:
        with open(os.path.join(ROOT, rel), "rb") as fh:
            data = fh.read()
        data = data.replace(tok, b"__HF_TOKEN__")
        data = TOKEN_RE.sub(b"__HF_TOKEN__", data)
        ops.append(CommitOperationAdd(path_in_repo=rel, path_or_fileobj=io.BytesIO(data)))
    ops.append(CommitOperationAdd(path_in_repo=f"{SNAP_DIR}/manifest.json",
                                  path_or_fileobj=io.BytesIO(json.dumps(manifest, indent=1).encode())))
    ops.append(CommitOperationAdd(path_in_repo=f"{SNAP_DIR}/environment.json",
                                  path_or_fileobj=io.BytesIO(json.dumps(env, indent=1).encode())))
    ops.append(CommitOperationAdd(path_in_repo=f"{SNAP_DIR}/pip_freeze.txt", path_or_fileobj=io.BytesIO(freeze.encode())))
    c = api.create_commit(repo_id=HF_REPO, repo_type=REPO_TYPE, operations=ops,
                          commit_message=f"snapshot: manifest, environment, {len(redact)} redacted files")
    commits.append(getattr(c, "commit_url", str(c)))

    # 3) verify: every local file must now exist on the Hub
    remote = set(api.list_repo_files(HF_REPO, repo_type=REPO_TYPE))
    missing = [m["path"] for m in manifest if m["path"] not in remote]
    url = f"https://huggingface.co/{HF_REPO}"
    print(f"[HF-ALL] done in {time.time() - t0:.0f}s, {len(remote)} files on the Hub, {len(missing)} local files missing")
    for r in missing[:50]:
        print(f"[HF-ALL]   missing on Hub: {r}")

    lst = lambda xs: "".join(f"\n- `{x}`" for x in xs[:30]) or " none"
    return mo.md(
        f"**Saved everything to [{HF_REPO}]({url})**: {len(manifest)} files, {total_mb:.1f} MB "
        f"({len(redact)} saved as token-redacted copies). Hub now holds {len(remote)} files.\n\n"
        f"**Verification:** {'every local file is on the Hub' if not missing else f'{len(missing)} files missing (listed in the log), re-run this cell'}\n\n"
        f"Manifest with SHA-256 of every file: `{SNAP_DIR}/manifest.json`; environment: `{SNAP_DIR}/environment.json`, `{SNAP_DIR}/pip_freeze.txt`\n\n"
        f"Commits: {', '.join(commits)}\n\n"
        f"Redacted copies:{lst(redact)}\n\nToo large for the Hub:{lst(too_big)}\n\nUnreadable:{lst(unreadable)}"
    )


_hf_save_everything()

In [ ]:
def _():
    import os, sys, subprocess
    import marimo as mo
    try:
        import matplotlib
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
    try:
        import huggingface_hub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

    OUT = os.path.join(os.getcwd(), "nsgnn_figures")
    os.makedirs(OUT, exist_ok=True)

    # categorical slots in fixed order (validated reference palette, light mode) + neutral ink
    BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
    INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df"

    plt.rcParams.update({
        "font.family": "serif", "font.serif": ["DejaVu Serif"], "mathtext.fontset": "dejavuserif",
        "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6, "lines.linewidth": 2,
        "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
    })


    def save(fig, name):
        fig.savefig(os.path.join(OUT, name + ".pdf"))
        fig.savefig(os.path.join(OUT, name + ".png"), dpi=300)
        plt.close(fig)
        print("wrote", name)


    def grid(ax, axis="both"):
        ax.grid(True, axis=axis, color=GRID, lw=0.6, zorder=0)
        ax.set_axisbelow(True)


    # ------------------------------------------------------------------------------------------------ Figure 2
    # (a) ZINC, run L1 (3 seeds): every rule of the library added to GIN and GINE.
    #     x = 1 - R^2_T (share of the rule the trained plain network does not decode), y = test-MAE reduction in %.
    ZINC_L1 = {  # rule: (R2_T gin, gain% gin, R2_T gine, gain% gine)
        "bond_hist":   (0.8559, 1.9865, 0.9837, -0.7139),
        "cycles":      (0.4923, 27.3878, 0.5919, 32.1818),
        "cyclomatic":  (0.9814, -0.5150, 0.9838, 3.5249),
        "degree_hist": (0.9784, 0.1327, 0.9828, -4.8588),
        "ring_atoms":  (0.9386, -1.2317, 0.9524, -3.2608),
        "size":        (0.9939, -1.1556, 0.9940, 2.2052),
        "spectrum":    (0.8483, -2.5696, 0.8615, 1.6615),
        "triangles":   (0.3976, 0.6982, 0.6379, -0.3952),
        "type_hist":   (0.9873, -1.6955, 0.9884, 2.0632),
        "walks":       (0.9932, 2.4621, 0.9938, 0.5501),
    }
    # (b) validation-selected discovered rule set vs the hand rule, gains over the plain network (paired by seed).
    #     relative improvement in % of the plain score (MAE: lower is better; ROC-AUC / accuracy: higher is better)
    #     (dataset, metric, plain, discovered gain, lo, hi, hand gain, hand lo, hand hi, hand name, held-out?)
    DISC = [
        ("ZINC", "MAE", 0.1911, 0.0716, 0.0647, 0.0773, 0.0605, 0.0558, 0.0651, "cycles", False),
        ("molhiv", "AUC", 0.7588, 0.0208, 0.0103, 0.0312, 0.0283, 0.0131, 0.0420, "cycles", False),
        ("COLLAB", "acc", 0.7996, 0.0068, 0.0000, 0.0160, 0.0076, 0.0012, 0.0140, "size", False),
        ("molbbbp", "AUC", 0.6421, 0.0347, 0.0251, 0.0445, 0.0189, 0.0113, 0.0266, "cycles", True),
        ("molbace", "AUC", 0.8084, -0.0124, -0.0285, 0.0076, -0.0010, -0.0186, 0.0184, "cycles", True),
        ("PROTEINS", "acc", 0.7786, -0.0027, -0.0152, 0.0080, -0.0089, -0.0232, 0.0054, "cycles", True),
        ("IMDB-B", "acc", 0.7590, -0.0070, -0.0230, 0.0100, -0.0030, -0.0160, 0.0090, "size", True),
        ("REDDIT-B", "acc", 0.9175, -0.0020, -0.0135, 0.0115, -0.0005, -0.0110, 0.0105, "size", True),
    ]


    def fig2():
        fig, (a, b) = plt.subplots(1, 2, figsize=(7.1, 2.75), gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.42})
        grid(a)
        for k, (fam, col, mk) in enumerate([("GIN", BLUE, "o"), ("GINE", ORANGE, "s")]):
            xs = [1 - v[2 * k] for v in ZINC_L1.values()]
            ys = [v[2 * k + 1] for v in ZINC_L1.values()]
            a.scatter(xs, ys, s=28, marker=mk, color=col, edgecolor="white", linewidth=0.8, zorder=3, label=fam)
        a.axhline(0, color=INK2, lw=0.6, zorder=1)
        a.annotate("", xy=(1 - 0.5919, 32.18), xytext=(0.30, 30.0), arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.annotate("", xy=(1 - 0.4923, 27.39), xytext=(0.30, 28.6), arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.text(0.29, 29.3, "ring counts: the\nring term of the target", fontsize=6.8, color=INK, ha="right", va="center")
        a.annotate("triangles: unlearned\nbut label-irrelevant", xy=(1 - 0.3976, 0.70), xytext=(0.50, 8.0), fontsize=6.8,
                   color=INK, ha="center", arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.annotate("bond counts: invisible\nto GIN, yet decoded\n($R^2_T=0.86$), worth 0", xy=(1 - 0.8559, 1.99),
                   xytext=(0.15, 16.5), fontsize=6.8, color=INK, ha="center",
                   arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.set_xlim(-0.02, 0.66)
        a.set_ylim(-7, 36)
        a.set_xlabel(r"unlearned share of the rule, $1-R^2_T$")
        a.set_ylabel("test-MAE reduction on ZINC (%)")
        a.set_title("(a) A rule pays only for what is not learned", loc="left")
        a.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.55))

        grid(b, "x")
        n = len(DISC)
        yy = np.arange(n)[::-1]
        cap = 8.5
        for y, (name, met, plain, g, lo, hi, hg, hlo, hhi, hname, held) in zip(yy, DISC):
            s = 100 / plain
            if g * s > cap:
                b.scatter([cap - 0.25], [y + 0.13], s=34, marker=">", color=BLUE, edgecolor="white", linewidth=0.8, zorder=4)
                b.scatter([cap - 0.25], [y - 0.13], s=30, marker=">", color=ORANGE, edgecolor="white", linewidth=0.8, zorder=4)
                b.text(cap - 0.6, y, f"off scale: discovered {g * s:.1f}%, expert {hg * s:.1f}%", fontsize=6.6,
                       color=INK, va="center", ha="right")
                continue
            b.plot([lo * s, hi * s], [y + 0.13] * 2, color=BLUE, lw=1.6, solid_capstyle="round", zorder=3)
            b.plot([hlo * s, hhi * s], [y - 0.13] * 2, color=ORANGE, lw=1.6, solid_capstyle="round", zorder=3)
            b.scatter([g * s], [y + 0.13], s=26, marker="o", color=BLUE, edgecolor="white", linewidth=0.8, zorder=4)
            b.scatter([hg * s], [y - 0.13], s=26, marker="s", color=ORANGE, edgecolor="white", linewidth=0.8, zorder=4)
        b.axvline(0, color=INK2, lw=0.6)
        b.axhline(4.5, color=MUTED, lw=0.6, ls=(0, (3, 2)))
        b.text(-4.0, 4.62, "development", fontsize=6.3, color=INK2, ha="left", va="bottom")
        b.text(-4.0, 4.38, "frozen method, unseen data", fontsize=6.3, color=INK2, ha="left", va="top")
        b.set_yticks(yy)
        b.set_yticklabels([f"{d[0]} ({d[1]})" for d in DISC])
        b.set_xlim(-4.2, cap)
        b.set_ylim(-0.6, n - 0.4)
        b.set_xlabel("improvement over the plain network (% of its score)")
        b.set_title("(b) Discovered rules vs. the expert rule", loc="left")
        b.scatter([], [], s=26, marker="o", color=BLUE, label="discovered (out-of-fold)")
        b.scatter([], [], s=26, marker="s", color=ORANGE, label="expert rule")
        b.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=2)
        save(fig, "fig2_learnability_discovery")


    # ------------------------------------------------------------------------------------------------ Figure 1 stand-in
    def fig1():
        """Vector stand-in for Figure 1 (the final art follows figure1_prompt.md). Four stages, left to right."""
        fig, ax = plt.subplots(figsize=(7.1, 2.5))
        ax.set_xlim(0, 100)
        ax.set_ylim(1, 31)
        ax.axis("off")
        boxes = [
            (1, "1  Plain GNN on\ntwo halves", "out-of-fold\npredictions $\\hat y^{(-k)}(G_i)$\nand codes $T_i$"),
            (26, "2  Out-of-fold\ndebt", "residual $r_i = y_i-\\hat y^{(-k)}(G_i)$\nunlearned part\n$\\tilde C_i = C_i-\\hat C(T_i)$"),
            (51, "3  Calibrated\nblock pursuit", "$E_b=\\|Q_b^{\\top} r\\|^2/n$\nscaled-$\\chi^2$ surprise\nWestfall-Young $p$"),
            (76, "4  Add the\nnamed rule", "logit $+\\,\\phi(S_{\\hat b})$\nnumber of blocks\nchosen on validation"),
        ]
        for x, title, body in boxes:
            ax.add_patch(FancyBboxPatch((x, 9), 22, 20, boxstyle="round,pad=0.4,rounding_size=1.6",
                                        facecolor="#f3f2ee", edgecolor=INK2, lw=0.8))
            ax.text(x + 11, 27.6, title, ha="center", va="top", fontsize=7.6, color=INK, weight="bold")
            ax.text(x + 11, 15.2, body, ha="center", va="center", fontsize=6.9, color=INK)
        for x in (23.4, 48.4, 73.4):
            ax.add_patch(FancyArrowPatch((x, 19), (x + 2.4, 19), arrowstyle="-|>", mutation_scale=9, color=INK2, lw=1))
        ax.text(50, 3.2, "ZINC: the first block is $\\max(0,\\ell_{\\max}-6)$, the hidden ring term of penalised logP "
                         "($p_{\\mathrm{adj}}=0.001$ in three independent runs)", ha="center", fontsize=7.5, color=INK2)
        save(fig, "fig1_pipeline")


    # ------------------------------------------------------------------------------------------------ Appendix A1
    # exact 1-WL analysis (run G4): share of graphs sharing a WL_t class with a graph of a different rule value,
    # maximum over the six beyond-WL rules (triangles, 4-cycles, components, cyclomatic number, clustering, triangle nodes)
    WL = {"MUTAG": [0.133, 0, 0, 0, 0], "PTC": [0.032, 0, 0, 0, 0], "PROTEINS": [0.4636, 0, 0, 0, 0],
          "NCI1": [0.3348, 0.0012, 0, 0, 0], "IMDB-B": [0, 0, 0, 0, 0], "IMDB-M": [0, 0, 0, 0, 0], "COLLAB": [0, 0, 0, 0, 0]}


    def figA1():
        """Three bars per dataset (t = 0, 1, 2). Zero-height bars are drawn as a coloured stub with a value label so that
        every round stays visible even where colour refinement has removed all ambiguity."""
        fig, ax = plt.subplots(figsize=(5.4, 2.5))
        grid(ax, "y")
        names = list(WL)
        x = np.arange(len(names))
        w = 0.26
        rounds = [(0, MUTED, "before message passing ($t=0$)"), (1, BLUE, "after one round ($t=1$)"),
                  (2, AQUA, "after two rounds ($t=2$)")]
        for k, (t, col, lab) in enumerate(rounds):
            xs = x + (k - 1) * (w + 0.02)
            vals = [100 * WL[d][t] for d in names]
            ax.bar(xs, vals, w, color=col, label=lab, zorder=3)
            for xi, v in zip(xs, vals):
                if v < 1.0:  # invisible bar: draw a stub and print the exact value
                    ax.plot([xi - w / 2, xi + w / 2], [0.35, 0.35], color=col, lw=2.2, solid_capstyle="butt", zorder=4)
                    ax.text(xi, 1.3, f"{v:.2g}%", ha="center", va="bottom", fontsize=5.6, color=col if t else INK2,
                            rotation=90)
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.set_ylim(0, 52)
        ax.set_ylabel("ambiguous graphs (%)")
        ax.set_title("Exact 1-WL ambiguity of beyond-1-WL rules, by refinement round", loc="left")
        ax.legend(frameon=False, loc="upper right")
        save(fig, "figA1_wl_ambiguity")


    # ------------------------------------------------------------------------------------------------ Appendix A2
    # gain ledger (runs G9/G10, 10-fold CV), WL-computable rules on GIN with a linear readout, nats per graph
    LEDGER = {"COLLAB": (0.0523, 0.0416, -0.0025, 0.0131), "NCI1": (0.0259, 0.0215, -0.0068, 0.0113),
              "PROTEINS": (0.0257, 0.0072, -0.0191, 0.0376)}
    # 95% CIs reported in Table 2 for the observed gain and the residual
    LEDGER_CI = {"COLLAB": ((0.038, 0.067), (0.003, 0.023)), "NCI1": ((0.011, 0.041), (-0.021, 0.043)),
                 "PROTEINS": ((0.007, 0.044), (-0.001, 0.076))}


    def figA2():
        fig, ax = plt.subplots(figsize=(5.0, 2.6))
        grid(ax, "y")
        names = list(LEDGER)
        x = np.arange(len(names))
        w = 0.2
        labs = ["observed gain", "readout part", "coordinate part", "residual"]
        cols = [INK2, BLUE, ORANGE, AQUA]
        for k in range(4):
            xs = x + (k - 1.5) * (w + 0.02)
            ax.bar(xs, [LEDGER[d][k] for d in names], w, color=cols[k], label=labs[k], zorder=3)
            if k in (0, 3):
                ci = [LEDGER_CI[d][0 if k == 0 else 1] for d in names]
                ax.errorbar(xs, [LEDGER[d][k] for d in names],
                            yerr=[[LEDGER[d][k] - c[0] for d, c in zip(names, ci)], [c[1] - LEDGER[d][k] for d, c in zip(names, ci)]],
                            fmt="none", ecolor=INK, elinewidth=0.8, capsize=2, zorder=4)
        ax.axhline(0, color=INK2, lw=0.6)
        ax.set_ylim(-0.03, 0.09)
        ax.set_title("Gain ledger of 1-WL-computable rules on GIN (95% CI)", loc="left")
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.set_ylabel("test log-loss gain (nats)")
        ax.legend(frameon=False, ncol=2, loc="upper left")
        save(fig, "figA2_gain_ledger")


    # ------------------------------------------------------------------------------------------------ Appendix A3
    # greedy discovery scores, steps 1-8 (run D1, in-sample residual), and the permutation null of run D3
    D1_SCORES = {"ZINC / GIN": [0.1857, 0.0354, 0.0289, 0.0251, 0.0241, 0.0218, 0.0228, 0.0249],
                 "ZINC / GINE": [0.1850, 0.0288, 0.0256, 0.0251, 0.0258, 0.0236, 0.0197, 0.0195],
                 "molhiv / GINE": [0.0156, 0.0131, 0.0077, 0.0076, 0.0068, 0.0063, 0.0063, 0.0063]}
    D3_NULL = {"ZINC / GIN": (0.252, 0.0396), "ZINC / GINE": (0.1763, 0.0347)}


    def figA3():
        fig, (a, b) = plt.subplots(1, 2, figsize=(6.6, 2.3), gridspec_kw={"wspace": 0.35})
        grid(a)
        for (k, v), col, mk in zip(D1_SCORES.items(), [BLUE, ORANGE, AQUA], ["o", "s", "^"]):
            a.plot(range(1, 9), v, color=col, lw=1.6, marker=mk, ms=4.5, label=k)
        a.set_yscale("log")
        a.set_xlabel("greedy step")
        a.set_ylabel("residual score")
        a.set_title("(a) one dominant pick on ZINC", loc="left")
        a.legend(frameon=False)
        grid(b, "y")
        names = list(D3_NULL)
        x = np.arange(len(names))
        b.bar(x - 0.19, [D3_NULL[n][0] for n in names], 0.36, color=BLUE, label="observed first pick", zorder=3)
        b.bar(x + 0.19, [D3_NULL[n][1] for n in names], 0.36, color=MUTED, label="null 95th percentile", zorder=3)
        b.set_xticks(x)
        b.set_xticklabels(names)
        b.set_ylabel("score")
        b.set_title("(b) 1000-permutation null", loc="left")
        b.legend(frameon=False)
        save(fig, "figA3_discovery_scores")


    # ------------------------------------------------------------------------------------------------ Appendix A4
    # run T1, random (in-distribution) split: mean test log-loss gain over the rule library vs the noise-input twin
    T1_RANDOM = {"IMDB-B": -0.0062, "NCI1": 0.0063, "PROTEINS": -0.0157, "molbace": -0.0195, "molbbbp": -0.0062,
                 "molhiv": 0.0021}


    def figA4():
        fig, ax = plt.subplots(figsize=(4.6, 2.1))
        grid(ax, "y")
        names = list(T1_RANDOM)
        ax.bar(names, [T1_RANDOM[n] for n in names], 0.5, color=BLUE, zorder=3)
        ax.axhline(0, color=INK2, lw=0.6)
        ax.axhspan(-0.02, 0.02, color=GRID, alpha=0.6, zorder=1)
        ax.set_ylim(-0.04, 0.04)
        ax.set_ylabel("mean rule gain (nats)")
        ax.set_title("Classical rules under random splits: gain over the noise-input twin", loc="left")
        save(fig, "figA4_in_distribution")



    TITLES = {
        "fig1_pipeline": "Figure 1. Out-of-fold rule discovery (vector stand-in)",
        "fig2_learnability_discovery": "Figure 2. Learnability on ZINC (a) and discovered vs expert rules (b)",
        "figA1_wl_ambiguity": "Figure D.1. Exact 1-WL ambiguity by refinement round",
        "figA2_gain_ledger": "Figure D.2. Gain ledger of 1-WL-computable rules on GIN",
        "figA3_discovery_scores": "Figure D.3. Discovery scores and permutation null",
        "figA4_in_distribution": "Figure D.4. Classical rules under random splits",
    }
    for _f in (fig1, fig2, figA1, figA2, figA3, figA4):
        _f()
    with open(os.path.join(OUT, "README.md"), "w") as _fh:
        _fh.write("# Paper figures\n\n" + "\n".join(f"- `{k}.pdf` / `{k}.png`: {v}" for k, v in TITLES.items()) + "\n")

    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, repo_type="model", folder_path=OUT, path_in_repo="paper/figures",
                      commit_message="paper figures: regenerated, titled")
    remote = [f for f in api.list_repo_files(HF_REPO, repo_type="model") if f.startswith("paper/figures/")]
    want = [f"paper/figures/{k}.{e}" for k in TITLES for e in ("pdf", "png")]
    missing = [w for w in want if w not in remote]
    mo.output.replace(mo.vstack([
        mo.md(f"**Figures saved to Hugging Face** ({HF_REPO}, paper/figures/): {len(want) - len(missing)}/{len(want)} files"
              + ("" if not missing else f". MISSING: {missing}")),
        *[mo.vstack([mo.md(f"**{v}**"), mo.image(src=os.path.join(OUT, k + ".png"), width=700)]) for k, v in TITLES.items()],
    ]))

_()

In [ ]:
def _():
    import os, sys, subprocess
    import marimo as mo
    try:
        import matplotlib
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
    try:
        import huggingface_hub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

    OUT = os.path.join(os.getcwd(), "nsgnn_figures")
    os.makedirs(OUT, exist_ok=True)

    # categorical slots in fixed order (validated reference palette, light mode) + neutral ink
    BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
    INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df"

    plt.rcParams.update({
        "font.family": "serif", "font.serif": ["DejaVu Serif"], "mathtext.fontset": "dejavuserif",
        "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5, "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6, "lines.linewidth": 2,
        "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42,
    })


    def save(fig, name):
        fig.savefig(os.path.join(OUT, name + ".pdf"))
        fig.savefig(os.path.join(OUT, name + ".png"), dpi=300)
        plt.close(fig)
        print("wrote", name)


    def grid(ax, axis="both"):
        ax.grid(True, axis=axis, color=GRID, lw=0.6, zorder=0)
        ax.set_axisbelow(True)


    # ------------------------------------------------------------------------------------------------ Figure 2
    # (a) ZINC, run L1 (3 seeds): every rule of the library added to GIN and GINE.
    #     x = 1 - R^2_T (share of the rule the trained plain network does not decode), y = test-MAE reduction in %.
    ZINC_L1 = {  # rule: (R2_T gin, gain% gin, R2_T gine, gain% gine)
        "bond_hist":   (0.8559, 1.9865, 0.9837, -0.7139),
        "cycles":      (0.4923, 27.3878, 0.5919, 32.1818),
        "cyclomatic":  (0.9814, -0.5150, 0.9838, 3.5249),
        "degree_hist": (0.9784, 0.1327, 0.9828, -4.8588),
        "ring_atoms":  (0.9386, -1.2317, 0.9524, -3.2608),
        "size":        (0.9939, -1.1556, 0.9940, 2.2052),
        "spectrum":    (0.8483, -2.5696, 0.8615, 1.6615),
        "triangles":   (0.3976, 0.6982, 0.6379, -0.3952),
        "type_hist":   (0.9873, -1.6955, 0.9884, 2.0632),
        "walks":       (0.9932, 2.4621, 0.9938, 0.5501),
    }
    # (b) validation-selected discovered rule set vs the hand rule, gains over the plain network (paired by seed).
    #     relative improvement in % of the plain score (MAE: lower is better; ROC-AUC / accuracy: higher is better)
    #     (dataset, metric, plain, discovered gain, lo, hi, hand gain, hand lo, hand hi, hand name, held-out?)
    DISC = [
        ("ZINC", "MAE", 0.1911, 0.0716, 0.0647, 0.0773, 0.0605, 0.0558, 0.0651, "cycles", False),
        ("molhiv", "AUC", 0.7588, 0.0208, 0.0103, 0.0312, 0.0283, 0.0131, 0.0420, "cycles", False),
        ("COLLAB", "acc", 0.7996, 0.0068, 0.0000, 0.0160, 0.0076, 0.0012, 0.0140, "size", False),
        ("molbbbp", "AUC", 0.6421, 0.0347, 0.0251, 0.0445, 0.0189, 0.0113, 0.0266, "cycles", True),
        ("molbace", "AUC", 0.8084, -0.0124, -0.0285, 0.0076, -0.0010, -0.0186, 0.0184, "cycles", True),
        ("PROTEINS", "acc", 0.7786, -0.0027, -0.0152, 0.0080, -0.0089, -0.0232, 0.0054, "cycles", True),
        ("IMDB-B", "acc", 0.7590, -0.0070, -0.0230, 0.0100, -0.0030, -0.0160, 0.0090, "size", True),
        ("REDDIT-B", "acc", 0.9175, -0.0020, -0.0135, 0.0115, -0.0005, -0.0110, 0.0105, "size", True),
    ]


    def fig2():
        fig, (a, b) = plt.subplots(1, 2, figsize=(7.1, 2.75), gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.42})
        grid(a)
        for k, (fam, col, mk) in enumerate([("GIN", BLUE, "o"), ("GINE", ORANGE, "s")]):
            xs = [1 - v[2 * k] for v in ZINC_L1.values()]
            ys = [v[2 * k + 1] for v in ZINC_L1.values()]
            a.scatter(xs, ys, s=28, marker=mk, color=col, edgecolor="white", linewidth=0.8, zorder=4 - k, label=fam)
        a.axhline(0, color=INK2, lw=0.6, zorder=1)
        a.annotate("", xy=(1 - 0.5919, 32.18), xytext=(0.30, 30.0), arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.annotate("", xy=(1 - 0.4923, 27.39), xytext=(0.30, 28.6), arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.text(0.29, 29.3, "ring counts: the\nring term of the target", fontsize=6.8, color=INK, ha="right", va="center")
        a.annotate("triangles: unlearned\nbut label-irrelevant", xy=(1 - 0.3976, 0.70), xytext=(0.50, 8.0), fontsize=6.8,
                   color=INK, ha="center", arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.annotate("bond counts: invisible\nto GIN, yet decoded\n($R^2_T=0.86$), worth 0", xy=(1 - 0.8559, 1.99),
                   xytext=(0.15, 16.5), fontsize=6.8, color=INK, ha="center",
                   arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
        a.set_xlim(-0.02, 0.66)
        a.set_ylim(-7, 36)
        a.set_xlabel(r"unlearned share of the rule, $1-R^2_T$")
        a.set_ylabel("test-MAE reduction on ZINC (%)")
        a.set_title("(a) A rule pays only for what is not learned", loc="left")
        a.legend(frameon=False, loc="center right", bbox_to_anchor=(1.0, 0.55))

        grid(b, "x")
        n = len(DISC)
        yy = np.arange(n)[::-1]
        cap = 8.5
        for y, (name, met, plain, g, lo, hi, hg, hlo, hhi, hname, held) in zip(yy, DISC):
            s = 100 / plain
            if g * s > cap:
                b.scatter([cap - 0.25], [y + 0.13], s=34, marker=">", color=BLUE, edgecolor="white", linewidth=0.8, zorder=4)
                b.scatter([cap - 0.25], [y - 0.13], s=30, marker=">", color=ORANGE, edgecolor="white", linewidth=0.8, zorder=4)
                b.text(cap - 0.6, y, f"off scale: discovered {g * s:.1f}%, expert {hg * s:.1f}%", fontsize=6.6,
                       color=INK, va="center", ha="right", zorder=5,
                       bbox=dict(facecolor="white", edgecolor="none", pad=1.0))
                continue
            b.plot([lo * s, hi * s], [y + 0.13] * 2, color=BLUE, lw=1.6, solid_capstyle="round", zorder=3)
            b.plot([hlo * s, hhi * s], [y - 0.13] * 2, color=ORANGE, lw=1.6, solid_capstyle="round", zorder=3)
            b.scatter([g * s], [y + 0.13], s=26, marker="o", color=BLUE, edgecolor="white", linewidth=0.8, zorder=4)
            b.scatter([hg * s], [y - 0.13], s=26, marker="s", color=ORANGE, edgecolor="white", linewidth=0.8, zorder=4)
        b.axvline(0, color=INK2, lw=0.6)
        b.axhline(4.5, color=MUTED, lw=0.6, ls=(0, (3, 2)))
        b.text(-4.0, 4.62, "development", fontsize=6.3, color=INK2, ha="left", va="bottom")
        b.text(-4.0, 4.38, "frozen method, unseen data", fontsize=6.3, color=INK2, ha="left", va="top")
        b.set_yticks(yy)
        b.set_yticklabels([f"{d[0]} ({d[1]})" for d in DISC])
        b.set_xlim(-4.2, cap)
        b.set_ylim(-0.6, n - 0.4)
        b.set_xlabel("improvement over the plain network (% of its score)")
        b.set_title("(b) Discovered rules vs. the expert rule", loc="left")
        b.scatter([], [], s=26, marker="o", color=BLUE, label="discovered (out-of-fold)")
        b.scatter([], [], s=26, marker="s", color=ORANGE, label="expert rule")
        b.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=2)
        save(fig, "fig2_learnability_discovery")


    # ------------------------------------------------------------------------------------------------ Figure 1 stand-in
    def fig1():
        """Vector stand-in for Figure 1 (the final art follows figure1_prompt.md). Four stages, left to right."""
        fig, ax = plt.subplots(figsize=(7.1, 2.5))
        ax.set_xlim(0, 100)
        ax.set_ylim(1, 31)
        ax.axis("off")
        boxes = [
            (1, "1  Plain GNN on\ntwo halves", "out-of-fold\npredictions $\\hat y^{(-k)}(G_i)$\nand codes $T_i$"),
            (26, "2  Out-of-fold\ndebt", "residual $r_i = y_i-\\hat y^{(-k)}(G_i)$\nunlearned part\n$\\tilde C_i = C_i-\\hat C(T_i)$"),
            (51, "3  Calibrated\nblock pursuit", "$E_b=\\|Q_b^{\\top} r\\|^2/n$\nscaled-$\\chi^2$ surprise\nWestfall-Young $p$"),
            (76, "4  Add the\nnamed rule", "logit $+\\,\\phi(S_{\\hat b})$\nnumber of blocks\nchosen on validation"),
        ]
        for x, title, body in boxes:
            ax.add_patch(FancyBboxPatch((x, 9), 22, 20, boxstyle="round,pad=0.4,rounding_size=1.6",
                                        facecolor="#f3f2ee", edgecolor=INK2, lw=0.8))
            ax.text(x + 11, 27.6, title, ha="center", va="top", fontsize=7.6, color=INK, weight="bold")
            ax.text(x + 11, 15.2, body, ha="center", va="center", fontsize=6.9, color=INK)
        for x in (23.4, 48.4, 73.4):
            ax.add_patch(FancyArrowPatch((x, 19), (x + 2.4, 19), arrowstyle="-|>", mutation_scale=9, color=INK2, lw=1))
        ax.text(50, 3.2, "ZINC: the first block is $\\max(0,\\ell_{\\max}-6)$, the hidden ring term of penalised logP "
                         "($p_{\\mathrm{adj}}=0.001$ in three independent runs)", ha="center", fontsize=7.5, color=INK2)
        save(fig, "fig1_pipeline")


    # ------------------------------------------------------------------------------------------------ Appendix A1
    # exact 1-WL analysis (run G4): share of graphs sharing a WL_t class with a graph of a different rule value,
    # maximum over the six beyond-WL rules (triangles, 4-cycles, components, cyclomatic number, clustering, triangle nodes)
    WL = {"MUTAG": [0.133, 0, 0, 0, 0], "PTC": [0.032, 0, 0, 0, 0], "PROTEINS": [0.4636, 0, 0, 0, 0],
          "NCI1": [0.3348, 0.0012, 0, 0, 0], "IMDB-B": [0, 0, 0, 0, 0], "IMDB-M": [0, 0, 0, 0, 0], "COLLAB": [0, 0, 0, 0, 0]}


    def figA1():
        """Three bars per dataset (t = 0, 1, 2). Zero-height bars are drawn as a coloured stub with a value label so that
        every round stays visible even where colour refinement has removed all ambiguity."""
        fig, ax = plt.subplots(figsize=(5.4, 2.5))
        grid(ax, "y")
        names = list(WL)
        x = np.arange(len(names))
        w = 0.26
        rounds = [(0, MUTED, "before message passing ($t=0$)"), (1, BLUE, "after one round ($t=1$)"),
                  (2, AQUA, "after two rounds ($t=2$)")]
        for k, (t, col, lab) in enumerate(rounds):
            xs = x + (k - 1) * (w + 0.02)
            vals = [100 * WL[d][t] for d in names]
            ax.bar(xs, vals, w, color=col, label=lab, zorder=3)
            for xi, v in zip(xs, vals):
                if v < 1.0:  # invisible bar: draw a stub and print the exact value
                    ax.plot([xi - w / 2, xi + w / 2], [0.35, 0.35], color=col, lw=2.2, solid_capstyle="butt", zorder=4)
                    ax.text(xi, 1.3, f"{v:.2g}%", ha="center", va="bottom", fontsize=5.6, color=col if t else INK2,
                            rotation=90)
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.set_ylim(0, 52)
        ax.set_ylabel("ambiguous graphs (%)")
        ax.set_title("Exact 1-WL ambiguity of beyond-1-WL rules, by refinement round", loc="left")
        ax.legend(frameon=False, loc="upper right")
        save(fig, "figA1_wl_ambiguity")


    # ------------------------------------------------------------------------------------------------ Appendix A2
    # gain ledger (runs G9/G10, 10-fold CV), WL-computable rules on GIN with a linear readout, nats per graph
    LEDGER = {"COLLAB": (0.0523, 0.0416, -0.0025, 0.0131), "NCI1": (0.0259, 0.0215, -0.0068, 0.0113),
              "PROTEINS": (0.0257, 0.0072, -0.0191, 0.0376)}
    # 95% CIs reported in Table 2 for the observed gain and the residual
    LEDGER_CI = {"COLLAB": ((0.038, 0.067), (0.003, 0.023)), "NCI1": ((0.011, 0.041), (-0.021, 0.043)),
                 "PROTEINS": ((0.007, 0.044), (-0.001, 0.076))}


    def figA2():
        fig, ax = plt.subplots(figsize=(5.0, 2.6))
        grid(ax, "y")
        names = list(LEDGER)
        x = np.arange(len(names))
        w = 0.2
        labs = ["observed gain", "readout part", "coordinate part", "residual"]
        cols = [INK2, BLUE, ORANGE, AQUA]
        for k in range(4):
            xs = x + (k - 1.5) * (w + 0.02)
            ax.bar(xs, [LEDGER[d][k] for d in names], w, color=cols[k], label=labs[k], zorder=3)
            if k in (0, 3):
                ci = [LEDGER_CI[d][0 if k == 0 else 1] for d in names]
                ax.errorbar(xs, [LEDGER[d][k] for d in names],
                            yerr=[[LEDGER[d][k] - c[0] for d, c in zip(names, ci)], [c[1] - LEDGER[d][k] for d, c in zip(names, ci)]],
                            fmt="none", ecolor=INK, elinewidth=0.8, capsize=2, zorder=4)
        ax.axhline(0, color=INK2, lw=0.6)
        ax.set_ylim(-0.03, 0.09)
        ax.set_title("Gain ledger of 1-WL-computable rules on GIN (95% CI)", loc="left")
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.set_ylabel("test log-loss gain (nats)")
        ax.legend(frameon=False, ncol=2, loc="upper left")
        save(fig, "figA2_gain_ledger")


    # ------------------------------------------------------------------------------------------------ Appendix A3
    # greedy discovery scores, steps 1-8 (run D1, in-sample residual), and the permutation null of run D3
    D1_SCORES = {"ZINC / GIN": [0.1857, 0.0354, 0.0289, 0.0251, 0.0241, 0.0218, 0.0228, 0.0249],
                 "ZINC / GINE": [0.1850, 0.0288, 0.0256, 0.0251, 0.0258, 0.0236, 0.0197, 0.0195],
                 "molhiv / GINE": [0.0156, 0.0131, 0.0077, 0.0076, 0.0068, 0.0063, 0.0063, 0.0063]}
    D3_NULL = {"ZINC / GIN": (0.252, 0.0396), "ZINC / GINE": (0.1763, 0.0347)}


    def figA3():
        fig, (a, b) = plt.subplots(1, 2, figsize=(6.6, 2.3), gridspec_kw={"wspace": 0.35})
        grid(a)
        for (k, v), col, mk in zip(D1_SCORES.items(), [BLUE, ORANGE, AQUA], ["o", "s", "^"]):
            a.plot(range(1, 9), v, color=col, lw=1.6, marker=mk, ms=4.5, label=k)
        a.set_yscale("log")
        a.set_xlabel("greedy step")
        a.set_ylabel("residual score")
        a.set_title("(a) one dominant pick on ZINC", loc="left")
        a.legend(frameon=False)
        grid(b, "y")
        names = list(D3_NULL)
        x = np.arange(len(names))
        b.bar(x - 0.19, [D3_NULL[n][0] for n in names], 0.36, color=BLUE, label="observed first pick", zorder=3)
        b.bar(x + 0.19, [D3_NULL[n][1] for n in names], 0.36, color=MUTED, label="null 95th percentile", zorder=3)
        b.set_xticks(x)
        b.set_xticklabels(names)
        b.set_ylabel("score")
        b.set_title("(b) 1000-permutation null", loc="left")
        b.legend(frameon=False)
        save(fig, "figA3_discovery_scores")


    # ------------------------------------------------------------------------------------------------ Appendix A4
    # run T1, random (in-distribution) split: mean test log-loss gain over the rule library vs the noise-input twin
    T1_RANDOM = {"IMDB-B": -0.0062, "NCI1": 0.0063, "PROTEINS": -0.0157, "molbace": -0.0195, "molbbbp": -0.0062,
                 "molhiv": 0.0021}


    def figA4():
        fig, ax = plt.subplots(figsize=(4.6, 2.1))
        grid(ax, "y")
        names = list(T1_RANDOM)
        ax.bar(names, [T1_RANDOM[n] for n in names], 0.5, color=BLUE, zorder=3)
        ax.axhline(0, color=INK2, lw=0.6)
        ax.axhspan(-0.02, 0.02, color=GRID, alpha=0.6, zorder=1)
        ax.set_ylim(-0.04, 0.04)
        ax.set_ylabel("mean rule gain (nats)")
        ax.set_title("Classical rules under random splits: gain over the noise-input twin", loc="left")
        save(fig, "figA4_in_distribution")



    TITLES = {
        "fig1_pipeline": "Figure 1. Out-of-fold rule discovery (vector stand-in)",
        "fig2_learnability_discovery": "Figure 2. Learnability on ZINC (a) and discovered vs expert rules (b)",
        "figA1_wl_ambiguity": "Figure D.1. Exact 1-WL ambiguity by refinement round",
        "figA2_gain_ledger": "Figure D.2. Gain ledger of 1-WL-computable rules on GIN",
        "figA3_discovery_scores": "Figure D.3. Discovery scores and permutation null",
        "figA4_in_distribution": "Figure D.4. Classical rules under random splits",
    }
    for _f in (fig1, fig2, figA1, figA2, figA3, figA4):
        _f()
    with open(os.path.join(OUT, "README.md"), "w") as _fh:
        _fh.write("# Paper figures\n\n" + "\n".join(f"- `{k}.pdf` / `{k}.png`: {v}" for k, v in TITLES.items()) + "\n")

    HF_REPO = "<HF_REPO>"
    HF_TOKEN = os.environ.get("HF_TOKEN") or "<YOUR_HF_TOKEN>"
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, repo_type="model", folder_path=OUT, path_in_repo="paper/figures",
                      commit_message="paper figures: regenerated, titled")
    remote = [f for f in api.list_repo_files(HF_REPO, repo_type="model") if f.startswith("paper/figures/")]
    want = [f"paper/figures/{k}.{e}" for k in TITLES for e in ("pdf", "png")]
    missing = [w for w in want if w not in remote]
    mo.output.replace(mo.vstack([
        mo.md(f"**Figures saved to Hugging Face** ({HF_REPO}, paper/figures/): {len(want) - len(missing)}/{len(want)} files"
              + ("" if not missing else f". MISSING: {missing}")),
        *[mo.vstack([mo.md(f"**{v}**"), mo.image(src=os.path.join(OUT, k + ".png"), width=700)]) for k, v in TITLES.items()],
    ]))
_()